# v30 — 처음부터 다시 만든 소형 앙상블

A는 새 표현의 소형 조합, B는 희귀 표현3개, C는 선택적 CNB4개, D는 검증된 문헌 파서/전처리를 공유하는4개를 비교합니다. 최종 활성 구성과 실제 결과는 아래에서 확인합니다. D는 과거의 좋은 전처리 정의/4개 recipe를 차용하지만 기존 fitted 모델을 가져오지 않습니다. 기존 학습 피클/분류기를 차용하지 않습니다. A의 최대4개 nested 조합 비교와 실패를 보존하고 B를 적응적 후속으로 구분합니다. B는 독립 외부 확인이 아닙니다.

논문50편 검토 기록에서 희귀 변이 공유 표현·표기 구분·event count·fold 내 선별/통계를 참고했습니다. 원문50편을 새로 정독하거나 논문 모델을 재현했다는 뜻은 아닙니다. test는 최종 고정 모델 추론에만 사용합니다. 데이터·requirements 환경이면 별도 프로젝트 py 없이 연구/학습/저장/추론/문서 생성이 모두 여기서 실행됩니다. runtime worker는 artifacts 아래 자동 생성합니다.

사용자 상한은 최종4개이며5개는 별도 문의입니다. 이번 최종은3~4개이며 실행 결과에 기록합니다. 상세 결과·실패/수정 이력·변수·문헌 연결은 analysis_report_v30.md와 selected_variables_v30.md에 기록합니다.

활성 후보 평균 MacroF1 0.489216, accuracy 49.0323%, v29 대비 +0.001564. 개선 기준 통과 False. Public 미확인.


In [1]:
CORE30="import os,sys,re,json,time,hashlib,warnings,platform,gc\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np,pandas as pd\nfrom scipy import sparse\nfrom sklearn.feature_extraction import DictVectorizer\nfrom sklearn.feature_selection import chi2\nfrom sklearn.preprocessing import MaxAbsScaler\nfrom sklearn.feature_extraction.text import TfidfTransformer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.naive_bayes import ComplementNB\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.utils.class_weight import compute_sample_weight\nfrom xgboost import XGBClassifier\nimport itertools\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix\nfrom lightgbm import LGBMClassifier\nfrom threadpoolctl import threadpool_limits\nimport cloudpickle\nthreadpool_limits(2)\nCONFIGS30=[{'name':'lr_strong','algo':'lr','view':'count','C':.1}, {'name':'lr_light','algo':'lr','view':'count','C':1.}, {'name':'lgb15','algo':'lgb','view':'count','leaves':15,'objective':'multiclass'}, {'name':'lgb31_ova','algo':'lgb','view':'count','leaves':31,'objective':'multiclassova','augment':True}, {'name':'xgb4','algo':'xgb','view':'count'}, {'name':'mlp128','algo':'mlp','view':'count','augment':True}]\ndef combinations30():\n return [tuple(c['name'] for c in combo) for k in range(1,5) for combo in itertools.combinations(CONFIGS30,k) if len(set(c['algo'] for c in combo))==k]\nCOMBINATIONS30=combinations30()\nPROTOCOL30={'version':30,'purpose':'from-scratch compact ensemble of1..4 classifiers; no inherited fitted classifiers','candidates':CONFIGS30,'max_final_classifiers':4,'five_requires_user_permission':True,'combination_count':len(COMBINATIONS30),'revision':'initial single-only interrupted on user clarification; later string-group-ID pilot exposed split mismatch and weak partial outer outcomes; final design corrected uint64 groups, shared AA/bin features and train-only augmentation after those observations; logs retained','outer_folds':5,'outer_seed':29032,'inner_folds':2,'inner_seed':30030,'features':'gene binary/count/type; shared gene-AA and position bins64/256; recurrent exact event; non-normalized burden summaries','fit_scope':'vocabulary, DF, chi2, MaxAbs, IDF only training fold','selection':'fixed equal weights; all35 family-distinct subsets of1..4; choose fewest models within0.002 of best mean inner F1, tie highest mean then fixed order; no outer-based selection','final_selection':'same two-fold inner procedure on development only, then selected configuration freshly fit all6201','exact_min_df':3,'exact_max_features':2048,'general_min_df':2,'general_max_features':6144,'legacy_X':'only full single-AA substitution A123X maps A123*; historical-stop assumption, not all X; retain raw files','augmentation':{'members':['lgb31_ova','mlp128'],'gene_mask_probability':.1,'rare_nuisance_poisson':10,'pool_df_max':2,'types':['missense','synonymous'],'weight':.5,'vocabulary_fit':'original training rows only','seed':30030},'test_for_research':False,'old_holdout_scored':False,'external_pretrained':False,'twin_rule':False,'historical_bias':'same repeatedly observed development patients; nested CV does not remove prior research decisions','reference':'v29 genuine out-of-fold cache only, exact training/validation and train hash checked; cache optional'}\ndef locate30():\n root=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/train.csv').exists()),None)\n if root is None:raise FileNotFoundError('data/train.csv required')\n work=root/'models/혜진' if (root/'models/혜진').is_dir() else root\n return root,work,work/'artifacts/v30'\ndef canonical_token30(token):\n token=token.strip()\n if re.fullmatch(r'[ACDEFGHIKLMNPQRSTVWY]\\d+X',token):token=token[:-1]+'*'\n return token\nSUB30=re.compile(r'^([ACDEFGHIKLMNPQRSTVWY*])(\\d+)([ACDEFGHIKLMNPQRSTVWY*])$')\ndef kind30(token):\n if 'fs' in token:return 'frameshift'\n if 'delins' in token:return 'delins'\n if 'del' in token:return 'deletion'\n if 'ins' in token:return 'insertion'\n m=SUB30.fullmatch(token)\n if m:\n  a,_,b=m.groups()\n  if a==b:return 'stop_retained' if a=='*' else 'synonymous'\n  if b=='*':return 'stop_gained'\n  if a=='*':return 'stop_lost'\n  return 'missense'\n return 'unparsed'\ndef row_events30(events,missing=()):\n d={k:1. for k in missing};events=sorted(set(events));pergene=Counter();types=Counter();unparsed=0\n for gene,token in events:\n  kind=kind30(token);pergene[gene]+=1;types[kind]+=1;unparsed+=kind=='unparsed';d['gene|'+gene]=1.;d['type|'+gene+'|'+kind]=1.;d['event|'+gene+'|'+token]=1.\n  m=SUB30.fullmatch(token)\n  if m:\n   aa,pos,bb=m.groups();d['meta|'+gene+'|aa|'+aa+'>'+bb]=1.;d['meta|'+gene+'|bin64|'+str((int(pos)-1)//64)]=1.;d['meta|'+gene+'|bin256|'+str((int(pos)-1)//256)]=1.\n for gene,n in pergene.items():d['count|'+gene]=float(np.log1p(n))\n d['summary|log_mutated_genes']=float(np.log1p(len(pergene)));d['summary|log_unique_events']=float(np.log1p(len(events)));d['summary|log_missing']=float(np.log1p(len(missing)));d['summary|log_unparsed']=float(np.log1p(unparsed))\n for kind,n in types.items():d['summary|log_type_'+kind]=float(np.log1p(n))\n return d\ndef parse30(frame,genes):\n if set(genes)-set(frame.columns):raise ValueError('Missing gene columns')\n rows=[]\n for vals in frame.loc[:,genes].itertuples(index=False,name=None):\n  events=[];missing=[]\n  for gene,cell in zip(genes,vals):\n   if pd.isna(cell) or str(cell).strip()=='':missing.append('missing|'+gene);continue\n   if str(cell).strip()=='WT':continue\n   events.extend((gene,canonical_token30(t)) for t in re.split(r'[\\s;,]+',str(cell).strip()) if t)\n  rows.append(row_events30(events,missing))\n return rows\ndef augment30(rows):\n rng=np.random.default_rng(30030);df=Counter(k for r in rows for k in r if k.startswith('event|'));pool=[tuple(k.split('|',2)[1:]) for k,n in df.items() if n<=2 and kind30(k.split('|',2)[2]) in ['missense','synonymous']];aug=[]\n for r in rows:\n  events=[tuple(k.split('|',2)[1:]) for k in r if k.startswith('event|')];masked={g for g in {g for g,t in events} if rng.random()<.1};events=[e for e in events if e[0] not in masked]\n  if pool:events.extend(pool[i] for i in rng.integers(0,len(pool),size=rng.poisson(10)))\n  aug.append(row_events30(events,[k for k in r if k.startswith('missing|')]))\n return aug\nclass Features30:\n def fit(self,rows,y):\n  self.vectorizer=DictVectorizer(dtype=np.float32,sort=True);m=self.vectorizer.fit_transform(rows).tocsr();names=np.asarray(self.vectorizer.get_feature_names_out());df=np.asarray((m>0).sum(0)).ravel();score=chi2(m,y)[0];score=np.nan_to_num(score,nan=-np.inf)\n  selected=[]\n  for prefix,min_df,budget in [('event|',3,2048),('meta|',3,2048),('general',2,6144)]:\n   mask=np.char.startswith(names.astype(str),prefix) if prefix!='general' else ~np.char.startswith(names.astype(str),'event|') & ~np.char.startswith(names.astype(str),'meta|') & ~np.char.startswith(names.astype(str),'summary|')\n   ix=np.flatnonzero(mask & (df>=min_df));ix=ix[np.argsort(-score[ix],kind='stable')[:budget]];selected.extend(ix.tolist())\n  selected.extend(np.flatnonzero(np.char.startswith(names.astype(str),'summary|')).tolist());self.selected_=np.sort(np.array(selected,dtype=int));self.names_=names[self.selected_];self.df_=df[self.selected_];self.scores_=score[self.selected_];return self\n def transform(self,rows):\n  m=self.vectorizer.transform(rows).tocsr()[:,self.selected_].astype(np.float32);m.eliminate_zeros();return m\nclass Single30:\n def __init__(self,config,genes,labels):self.config=dict(config);self.genes_=list(genes);self.classes_=np.asarray(labels)\n def fit(self,rows,y):\n  self.pre=Features30().fit(rows,y);m=self.pre.transform(rows);self.transformer=TfidfTransformer(norm=None,sublinear_tf=True) if self.config['view']=='tfidf' else MaxAbsScaler();m=self.transformer.fit_transform(m)\n  c=self.config;weights=compute_sample_weight('balanced',y);target=np.asarray(y)\n  if c.get('augment',False):\n   aug=self.transformer.transform(self.pre.transform(augment30(rows)));m=sparse.vstack([m,aug],format='csr');target=np.concatenate([target,target]);weights=np.concatenate([weights,.5*weights])\n  if c['algo']=='lr':self.estimator=LogisticRegression(C=c['C'],solver='lbfgs',max_iter=700,class_weight='balanced',random_state=30030)\n  elif c['algo']=='cnb':self.estimator=ComplementNB(alpha=c['alpha'])\n  elif c['algo']=='xgb':self.estimator=XGBClassifier(n_estimators=300,max_depth=4,learning_rate=.05,min_child_weight=3,subsample=.85,colsample_bytree=.7,reg_lambda=5.,objective='multi:softprob',num_class=len(self.classes_),tree_method='hist',n_jobs=2,random_state=30030)\n  elif c['algo']=='mlp':self.estimator=MLPClassifier(hidden_layer_sizes=(128,),alpha=1.,batch_size=128,learning_rate_init=.001,max_iter=150,early_stopping=False,random_state=30030)\n  else:self.estimator=LGBMClassifier(objective=c['objective'],num_class=len(self.classes_),n_estimators=400 if c.get('augment') else 260,learning_rate=.05,num_leaves=c['leaves'],max_depth=-1,min_child_samples=25,colsample_bytree=.7,subsample=.85,subsample_freq=1,reg_alpha=.1,reg_lambda=3.,class_weight=None if c.get('augment') else 'balanced',n_jobs=2,random_state=30030,verbosity=-1,force_col_wise=True,zero_as_missing=False)\n  # Numeric feature names avoid LightGBM JSON-character restrictions on mutation tokens.\n  if c['algo'] in ['xgb','mlp'] or c.get('augment',False):self.estimator.fit(m,target,sample_weight=weights)\n  else:self.estimator.fit(m,target)\n  self.feature_count_=m.shape[1];return self\n def predict_rows(self,rows):\n  p=self.estimator.predict_proba(self.transformer.transform(self.pre.transform(rows)));p=np.maximum(np.asarray(p,dtype=float),1e-15);p/=p.sum(1,keepdims=True);assert p.shape==(len(rows),len(self.classes_)) and np.isfinite(p).all();return p\n def predict_frame(self,frame):return self.predict_rows(parse30(frame,self.genes_))\ndef score30(y,p):return float(f1_score(y,p.argmax(1),labels=np.arange(26),average='macro',zero_division=0))\nclass Compact30:\n def __init__(self,selected,genes,labels):self.selected_=list(selected);self.genes_=list(genes);self.classes_=np.asarray(labels)\n def fit(self,rows,y):\n  self.members_={};self.fit_seconds_={}\n  for name in self.selected_:\n   cfg=next(c for c in CONFIGS30 if c['name']==name);a=time.perf_counter();self.members_[name]=Single30(cfg,self.genes_,self.classes_).fit(rows,y);self.fit_seconds_[name]=time.perf_counter()-a\n  assert 1<=len(self.members_)<=4;self.feature_count_=sum(m.feature_count_ for m in self.members_.values());return self\n def predict_rows(self,rows):return np.mean([m.predict_rows(rows) for m in self.members_.values()],axis=0)\n def predict_frame(self,frame):return self.predict_rows(parse30(frame,self.genes_))\ndef choose30(rows,y,groups,genes,labels,seed,log=None):\n metrics=[];combometrics=[]\n for fold,(a,b) in enumerate(StratifiedGroupKFold(2,shuffle=True,random_state=seed).split(np.zeros(len(y)),y,groups)):\n  assert not set(groups[a])&set(groups[b]);ra=[rows[i] for i in a];rb=[rows[i] for i in b];probs={}\n  for cfg in CONFIGS30:\n   start=time.perf_counter();model=Single30(cfg,genes,labels).fit(ra,y[a]);p=model.predict_rows(rb);probs[cfg['name']]=p;record={'inner_fold':fold,'candidate':cfg['name'],'f1':score30(y[b],p),'accuracy':float(accuracy_score(y[b],p.argmax(1))),'seconds':time.perf_counter()-start,'features':model.feature_count_};metrics.append(record);print('INNER',record,flush=True)\n   if log:pd.DataFrame(metrics).to_csv(log,index=False)\n   del model;gc.collect()\n  for names in COMBINATIONS30:\n   p=np.mean([probs[k] for k in names],axis=0);combometrics.append({'inner_fold':fold,'members':'|'.join(names),'count':len(names),'f1':score30(y[b],p)})\n  if log:pd.DataFrame(combometrics).to_csv(Path(log).with_name(Path(log).stem+'_combinations.csv'),index=False)\n mean={names:float(np.mean([r['f1'] for r in combometrics if r['members']=='|'.join(names)])) for names in COMBINATIONS30};best=max(mean.values());eligible=[n for n in COMBINATIONS30 if mean[n]>=best-.002];selected=min(eligible,key=lambda n:(len(n),-mean[n],COMBINATIONS30.index(n)));print('INNER SELECTION',selected,'mean',mean[selected],'best',best,flush=True);return {'name':'|'.join(selected),'members':list(selected),'count':len(selected),'inner_f1':mean[selected],'inner_best_f1':best},metrics\n"
WORKER30="import sys,json,time\nfrom pathlib import Path\nout=Path(sys.argv[1]);fold=int(sys.argv[2]);exec((out/'core30.py').read_text())\ndef guard30(event,args):\n if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('Research cannot open test/sample')\nsys.addaudithook(guard30)\nwith (out/'parsed_train.pkl').open('rb') as f:data=cloudpickle.load(f)\nwith np.load(out/f'split{fold}.npz') as q:tr=q['tr'];va=q['va']\nrows=data['rows'];y=data['y'];groups=data['groups'];assert not set(groups[tr])&set(groups[va]);selected,metrics=choose30([rows[i] for i in tr],y[tr],groups[tr],data['genes'],data['labels'],30030+fold,out/f'inner_fold{fold}.csv');start=time.perf_counter();model=Compact30(selected['members'],data['genes'],data['labels']).fit([rows[i] for i in tr],y[tr]);fitseconds=time.perf_counter()-start;start=time.perf_counter();p=model.predict_rows([rows[i] for i in va]);seconds=time.perf_counter()-start;np.savez_compressed(out/f'fold{fold}.npz',tr=tr,va=va,probabilities=p);result={'fold':fold,'selected':selected['name'],'f1':score30(y[va],p),'accuracy':float(accuracy_score(y[va],p.argmax(1))),'features':model.feature_count_,'fit_seconds':fitseconds,'prediction_seconds':seconds};(out/f'fold{fold}.json').write_text(json.dumps(result,indent=2));print('OUTER RESULT',result,flush=True)\n"
COHORT30={'ids': ['TRAIN_0000', 'TRAIN_0001', 'TRAIN_0002', 'TRAIN_0003', 'TRAIN_0004', 'TRAIN_0005', 'TRAIN_0006', 'TRAIN_0007', 'TRAIN_0008', 'TRAIN_0009', 'TRAIN_0010', 'TRAIN_0011', 'TRAIN_0012', 'TRAIN_0013', 'TRAIN_0014', 'TRAIN_0015', 'TRAIN_0016', 'TRAIN_0017', 'TRAIN_0018', 'TRAIN_0019', 'TRAIN_0020', 'TRAIN_0021', 'TRAIN_0022', 'TRAIN_0023', 'TRAIN_0024', 'TRAIN_0025', 'TRAIN_0026', 'TRAIN_0027', 'TRAIN_0028', 'TRAIN_0029', 'TRAIN_0030', 'TRAIN_0031', 'TRAIN_0032', 'TRAIN_0033', 'TRAIN_0034', 'TRAIN_0035', 'TRAIN_0036', 'TRAIN_0037', 'TRAIN_0038', 'TRAIN_0039', 'TRAIN_0040', 'TRAIN_0041', 'TRAIN_0042', 'TRAIN_0043', 'TRAIN_0044', 'TRAIN_0045', 'TRAIN_0046', 'TRAIN_0047', 'TRAIN_0048', 'TRAIN_0049', 'TRAIN_0050', 'TRAIN_0051', 'TRAIN_0052', 'TRAIN_0053', 'TRAIN_0054', 'TRAIN_0055', 'TRAIN_0056', 'TRAIN_0057', 'TRAIN_0058', 'TRAIN_0059', 'TRAIN_0060', 'TRAIN_0061', 'TRAIN_0062', 'TRAIN_0063', 'TRAIN_0064', 'TRAIN_0065', 'TRAIN_0066', 'TRAIN_0067', 'TRAIN_0068', 'TRAIN_0069', 'TRAIN_0070', 'TRAIN_0071', 'TRAIN_0072', 'TRAIN_0073', 'TRAIN_0074', 'TRAIN_0075', 'TRAIN_0076', 'TRAIN_0077', 'TRAIN_0078', 'TRAIN_0079', 'TRAIN_0080', 'TRAIN_0081', 'TRAIN_0082', 'TRAIN_0083', 'TRAIN_0084', 'TRAIN_0085', 'TRAIN_0086', 'TRAIN_0087', 'TRAIN_0088', 'TRAIN_0089', 'TRAIN_0090', 'TRAIN_0091', 'TRAIN_0092', 'TRAIN_0093', 'TRAIN_0094', 'TRAIN_0095', 'TRAIN_0096', 'TRAIN_0097', 'TRAIN_0098', 'TRAIN_0099', 'TRAIN_0100', 'TRAIN_0101', 'TRAIN_0102', 'TRAIN_0103', 'TRAIN_0104', 'TRAIN_0105', 'TRAIN_0106', 'TRAIN_0107', 'TRAIN_0108', 'TRAIN_0109', 'TRAIN_0110', 'TRAIN_0111', 'TRAIN_0112', 'TRAIN_0113', 'TRAIN_0114', 'TRAIN_0115', 'TRAIN_0116', 'TRAIN_0117', 'TRAIN_0118', 'TRAIN_0119', 'TRAIN_0120', 'TRAIN_0121', 'TRAIN_0122', 'TRAIN_0123', 'TRAIN_0124', 'TRAIN_0125', 'TRAIN_0126', 'TRAIN_0127', 'TRAIN_0128', 'TRAIN_0129', 'TRAIN_0130', 'TRAIN_0131', 'TRAIN_0132', 'TRAIN_0133', 'TRAIN_0134', 'TRAIN_0135', 'TRAIN_0136', 'TRAIN_0137', 'TRAIN_0138', 'TRAIN_0139', 'TRAIN_0140', 'TRAIN_0141', 'TRAIN_0142', 'TRAIN_0143', 'TRAIN_0144', 'TRAIN_0145', 'TRAIN_0146', 'TRAIN_0147', 'TRAIN_0148', 'TRAIN_0149', 'TRAIN_0150', 'TRAIN_0151', 'TRAIN_0152', 'TRAIN_0153', 'TRAIN_0154', 'TRAIN_0155', 'TRAIN_0156', 'TRAIN_0157', 'TRAIN_0158', 'TRAIN_0159', 'TRAIN_0160', 'TRAIN_0161', 'TRAIN_0162', 'TRAIN_0163', 'TRAIN_0164', 'TRAIN_0165', 'TRAIN_0166', 'TRAIN_0167', 'TRAIN_0168', 'TRAIN_0169', 'TRAIN_0170', 'TRAIN_0171', 'TRAIN_0172', 'TRAIN_0173', 'TRAIN_0174', 'TRAIN_0175', 'TRAIN_0176', 'TRAIN_0177', 'TRAIN_0178', 'TRAIN_0179', 'TRAIN_0180', 'TRAIN_0181', 'TRAIN_0182', 'TRAIN_0183', 'TRAIN_0184', 'TRAIN_0185', 'TRAIN_0186', 'TRAIN_0187', 'TRAIN_0188', 'TRAIN_0189', 'TRAIN_0190', 'TRAIN_0191', 'TRAIN_0192', 'TRAIN_0193', 'TRAIN_0194', 'TRAIN_0195', 'TRAIN_0196', 'TRAIN_0197', 'TRAIN_0198', 'TRAIN_0199', 'TRAIN_0200', 'TRAIN_0201', 'TRAIN_0202', 'TRAIN_0203', 'TRAIN_0204', 'TRAIN_0205', 'TRAIN_0206', 'TRAIN_0207', 'TRAIN_0208', 'TRAIN_0209', 'TRAIN_0210', 'TRAIN_0211', 'TRAIN_0212', 'TRAIN_0213', 'TRAIN_0214', 'TRAIN_0215', 'TRAIN_0216', 'TRAIN_0217', 'TRAIN_0218', 'TRAIN_0219', 'TRAIN_0220', 'TRAIN_0221', 'TRAIN_0222', 'TRAIN_0223', 'TRAIN_0224', 'TRAIN_0225', 'TRAIN_0226', 'TRAIN_0227', 'TRAIN_0228', 'TRAIN_0229', 'TRAIN_0230', 'TRAIN_0231', 'TRAIN_0232', 'TRAIN_0233', 'TRAIN_0234', 'TRAIN_0235', 'TRAIN_0236', 'TRAIN_0237', 'TRAIN_0238', 'TRAIN_0239', 'TRAIN_0240', 'TRAIN_0241', 'TRAIN_0242', 'TRAIN_0243', 'TRAIN_0244', 'TRAIN_0245', 'TRAIN_0246', 'TRAIN_0247', 'TRAIN_0248', 'TRAIN_0249', 'TRAIN_0250', 'TRAIN_0251', 'TRAIN_0252', 'TRAIN_0253', 'TRAIN_0254', 'TRAIN_0255', 'TRAIN_0256', 'TRAIN_0257', 'TRAIN_0258', 'TRAIN_0259', 'TRAIN_0260', 'TRAIN_0261', 'TRAIN_0262', 'TRAIN_0263', 'TRAIN_0264', 'TRAIN_0265', 'TRAIN_0266', 'TRAIN_0267', 'TRAIN_0268', 'TRAIN_0269', 'TRAIN_0270', 'TRAIN_0271', 'TRAIN_0272', 'TRAIN_0273', 'TRAIN_0274', 'TRAIN_0275', 'TRAIN_0276', 'TRAIN_0277', 'TRAIN_0278', 'TRAIN_0279', 'TRAIN_0280', 'TRAIN_0281', 'TRAIN_0282', 'TRAIN_0283', 'TRAIN_0284', 'TRAIN_0285', 'TRAIN_0286', 'TRAIN_0287', 'TRAIN_0288', 'TRAIN_0289', 'TRAIN_0290', 'TRAIN_0291', 'TRAIN_0292', 'TRAIN_0293', 'TRAIN_0294', 'TRAIN_0295', 'TRAIN_0296', 'TRAIN_0297', 'TRAIN_0298', 'TRAIN_0299', 'TRAIN_0300', 'TRAIN_0301', 'TRAIN_0302', 'TRAIN_0303', 'TRAIN_0304', 'TRAIN_0305', 'TRAIN_0306', 'TRAIN_0307', 'TRAIN_0308', 'TRAIN_0309', 'TRAIN_0310', 'TRAIN_0311', 'TRAIN_0312', 'TRAIN_0313', 'TRAIN_0314', 'TRAIN_0315', 'TRAIN_0316', 'TRAIN_0317', 'TRAIN_0318', 'TRAIN_0319', 'TRAIN_0320', 'TRAIN_0321', 'TRAIN_0322', 'TRAIN_0323', 'TRAIN_0324', 'TRAIN_0325', 'TRAIN_0326', 'TRAIN_0327', 'TRAIN_0328', 'TRAIN_0329', 'TRAIN_0330', 'TRAIN_0331', 'TRAIN_0332', 'TRAIN_0333', 'TRAIN_0334', 'TRAIN_0335', 'TRAIN_0336', 'TRAIN_0337', 'TRAIN_0338', 'TRAIN_0339', 'TRAIN_0340', 'TRAIN_0341', 'TRAIN_0342', 'TRAIN_0343', 'TRAIN_0344', 'TRAIN_0345', 'TRAIN_0346', 'TRAIN_0347', 'TRAIN_0348', 'TRAIN_0349', 'TRAIN_0350', 'TRAIN_0351', 'TRAIN_0352', 'TRAIN_0353', 'TRAIN_0354', 'TRAIN_0355', 'TRAIN_0356', 'TRAIN_0357', 'TRAIN_0358', 'TRAIN_0359', 'TRAIN_0360', 'TRAIN_0361', 'TRAIN_0362', 'TRAIN_0363', 'TRAIN_0364', 'TRAIN_0365', 'TRAIN_0366', 'TRAIN_0367', 'TRAIN_0368', 'TRAIN_0369', 'TRAIN_0370', 'TRAIN_0371', 'TRAIN_0372', 'TRAIN_0373', 'TRAIN_0374', 'TRAIN_0375', 'TRAIN_0376', 'TRAIN_0377', 'TRAIN_0378', 'TRAIN_0379', 'TRAIN_0380', 'TRAIN_0381', 'TRAIN_0382', 'TRAIN_0383', 'TRAIN_0384', 'TRAIN_0385', 'TRAIN_0386', 'TRAIN_0387', 'TRAIN_0388', 'TRAIN_0389', 'TRAIN_0390', 'TRAIN_0391', 'TRAIN_0392', 'TRAIN_0393', 'TRAIN_0394', 'TRAIN_0395', 'TRAIN_0396', 'TRAIN_0397', 'TRAIN_0398', 'TRAIN_0399', 'TRAIN_0400', 'TRAIN_0401', 'TRAIN_0402', 'TRAIN_0403', 'TRAIN_0404', 'TRAIN_0405', 'TRAIN_0406', 'TRAIN_0407', 'TRAIN_0408', 'TRAIN_0409', 'TRAIN_0410', 'TRAIN_0411', 'TRAIN_0412', 'TRAIN_0413', 'TRAIN_0414', 'TRAIN_0415', 'TRAIN_0416', 'TRAIN_0417', 'TRAIN_0418', 'TRAIN_0419', 'TRAIN_0420', 'TRAIN_0421', 'TRAIN_0422', 'TRAIN_0423', 'TRAIN_0424', 'TRAIN_0425', 'TRAIN_0426', 'TRAIN_0427', 'TRAIN_0428', 'TRAIN_0429', 'TRAIN_0430', 'TRAIN_0431', 'TRAIN_0432', 'TRAIN_0433', 'TRAIN_0434', 'TRAIN_0435', 'TRAIN_0436', 'TRAIN_0437', 'TRAIN_0438', 'TRAIN_0439', 'TRAIN_0440', 'TRAIN_0441', 'TRAIN_0442', 'TRAIN_0443', 'TRAIN_0444', 'TRAIN_0445', 'TRAIN_0446', 'TRAIN_0447', 'TRAIN_0448', 'TRAIN_0449', 'TRAIN_0450', 'TRAIN_0451', 'TRAIN_0452', 'TRAIN_0453', 'TRAIN_0454', 'TRAIN_0455', 'TRAIN_0456', 'TRAIN_0457', 'TRAIN_0458', 'TRAIN_0459', 'TRAIN_0460', 'TRAIN_0461', 'TRAIN_0462', 'TRAIN_0463', 'TRAIN_0464', 'TRAIN_0465', 'TRAIN_0466', 'TRAIN_0467', 'TRAIN_0468', 'TRAIN_0469', 'TRAIN_0470', 'TRAIN_0471', 'TRAIN_0472', 'TRAIN_0473', 'TRAIN_0474', 'TRAIN_0475', 'TRAIN_0476', 'TRAIN_0477', 'TRAIN_0478', 'TRAIN_0479', 'TRAIN_0480', 'TRAIN_0481', 'TRAIN_0482', 'TRAIN_0483', 'TRAIN_0484', 'TRAIN_0485', 'TRAIN_0486', 'TRAIN_0487', 'TRAIN_0488', 'TRAIN_0489', 'TRAIN_0490', 'TRAIN_0491', 'TRAIN_0492', 'TRAIN_0493', 'TRAIN_0494', 'TRAIN_0495', 'TRAIN_0496', 'TRAIN_0497', 'TRAIN_0498', 'TRAIN_0499', 'TRAIN_0500', 'TRAIN_0501', 'TRAIN_0502', 'TRAIN_0503', 'TRAIN_0504', 'TRAIN_0505', 'TRAIN_0506', 'TRAIN_0507', 'TRAIN_0508', 'TRAIN_0509', 'TRAIN_0510', 'TRAIN_0511', 'TRAIN_0512', 'TRAIN_0513', 'TRAIN_0514', 'TRAIN_0515', 'TRAIN_0516', 'TRAIN_0517', 'TRAIN_0518', 'TRAIN_0519', 'TRAIN_0520', 'TRAIN_0521', 'TRAIN_0522', 'TRAIN_0523', 'TRAIN_0524', 'TRAIN_0525', 'TRAIN_0526', 'TRAIN_0527', 'TRAIN_0528', 'TRAIN_0529', 'TRAIN_0530', 'TRAIN_0531', 'TRAIN_0532', 'TRAIN_0533', 'TRAIN_0534', 'TRAIN_0535', 'TRAIN_0536', 'TRAIN_0537', 'TRAIN_0538', 'TRAIN_0539', 'TRAIN_0540', 'TRAIN_0541', 'TRAIN_0542', 'TRAIN_0543', 'TRAIN_0544', 'TRAIN_0545', 'TRAIN_0546', 'TRAIN_0547', 'TRAIN_0548', 'TRAIN_0549', 'TRAIN_0550', 'TRAIN_0551', 'TRAIN_0552', 'TRAIN_0553', 'TRAIN_0554', 'TRAIN_0555', 'TRAIN_0556', 'TRAIN_0557', 'TRAIN_0558', 'TRAIN_0559', 'TRAIN_0560', 'TRAIN_0561', 'TRAIN_0562', 'TRAIN_0563', 'TRAIN_0564', 'TRAIN_0565', 'TRAIN_0566', 'TRAIN_0567', 'TRAIN_0568', 'TRAIN_0569', 'TRAIN_0570', 'TRAIN_0571', 'TRAIN_0572', 'TRAIN_0573', 'TRAIN_0574', 'TRAIN_0575', 'TRAIN_0576', 'TRAIN_0577', 'TRAIN_0578', 'TRAIN_0579', 'TRAIN_0580', 'TRAIN_0581', 'TRAIN_0582', 'TRAIN_0583', 'TRAIN_0584', 'TRAIN_0585', 'TRAIN_0586', 'TRAIN_0587', 'TRAIN_0588', 'TRAIN_0589', 'TRAIN_0590', 'TRAIN_0591', 'TRAIN_0592', 'TRAIN_0593', 'TRAIN_0594', 'TRAIN_0595', 'TRAIN_0596', 'TRAIN_0597', 'TRAIN_0598', 'TRAIN_0599', 'TRAIN_0600', 'TRAIN_0601', 'TRAIN_0602', 'TRAIN_0603', 'TRAIN_0604', 'TRAIN_0605', 'TRAIN_0606', 'TRAIN_0607', 'TRAIN_0608', 'TRAIN_0609', 'TRAIN_0610', 'TRAIN_0611', 'TRAIN_0612', 'TRAIN_0613', 'TRAIN_0614', 'TRAIN_0615', 'TRAIN_0616', 'TRAIN_0617', 'TRAIN_0618', 'TRAIN_0619', 'TRAIN_0620', 'TRAIN_0621', 'TRAIN_0622', 'TRAIN_0623', 'TRAIN_0624', 'TRAIN_0625', 'TRAIN_0626', 'TRAIN_0627', 'TRAIN_0628', 'TRAIN_0629', 'TRAIN_0630', 'TRAIN_0631', 'TRAIN_0632', 'TRAIN_0633', 'TRAIN_0634', 'TRAIN_0635', 'TRAIN_0636', 'TRAIN_0637', 'TRAIN_0638', 'TRAIN_0639', 'TRAIN_0640', 'TRAIN_0641', 'TRAIN_0642', 'TRAIN_0643', 'TRAIN_0644', 'TRAIN_0645', 'TRAIN_0646', 'TRAIN_0647', 'TRAIN_0648', 'TRAIN_0649', 'TRAIN_0650', 'TRAIN_0651', 'TRAIN_0652', 'TRAIN_0653', 'TRAIN_0654', 'TRAIN_0655', 'TRAIN_0656', 'TRAIN_0657', 'TRAIN_0658', 'TRAIN_0659', 'TRAIN_0660', 'TRAIN_0661', 'TRAIN_0662', 'TRAIN_0663', 'TRAIN_0664', 'TRAIN_0665', 'TRAIN_0666', 'TRAIN_0667', 'TRAIN_0668', 'TRAIN_0669', 'TRAIN_0670', 'TRAIN_0671', 'TRAIN_0672', 'TRAIN_0673', 'TRAIN_0674', 'TRAIN_0675', 'TRAIN_0676', 'TRAIN_0677', 'TRAIN_0678', 'TRAIN_0679', 'TRAIN_0680', 'TRAIN_0681', 'TRAIN_0682', 'TRAIN_0683', 'TRAIN_0684', 'TRAIN_0685', 'TRAIN_0686', 'TRAIN_0687', 'TRAIN_0688', 'TRAIN_0689', 'TRAIN_0690', 'TRAIN_0691', 'TRAIN_0692', 'TRAIN_0693', 'TRAIN_0694', 'TRAIN_0695', 'TRAIN_0696', 'TRAIN_0697', 'TRAIN_0698', 'TRAIN_0699', 'TRAIN_0700', 'TRAIN_0701', 'TRAIN_0702', 'TRAIN_0703', 'TRAIN_0704', 'TRAIN_0705', 'TRAIN_0706', 'TRAIN_0707', 'TRAIN_0708', 'TRAIN_0709', 'TRAIN_0710', 'TRAIN_0711', 'TRAIN_0712', 'TRAIN_0713', 'TRAIN_0714', 'TRAIN_0715', 'TRAIN_0716', 'TRAIN_0717', 'TRAIN_0718', 'TRAIN_0719', 'TRAIN_0720', 'TRAIN_0721', 'TRAIN_0722', 'TRAIN_0723', 'TRAIN_0724', 'TRAIN_0725', 'TRAIN_0726', 'TRAIN_0727', 'TRAIN_0728', 'TRAIN_0729', 'TRAIN_0730', 'TRAIN_0731', 'TRAIN_0732', 'TRAIN_0733', 'TRAIN_0734', 'TRAIN_0735', 'TRAIN_0736', 'TRAIN_0737', 'TRAIN_0738', 'TRAIN_0739', 'TRAIN_0740', 'TRAIN_0741', 'TRAIN_0742', 'TRAIN_0743', 'TRAIN_0744', 'TRAIN_0745', 'TRAIN_0746', 'TRAIN_0747', 'TRAIN_0748', 'TRAIN_0749', 'TRAIN_0750', 'TRAIN_0751', 'TRAIN_0752', 'TRAIN_0753', 'TRAIN_0754', 'TRAIN_0755', 'TRAIN_0756', 'TRAIN_0757', 'TRAIN_0758', 'TRAIN_0759', 'TRAIN_0760', 'TRAIN_0761', 'TRAIN_0762', 'TRAIN_0763', 'TRAIN_0764', 'TRAIN_0765', 'TRAIN_0766', 'TRAIN_0767', 'TRAIN_0768', 'TRAIN_0769', 'TRAIN_0770', 'TRAIN_0771', 'TRAIN_0772', 'TRAIN_0773', 'TRAIN_0774', 'TRAIN_0775', 'TRAIN_0776', 'TRAIN_0777', 'TRAIN_0778', 'TRAIN_0779', 'TRAIN_0780', 'TRAIN_0781', 'TRAIN_0782', 'TRAIN_0783', 'TRAIN_0784', 'TRAIN_0785', 'TRAIN_0786', 'TRAIN_0787', 'TRAIN_0788', 'TRAIN_0789', 'TRAIN_0790', 'TRAIN_0791', 'TRAIN_0792', 'TRAIN_0793', 'TRAIN_0794', 'TRAIN_0795', 'TRAIN_0796', 'TRAIN_0797', 'TRAIN_0798', 'TRAIN_0799', 'TRAIN_0800', 'TRAIN_0801', 'TRAIN_0802', 'TRAIN_0803', 'TRAIN_0804', 'TRAIN_0805', 'TRAIN_0806', 'TRAIN_0807', 'TRAIN_0808', 'TRAIN_0809', 'TRAIN_0810', 'TRAIN_0811', 'TRAIN_0812', 'TRAIN_0813', 'TRAIN_0814', 'TRAIN_0815', 'TRAIN_0816', 'TRAIN_0817', 'TRAIN_0818', 'TRAIN_0819', 'TRAIN_0820', 'TRAIN_0821', 'TRAIN_0822', 'TRAIN_0823', 'TRAIN_0824', 'TRAIN_0825', 'TRAIN_0826', 'TRAIN_0827', 'TRAIN_0828', 'TRAIN_0829', 'TRAIN_0830', 'TRAIN_0831', 'TRAIN_0832', 'TRAIN_0833', 'TRAIN_0834', 'TRAIN_0835', 'TRAIN_0836', 'TRAIN_0837', 'TRAIN_0838', 'TRAIN_0839', 'TRAIN_0840', 'TRAIN_0841', 'TRAIN_0842', 'TRAIN_0843', 'TRAIN_0844', 'TRAIN_0845', 'TRAIN_0846', 'TRAIN_0847', 'TRAIN_0848', 'TRAIN_0849', 'TRAIN_0850', 'TRAIN_0851', 'TRAIN_0852', 'TRAIN_0853', 'TRAIN_0854', 'TRAIN_0855', 'TRAIN_0856', 'TRAIN_0857', 'TRAIN_0858', 'TRAIN_0859', 'TRAIN_0860', 'TRAIN_0861', 'TRAIN_0862', 'TRAIN_0863', 'TRAIN_0864', 'TRAIN_0865', 'TRAIN_0866', 'TRAIN_0867', 'TRAIN_0868', 'TRAIN_0869', 'TRAIN_0870', 'TRAIN_0871', 'TRAIN_0872', 'TRAIN_0873', 'TRAIN_0874', 'TRAIN_0875', 'TRAIN_0876', 'TRAIN_0877', 'TRAIN_0878', 'TRAIN_0879', 'TRAIN_0880', 'TRAIN_0881', 'TRAIN_0882', 'TRAIN_0883', 'TRAIN_0884', 'TRAIN_0885', 'TRAIN_0886', 'TRAIN_0887', 'TRAIN_0888', 'TRAIN_0889', 'TRAIN_0890', 'TRAIN_0891', 'TRAIN_0892', 'TRAIN_0893', 'TRAIN_0894', 'TRAIN_0895', 'TRAIN_0896', 'TRAIN_0897', 'TRAIN_0898', 'TRAIN_0899', 'TRAIN_0900', 'TRAIN_0901', 'TRAIN_0902', 'TRAIN_0903', 'TRAIN_0904', 'TRAIN_0905', 'TRAIN_0906', 'TRAIN_0907', 'TRAIN_0908', 'TRAIN_0909', 'TRAIN_0910', 'TRAIN_0911', 'TRAIN_0912', 'TRAIN_0913', 'TRAIN_0914', 'TRAIN_0915', 'TRAIN_0916', 'TRAIN_0917', 'TRAIN_0918', 'TRAIN_0919', 'TRAIN_0920', 'TRAIN_0921', 'TRAIN_0922', 'TRAIN_0923', 'TRAIN_0924', 'TRAIN_0925', 'TRAIN_0926', 'TRAIN_0927', 'TRAIN_0928', 'TRAIN_0929', 'TRAIN_0930', 'TRAIN_0931', 'TRAIN_0932', 'TRAIN_0933', 'TRAIN_0934', 'TRAIN_0935', 'TRAIN_0936', 'TRAIN_0937', 'TRAIN_0938', 'TRAIN_0939', 'TRAIN_0940', 'TRAIN_0941', 'TRAIN_0942', 'TRAIN_0943', 'TRAIN_0944', 'TRAIN_0945', 'TRAIN_0946', 'TRAIN_0947', 'TRAIN_0948', 'TRAIN_0949', 'TRAIN_0950', 'TRAIN_0951', 'TRAIN_0952', 'TRAIN_0953', 'TRAIN_0954', 'TRAIN_0955', 'TRAIN_0956', 'TRAIN_0957', 'TRAIN_0958', 'TRAIN_0959', 'TRAIN_0960', 'TRAIN_0961', 'TRAIN_0962', 'TRAIN_0963', 'TRAIN_0964', 'TRAIN_0965', 'TRAIN_0966', 'TRAIN_0967', 'TRAIN_0968', 'TRAIN_0969', 'TRAIN_0970', 'TRAIN_0971', 'TRAIN_0972', 'TRAIN_0973', 'TRAIN_0974', 'TRAIN_0975', 'TRAIN_0976', 'TRAIN_0977', 'TRAIN_0978', 'TRAIN_0979', 'TRAIN_0980', 'TRAIN_0981', 'TRAIN_0982', 'TRAIN_0983', 'TRAIN_0984', 'TRAIN_0985', 'TRAIN_0986', 'TRAIN_0987', 'TRAIN_0988', 'TRAIN_0989', 'TRAIN_0990', 'TRAIN_0991', 'TRAIN_0992', 'TRAIN_0993', 'TRAIN_0994', 'TRAIN_0995', 'TRAIN_0996', 'TRAIN_0997', 'TRAIN_0998', 'TRAIN_0999', 'TRAIN_1000', 'TRAIN_1001', 'TRAIN_1002', 'TRAIN_1003', 'TRAIN_1004', 'TRAIN_1005', 'TRAIN_1006', 'TRAIN_1007', 'TRAIN_1008', 'TRAIN_1009', 'TRAIN_1010', 'TRAIN_1011', 'TRAIN_1012', 'TRAIN_1013', 'TRAIN_1014', 'TRAIN_1015', 'TRAIN_1016', 'TRAIN_1017', 'TRAIN_1018', 'TRAIN_1019', 'TRAIN_1020', 'TRAIN_1021', 'TRAIN_1022', 'TRAIN_1023', 'TRAIN_1024', 'TRAIN_1025', 'TRAIN_1026', 'TRAIN_1027', 'TRAIN_1028', 'TRAIN_1029', 'TRAIN_1030', 'TRAIN_1031', 'TRAIN_1032', 'TRAIN_1033', 'TRAIN_1034', 'TRAIN_1035', 'TRAIN_1036', 'TRAIN_1037', 'TRAIN_1038', 'TRAIN_1039', 'TRAIN_1040', 'TRAIN_1041', 'TRAIN_1042', 'TRAIN_1043', 'TRAIN_1044', 'TRAIN_1045', 'TRAIN_1046', 'TRAIN_1047', 'TRAIN_1048', 'TRAIN_1049', 'TRAIN_1050', 'TRAIN_1051', 'TRAIN_1052', 'TRAIN_1053', 'TRAIN_1054', 'TRAIN_1055', 'TRAIN_1056', 'TRAIN_1057', 'TRAIN_1058', 'TRAIN_1059', 'TRAIN_1060', 'TRAIN_1061', 'TRAIN_1062', 'TRAIN_1063', 'TRAIN_1064', 'TRAIN_1065', 'TRAIN_1066', 'TRAIN_1067', 'TRAIN_1068', 'TRAIN_1069', 'TRAIN_1070', 'TRAIN_1071', 'TRAIN_1072', 'TRAIN_1073', 'TRAIN_1074', 'TRAIN_1075', 'TRAIN_1076', 'TRAIN_1077', 'TRAIN_1078', 'TRAIN_1079', 'TRAIN_1080', 'TRAIN_1081', 'TRAIN_1082', 'TRAIN_1083', 'TRAIN_1084', 'TRAIN_1085', 'TRAIN_1086', 'TRAIN_1087', 'TRAIN_1088', 'TRAIN_1089', 'TRAIN_1090', 'TRAIN_1091', 'TRAIN_1092', 'TRAIN_1093', 'TRAIN_1094', 'TRAIN_1095', 'TRAIN_1096', 'TRAIN_1097', 'TRAIN_1098', 'TRAIN_1099', 'TRAIN_1100', 'TRAIN_1101', 'TRAIN_1102', 'TRAIN_1103', 'TRAIN_1104', 'TRAIN_1105', 'TRAIN_1106', 'TRAIN_1107', 'TRAIN_1108', 'TRAIN_1109', 'TRAIN_1110', 'TRAIN_1111', 'TRAIN_1112', 'TRAIN_1113', 'TRAIN_1114', 'TRAIN_1115', 'TRAIN_1116', 'TRAIN_1117', 'TRAIN_1118', 'TRAIN_1119', 'TRAIN_1120', 'TRAIN_1121', 'TRAIN_1122', 'TRAIN_1123', 'TRAIN_1124', 'TRAIN_1125', 'TRAIN_1126', 'TRAIN_1127', 'TRAIN_1128', 'TRAIN_1129', 'TRAIN_1130', 'TRAIN_1131', 'TRAIN_1132', 'TRAIN_1133', 'TRAIN_1134', 'TRAIN_1135', 'TRAIN_1136', 'TRAIN_1137', 'TRAIN_1138', 'TRAIN_1139', 'TRAIN_1140', 'TRAIN_1141', 'TRAIN_1142', 'TRAIN_1143', 'TRAIN_1144', 'TRAIN_1145', 'TRAIN_1146', 'TRAIN_1147', 'TRAIN_1148', 'TRAIN_1149', 'TRAIN_1150', 'TRAIN_1151', 'TRAIN_1152', 'TRAIN_1153', 'TRAIN_1154', 'TRAIN_1155', 'TRAIN_1156', 'TRAIN_1157', 'TRAIN_1158', 'TRAIN_1159', 'TRAIN_1160', 'TRAIN_1161', 'TRAIN_1162', 'TRAIN_1163', 'TRAIN_1164', 'TRAIN_1165', 'TRAIN_1166', 'TRAIN_1167', 'TRAIN_1168', 'TRAIN_1169', 'TRAIN_1170', 'TRAIN_1171', 'TRAIN_1172', 'TRAIN_1173', 'TRAIN_1174', 'TRAIN_1175', 'TRAIN_1176', 'TRAIN_1177', 'TRAIN_1178', 'TRAIN_1179', 'TRAIN_1180', 'TRAIN_1181', 'TRAIN_1182', 'TRAIN_1183', 'TRAIN_1184', 'TRAIN_1185', 'TRAIN_1186', 'TRAIN_1187', 'TRAIN_1188', 'TRAIN_1189', 'TRAIN_1190', 'TRAIN_1191', 'TRAIN_1192', 'TRAIN_1193', 'TRAIN_1194', 'TRAIN_1195', 'TRAIN_1196', 'TRAIN_1197', 'TRAIN_1198', 'TRAIN_1199', 'TRAIN_1200', 'TRAIN_1201', 'TRAIN_1202', 'TRAIN_1203', 'TRAIN_1204', 'TRAIN_1205', 'TRAIN_1206', 'TRAIN_1207', 'TRAIN_1208', 'TRAIN_1209', 'TRAIN_1210', 'TRAIN_1211', 'TRAIN_1212', 'TRAIN_1213', 'TRAIN_1214', 'TRAIN_1215', 'TRAIN_1216', 'TRAIN_1217', 'TRAIN_1218', 'TRAIN_1219', 'TRAIN_1220', 'TRAIN_1221', 'TRAIN_1222', 'TRAIN_1223', 'TRAIN_1224', 'TRAIN_1225', 'TRAIN_1226', 'TRAIN_1227', 'TRAIN_1228', 'TRAIN_1229', 'TRAIN_1230', 'TRAIN_1231', 'TRAIN_1232', 'TRAIN_1233', 'TRAIN_1234', 'TRAIN_1235', 'TRAIN_1236', 'TRAIN_1237', 'TRAIN_1238', 'TRAIN_1239', 'TRAIN_1240', 'TRAIN_1241', 'TRAIN_1242', 'TRAIN_1243', 'TRAIN_1244', 'TRAIN_1245', 'TRAIN_1246', 'TRAIN_1247', 'TRAIN_1248', 'TRAIN_1249', 'TRAIN_1250', 'TRAIN_1251', 'TRAIN_1252', 'TRAIN_1253', 'TRAIN_1254', 'TRAIN_1255', 'TRAIN_1256', 'TRAIN_1257', 'TRAIN_1258', 'TRAIN_1259', 'TRAIN_1260', 'TRAIN_1261', 'TRAIN_1262', 'TRAIN_1263', 'TRAIN_1264', 'TRAIN_1265', 'TRAIN_1266', 'TRAIN_1267', 'TRAIN_1268', 'TRAIN_1269', 'TRAIN_1270', 'TRAIN_1271', 'TRAIN_1272', 'TRAIN_1273', 'TRAIN_1274', 'TRAIN_1275', 'TRAIN_1276', 'TRAIN_1277', 'TRAIN_1278', 'TRAIN_1279', 'TRAIN_1280', 'TRAIN_1281', 'TRAIN_1282', 'TRAIN_1283', 'TRAIN_1284', 'TRAIN_1285', 'TRAIN_1286', 'TRAIN_1287', 'TRAIN_1288', 'TRAIN_1289', 'TRAIN_1290', 'TRAIN_1291', 'TRAIN_1292', 'TRAIN_1293', 'TRAIN_1294', 'TRAIN_1295', 'TRAIN_1296', 'TRAIN_1297', 'TRAIN_1298', 'TRAIN_1299', 'TRAIN_1300', 'TRAIN_1301', 'TRAIN_1302', 'TRAIN_1303', 'TRAIN_1304', 'TRAIN_1305', 'TRAIN_1306', 'TRAIN_1307', 'TRAIN_1308', 'TRAIN_1309', 'TRAIN_1310', 'TRAIN_1311', 'TRAIN_1312', 'TRAIN_1313', 'TRAIN_1314', 'TRAIN_1315', 'TRAIN_1316', 'TRAIN_1317', 'TRAIN_1318', 'TRAIN_1319', 'TRAIN_1320', 'TRAIN_1321', 'TRAIN_1322', 'TRAIN_1323', 'TRAIN_1324', 'TRAIN_1325', 'TRAIN_1326', 'TRAIN_1327', 'TRAIN_1328', 'TRAIN_1329', 'TRAIN_1330', 'TRAIN_1331', 'TRAIN_1332', 'TRAIN_1333', 'TRAIN_1334', 'TRAIN_1335', 'TRAIN_1336', 'TRAIN_1337', 'TRAIN_1338', 'TRAIN_1339', 'TRAIN_1340', 'TRAIN_1341', 'TRAIN_1342', 'TRAIN_1343', 'TRAIN_1344', 'TRAIN_1345', 'TRAIN_1346', 'TRAIN_1347', 'TRAIN_1348', 'TRAIN_1349', 'TRAIN_1350', 'TRAIN_1351', 'TRAIN_1352', 'TRAIN_1353', 'TRAIN_1354', 'TRAIN_1355', 'TRAIN_1356', 'TRAIN_1357', 'TRAIN_1358', 'TRAIN_1359', 'TRAIN_1360', 'TRAIN_1361', 'TRAIN_1362', 'TRAIN_1363', 'TRAIN_1364', 'TRAIN_1365', 'TRAIN_1366', 'TRAIN_1367', 'TRAIN_1368', 'TRAIN_1369', 'TRAIN_1370', 'TRAIN_1371', 'TRAIN_1372', 'TRAIN_1373', 'TRAIN_1374', 'TRAIN_1375', 'TRAIN_1376', 'TRAIN_1377', 'TRAIN_1378', 'TRAIN_1379', 'TRAIN_1380', 'TRAIN_1381', 'TRAIN_1382', 'TRAIN_1383', 'TRAIN_1384', 'TRAIN_1385', 'TRAIN_1386', 'TRAIN_1387', 'TRAIN_1388', 'TRAIN_1389', 'TRAIN_1390', 'TRAIN_1391', 'TRAIN_1392', 'TRAIN_1393', 'TRAIN_1394', 'TRAIN_1395', 'TRAIN_1396', 'TRAIN_1397', 'TRAIN_1398', 'TRAIN_1399', 'TRAIN_1400', 'TRAIN_1401', 'TRAIN_1402', 'TRAIN_1403', 'TRAIN_1404', 'TRAIN_1405', 'TRAIN_1406', 'TRAIN_1407', 'TRAIN_1408', 'TRAIN_1409', 'TRAIN_1410', 'TRAIN_1411', 'TRAIN_1412', 'TRAIN_1413', 'TRAIN_1414', 'TRAIN_1415', 'TRAIN_1416', 'TRAIN_1417', 'TRAIN_1418', 'TRAIN_1419', 'TRAIN_1420', 'TRAIN_1421', 'TRAIN_1422', 'TRAIN_1423', 'TRAIN_1424', 'TRAIN_1425', 'TRAIN_1426', 'TRAIN_1427', 'TRAIN_1428', 'TRAIN_1429', 'TRAIN_1430', 'TRAIN_1431', 'TRAIN_1432', 'TRAIN_1433', 'TRAIN_1434', 'TRAIN_1435', 'TRAIN_1436', 'TRAIN_1437', 'TRAIN_1438', 'TRAIN_1439', 'TRAIN_1440', 'TRAIN_1441', 'TRAIN_1442', 'TRAIN_1443', 'TRAIN_1444', 'TRAIN_1445', 'TRAIN_1446', 'TRAIN_1447', 'TRAIN_1448', 'TRAIN_1449', 'TRAIN_1450', 'TRAIN_1451', 'TRAIN_1452', 'TRAIN_1453', 'TRAIN_1454', 'TRAIN_1455', 'TRAIN_1456', 'TRAIN_1457', 'TRAIN_1458', 'TRAIN_1459', 'TRAIN_1460', 'TRAIN_1461', 'TRAIN_1462', 'TRAIN_1463', 'TRAIN_1464', 'TRAIN_1465', 'TRAIN_1466', 'TRAIN_1467', 'TRAIN_1468', 'TRAIN_1469', 'TRAIN_1470', 'TRAIN_1471', 'TRAIN_1472', 'TRAIN_1473', 'TRAIN_1474', 'TRAIN_1475', 'TRAIN_1476', 'TRAIN_1477', 'TRAIN_1478', 'TRAIN_1479', 'TRAIN_1480', 'TRAIN_1481', 'TRAIN_1482', 'TRAIN_1483', 'TRAIN_1484', 'TRAIN_1485', 'TRAIN_1486', 'TRAIN_1487', 'TRAIN_1488', 'TRAIN_1489', 'TRAIN_1490', 'TRAIN_1491', 'TRAIN_1492', 'TRAIN_1493', 'TRAIN_1494', 'TRAIN_1495', 'TRAIN_1496', 'TRAIN_1497', 'TRAIN_1498', 'TRAIN_1499', 'TRAIN_1500', 'TRAIN_1501', 'TRAIN_1502', 'TRAIN_1503', 'TRAIN_1504', 'TRAIN_1505', 'TRAIN_1506', 'TRAIN_1507', 'TRAIN_1508', 'TRAIN_1509', 'TRAIN_1510', 'TRAIN_1511', 'TRAIN_1512', 'TRAIN_1513', 'TRAIN_1514', 'TRAIN_1515', 'TRAIN_1516', 'TRAIN_1517', 'TRAIN_1518', 'TRAIN_1519', 'TRAIN_1520', 'TRAIN_1521', 'TRAIN_1522', 'TRAIN_1523', 'TRAIN_1524', 'TRAIN_1525', 'TRAIN_1526', 'TRAIN_1527', 'TRAIN_1528', 'TRAIN_1529', 'TRAIN_1530', 'TRAIN_1531', 'TRAIN_1532', 'TRAIN_1533', 'TRAIN_1534', 'TRAIN_1535', 'TRAIN_1536', 'TRAIN_1537', 'TRAIN_1538', 'TRAIN_1539', 'TRAIN_1540', 'TRAIN_1541', 'TRAIN_1542', 'TRAIN_1543', 'TRAIN_1544', 'TRAIN_1545', 'TRAIN_1546', 'TRAIN_1547', 'TRAIN_1548', 'TRAIN_1549', 'TRAIN_1550', 'TRAIN_1551', 'TRAIN_1552', 'TRAIN_1553', 'TRAIN_1554', 'TRAIN_1555', 'TRAIN_1556', 'TRAIN_1557', 'TRAIN_1558', 'TRAIN_1559', 'TRAIN_1560', 'TRAIN_1561', 'TRAIN_1562', 'TRAIN_1563', 'TRAIN_1564', 'TRAIN_1565', 'TRAIN_1566', 'TRAIN_1567', 'TRAIN_1568', 'TRAIN_1569', 'TRAIN_1570', 'TRAIN_1571', 'TRAIN_1572', 'TRAIN_1573', 'TRAIN_1574', 'TRAIN_1575', 'TRAIN_1576', 'TRAIN_1577', 'TRAIN_1578', 'TRAIN_1579', 'TRAIN_1580', 'TRAIN_1581', 'TRAIN_1582', 'TRAIN_1583', 'TRAIN_1584', 'TRAIN_1585', 'TRAIN_1586', 'TRAIN_1587', 'TRAIN_1588', 'TRAIN_1589', 'TRAIN_1590', 'TRAIN_1591', 'TRAIN_1592', 'TRAIN_1593', 'TRAIN_1594', 'TRAIN_1595', 'TRAIN_1596', 'TRAIN_1597', 'TRAIN_1598', 'TRAIN_1599', 'TRAIN_1600', 'TRAIN_1601', 'TRAIN_1602', 'TRAIN_1603', 'TRAIN_1604', 'TRAIN_1605', 'TRAIN_1606', 'TRAIN_1607', 'TRAIN_1608', 'TRAIN_1609', 'TRAIN_1610', 'TRAIN_1611', 'TRAIN_1612', 'TRAIN_1613', 'TRAIN_1614', 'TRAIN_1615', 'TRAIN_1616', 'TRAIN_1617', 'TRAIN_1618', 'TRAIN_1619', 'TRAIN_1620', 'TRAIN_1621', 'TRAIN_1622', 'TRAIN_1623', 'TRAIN_1624', 'TRAIN_1625', 'TRAIN_1626', 'TRAIN_1627', 'TRAIN_1628', 'TRAIN_1629', 'TRAIN_1630', 'TRAIN_1631', 'TRAIN_1632', 'TRAIN_1633', 'TRAIN_1634', 'TRAIN_1635', 'TRAIN_1636', 'TRAIN_1637', 'TRAIN_1638', 'TRAIN_1639', 'TRAIN_1640', 'TRAIN_1641', 'TRAIN_1642', 'TRAIN_1643', 'TRAIN_1644', 'TRAIN_1645', 'TRAIN_1646', 'TRAIN_1647', 'TRAIN_1648', 'TRAIN_1649', 'TRAIN_1650', 'TRAIN_1651', 'TRAIN_1652', 'TRAIN_1653', 'TRAIN_1654', 'TRAIN_1655', 'TRAIN_1656', 'TRAIN_1657', 'TRAIN_1658', 'TRAIN_1659', 'TRAIN_1660', 'TRAIN_1661', 'TRAIN_1662', 'TRAIN_1663', 'TRAIN_1664', 'TRAIN_1665', 'TRAIN_1666', 'TRAIN_1667', 'TRAIN_1668', 'TRAIN_1669', 'TRAIN_1670', 'TRAIN_1671', 'TRAIN_1672', 'TRAIN_1673', 'TRAIN_1674', 'TRAIN_1675', 'TRAIN_1676', 'TRAIN_1677', 'TRAIN_1678', 'TRAIN_1679', 'TRAIN_1680', 'TRAIN_1681', 'TRAIN_1682', 'TRAIN_1683', 'TRAIN_1684', 'TRAIN_1685', 'TRAIN_1686', 'TRAIN_1687', 'TRAIN_1688', 'TRAIN_1689', 'TRAIN_1690', 'TRAIN_1691', 'TRAIN_1692', 'TRAIN_1693', 'TRAIN_1694', 'TRAIN_1695', 'TRAIN_1696', 'TRAIN_1697', 'TRAIN_1698', 'TRAIN_1699', 'TRAIN_1700', 'TRAIN_1701', 'TRAIN_1702', 'TRAIN_1703', 'TRAIN_1704', 'TRAIN_1705', 'TRAIN_1706', 'TRAIN_1707', 'TRAIN_1708', 'TRAIN_1709', 'TRAIN_1710', 'TRAIN_1711', 'TRAIN_1712', 'TRAIN_1713', 'TRAIN_1714', 'TRAIN_1715', 'TRAIN_1716', 'TRAIN_1717', 'TRAIN_1718', 'TRAIN_1719', 'TRAIN_1720', 'TRAIN_1721', 'TRAIN_1722', 'TRAIN_1723', 'TRAIN_1724', 'TRAIN_1725', 'TRAIN_1726', 'TRAIN_1727', 'TRAIN_1728', 'TRAIN_1729', 'TRAIN_1730', 'TRAIN_1731', 'TRAIN_1732', 'TRAIN_1733', 'TRAIN_1734', 'TRAIN_1735', 'TRAIN_1736', 'TRAIN_1737', 'TRAIN_1738', 'TRAIN_1739', 'TRAIN_1740', 'TRAIN_1741', 'TRAIN_1742', 'TRAIN_1743', 'TRAIN_1744', 'TRAIN_1745', 'TRAIN_1746', 'TRAIN_1747', 'TRAIN_1748', 'TRAIN_1749', 'TRAIN_1750', 'TRAIN_1751', 'TRAIN_1752', 'TRAIN_1753', 'TRAIN_1754', 'TRAIN_1755', 'TRAIN_1756', 'TRAIN_1757', 'TRAIN_1758', 'TRAIN_1759', 'TRAIN_1760', 'TRAIN_1761', 'TRAIN_1762', 'TRAIN_1763', 'TRAIN_1764', 'TRAIN_1765', 'TRAIN_1766', 'TRAIN_1767', 'TRAIN_1768', 'TRAIN_1769', 'TRAIN_1770', 'TRAIN_1771', 'TRAIN_1772', 'TRAIN_1773', 'TRAIN_1774', 'TRAIN_1775', 'TRAIN_1776', 'TRAIN_1777', 'TRAIN_1778', 'TRAIN_1779', 'TRAIN_1780', 'TRAIN_1781', 'TRAIN_1782', 'TRAIN_1783', 'TRAIN_1784', 'TRAIN_1785', 'TRAIN_1786', 'TRAIN_1787', 'TRAIN_1788', 'TRAIN_1789', 'TRAIN_1790', 'TRAIN_1791', 'TRAIN_1792', 'TRAIN_1793', 'TRAIN_1794', 'TRAIN_1795', 'TRAIN_1796', 'TRAIN_1797', 'TRAIN_1798', 'TRAIN_1799', 'TRAIN_1800', 'TRAIN_1801', 'TRAIN_1802', 'TRAIN_1803', 'TRAIN_1804', 'TRAIN_1805', 'TRAIN_1806', 'TRAIN_1807', 'TRAIN_1808', 'TRAIN_1809', 'TRAIN_1810', 'TRAIN_1811', 'TRAIN_1812', 'TRAIN_1813', 'TRAIN_1814', 'TRAIN_1815', 'TRAIN_1816', 'TRAIN_1817', 'TRAIN_1818', 'TRAIN_1819', 'TRAIN_1820', 'TRAIN_1821', 'TRAIN_1822', 'TRAIN_1823', 'TRAIN_1824', 'TRAIN_1825', 'TRAIN_1826', 'TRAIN_1827', 'TRAIN_1828', 'TRAIN_1829', 'TRAIN_1830', 'TRAIN_1831', 'TRAIN_1832', 'TRAIN_1833', 'TRAIN_1834', 'TRAIN_1835', 'TRAIN_1836', 'TRAIN_1837', 'TRAIN_1838', 'TRAIN_1839', 'TRAIN_1840', 'TRAIN_1841', 'TRAIN_1842', 'TRAIN_1843', 'TRAIN_1844', 'TRAIN_1845', 'TRAIN_1846', 'TRAIN_1847', 'TRAIN_1848', 'TRAIN_1849', 'TRAIN_1850', 'TRAIN_1851', 'TRAIN_1852', 'TRAIN_1853', 'TRAIN_1854', 'TRAIN_1855', 'TRAIN_1856', 'TRAIN_1857', 'TRAIN_1858', 'TRAIN_1859', 'TRAIN_1860', 'TRAIN_1861', 'TRAIN_1862', 'TRAIN_1863', 'TRAIN_1864', 'TRAIN_1865', 'TRAIN_1866', 'TRAIN_1867', 'TRAIN_1868', 'TRAIN_1869', 'TRAIN_1870', 'TRAIN_1871', 'TRAIN_1872', 'TRAIN_1873', 'TRAIN_1874', 'TRAIN_1875', 'TRAIN_1876', 'TRAIN_1877', 'TRAIN_1878', 'TRAIN_1879', 'TRAIN_1880', 'TRAIN_1881', 'TRAIN_1882', 'TRAIN_1883', 'TRAIN_1884', 'TRAIN_1885', 'TRAIN_1886', 'TRAIN_1887', 'TRAIN_1888', 'TRAIN_1889', 'TRAIN_1890', 'TRAIN_1891', 'TRAIN_1892', 'TRAIN_1893', 'TRAIN_1894', 'TRAIN_1895', 'TRAIN_1896', 'TRAIN_1897', 'TRAIN_1898', 'TRAIN_1899', 'TRAIN_1900', 'TRAIN_1901', 'TRAIN_1902', 'TRAIN_1903', 'TRAIN_1904', 'TRAIN_1905', 'TRAIN_1906', 'TRAIN_1907', 'TRAIN_1908', 'TRAIN_1909', 'TRAIN_1910', 'TRAIN_1911', 'TRAIN_1912', 'TRAIN_1913', 'TRAIN_1914', 'TRAIN_1915', 'TRAIN_1916', 'TRAIN_1917', 'TRAIN_1918', 'TRAIN_1919', 'TRAIN_1920', 'TRAIN_1921', 'TRAIN_1922', 'TRAIN_1923', 'TRAIN_1924', 'TRAIN_1925', 'TRAIN_1926', 'TRAIN_1927', 'TRAIN_1928', 'TRAIN_1929', 'TRAIN_1930', 'TRAIN_1931', 'TRAIN_1932', 'TRAIN_1933', 'TRAIN_1934', 'TRAIN_1935', 'TRAIN_1936', 'TRAIN_1937', 'TRAIN_1938', 'TRAIN_1939', 'TRAIN_1940', 'TRAIN_1941', 'TRAIN_1942', 'TRAIN_1943', 'TRAIN_1944', 'TRAIN_1945', 'TRAIN_1946', 'TRAIN_1947', 'TRAIN_1948', 'TRAIN_1949', 'TRAIN_1950', 'TRAIN_1951', 'TRAIN_1952', 'TRAIN_1953', 'TRAIN_1954', 'TRAIN_1955', 'TRAIN_1956', 'TRAIN_1957', 'TRAIN_1958', 'TRAIN_1959', 'TRAIN_1960', 'TRAIN_1961', 'TRAIN_1962', 'TRAIN_1963', 'TRAIN_1964', 'TRAIN_1965', 'TRAIN_1966', 'TRAIN_1967', 'TRAIN_1968', 'TRAIN_1969', 'TRAIN_1970', 'TRAIN_1971', 'TRAIN_1972', 'TRAIN_1973', 'TRAIN_1974', 'TRAIN_1975', 'TRAIN_1976', 'TRAIN_1977', 'TRAIN_1978', 'TRAIN_1979', 'TRAIN_1980', 'TRAIN_1981', 'TRAIN_1982', 'TRAIN_1983', 'TRAIN_1984', 'TRAIN_1985', 'TRAIN_1986', 'TRAIN_1987', 'TRAIN_1988', 'TRAIN_1989', 'TRAIN_1990', 'TRAIN_1991', 'TRAIN_1992', 'TRAIN_1993', 'TRAIN_1994', 'TRAIN_1995', 'TRAIN_1996', 'TRAIN_1997', 'TRAIN_1998', 'TRAIN_1999', 'TRAIN_2000', 'TRAIN_2001', 'TRAIN_2002', 'TRAIN_2003', 'TRAIN_2004', 'TRAIN_2005', 'TRAIN_2006', 'TRAIN_2007', 'TRAIN_2008', 'TRAIN_2009', 'TRAIN_2010', 'TRAIN_2011', 'TRAIN_2012', 'TRAIN_2013', 'TRAIN_2014', 'TRAIN_2015', 'TRAIN_2016', 'TRAIN_2017', 'TRAIN_2018', 'TRAIN_2019', 'TRAIN_2020', 'TRAIN_2021', 'TRAIN_2022', 'TRAIN_2023', 'TRAIN_2024', 'TRAIN_2025', 'TRAIN_2026', 'TRAIN_2027', 'TRAIN_2028', 'TRAIN_2029', 'TRAIN_2030', 'TRAIN_2031', 'TRAIN_2032', 'TRAIN_2033', 'TRAIN_2034', 'TRAIN_2035', 'TRAIN_2036', 'TRAIN_2037', 'TRAIN_2038', 'TRAIN_2039', 'TRAIN_2040', 'TRAIN_2041', 'TRAIN_2042', 'TRAIN_2043', 'TRAIN_2044', 'TRAIN_2045', 'TRAIN_2046', 'TRAIN_2047', 'TRAIN_2048', 'TRAIN_2049', 'TRAIN_2050', 'TRAIN_2051', 'TRAIN_2052', 'TRAIN_2053', 'TRAIN_2054', 'TRAIN_2055', 'TRAIN_2056', 'TRAIN_2057', 'TRAIN_2058', 'TRAIN_2059', 'TRAIN_2060', 'TRAIN_2061', 'TRAIN_2062', 'TRAIN_2063', 'TRAIN_2064', 'TRAIN_2065', 'TRAIN_2066', 'TRAIN_2067', 'TRAIN_2068', 'TRAIN_2069', 'TRAIN_2070', 'TRAIN_2071', 'TRAIN_2072', 'TRAIN_2073', 'TRAIN_2074', 'TRAIN_2075', 'TRAIN_2076', 'TRAIN_2077', 'TRAIN_2078', 'TRAIN_2079', 'TRAIN_2080', 'TRAIN_2081', 'TRAIN_2082', 'TRAIN_2083', 'TRAIN_2084', 'TRAIN_2085', 'TRAIN_2086', 'TRAIN_2087', 'TRAIN_2088', 'TRAIN_2089', 'TRAIN_2090', 'TRAIN_2091', 'TRAIN_2092', 'TRAIN_2093', 'TRAIN_2094', 'TRAIN_2095', 'TRAIN_2096', 'TRAIN_2097', 'TRAIN_2098', 'TRAIN_2099', 'TRAIN_2100', 'TRAIN_2101', 'TRAIN_2102', 'TRAIN_2103', 'TRAIN_2104', 'TRAIN_2105', 'TRAIN_2106', 'TRAIN_2107', 'TRAIN_2108', 'TRAIN_2109', 'TRAIN_2110', 'TRAIN_2111', 'TRAIN_2112', 'TRAIN_2113', 'TRAIN_2114', 'TRAIN_2115', 'TRAIN_2116', 'TRAIN_2117', 'TRAIN_2118', 'TRAIN_2119', 'TRAIN_2120', 'TRAIN_2121', 'TRAIN_2122', 'TRAIN_2123', 'TRAIN_2124', 'TRAIN_2125', 'TRAIN_2126', 'TRAIN_2127', 'TRAIN_2128', 'TRAIN_2129', 'TRAIN_2130', 'TRAIN_2131', 'TRAIN_2132', 'TRAIN_2133', 'TRAIN_2134', 'TRAIN_2135', 'TRAIN_2136', 'TRAIN_2137', 'TRAIN_2138', 'TRAIN_2139', 'TRAIN_2140', 'TRAIN_2141', 'TRAIN_2142', 'TRAIN_2143', 'TRAIN_2144', 'TRAIN_2145', 'TRAIN_2146', 'TRAIN_2147', 'TRAIN_2148', 'TRAIN_2149', 'TRAIN_2150', 'TRAIN_2151', 'TRAIN_2152', 'TRAIN_2153', 'TRAIN_2154', 'TRAIN_2155', 'TRAIN_2156', 'TRAIN_2157', 'TRAIN_2158', 'TRAIN_2159', 'TRAIN_2160', 'TRAIN_2161', 'TRAIN_2162', 'TRAIN_2163', 'TRAIN_2164', 'TRAIN_2165', 'TRAIN_2166', 'TRAIN_2167', 'TRAIN_2168', 'TRAIN_2169', 'TRAIN_2170', 'TRAIN_2171', 'TRAIN_2172', 'TRAIN_2173', 'TRAIN_2174', 'TRAIN_2175', 'TRAIN_2176', 'TRAIN_2177', 'TRAIN_2178', 'TRAIN_2179', 'TRAIN_2180', 'TRAIN_2181', 'TRAIN_2182', 'TRAIN_2183', 'TRAIN_2184', 'TRAIN_2185', 'TRAIN_2186', 'TRAIN_2187', 'TRAIN_2188', 'TRAIN_2189', 'TRAIN_2190', 'TRAIN_2191', 'TRAIN_2192', 'TRAIN_2193', 'TRAIN_2194', 'TRAIN_2195', 'TRAIN_2196', 'TRAIN_2197', 'TRAIN_2198', 'TRAIN_2199', 'TRAIN_2200', 'TRAIN_2201', 'TRAIN_2202', 'TRAIN_2203', 'TRAIN_2204', 'TRAIN_2205', 'TRAIN_2206', 'TRAIN_2207', 'TRAIN_2208', 'TRAIN_2209', 'TRAIN_2210', 'TRAIN_2211', 'TRAIN_2212', 'TRAIN_2213', 'TRAIN_2214', 'TRAIN_2215', 'TRAIN_2216', 'TRAIN_2217', 'TRAIN_2218', 'TRAIN_2219', 'TRAIN_2220', 'TRAIN_2221', 'TRAIN_2222', 'TRAIN_2223', 'TRAIN_2224', 'TRAIN_2225', 'TRAIN_2226', 'TRAIN_2227', 'TRAIN_2228', 'TRAIN_2229', 'TRAIN_2230', 'TRAIN_2231', 'TRAIN_2232', 'TRAIN_2233', 'TRAIN_2234', 'TRAIN_2235', 'TRAIN_2236', 'TRAIN_2237', 'TRAIN_2238', 'TRAIN_2239', 'TRAIN_2240', 'TRAIN_2241', 'TRAIN_2242', 'TRAIN_2243', 'TRAIN_2244', 'TRAIN_2245', 'TRAIN_2246', 'TRAIN_2247', 'TRAIN_2248', 'TRAIN_2249', 'TRAIN_2250', 'TRAIN_2251', 'TRAIN_2252', 'TRAIN_2253', 'TRAIN_2254', 'TRAIN_2255', 'TRAIN_2256', 'TRAIN_2257', 'TRAIN_2258', 'TRAIN_2259', 'TRAIN_2260', 'TRAIN_2261', 'TRAIN_2262', 'TRAIN_2263', 'TRAIN_2264', 'TRAIN_2265', 'TRAIN_2266', 'TRAIN_2267', 'TRAIN_2268', 'TRAIN_2269', 'TRAIN_2270', 'TRAIN_2271', 'TRAIN_2272', 'TRAIN_2273', 'TRAIN_2274', 'TRAIN_2275', 'TRAIN_2276', 'TRAIN_2277', 'TRAIN_2278', 'TRAIN_2279', 'TRAIN_2280', 'TRAIN_2281', 'TRAIN_2282', 'TRAIN_2283', 'TRAIN_2284', 'TRAIN_2285', 'TRAIN_2286', 'TRAIN_2287', 'TRAIN_2288', 'TRAIN_2289', 'TRAIN_2290', 'TRAIN_2291', 'TRAIN_2292', 'TRAIN_2293', 'TRAIN_2294', 'TRAIN_2295', 'TRAIN_2296', 'TRAIN_2297', 'TRAIN_2298', 'TRAIN_2299', 'TRAIN_2300', 'TRAIN_2301', 'TRAIN_2302', 'TRAIN_2303', 'TRAIN_2304', 'TRAIN_2305', 'TRAIN_2306', 'TRAIN_2307', 'TRAIN_2308', 'TRAIN_2309', 'TRAIN_2310', 'TRAIN_2311', 'TRAIN_2312', 'TRAIN_2313', 'TRAIN_2314', 'TRAIN_2315', 'TRAIN_2316', 'TRAIN_2317', 'TRAIN_2318', 'TRAIN_2319', 'TRAIN_2320', 'TRAIN_2321', 'TRAIN_2322', 'TRAIN_2323', 'TRAIN_2324', 'TRAIN_2325', 'TRAIN_2326', 'TRAIN_2327', 'TRAIN_2328', 'TRAIN_2329', 'TRAIN_2330', 'TRAIN_2331', 'TRAIN_2332', 'TRAIN_2333', 'TRAIN_2334', 'TRAIN_2335', 'TRAIN_2336', 'TRAIN_2337', 'TRAIN_2338', 'TRAIN_2339', 'TRAIN_2340', 'TRAIN_2341', 'TRAIN_2342', 'TRAIN_2343', 'TRAIN_2344', 'TRAIN_2345', 'TRAIN_2346', 'TRAIN_2347', 'TRAIN_2348', 'TRAIN_2349', 'TRAIN_2350', 'TRAIN_2351', 'TRAIN_2352', 'TRAIN_2353', 'TRAIN_2354', 'TRAIN_2355', 'TRAIN_2356', 'TRAIN_2357', 'TRAIN_2358', 'TRAIN_2359', 'TRAIN_2360', 'TRAIN_2361', 'TRAIN_2362', 'TRAIN_2363', 'TRAIN_2364', 'TRAIN_2365', 'TRAIN_2366', 'TRAIN_2367', 'TRAIN_2368', 'TRAIN_2369', 'TRAIN_2370', 'TRAIN_2371', 'TRAIN_2372', 'TRAIN_2373', 'TRAIN_2374', 'TRAIN_2375', 'TRAIN_2376', 'TRAIN_2377', 'TRAIN_2378', 'TRAIN_2379', 'TRAIN_2380', 'TRAIN_2381', 'TRAIN_2382', 'TRAIN_2383', 'TRAIN_2384', 'TRAIN_2385', 'TRAIN_2386', 'TRAIN_2387', 'TRAIN_2388', 'TRAIN_2389', 'TRAIN_2390', 'TRAIN_2391', 'TRAIN_2392', 'TRAIN_2393', 'TRAIN_2394', 'TRAIN_2395', 'TRAIN_2396', 'TRAIN_2397', 'TRAIN_2398', 'TRAIN_2399', 'TRAIN_2400', 'TRAIN_2401', 'TRAIN_2402', 'TRAIN_2403', 'TRAIN_2404', 'TRAIN_2405', 'TRAIN_2406', 'TRAIN_2407', 'TRAIN_2408', 'TRAIN_2409', 'TRAIN_2410', 'TRAIN_2411', 'TRAIN_2412', 'TRAIN_2413', 'TRAIN_2414', 'TRAIN_2415', 'TRAIN_2416', 'TRAIN_2417', 'TRAIN_2418', 'TRAIN_2419', 'TRAIN_2420', 'TRAIN_2421', 'TRAIN_2422', 'TRAIN_2423', 'TRAIN_2424', 'TRAIN_2425', 'TRAIN_2426', 'TRAIN_2427', 'TRAIN_2428', 'TRAIN_2429', 'TRAIN_2430', 'TRAIN_2431', 'TRAIN_2432', 'TRAIN_2433', 'TRAIN_2434', 'TRAIN_2435', 'TRAIN_2436', 'TRAIN_2437', 'TRAIN_2438', 'TRAIN_2439', 'TRAIN_2440', 'TRAIN_2441', 'TRAIN_2442', 'TRAIN_2443', 'TRAIN_2444', 'TRAIN_2445', 'TRAIN_2446', 'TRAIN_2447', 'TRAIN_2448', 'TRAIN_2449', 'TRAIN_2450', 'TRAIN_2451', 'TRAIN_2452', 'TRAIN_2453', 'TRAIN_2454', 'TRAIN_2455', 'TRAIN_2456', 'TRAIN_2457', 'TRAIN_2458', 'TRAIN_2459', 'TRAIN_2460', 'TRAIN_2461', 'TRAIN_2462', 'TRAIN_2463', 'TRAIN_2464', 'TRAIN_2465', 'TRAIN_2466', 'TRAIN_2467', 'TRAIN_2468', 'TRAIN_2469', 'TRAIN_2470', 'TRAIN_2471', 'TRAIN_2472', 'TRAIN_2473', 'TRAIN_2474', 'TRAIN_2475', 'TRAIN_2476', 'TRAIN_2477', 'TRAIN_2478', 'TRAIN_2479', 'TRAIN_2480', 'TRAIN_2481', 'TRAIN_2482', 'TRAIN_2483', 'TRAIN_2484', 'TRAIN_2485', 'TRAIN_2486', 'TRAIN_2487', 'TRAIN_2488', 'TRAIN_2489', 'TRAIN_2490', 'TRAIN_2491', 'TRAIN_2492', 'TRAIN_2493', 'TRAIN_2494', 'TRAIN_2495', 'TRAIN_2496', 'TRAIN_2497', 'TRAIN_2498', 'TRAIN_2499', 'TRAIN_2500', 'TRAIN_2501', 'TRAIN_2502', 'TRAIN_2503', 'TRAIN_2504', 'TRAIN_2505', 'TRAIN_2506', 'TRAIN_2507', 'TRAIN_2508', 'TRAIN_2509', 'TRAIN_2510', 'TRAIN_2511', 'TRAIN_2512', 'TRAIN_2513', 'TRAIN_2514', 'TRAIN_2515', 'TRAIN_2516', 'TRAIN_2517', 'TRAIN_2518', 'TRAIN_2519', 'TRAIN_2520', 'TRAIN_2521', 'TRAIN_2522', 'TRAIN_2523', 'TRAIN_2524', 'TRAIN_2525', 'TRAIN_2526', 'TRAIN_2527', 'TRAIN_2528', 'TRAIN_2529', 'TRAIN_2530', 'TRAIN_2531', 'TRAIN_2532', 'TRAIN_2533', 'TRAIN_2534', 'TRAIN_2535', 'TRAIN_2536', 'TRAIN_2537', 'TRAIN_2538', 'TRAIN_2539', 'TRAIN_2540', 'TRAIN_2541', 'TRAIN_2542', 'TRAIN_2543', 'TRAIN_2544', 'TRAIN_2545', 'TRAIN_2546', 'TRAIN_2547', 'TRAIN_2548', 'TRAIN_2549', 'TRAIN_2550', 'TRAIN_2551', 'TRAIN_2552', 'TRAIN_2553', 'TRAIN_2554', 'TRAIN_2555', 'TRAIN_2556', 'TRAIN_2557', 'TRAIN_2558', 'TRAIN_2559', 'TRAIN_2560', 'TRAIN_2561', 'TRAIN_2562', 'TRAIN_2563', 'TRAIN_2564', 'TRAIN_2565', 'TRAIN_2566', 'TRAIN_2567', 'TRAIN_2568', 'TRAIN_2569', 'TRAIN_2570', 'TRAIN_2571', 'TRAIN_2572', 'TRAIN_2573', 'TRAIN_2574', 'TRAIN_2575', 'TRAIN_2576', 'TRAIN_2577', 'TRAIN_2578', 'TRAIN_2579', 'TRAIN_2580', 'TRAIN_2581', 'TRAIN_2582', 'TRAIN_2583', 'TRAIN_2584', 'TRAIN_2585', 'TRAIN_2586', 'TRAIN_2587', 'TRAIN_2588', 'TRAIN_2589', 'TRAIN_2590', 'TRAIN_2591', 'TRAIN_2592', 'TRAIN_2593', 'TRAIN_2594', 'TRAIN_2595', 'TRAIN_2596', 'TRAIN_2597', 'TRAIN_2598', 'TRAIN_2599', 'TRAIN_2600', 'TRAIN_2601', 'TRAIN_2602', 'TRAIN_2603', 'TRAIN_2604', 'TRAIN_2605', 'TRAIN_2606', 'TRAIN_2607', 'TRAIN_2608', 'TRAIN_2609', 'TRAIN_2610', 'TRAIN_2611', 'TRAIN_2612', 'TRAIN_2613', 'TRAIN_2614', 'TRAIN_2615', 'TRAIN_2616', 'TRAIN_2617', 'TRAIN_2618', 'TRAIN_2619', 'TRAIN_2620', 'TRAIN_2621', 'TRAIN_2622', 'TRAIN_2623', 'TRAIN_2624', 'TRAIN_2625', 'TRAIN_2626', 'TRAIN_2627', 'TRAIN_2628', 'TRAIN_2629', 'TRAIN_2630', 'TRAIN_2631', 'TRAIN_2632', 'TRAIN_2633', 'TRAIN_2634', 'TRAIN_2635', 'TRAIN_2636', 'TRAIN_2637', 'TRAIN_2638', 'TRAIN_2639', 'TRAIN_2640', 'TRAIN_2641', 'TRAIN_2642', 'TRAIN_2643', 'TRAIN_2644', 'TRAIN_2645', 'TRAIN_2646', 'TRAIN_2647', 'TRAIN_2648', 'TRAIN_2649', 'TRAIN_2650', 'TRAIN_2651', 'TRAIN_2652', 'TRAIN_2653', 'TRAIN_2654', 'TRAIN_2655', 'TRAIN_2656', 'TRAIN_2657', 'TRAIN_2658', 'TRAIN_2659', 'TRAIN_2660', 'TRAIN_2661', 'TRAIN_2662', 'TRAIN_2663', 'TRAIN_2664', 'TRAIN_2665', 'TRAIN_2666', 'TRAIN_2667', 'TRAIN_2668', 'TRAIN_2669', 'TRAIN_2670', 'TRAIN_2671', 'TRAIN_2672', 'TRAIN_2673', 'TRAIN_2674', 'TRAIN_2675', 'TRAIN_2676', 'TRAIN_2677', 'TRAIN_2678', 'TRAIN_2679', 'TRAIN_2680', 'TRAIN_2681', 'TRAIN_2682', 'TRAIN_2683', 'TRAIN_2684', 'TRAIN_2685', 'TRAIN_2686', 'TRAIN_2687', 'TRAIN_2688', 'TRAIN_2689', 'TRAIN_2690', 'TRAIN_2691', 'TRAIN_2692', 'TRAIN_2693', 'TRAIN_2694', 'TRAIN_2695', 'TRAIN_2696', 'TRAIN_2697', 'TRAIN_2698', 'TRAIN_2699', 'TRAIN_2700', 'TRAIN_2701', 'TRAIN_2702', 'TRAIN_2703', 'TRAIN_2704', 'TRAIN_2705', 'TRAIN_2706', 'TRAIN_2707', 'TRAIN_2708', 'TRAIN_2709', 'TRAIN_2710', 'TRAIN_2711', 'TRAIN_2712', 'TRAIN_2713', 'TRAIN_2714', 'TRAIN_2715', 'TRAIN_2716', 'TRAIN_2717', 'TRAIN_2718', 'TRAIN_2719', 'TRAIN_2720', 'TRAIN_2721', 'TRAIN_2722', 'TRAIN_2723', 'TRAIN_2724', 'TRAIN_2725', 'TRAIN_2726', 'TRAIN_2727', 'TRAIN_2728', 'TRAIN_2729', 'TRAIN_2730', 'TRAIN_2731', 'TRAIN_2732', 'TRAIN_2733', 'TRAIN_2734', 'TRAIN_2735', 'TRAIN_2736', 'TRAIN_2737', 'TRAIN_2738', 'TRAIN_2739', 'TRAIN_2740', 'TRAIN_2741', 'TRAIN_2742', 'TRAIN_2743', 'TRAIN_2744', 'TRAIN_2745', 'TRAIN_2746', 'TRAIN_2747', 'TRAIN_2748', 'TRAIN_2749', 'TRAIN_2750', 'TRAIN_2751', 'TRAIN_2752', 'TRAIN_2753', 'TRAIN_2754', 'TRAIN_2755', 'TRAIN_2756', 'TRAIN_2757', 'TRAIN_2758', 'TRAIN_2759', 'TRAIN_2760', 'TRAIN_2761', 'TRAIN_2762', 'TRAIN_2763', 'TRAIN_2764', 'TRAIN_2765', 'TRAIN_2766', 'TRAIN_2767', 'TRAIN_2768', 'TRAIN_2769', 'TRAIN_2770', 'TRAIN_2771', 'TRAIN_2772', 'TRAIN_2773', 'TRAIN_2774', 'TRAIN_2775', 'TRAIN_2776', 'TRAIN_2777', 'TRAIN_2778', 'TRAIN_2779', 'TRAIN_2780', 'TRAIN_2781', 'TRAIN_2782', 'TRAIN_2783', 'TRAIN_2784', 'TRAIN_2785', 'TRAIN_2786', 'TRAIN_2787', 'TRAIN_2788', 'TRAIN_2789', 'TRAIN_2790', 'TRAIN_2791', 'TRAIN_2792', 'TRAIN_2793', 'TRAIN_2794', 'TRAIN_2795', 'TRAIN_2796', 'TRAIN_2797', 'TRAIN_2798', 'TRAIN_2799', 'TRAIN_2800', 'TRAIN_2801', 'TRAIN_2802', 'TRAIN_2803', 'TRAIN_2804', 'TRAIN_2805', 'TRAIN_2806', 'TRAIN_2807', 'TRAIN_2808', 'TRAIN_2809', 'TRAIN_2810', 'TRAIN_2811', 'TRAIN_2812', 'TRAIN_2813', 'TRAIN_2814', 'TRAIN_2815', 'TRAIN_2816', 'TRAIN_2817', 'TRAIN_2818', 'TRAIN_2819', 'TRAIN_2820', 'TRAIN_2821', 'TRAIN_2822', 'TRAIN_2823', 'TRAIN_2824', 'TRAIN_2825', 'TRAIN_2826', 'TRAIN_2827', 'TRAIN_2828', 'TRAIN_2829', 'TRAIN_2830', 'TRAIN_2831', 'TRAIN_2832', 'TRAIN_2833', 'TRAIN_2834', 'TRAIN_2835', 'TRAIN_2836', 'TRAIN_2837', 'TRAIN_2838', 'TRAIN_2839', 'TRAIN_2840', 'TRAIN_2841', 'TRAIN_2842', 'TRAIN_2843', 'TRAIN_2844', 'TRAIN_2845', 'TRAIN_2846', 'TRAIN_2847', 'TRAIN_2848', 'TRAIN_2849', 'TRAIN_2850', 'TRAIN_2851', 'TRAIN_2852', 'TRAIN_2853', 'TRAIN_2854', 'TRAIN_2855', 'TRAIN_2856', 'TRAIN_2857', 'TRAIN_2858', 'TRAIN_2859', 'TRAIN_2860', 'TRAIN_2861', 'TRAIN_2862', 'TRAIN_2863', 'TRAIN_2864', 'TRAIN_2865', 'TRAIN_2866', 'TRAIN_2867', 'TRAIN_2868', 'TRAIN_2869', 'TRAIN_2870', 'TRAIN_2871', 'TRAIN_2872', 'TRAIN_2873', 'TRAIN_2874', 'TRAIN_2875', 'TRAIN_2876', 'TRAIN_2877', 'TRAIN_2878', 'TRAIN_2879', 'TRAIN_2880', 'TRAIN_2881', 'TRAIN_2882', 'TRAIN_2883', 'TRAIN_2884', 'TRAIN_2885', 'TRAIN_2886', 'TRAIN_2887', 'TRAIN_2888', 'TRAIN_2889', 'TRAIN_2890', 'TRAIN_2891', 'TRAIN_2892', 'TRAIN_2893', 'TRAIN_2894', 'TRAIN_2895', 'TRAIN_2896', 'TRAIN_2897', 'TRAIN_2898', 'TRAIN_2899', 'TRAIN_2900', 'TRAIN_2901', 'TRAIN_2902', 'TRAIN_2903', 'TRAIN_2904', 'TRAIN_2905', 'TRAIN_2906', 'TRAIN_2907', 'TRAIN_2908', 'TRAIN_2909', 'TRAIN_2910', 'TRAIN_2911', 'TRAIN_2912', 'TRAIN_2913', 'TRAIN_2914', 'TRAIN_2915', 'TRAIN_2916', 'TRAIN_2917', 'TRAIN_2918', 'TRAIN_2919', 'TRAIN_2920', 'TRAIN_2921', 'TRAIN_2922', 'TRAIN_2923', 'TRAIN_2924', 'TRAIN_2925', 'TRAIN_2926', 'TRAIN_2927', 'TRAIN_2928', 'TRAIN_2929', 'TRAIN_2930', 'TRAIN_2931', 'TRAIN_2932', 'TRAIN_2933', 'TRAIN_2934', 'TRAIN_2935', 'TRAIN_2936', 'TRAIN_2937', 'TRAIN_2938', 'TRAIN_2939', 'TRAIN_2940', 'TRAIN_2941', 'TRAIN_2942', 'TRAIN_2943', 'TRAIN_2944', 'TRAIN_2945', 'TRAIN_2946', 'TRAIN_2947', 'TRAIN_2948', 'TRAIN_2949', 'TRAIN_2950', 'TRAIN_2951', 'TRAIN_2952', 'TRAIN_2953', 'TRAIN_2954', 'TRAIN_2955', 'TRAIN_2956', 'TRAIN_2957', 'TRAIN_2958', 'TRAIN_2959', 'TRAIN_2960', 'TRAIN_2961', 'TRAIN_2962', 'TRAIN_2963', 'TRAIN_2964', 'TRAIN_2965', 'TRAIN_2966', 'TRAIN_2967', 'TRAIN_2968', 'TRAIN_2969', 'TRAIN_2970', 'TRAIN_2971', 'TRAIN_2972', 'TRAIN_2973', 'TRAIN_2974', 'TRAIN_2975', 'TRAIN_2976', 'TRAIN_2977', 'TRAIN_2978', 'TRAIN_2979', 'TRAIN_2980', 'TRAIN_2981', 'TRAIN_2982', 'TRAIN_2983', 'TRAIN_2984', 'TRAIN_2985', 'TRAIN_2986', 'TRAIN_2987', 'TRAIN_2988', 'TRAIN_2989', 'TRAIN_2990', 'TRAIN_2991', 'TRAIN_2992', 'TRAIN_2993', 'TRAIN_2994', 'TRAIN_2995', 'TRAIN_2996', 'TRAIN_2997', 'TRAIN_2998', 'TRAIN_2999', 'TRAIN_3000', 'TRAIN_3001', 'TRAIN_3002', 'TRAIN_3003', 'TRAIN_3004', 'TRAIN_3005', 'TRAIN_3006', 'TRAIN_3007', 'TRAIN_3008', 'TRAIN_3009', 'TRAIN_3010', 'TRAIN_3011', 'TRAIN_3012', 'TRAIN_3013', 'TRAIN_3014', 'TRAIN_3015', 'TRAIN_3016', 'TRAIN_3017', 'TRAIN_3018', 'TRAIN_3019', 'TRAIN_3020', 'TRAIN_3021', 'TRAIN_3022', 'TRAIN_3023', 'TRAIN_3024', 'TRAIN_3025', 'TRAIN_3026', 'TRAIN_3027', 'TRAIN_3028', 'TRAIN_3029', 'TRAIN_3030', 'TRAIN_3031', 'TRAIN_3032', 'TRAIN_3033', 'TRAIN_3034', 'TRAIN_3035', 'TRAIN_3036', 'TRAIN_3037', 'TRAIN_3038', 'TRAIN_3039', 'TRAIN_3040', 'TRAIN_3041', 'TRAIN_3042', 'TRAIN_3043', 'TRAIN_3044', 'TRAIN_3045', 'TRAIN_3046', 'TRAIN_3047', 'TRAIN_3048', 'TRAIN_3049', 'TRAIN_3050', 'TRAIN_3051', 'TRAIN_3052', 'TRAIN_3053', 'TRAIN_3054', 'TRAIN_3055', 'TRAIN_3056', 'TRAIN_3057', 'TRAIN_3058', 'TRAIN_3059', 'TRAIN_3060', 'TRAIN_3061', 'TRAIN_3062', 'TRAIN_3063', 'TRAIN_3064', 'TRAIN_3065', 'TRAIN_3066', 'TRAIN_3067', 'TRAIN_3068', 'TRAIN_3069', 'TRAIN_3070', 'TRAIN_3071', 'TRAIN_3072', 'TRAIN_3073', 'TRAIN_3074', 'TRAIN_3075', 'TRAIN_3076', 'TRAIN_3077', 'TRAIN_3078', 'TRAIN_3079', 'TRAIN_3080', 'TRAIN_3081', 'TRAIN_3082', 'TRAIN_3083', 'TRAIN_3084', 'TRAIN_3085', 'TRAIN_3086', 'TRAIN_3087', 'TRAIN_3088', 'TRAIN_3089', 'TRAIN_3090', 'TRAIN_3091', 'TRAIN_3092', 'TRAIN_3093', 'TRAIN_3094', 'TRAIN_3095', 'TRAIN_3096', 'TRAIN_3097', 'TRAIN_3098', 'TRAIN_3099', 'TRAIN_3100', 'TRAIN_3101', 'TRAIN_3102', 'TRAIN_3103', 'TRAIN_3104', 'TRAIN_3105', 'TRAIN_3106', 'TRAIN_3107', 'TRAIN_3108', 'TRAIN_3109', 'TRAIN_3110', 'TRAIN_3111', 'TRAIN_3112', 'TRAIN_3113', 'TRAIN_3114', 'TRAIN_3115', 'TRAIN_3116', 'TRAIN_3117', 'TRAIN_3118', 'TRAIN_3119', 'TRAIN_3120', 'TRAIN_3121', 'TRAIN_3122', 'TRAIN_3123', 'TRAIN_3124', 'TRAIN_3125', 'TRAIN_3126', 'TRAIN_3127', 'TRAIN_3128', 'TRAIN_3129', 'TRAIN_3130', 'TRAIN_3131', 'TRAIN_3132', 'TRAIN_3133', 'TRAIN_3134', 'TRAIN_3135', 'TRAIN_3136', 'TRAIN_3137', 'TRAIN_3138', 'TRAIN_3139', 'TRAIN_3140', 'TRAIN_3141', 'TRAIN_3142', 'TRAIN_3143', 'TRAIN_3144', 'TRAIN_3145', 'TRAIN_3146', 'TRAIN_3147', 'TRAIN_3148', 'TRAIN_3149', 'TRAIN_3150', 'TRAIN_3151', 'TRAIN_3152', 'TRAIN_3153', 'TRAIN_3154', 'TRAIN_3155', 'TRAIN_3156', 'TRAIN_3157', 'TRAIN_3158', 'TRAIN_3159', 'TRAIN_3160', 'TRAIN_3161', 'TRAIN_3162', 'TRAIN_3163', 'TRAIN_3164', 'TRAIN_3165', 'TRAIN_3166', 'TRAIN_3167', 'TRAIN_3168', 'TRAIN_3169', 'TRAIN_3170', 'TRAIN_3171', 'TRAIN_3172', 'TRAIN_3173', 'TRAIN_3174', 'TRAIN_3175', 'TRAIN_3176', 'TRAIN_3177', 'TRAIN_3178', 'TRAIN_3179', 'TRAIN_3180', 'TRAIN_3181', 'TRAIN_3182', 'TRAIN_3183', 'TRAIN_3184', 'TRAIN_3185', 'TRAIN_3186', 'TRAIN_3187', 'TRAIN_3188', 'TRAIN_3189', 'TRAIN_3190', 'TRAIN_3191', 'TRAIN_3192', 'TRAIN_3193', 'TRAIN_3194', 'TRAIN_3195', 'TRAIN_3196', 'TRAIN_3197', 'TRAIN_3198', 'TRAIN_3199', 'TRAIN_3200', 'TRAIN_3201', 'TRAIN_3202', 'TRAIN_3203', 'TRAIN_3204', 'TRAIN_3205', 'TRAIN_3206', 'TRAIN_3207', 'TRAIN_3208', 'TRAIN_3209', 'TRAIN_3210', 'TRAIN_3211', 'TRAIN_3212', 'TRAIN_3213', 'TRAIN_3214', 'TRAIN_3215', 'TRAIN_3216', 'TRAIN_3217', 'TRAIN_3218', 'TRAIN_3219', 'TRAIN_3220', 'TRAIN_3221', 'TRAIN_3222', 'TRAIN_3223', 'TRAIN_3224', 'TRAIN_3225', 'TRAIN_3226', 'TRAIN_3227', 'TRAIN_3228', 'TRAIN_3229', 'TRAIN_3230', 'TRAIN_3231', 'TRAIN_3232', 'TRAIN_3233', 'TRAIN_3234', 'TRAIN_3235', 'TRAIN_3236', 'TRAIN_3237', 'TRAIN_3238', 'TRAIN_3239', 'TRAIN_3240', 'TRAIN_3241', 'TRAIN_3242', 'TRAIN_3243', 'TRAIN_3244', 'TRAIN_3245', 'TRAIN_3246', 'TRAIN_3247', 'TRAIN_3248', 'TRAIN_3249', 'TRAIN_3250', 'TRAIN_3251', 'TRAIN_3252', 'TRAIN_3253', 'TRAIN_3254', 'TRAIN_3255', 'TRAIN_3256', 'TRAIN_3257', 'TRAIN_3258', 'TRAIN_3259', 'TRAIN_3260', 'TRAIN_3261', 'TRAIN_3262', 'TRAIN_3263', 'TRAIN_3264', 'TRAIN_3265', 'TRAIN_3266', 'TRAIN_3267', 'TRAIN_3268', 'TRAIN_3269', 'TRAIN_3270', 'TRAIN_3271', 'TRAIN_3272', 'TRAIN_3273', 'TRAIN_3274', 'TRAIN_3275', 'TRAIN_3276', 'TRAIN_3277', 'TRAIN_3278', 'TRAIN_3279', 'TRAIN_3280', 'TRAIN_3281', 'TRAIN_3282', 'TRAIN_3283', 'TRAIN_3284', 'TRAIN_3285', 'TRAIN_3286', 'TRAIN_3287', 'TRAIN_3288', 'TRAIN_3289', 'TRAIN_3290', 'TRAIN_3291', 'TRAIN_3292', 'TRAIN_3293', 'TRAIN_3294', 'TRAIN_3295', 'TRAIN_3296', 'TRAIN_3297', 'TRAIN_3298', 'TRAIN_3299', 'TRAIN_3300', 'TRAIN_3301', 'TRAIN_3302', 'TRAIN_3303', 'TRAIN_3304', 'TRAIN_3305', 'TRAIN_3306', 'TRAIN_3307', 'TRAIN_3308', 'TRAIN_3309', 'TRAIN_3310', 'TRAIN_3311', 'TRAIN_3312', 'TRAIN_3313', 'TRAIN_3314', 'TRAIN_3315', 'TRAIN_3316', 'TRAIN_3317', 'TRAIN_3318', 'TRAIN_3319', 'TRAIN_3320', 'TRAIN_3321', 'TRAIN_3322', 'TRAIN_3323', 'TRAIN_3324', 'TRAIN_3325', 'TRAIN_3326', 'TRAIN_3327', 'TRAIN_3328', 'TRAIN_3329', 'TRAIN_3330', 'TRAIN_3331', 'TRAIN_3332', 'TRAIN_3333', 'TRAIN_3334', 'TRAIN_3335', 'TRAIN_3336', 'TRAIN_3337', 'TRAIN_3338', 'TRAIN_3339', 'TRAIN_3340', 'TRAIN_3341', 'TRAIN_3342', 'TRAIN_3343', 'TRAIN_3344', 'TRAIN_3345', 'TRAIN_3346', 'TRAIN_3347', 'TRAIN_3348', 'TRAIN_3349', 'TRAIN_3350', 'TRAIN_3351', 'TRAIN_3352', 'TRAIN_3353', 'TRAIN_3354', 'TRAIN_3355', 'TRAIN_3356', 'TRAIN_3357', 'TRAIN_3358', 'TRAIN_3359', 'TRAIN_3360', 'TRAIN_3361', 'TRAIN_3362', 'TRAIN_3363', 'TRAIN_3364', 'TRAIN_3365', 'TRAIN_3366', 'TRAIN_3367', 'TRAIN_3368', 'TRAIN_3369', 'TRAIN_3370', 'TRAIN_3371', 'TRAIN_3372', 'TRAIN_3373', 'TRAIN_3374', 'TRAIN_3375', 'TRAIN_3376', 'TRAIN_3377', 'TRAIN_3378', 'TRAIN_3379', 'TRAIN_3380', 'TRAIN_3381', 'TRAIN_3382', 'TRAIN_3383', 'TRAIN_3384', 'TRAIN_3385', 'TRAIN_3386', 'TRAIN_3387', 'TRAIN_3388', 'TRAIN_3389', 'TRAIN_3390', 'TRAIN_3391', 'TRAIN_3392', 'TRAIN_3393', 'TRAIN_3394', 'TRAIN_3395', 'TRAIN_3396', 'TRAIN_3397', 'TRAIN_3398', 'TRAIN_3399', 'TRAIN_3400', 'TRAIN_3401', 'TRAIN_3402', 'TRAIN_3403', 'TRAIN_3404', 'TRAIN_3405', 'TRAIN_3406', 'TRAIN_3407', 'TRAIN_3408', 'TRAIN_3409', 'TRAIN_3410', 'TRAIN_3411', 'TRAIN_3412', 'TRAIN_3413', 'TRAIN_3414', 'TRAIN_3415', 'TRAIN_3416', 'TRAIN_3417', 'TRAIN_3418', 'TRAIN_3419', 'TRAIN_3420', 'TRAIN_3421', 'TRAIN_3422', 'TRAIN_3423', 'TRAIN_3424', 'TRAIN_3425', 'TRAIN_3426', 'TRAIN_3427', 'TRAIN_3428', 'TRAIN_3429', 'TRAIN_3430', 'TRAIN_3431', 'TRAIN_3432', 'TRAIN_3433', 'TRAIN_3434', 'TRAIN_3435', 'TRAIN_3436', 'TRAIN_3437', 'TRAIN_3438', 'TRAIN_3439', 'TRAIN_3440', 'TRAIN_3441', 'TRAIN_3442', 'TRAIN_3443', 'TRAIN_3444', 'TRAIN_3445', 'TRAIN_3446', 'TRAIN_3447', 'TRAIN_3448', 'TRAIN_3449', 'TRAIN_3450', 'TRAIN_3451', 'TRAIN_3452', 'TRAIN_3453', 'TRAIN_3454', 'TRAIN_3455', 'TRAIN_3456', 'TRAIN_3457', 'TRAIN_3458', 'TRAIN_3459', 'TRAIN_3460', 'TRAIN_3461', 'TRAIN_3462', 'TRAIN_3463', 'TRAIN_3464', 'TRAIN_3465', 'TRAIN_3466', 'TRAIN_3467', 'TRAIN_3468', 'TRAIN_3469', 'TRAIN_3470', 'TRAIN_3471', 'TRAIN_3472', 'TRAIN_3473', 'TRAIN_3474', 'TRAIN_3475', 'TRAIN_3476', 'TRAIN_3477', 'TRAIN_3478', 'TRAIN_3479', 'TRAIN_3480', 'TRAIN_3481', 'TRAIN_3482', 'TRAIN_3483', 'TRAIN_3484', 'TRAIN_3485', 'TRAIN_3486', 'TRAIN_3487', 'TRAIN_3488', 'TRAIN_3489', 'TRAIN_3490', 'TRAIN_3491', 'TRAIN_3492', 'TRAIN_3493', 'TRAIN_3494', 'TRAIN_3495', 'TRAIN_3496', 'TRAIN_3497', 'TRAIN_3498', 'TRAIN_3499', 'TRAIN_3500', 'TRAIN_3501', 'TRAIN_3502', 'TRAIN_3503', 'TRAIN_3504', 'TRAIN_3505', 'TRAIN_3506', 'TRAIN_3507', 'TRAIN_3508', 'TRAIN_3509', 'TRAIN_3510', 'TRAIN_3511', 'TRAIN_3512', 'TRAIN_3513', 'TRAIN_3514', 'TRAIN_3515', 'TRAIN_3516', 'TRAIN_3517', 'TRAIN_3518', 'TRAIN_3519', 'TRAIN_3520', 'TRAIN_3521', 'TRAIN_3522', 'TRAIN_3523', 'TRAIN_3524', 'TRAIN_3525', 'TRAIN_3526', 'TRAIN_3527', 'TRAIN_3528', 'TRAIN_3529', 'TRAIN_3530', 'TRAIN_3531', 'TRAIN_3532', 'TRAIN_3533', 'TRAIN_3534', 'TRAIN_3535', 'TRAIN_3536', 'TRAIN_3537', 'TRAIN_3538', 'TRAIN_3539', 'TRAIN_3540', 'TRAIN_3541', 'TRAIN_3542', 'TRAIN_3543', 'TRAIN_3544', 'TRAIN_3545', 'TRAIN_3546', 'TRAIN_3547', 'TRAIN_3548', 'TRAIN_3549', 'TRAIN_3550', 'TRAIN_3551', 'TRAIN_3552', 'TRAIN_3553', 'TRAIN_3554', 'TRAIN_3555', 'TRAIN_3556', 'TRAIN_3557', 'TRAIN_3558', 'TRAIN_3559', 'TRAIN_3560', 'TRAIN_3561', 'TRAIN_3562', 'TRAIN_3563', 'TRAIN_3564', 'TRAIN_3565', 'TRAIN_3566', 'TRAIN_3567', 'TRAIN_3568', 'TRAIN_3569', 'TRAIN_3570', 'TRAIN_3571', 'TRAIN_3572', 'TRAIN_3573', 'TRAIN_3574', 'TRAIN_3575', 'TRAIN_3576', 'TRAIN_3577', 'TRAIN_3578', 'TRAIN_3579', 'TRAIN_3580', 'TRAIN_3581', 'TRAIN_3582', 'TRAIN_3583', 'TRAIN_3584', 'TRAIN_3585', 'TRAIN_3586', 'TRAIN_3587', 'TRAIN_3588', 'TRAIN_3589', 'TRAIN_3590', 'TRAIN_3591', 'TRAIN_3592', 'TRAIN_3593', 'TRAIN_3594', 'TRAIN_3595', 'TRAIN_3596', 'TRAIN_3597', 'TRAIN_3598', 'TRAIN_3599', 'TRAIN_3600', 'TRAIN_3601', 'TRAIN_3602', 'TRAIN_3603', 'TRAIN_3604', 'TRAIN_3605', 'TRAIN_3606', 'TRAIN_3607', 'TRAIN_3608', 'TRAIN_3609', 'TRAIN_3610', 'TRAIN_3611', 'TRAIN_3612', 'TRAIN_3613', 'TRAIN_3614', 'TRAIN_3615', 'TRAIN_3616', 'TRAIN_3617', 'TRAIN_3618', 'TRAIN_3619', 'TRAIN_3620', 'TRAIN_3621', 'TRAIN_3622', 'TRAIN_3623', 'TRAIN_3624', 'TRAIN_3625', 'TRAIN_3626', 'TRAIN_3627', 'TRAIN_3628', 'TRAIN_3629', 'TRAIN_3630', 'TRAIN_3631', 'TRAIN_3632', 'TRAIN_3633', 'TRAIN_3634', 'TRAIN_3635', 'TRAIN_3636', 'TRAIN_3637', 'TRAIN_3638', 'TRAIN_3639', 'TRAIN_3640', 'TRAIN_3641', 'TRAIN_3642', 'TRAIN_3643', 'TRAIN_3644', 'TRAIN_3645', 'TRAIN_3646', 'TRAIN_3647', 'TRAIN_3648', 'TRAIN_3649', 'TRAIN_3650', 'TRAIN_3651', 'TRAIN_3652', 'TRAIN_3653', 'TRAIN_3654', 'TRAIN_3655', 'TRAIN_3656', 'TRAIN_3657', 'TRAIN_3658', 'TRAIN_3659', 'TRAIN_3660', 'TRAIN_3661', 'TRAIN_3662', 'TRAIN_3663', 'TRAIN_3664', 'TRAIN_3665', 'TRAIN_3666', 'TRAIN_3667', 'TRAIN_3668', 'TRAIN_3669', 'TRAIN_3670', 'TRAIN_3671', 'TRAIN_3672', 'TRAIN_3673', 'TRAIN_3674', 'TRAIN_3675', 'TRAIN_3676', 'TRAIN_3677', 'TRAIN_3678', 'TRAIN_3679', 'TRAIN_3680', 'TRAIN_3681', 'TRAIN_3682', 'TRAIN_3683', 'TRAIN_3684', 'TRAIN_3685', 'TRAIN_3686', 'TRAIN_3687', 'TRAIN_3688', 'TRAIN_3689', 'TRAIN_3690', 'TRAIN_3691', 'TRAIN_3692', 'TRAIN_3693', 'TRAIN_3694', 'TRAIN_3695', 'TRAIN_3696', 'TRAIN_3697', 'TRAIN_3698', 'TRAIN_3699', 'TRAIN_3700', 'TRAIN_3701', 'TRAIN_3702', 'TRAIN_3703', 'TRAIN_3704', 'TRAIN_3705', 'TRAIN_3706', 'TRAIN_3707', 'TRAIN_3708', 'TRAIN_3709', 'TRAIN_3710', 'TRAIN_3711', 'TRAIN_3712', 'TRAIN_3713', 'TRAIN_3714', 'TRAIN_3715', 'TRAIN_3716', 'TRAIN_3717', 'TRAIN_3718', 'TRAIN_3719', 'TRAIN_3720', 'TRAIN_3721', 'TRAIN_3722', 'TRAIN_3723', 'TRAIN_3724', 'TRAIN_3725', 'TRAIN_3726', 'TRAIN_3727', 'TRAIN_3728', 'TRAIN_3729', 'TRAIN_3730', 'TRAIN_3731', 'TRAIN_3732', 'TRAIN_3733', 'TRAIN_3734', 'TRAIN_3735', 'TRAIN_3736', 'TRAIN_3737', 'TRAIN_3738', 'TRAIN_3739', 'TRAIN_3740', 'TRAIN_3741', 'TRAIN_3742', 'TRAIN_3743', 'TRAIN_3744', 'TRAIN_3745', 'TRAIN_3746', 'TRAIN_3747', 'TRAIN_3748', 'TRAIN_3749', 'TRAIN_3750', 'TRAIN_3751', 'TRAIN_3752', 'TRAIN_3753', 'TRAIN_3754', 'TRAIN_3755', 'TRAIN_3756', 'TRAIN_3757', 'TRAIN_3758', 'TRAIN_3759', 'TRAIN_3760', 'TRAIN_3761', 'TRAIN_3762', 'TRAIN_3763', 'TRAIN_3764', 'TRAIN_3765', 'TRAIN_3766', 'TRAIN_3767', 'TRAIN_3768', 'TRAIN_3769', 'TRAIN_3770', 'TRAIN_3771', 'TRAIN_3772', 'TRAIN_3773', 'TRAIN_3774', 'TRAIN_3775', 'TRAIN_3776', 'TRAIN_3777', 'TRAIN_3778', 'TRAIN_3779', 'TRAIN_3780', 'TRAIN_3781', 'TRAIN_3782', 'TRAIN_3783', 'TRAIN_3784', 'TRAIN_3785', 'TRAIN_3786', 'TRAIN_3787', 'TRAIN_3788', 'TRAIN_3789', 'TRAIN_3790', 'TRAIN_3791', 'TRAIN_3792', 'TRAIN_3793', 'TRAIN_3794', 'TRAIN_3795', 'TRAIN_3796', 'TRAIN_3797', 'TRAIN_3798', 'TRAIN_3799', 'TRAIN_3800', 'TRAIN_3801', 'TRAIN_3802', 'TRAIN_3803', 'TRAIN_3804', 'TRAIN_3805', 'TRAIN_3806', 'TRAIN_3807', 'TRAIN_3808', 'TRAIN_3809', 'TRAIN_3810', 'TRAIN_3811', 'TRAIN_3812', 'TRAIN_3813', 'TRAIN_3814', 'TRAIN_3815', 'TRAIN_3816', 'TRAIN_3817', 'TRAIN_3818', 'TRAIN_3819', 'TRAIN_3820', 'TRAIN_3821', 'TRAIN_3822', 'TRAIN_3823', 'TRAIN_3824', 'TRAIN_3825', 'TRAIN_3826', 'TRAIN_3827', 'TRAIN_3828', 'TRAIN_3829', 'TRAIN_3830', 'TRAIN_3831', 'TRAIN_3832', 'TRAIN_3833', 'TRAIN_3834', 'TRAIN_3835', 'TRAIN_3836', 'TRAIN_3837', 'TRAIN_3838', 'TRAIN_3839', 'TRAIN_3840', 'TRAIN_3841', 'TRAIN_3842', 'TRAIN_3843', 'TRAIN_3844', 'TRAIN_3845', 'TRAIN_3846', 'TRAIN_3847', 'TRAIN_3848', 'TRAIN_3849', 'TRAIN_3850', 'TRAIN_3851', 'TRAIN_3852', 'TRAIN_3853', 'TRAIN_3854', 'TRAIN_3855', 'TRAIN_3856', 'TRAIN_3857', 'TRAIN_3858', 'TRAIN_3859', 'TRAIN_3860', 'TRAIN_3861', 'TRAIN_3862', 'TRAIN_3863', 'TRAIN_3864', 'TRAIN_3865', 'TRAIN_3866', 'TRAIN_3867', 'TRAIN_3868', 'TRAIN_3869', 'TRAIN_3870', 'TRAIN_3871', 'TRAIN_3872', 'TRAIN_3873', 'TRAIN_3874', 'TRAIN_3875', 'TRAIN_3876', 'TRAIN_3877', 'TRAIN_3878', 'TRAIN_3879', 'TRAIN_3880', 'TRAIN_3881', 'TRAIN_3882', 'TRAIN_3883', 'TRAIN_3884', 'TRAIN_3885', 'TRAIN_3886', 'TRAIN_3887', 'TRAIN_3888', 'TRAIN_3889', 'TRAIN_3890', 'TRAIN_3891', 'TRAIN_3892', 'TRAIN_3893', 'TRAIN_3894', 'TRAIN_3895', 'TRAIN_3896', 'TRAIN_3897', 'TRAIN_3898', 'TRAIN_3899', 'TRAIN_3900', 'TRAIN_3901', 'TRAIN_3902', 'TRAIN_3903', 'TRAIN_3904', 'TRAIN_3905', 'TRAIN_3906', 'TRAIN_3907', 'TRAIN_3908', 'TRAIN_3909', 'TRAIN_3910', 'TRAIN_3911', 'TRAIN_3912', 'TRAIN_3913', 'TRAIN_3914', 'TRAIN_3915', 'TRAIN_3916', 'TRAIN_3917', 'TRAIN_3918', 'TRAIN_3919', 'TRAIN_3920', 'TRAIN_3921', 'TRAIN_3922', 'TRAIN_3923', 'TRAIN_3924', 'TRAIN_3925', 'TRAIN_3926', 'TRAIN_3927', 'TRAIN_3928', 'TRAIN_3929', 'TRAIN_3930', 'TRAIN_3931', 'TRAIN_3932', 'TRAIN_3933', 'TRAIN_3934', 'TRAIN_3935', 'TRAIN_3936', 'TRAIN_3937', 'TRAIN_3938', 'TRAIN_3939', 'TRAIN_3940', 'TRAIN_3941', 'TRAIN_3942', 'TRAIN_3943', 'TRAIN_3944', 'TRAIN_3945', 'TRAIN_3946', 'TRAIN_3947', 'TRAIN_3948', 'TRAIN_3949', 'TRAIN_3950', 'TRAIN_3951', 'TRAIN_3952', 'TRAIN_3953', 'TRAIN_3954', 'TRAIN_3955', 'TRAIN_3956', 'TRAIN_3957', 'TRAIN_3958', 'TRAIN_3959', 'TRAIN_3960', 'TRAIN_3961', 'TRAIN_3962', 'TRAIN_3963', 'TRAIN_3964', 'TRAIN_3965', 'TRAIN_3966', 'TRAIN_3967', 'TRAIN_3968', 'TRAIN_3969', 'TRAIN_3970', 'TRAIN_3971', 'TRAIN_3972', 'TRAIN_3973', 'TRAIN_3974', 'TRAIN_3975', 'TRAIN_3976', 'TRAIN_3977', 'TRAIN_3978', 'TRAIN_3979', 'TRAIN_3980', 'TRAIN_3981', 'TRAIN_3982', 'TRAIN_3983', 'TRAIN_3984', 'TRAIN_3985', 'TRAIN_3986', 'TRAIN_3987', 'TRAIN_3988', 'TRAIN_3989', 'TRAIN_3990', 'TRAIN_3991', 'TRAIN_3992', 'TRAIN_3993', 'TRAIN_3994', 'TRAIN_3995', 'TRAIN_3996', 'TRAIN_3997', 'TRAIN_3998', 'TRAIN_3999', 'TRAIN_4000', 'TRAIN_4001', 'TRAIN_4002', 'TRAIN_4003', 'TRAIN_4004', 'TRAIN_4005', 'TRAIN_4006', 'TRAIN_4007', 'TRAIN_4008', 'TRAIN_4009', 'TRAIN_4010', 'TRAIN_4011', 'TRAIN_4012', 'TRAIN_4013', 'TRAIN_4014', 'TRAIN_4015', 'TRAIN_4016', 'TRAIN_4017', 'TRAIN_4018', 'TRAIN_4019', 'TRAIN_4020', 'TRAIN_4021', 'TRAIN_4022', 'TRAIN_4023', 'TRAIN_4024', 'TRAIN_4025', 'TRAIN_4026', 'TRAIN_4027', 'TRAIN_4028', 'TRAIN_4029', 'TRAIN_4030', 'TRAIN_4031', 'TRAIN_4032', 'TRAIN_4033', 'TRAIN_4034', 'TRAIN_4035', 'TRAIN_4036', 'TRAIN_4037', 'TRAIN_4038', 'TRAIN_4039', 'TRAIN_4040', 'TRAIN_4041', 'TRAIN_4042', 'TRAIN_4043', 'TRAIN_4044', 'TRAIN_4045', 'TRAIN_4046', 'TRAIN_4047', 'TRAIN_4048', 'TRAIN_4049', 'TRAIN_4050', 'TRAIN_4051', 'TRAIN_4052', 'TRAIN_4053', 'TRAIN_4054', 'TRAIN_4055', 'TRAIN_4056', 'TRAIN_4057', 'TRAIN_4058', 'TRAIN_4059', 'TRAIN_4060', 'TRAIN_4061', 'TRAIN_4062', 'TRAIN_4063', 'TRAIN_4064', 'TRAIN_4065', 'TRAIN_4066', 'TRAIN_4067', 'TRAIN_4068', 'TRAIN_4069', 'TRAIN_4070', 'TRAIN_4071', 'TRAIN_4072', 'TRAIN_4073', 'TRAIN_4074', 'TRAIN_4075', 'TRAIN_4076', 'TRAIN_4077', 'TRAIN_4078', 'TRAIN_4079', 'TRAIN_4080', 'TRAIN_4081', 'TRAIN_4082', 'TRAIN_4083', 'TRAIN_4084', 'TRAIN_4085', 'TRAIN_4086', 'TRAIN_4087', 'TRAIN_4088', 'TRAIN_4089', 'TRAIN_4090', 'TRAIN_4091', 'TRAIN_4092', 'TRAIN_4093', 'TRAIN_4094', 'TRAIN_4095', 'TRAIN_4096', 'TRAIN_4097', 'TRAIN_4098', 'TRAIN_4099', 'TRAIN_4100', 'TRAIN_4101', 'TRAIN_4102', 'TRAIN_4103', 'TRAIN_4104', 'TRAIN_4105', 'TRAIN_4106', 'TRAIN_4107', 'TRAIN_4108', 'TRAIN_4109', 'TRAIN_4110', 'TRAIN_4111', 'TRAIN_4112', 'TRAIN_4113', 'TRAIN_4114', 'TRAIN_4115', 'TRAIN_4116', 'TRAIN_4117', 'TRAIN_4118', 'TRAIN_4119', 'TRAIN_4120', 'TRAIN_4121', 'TRAIN_4122', 'TRAIN_4123', 'TRAIN_4124', 'TRAIN_4125', 'TRAIN_4126', 'TRAIN_4127', 'TRAIN_4128', 'TRAIN_4129', 'TRAIN_4130', 'TRAIN_4131', 'TRAIN_4132', 'TRAIN_4133', 'TRAIN_4134', 'TRAIN_4135', 'TRAIN_4136', 'TRAIN_4137', 'TRAIN_4138', 'TRAIN_4139', 'TRAIN_4140', 'TRAIN_4141', 'TRAIN_4142', 'TRAIN_4143', 'TRAIN_4144', 'TRAIN_4145', 'TRAIN_4146', 'TRAIN_4147', 'TRAIN_4148', 'TRAIN_4149', 'TRAIN_4150', 'TRAIN_4151', 'TRAIN_4152', 'TRAIN_4153', 'TRAIN_4154', 'TRAIN_4155', 'TRAIN_4156', 'TRAIN_4157', 'TRAIN_4158', 'TRAIN_4159', 'TRAIN_4160', 'TRAIN_4161', 'TRAIN_4162', 'TRAIN_4163', 'TRAIN_4164', 'TRAIN_4165', 'TRAIN_4166', 'TRAIN_4167', 'TRAIN_4168', 'TRAIN_4169', 'TRAIN_4170', 'TRAIN_4171', 'TRAIN_4172', 'TRAIN_4173', 'TRAIN_4174', 'TRAIN_4175', 'TRAIN_4176', 'TRAIN_4177', 'TRAIN_4178', 'TRAIN_4179', 'TRAIN_4180', 'TRAIN_4181', 'TRAIN_4182', 'TRAIN_4183', 'TRAIN_4184', 'TRAIN_4185', 'TRAIN_4186', 'TRAIN_4187', 'TRAIN_4188', 'TRAIN_4189', 'TRAIN_4190', 'TRAIN_4191', 'TRAIN_4192', 'TRAIN_4193', 'TRAIN_4194', 'TRAIN_4195', 'TRAIN_4196', 'TRAIN_4197', 'TRAIN_4198', 'TRAIN_4199', 'TRAIN_4200', 'TRAIN_4201', 'TRAIN_4202', 'TRAIN_4203', 'TRAIN_4204', 'TRAIN_4205', 'TRAIN_4206', 'TRAIN_4207', 'TRAIN_4208', 'TRAIN_4209', 'TRAIN_4210', 'TRAIN_4211', 'TRAIN_4212', 'TRAIN_4213', 'TRAIN_4214', 'TRAIN_4215', 'TRAIN_4216', 'TRAIN_4217', 'TRAIN_4218', 'TRAIN_4219', 'TRAIN_4220', 'TRAIN_4221', 'TRAIN_4222', 'TRAIN_4223', 'TRAIN_4224', 'TRAIN_4225', 'TRAIN_4226', 'TRAIN_4227', 'TRAIN_4228', 'TRAIN_4229', 'TRAIN_4230', 'TRAIN_4231', 'TRAIN_4232', 'TRAIN_4233', 'TRAIN_4234', 'TRAIN_4235', 'TRAIN_4236', 'TRAIN_4237', 'TRAIN_4238', 'TRAIN_4239', 'TRAIN_4240', 'TRAIN_4241', 'TRAIN_4242', 'TRAIN_4243', 'TRAIN_4244', 'TRAIN_4245', 'TRAIN_4246', 'TRAIN_4247', 'TRAIN_4248', 'TRAIN_4249', 'TRAIN_4250', 'TRAIN_4251', 'TRAIN_4252', 'TRAIN_4253', 'TRAIN_4254', 'TRAIN_4255', 'TRAIN_4256', 'TRAIN_4257', 'TRAIN_4258', 'TRAIN_4259', 'TRAIN_4260', 'TRAIN_4261', 'TRAIN_4262', 'TRAIN_4263', 'TRAIN_4264', 'TRAIN_4265', 'TRAIN_4266', 'TRAIN_4267', 'TRAIN_4268', 'TRAIN_4269', 'TRAIN_4270', 'TRAIN_4271', 'TRAIN_4272', 'TRAIN_4273', 'TRAIN_4274', 'TRAIN_4275', 'TRAIN_4276', 'TRAIN_4277', 'TRAIN_4278', 'TRAIN_4279', 'TRAIN_4280', 'TRAIN_4281', 'TRAIN_4282', 'TRAIN_4283', 'TRAIN_4284', 'TRAIN_4285', 'TRAIN_4286', 'TRAIN_4287', 'TRAIN_4288', 'TRAIN_4289', 'TRAIN_4290', 'TRAIN_4291', 'TRAIN_4292', 'TRAIN_4293', 'TRAIN_4294', 'TRAIN_4295', 'TRAIN_4296', 'TRAIN_4297', 'TRAIN_4298', 'TRAIN_4299', 'TRAIN_4300', 'TRAIN_4301', 'TRAIN_4302', 'TRAIN_4303', 'TRAIN_4304', 'TRAIN_4305', 'TRAIN_4306', 'TRAIN_4307', 'TRAIN_4308', 'TRAIN_4309', 'TRAIN_4310', 'TRAIN_4311', 'TRAIN_4312', 'TRAIN_4313', 'TRAIN_4314', 'TRAIN_4315', 'TRAIN_4316', 'TRAIN_4317', 'TRAIN_4318', 'TRAIN_4319', 'TRAIN_4320', 'TRAIN_4321', 'TRAIN_4322', 'TRAIN_4323', 'TRAIN_4324', 'TRAIN_4325', 'TRAIN_4326', 'TRAIN_4327', 'TRAIN_4328', 'TRAIN_4329', 'TRAIN_4330', 'TRAIN_4331', 'TRAIN_4332', 'TRAIN_4333', 'TRAIN_4334', 'TRAIN_4335', 'TRAIN_4336', 'TRAIN_4337', 'TRAIN_4338', 'TRAIN_4339', 'TRAIN_4340', 'TRAIN_4341', 'TRAIN_4342', 'TRAIN_4343', 'TRAIN_4344', 'TRAIN_4345', 'TRAIN_4346', 'TRAIN_4347', 'TRAIN_4348', 'TRAIN_4349', 'TRAIN_4350', 'TRAIN_4351', 'TRAIN_4352', 'TRAIN_4353', 'TRAIN_4354', 'TRAIN_4355', 'TRAIN_4356', 'TRAIN_4357', 'TRAIN_4358', 'TRAIN_4359', 'TRAIN_4360', 'TRAIN_4361', 'TRAIN_4362', 'TRAIN_4363', 'TRAIN_4364', 'TRAIN_4365', 'TRAIN_4366', 'TRAIN_4367', 'TRAIN_4368', 'TRAIN_4369', 'TRAIN_4370', 'TRAIN_4371', 'TRAIN_4372', 'TRAIN_4373', 'TRAIN_4374', 'TRAIN_4375', 'TRAIN_4376', 'TRAIN_4377', 'TRAIN_4378', 'TRAIN_4379', 'TRAIN_4380', 'TRAIN_4381', 'TRAIN_4382', 'TRAIN_4383', 'TRAIN_4384', 'TRAIN_4385', 'TRAIN_4386', 'TRAIN_4387', 'TRAIN_4388', 'TRAIN_4389', 'TRAIN_4390', 'TRAIN_4391', 'TRAIN_4392', 'TRAIN_4393', 'TRAIN_4394', 'TRAIN_4395', 'TRAIN_4396', 'TRAIN_4397', 'TRAIN_4398', 'TRAIN_4399', 'TRAIN_4400', 'TRAIN_4401', 'TRAIN_4402', 'TRAIN_4403', 'TRAIN_4404', 'TRAIN_4405', 'TRAIN_4406', 'TRAIN_4407', 'TRAIN_4408', 'TRAIN_4409', 'TRAIN_4410', 'TRAIN_4411', 'TRAIN_4412', 'TRAIN_4413', 'TRAIN_4414', 'TRAIN_4415', 'TRAIN_4416', 'TRAIN_4417', 'TRAIN_4418', 'TRAIN_4419', 'TRAIN_4420', 'TRAIN_4421', 'TRAIN_4422', 'TRAIN_4423', 'TRAIN_4424', 'TRAIN_4425', 'TRAIN_4426', 'TRAIN_4427', 'TRAIN_4428', 'TRAIN_4429', 'TRAIN_4430', 'TRAIN_4431', 'TRAIN_4432', 'TRAIN_4433', 'TRAIN_4434', 'TRAIN_4435', 'TRAIN_4436', 'TRAIN_4437', 'TRAIN_4438', 'TRAIN_4439', 'TRAIN_4440', 'TRAIN_4441', 'TRAIN_4442', 'TRAIN_4443', 'TRAIN_4444', 'TRAIN_4445', 'TRAIN_4446', 'TRAIN_4447', 'TRAIN_4448', 'TRAIN_4449', 'TRAIN_4450', 'TRAIN_4451', 'TRAIN_4452', 'TRAIN_4453', 'TRAIN_4454', 'TRAIN_4455', 'TRAIN_4456', 'TRAIN_4457', 'TRAIN_4458', 'TRAIN_4459', 'TRAIN_4460', 'TRAIN_4461', 'TRAIN_4462', 'TRAIN_4463', 'TRAIN_4464', 'TRAIN_4465', 'TRAIN_4466', 'TRAIN_4467', 'TRAIN_4468', 'TRAIN_4469', 'TRAIN_4470', 'TRAIN_4471', 'TRAIN_4472', 'TRAIN_4473', 'TRAIN_4474', 'TRAIN_4475', 'TRAIN_4476', 'TRAIN_4477', 'TRAIN_4478', 'TRAIN_4479', 'TRAIN_4480', 'TRAIN_4481', 'TRAIN_4482', 'TRAIN_4483', 'TRAIN_4484', 'TRAIN_4485', 'TRAIN_4486', 'TRAIN_4487', 'TRAIN_4488', 'TRAIN_4489', 'TRAIN_4490', 'TRAIN_4491', 'TRAIN_4492', 'TRAIN_4493', 'TRAIN_4494', 'TRAIN_4495', 'TRAIN_4496', 'TRAIN_4497', 'TRAIN_4498', 'TRAIN_4499', 'TRAIN_4500', 'TRAIN_4501', 'TRAIN_4502', 'TRAIN_4503', 'TRAIN_4504', 'TRAIN_4505', 'TRAIN_4506', 'TRAIN_4507', 'TRAIN_4508', 'TRAIN_4509', 'TRAIN_4510', 'TRAIN_4511', 'TRAIN_4512', 'TRAIN_4513', 'TRAIN_4514', 'TRAIN_4515', 'TRAIN_4516', 'TRAIN_4517', 'TRAIN_4518', 'TRAIN_4519', 'TRAIN_4520', 'TRAIN_4521', 'TRAIN_4522', 'TRAIN_4523', 'TRAIN_4524', 'TRAIN_4525', 'TRAIN_4526', 'TRAIN_4527', 'TRAIN_4528', 'TRAIN_4529', 'TRAIN_4530', 'TRAIN_4531', 'TRAIN_4532', 'TRAIN_4533', 'TRAIN_4534', 'TRAIN_4535', 'TRAIN_4536', 'TRAIN_4537', 'TRAIN_4538', 'TRAIN_4539', 'TRAIN_4540', 'TRAIN_4541', 'TRAIN_4542', 'TRAIN_4543', 'TRAIN_4544', 'TRAIN_4545', 'TRAIN_4546', 'TRAIN_4547', 'TRAIN_4548', 'TRAIN_4549', 'TRAIN_4550', 'TRAIN_4551', 'TRAIN_4552', 'TRAIN_4553', 'TRAIN_4554', 'TRAIN_4555', 'TRAIN_4556', 'TRAIN_4557', 'TRAIN_4558', 'TRAIN_4559', 'TRAIN_4560', 'TRAIN_4561', 'TRAIN_4562', 'TRAIN_4563', 'TRAIN_4564', 'TRAIN_4565', 'TRAIN_4566', 'TRAIN_4567', 'TRAIN_4568', 'TRAIN_4569', 'TRAIN_4570', 'TRAIN_4571', 'TRAIN_4572', 'TRAIN_4573', 'TRAIN_4574', 'TRAIN_4575', 'TRAIN_4576', 'TRAIN_4577', 'TRAIN_4578', 'TRAIN_4579', 'TRAIN_4580', 'TRAIN_4581', 'TRAIN_4582', 'TRAIN_4583', 'TRAIN_4584', 'TRAIN_4585', 'TRAIN_4586', 'TRAIN_4587', 'TRAIN_4588', 'TRAIN_4589', 'TRAIN_4590', 'TRAIN_4591', 'TRAIN_4592', 'TRAIN_4593', 'TRAIN_4594', 'TRAIN_4595', 'TRAIN_4596', 'TRAIN_4597', 'TRAIN_4598', 'TRAIN_4599', 'TRAIN_4600', 'TRAIN_4601', 'TRAIN_4602', 'TRAIN_4603', 'TRAIN_4604', 'TRAIN_4605', 'TRAIN_4606', 'TRAIN_4607', 'TRAIN_4608', 'TRAIN_4609', 'TRAIN_4610', 'TRAIN_4611', 'TRAIN_4612', 'TRAIN_4613', 'TRAIN_4614', 'TRAIN_4615', 'TRAIN_4616', 'TRAIN_4617', 'TRAIN_4618', 'TRAIN_4619', 'TRAIN_4620', 'TRAIN_4621', 'TRAIN_4622', 'TRAIN_4623', 'TRAIN_4624', 'TRAIN_4625', 'TRAIN_4626', 'TRAIN_4627', 'TRAIN_4628', 'TRAIN_4629', 'TRAIN_4630', 'TRAIN_4631', 'TRAIN_4632', 'TRAIN_4633', 'TRAIN_4634', 'TRAIN_4635', 'TRAIN_4636', 'TRAIN_4637', 'TRAIN_4638', 'TRAIN_4639', 'TRAIN_4640', 'TRAIN_4641', 'TRAIN_4642', 'TRAIN_4643', 'TRAIN_4644', 'TRAIN_4645', 'TRAIN_4646', 'TRAIN_4647', 'TRAIN_4648', 'TRAIN_4649', 'TRAIN_4650', 'TRAIN_4651', 'TRAIN_4652', 'TRAIN_4653', 'TRAIN_4654', 'TRAIN_4655', 'TRAIN_4656', 'TRAIN_4657', 'TRAIN_4658', 'TRAIN_4659', 'TRAIN_4660', 'TRAIN_4661', 'TRAIN_4662', 'TRAIN_4663', 'TRAIN_4664', 'TRAIN_4665', 'TRAIN_4666', 'TRAIN_4667', 'TRAIN_4668', 'TRAIN_4669', 'TRAIN_4670', 'TRAIN_4671', 'TRAIN_4672', 'TRAIN_4673', 'TRAIN_4674', 'TRAIN_4675', 'TRAIN_4676', 'TRAIN_4677', 'TRAIN_4678', 'TRAIN_4679', 'TRAIN_4680', 'TRAIN_4681', 'TRAIN_4682', 'TRAIN_4683', 'TRAIN_4684', 'TRAIN_4685', 'TRAIN_4686', 'TRAIN_4687', 'TRAIN_4688', 'TRAIN_4689', 'TRAIN_4690', 'TRAIN_4691', 'TRAIN_4692', 'TRAIN_4693', 'TRAIN_4694', 'TRAIN_4695', 'TRAIN_4696', 'TRAIN_4697', 'TRAIN_4698', 'TRAIN_4699', 'TRAIN_4700', 'TRAIN_4701', 'TRAIN_4702', 'TRAIN_4703', 'TRAIN_4704', 'TRAIN_4705', 'TRAIN_4706', 'TRAIN_4707', 'TRAIN_4708', 'TRAIN_4709', 'TRAIN_4710', 'TRAIN_4711', 'TRAIN_4712', 'TRAIN_4713', 'TRAIN_4714', 'TRAIN_4715', 'TRAIN_4716', 'TRAIN_4717', 'TRAIN_4718', 'TRAIN_4719', 'TRAIN_4720', 'TRAIN_4721', 'TRAIN_4722', 'TRAIN_4723', 'TRAIN_4724', 'TRAIN_4725', 'TRAIN_4726', 'TRAIN_4727', 'TRAIN_4728', 'TRAIN_4729', 'TRAIN_4730', 'TRAIN_4731', 'TRAIN_4732', 'TRAIN_4733', 'TRAIN_4734', 'TRAIN_4735', 'TRAIN_4736', 'TRAIN_4737', 'TRAIN_4738', 'TRAIN_4739', 'TRAIN_4740', 'TRAIN_4741', 'TRAIN_4742', 'TRAIN_4743', 'TRAIN_4744', 'TRAIN_4745', 'TRAIN_4746', 'TRAIN_4747', 'TRAIN_4748', 'TRAIN_4749', 'TRAIN_4750', 'TRAIN_4751', 'TRAIN_4752', 'TRAIN_4753', 'TRAIN_4754', 'TRAIN_4755', 'TRAIN_4756', 'TRAIN_4757', 'TRAIN_4758', 'TRAIN_4759', 'TRAIN_4760', 'TRAIN_4761', 'TRAIN_4762', 'TRAIN_4763', 'TRAIN_4764', 'TRAIN_4765', 'TRAIN_4766', 'TRAIN_4767', 'TRAIN_4768', 'TRAIN_4769', 'TRAIN_4770', 'TRAIN_4771', 'TRAIN_4772', 'TRAIN_4773', 'TRAIN_4774', 'TRAIN_4775', 'TRAIN_4776', 'TRAIN_4777', 'TRAIN_4778', 'TRAIN_4779', 'TRAIN_4780', 'TRAIN_4781', 'TRAIN_4782', 'TRAIN_4783', 'TRAIN_4784', 'TRAIN_4785', 'TRAIN_4786', 'TRAIN_4787', 'TRAIN_4788', 'TRAIN_4789', 'TRAIN_4790', 'TRAIN_4791', 'TRAIN_4792', 'TRAIN_4793', 'TRAIN_4794', 'TRAIN_4795', 'TRAIN_4796', 'TRAIN_4797', 'TRAIN_4798', 'TRAIN_4799', 'TRAIN_4800', 'TRAIN_4801', 'TRAIN_4802', 'TRAIN_4803', 'TRAIN_4804', 'TRAIN_4805', 'TRAIN_4806', 'TRAIN_4807', 'TRAIN_4808', 'TRAIN_4809', 'TRAIN_4810', 'TRAIN_4811', 'TRAIN_4812', 'TRAIN_4813', 'TRAIN_4814', 'TRAIN_4815', 'TRAIN_4816', 'TRAIN_4817', 'TRAIN_4818', 'TRAIN_4819', 'TRAIN_4820', 'TRAIN_4821', 'TRAIN_4822', 'TRAIN_4823', 'TRAIN_4824', 'TRAIN_4825', 'TRAIN_4826', 'TRAIN_4827', 'TRAIN_4828', 'TRAIN_4829', 'TRAIN_4830', 'TRAIN_4831', 'TRAIN_4832', 'TRAIN_4833', 'TRAIN_4834', 'TRAIN_4835', 'TRAIN_4836', 'TRAIN_4837', 'TRAIN_4838', 'TRAIN_4839', 'TRAIN_4840', 'TRAIN_4841', 'TRAIN_4842', 'TRAIN_4843', 'TRAIN_4844', 'TRAIN_4845', 'TRAIN_4846', 'TRAIN_4847', 'TRAIN_4848', 'TRAIN_4849', 'TRAIN_4850', 'TRAIN_4851', 'TRAIN_4852', 'TRAIN_4853', 'TRAIN_4854', 'TRAIN_4855', 'TRAIN_4856', 'TRAIN_4857', 'TRAIN_4858', 'TRAIN_4859', 'TRAIN_4860', 'TRAIN_4861', 'TRAIN_4862', 'TRAIN_4863', 'TRAIN_4864', 'TRAIN_4865', 'TRAIN_4866', 'TRAIN_4867', 'TRAIN_4868', 'TRAIN_4869', 'TRAIN_4870', 'TRAIN_4871', 'TRAIN_4872', 'TRAIN_4873', 'TRAIN_4874', 'TRAIN_4875', 'TRAIN_4876', 'TRAIN_4877', 'TRAIN_4878', 'TRAIN_4879', 'TRAIN_4880', 'TRAIN_4881', 'TRAIN_4882', 'TRAIN_4883', 'TRAIN_4884', 'TRAIN_4885', 'TRAIN_4886', 'TRAIN_4887', 'TRAIN_4888', 'TRAIN_4889', 'TRAIN_4890', 'TRAIN_4891', 'TRAIN_4892', 'TRAIN_4893', 'TRAIN_4894', 'TRAIN_4895', 'TRAIN_4896', 'TRAIN_4897', 'TRAIN_4898', 'TRAIN_4899', 'TRAIN_4900', 'TRAIN_4901', 'TRAIN_4902', 'TRAIN_4903', 'TRAIN_4904', 'TRAIN_4905', 'TRAIN_4906', 'TRAIN_4907', 'TRAIN_4908', 'TRAIN_4909', 'TRAIN_4910', 'TRAIN_4911', 'TRAIN_4912', 'TRAIN_4913', 'TRAIN_4914', 'TRAIN_4915', 'TRAIN_4916', 'TRAIN_4917', 'TRAIN_4918', 'TRAIN_4919', 'TRAIN_4920', 'TRAIN_4921', 'TRAIN_4922', 'TRAIN_4923', 'TRAIN_4924', 'TRAIN_4925', 'TRAIN_4926', 'TRAIN_4927', 'TRAIN_4928', 'TRAIN_4929', 'TRAIN_4930', 'TRAIN_4931', 'TRAIN_4932', 'TRAIN_4933', 'TRAIN_4934', 'TRAIN_4935', 'TRAIN_4936', 'TRAIN_4937', 'TRAIN_4938', 'TRAIN_4939', 'TRAIN_4940', 'TRAIN_4941', 'TRAIN_4942', 'TRAIN_4943', 'TRAIN_4944', 'TRAIN_4945', 'TRAIN_4946', 'TRAIN_4947', 'TRAIN_4948', 'TRAIN_4949', 'TRAIN_4950', 'TRAIN_4951', 'TRAIN_4952', 'TRAIN_4953', 'TRAIN_4954', 'TRAIN_4955', 'TRAIN_4956', 'TRAIN_4957', 'TRAIN_4958', 'TRAIN_4959', 'TRAIN_4960', 'TRAIN_4961', 'TRAIN_4962', 'TRAIN_4963', 'TRAIN_4964', 'TRAIN_4965', 'TRAIN_4966', 'TRAIN_4967', 'TRAIN_4968', 'TRAIN_4969', 'TRAIN_4970', 'TRAIN_4971', 'TRAIN_4972', 'TRAIN_4973', 'TRAIN_4974', 'TRAIN_4975', 'TRAIN_4976', 'TRAIN_4977', 'TRAIN_4978', 'TRAIN_4979', 'TRAIN_4980', 'TRAIN_4981', 'TRAIN_4982', 'TRAIN_4983', 'TRAIN_4984', 'TRAIN_4985', 'TRAIN_4986', 'TRAIN_4987', 'TRAIN_4988', 'TRAIN_4989', 'TRAIN_4990', 'TRAIN_4991', 'TRAIN_4992', 'TRAIN_4993', 'TRAIN_4994', 'TRAIN_4995', 'TRAIN_4996', 'TRAIN_4997', 'TRAIN_4998', 'TRAIN_4999', 'TRAIN_5000', 'TRAIN_5001', 'TRAIN_5002', 'TRAIN_5003', 'TRAIN_5004', 'TRAIN_5005', 'TRAIN_5006', 'TRAIN_5007', 'TRAIN_5008', 'TRAIN_5009', 'TRAIN_5010', 'TRAIN_5011', 'TRAIN_5012', 'TRAIN_5013', 'TRAIN_5014', 'TRAIN_5015', 'TRAIN_5016', 'TRAIN_5017', 'TRAIN_5018', 'TRAIN_5019', 'TRAIN_5020', 'TRAIN_5021', 'TRAIN_5022', 'TRAIN_5023', 'TRAIN_5024', 'TRAIN_5025', 'TRAIN_5026', 'TRAIN_5027', 'TRAIN_5028', 'TRAIN_5029', 'TRAIN_5030', 'TRAIN_5031', 'TRAIN_5032', 'TRAIN_5033', 'TRAIN_5034', 'TRAIN_5035', 'TRAIN_5036', 'TRAIN_5037', 'TRAIN_5038', 'TRAIN_5039', 'TRAIN_5040', 'TRAIN_5041', 'TRAIN_5042', 'TRAIN_5043', 'TRAIN_5044', 'TRAIN_5045', 'TRAIN_5046', 'TRAIN_5047', 'TRAIN_5048', 'TRAIN_5049', 'TRAIN_5050', 'TRAIN_5051', 'TRAIN_5052', 'TRAIN_5053', 'TRAIN_5054', 'TRAIN_5055', 'TRAIN_5056', 'TRAIN_5057', 'TRAIN_5058', 'TRAIN_5059', 'TRAIN_5060', 'TRAIN_5061', 'TRAIN_5062', 'TRAIN_5063', 'TRAIN_5064', 'TRAIN_5065', 'TRAIN_5066', 'TRAIN_5067', 'TRAIN_5068', 'TRAIN_5069', 'TRAIN_5070', 'TRAIN_5071', 'TRAIN_5072', 'TRAIN_5073', 'TRAIN_5074', 'TRAIN_5075', 'TRAIN_5076', 'TRAIN_5077', 'TRAIN_5078', 'TRAIN_5079', 'TRAIN_5080', 'TRAIN_5081', 'TRAIN_5082', 'TRAIN_5083', 'TRAIN_5084', 'TRAIN_5085', 'TRAIN_5086', 'TRAIN_5087', 'TRAIN_5088', 'TRAIN_5089', 'TRAIN_5090', 'TRAIN_5091', 'TRAIN_5092', 'TRAIN_5093', 'TRAIN_5094', 'TRAIN_5095', 'TRAIN_5096', 'TRAIN_5097', 'TRAIN_5098', 'TRAIN_5099', 'TRAIN_5100', 'TRAIN_5101', 'TRAIN_5102', 'TRAIN_5103', 'TRAIN_5104', 'TRAIN_5105', 'TRAIN_5106', 'TRAIN_5107', 'TRAIN_5108', 'TRAIN_5109', 'TRAIN_5110', 'TRAIN_5111', 'TRAIN_5112', 'TRAIN_5113', 'TRAIN_5114', 'TRAIN_5115', 'TRAIN_5116', 'TRAIN_5117', 'TRAIN_5118', 'TRAIN_5119', 'TRAIN_5120', 'TRAIN_5121', 'TRAIN_5122', 'TRAIN_5123', 'TRAIN_5124', 'TRAIN_5125', 'TRAIN_5126', 'TRAIN_5127', 'TRAIN_5128', 'TRAIN_5129', 'TRAIN_5130', 'TRAIN_5131', 'TRAIN_5132', 'TRAIN_5133', 'TRAIN_5134', 'TRAIN_5135', 'TRAIN_5136', 'TRAIN_5137', 'TRAIN_5138', 'TRAIN_5139', 'TRAIN_5140', 'TRAIN_5141', 'TRAIN_5142', 'TRAIN_5143', 'TRAIN_5144', 'TRAIN_5145', 'TRAIN_5146', 'TRAIN_5147', 'TRAIN_5148', 'TRAIN_5149', 'TRAIN_5150', 'TRAIN_5151', 'TRAIN_5152', 'TRAIN_5153', 'TRAIN_5154', 'TRAIN_5155', 'TRAIN_5156', 'TRAIN_5157', 'TRAIN_5158', 'TRAIN_5159', 'TRAIN_5160', 'TRAIN_5161', 'TRAIN_5162', 'TRAIN_5163', 'TRAIN_5164', 'TRAIN_5165', 'TRAIN_5166', 'TRAIN_5167', 'TRAIN_5168', 'TRAIN_5169', 'TRAIN_5170', 'TRAIN_5171', 'TRAIN_5172', 'TRAIN_5173', 'TRAIN_5174', 'TRAIN_5175', 'TRAIN_5176', 'TRAIN_5177', 'TRAIN_5178', 'TRAIN_5179', 'TRAIN_5180', 'TRAIN_5181', 'TRAIN_5182', 'TRAIN_5183', 'TRAIN_5184', 'TRAIN_5185', 'TRAIN_5186', 'TRAIN_5187', 'TRAIN_5188', 'TRAIN_5189', 'TRAIN_5190', 'TRAIN_5191', 'TRAIN_5192', 'TRAIN_5193', 'TRAIN_5194', 'TRAIN_5195', 'TRAIN_5196', 'TRAIN_5197', 'TRAIN_5198', 'TRAIN_5199', 'TRAIN_5200', 'TRAIN_5201', 'TRAIN_5202', 'TRAIN_5203', 'TRAIN_5204', 'TRAIN_5205', 'TRAIN_5206', 'TRAIN_5207', 'TRAIN_5208', 'TRAIN_5209', 'TRAIN_5210', 'TRAIN_5211', 'TRAIN_5212', 'TRAIN_5213', 'TRAIN_5214', 'TRAIN_5215', 'TRAIN_5216', 'TRAIN_5217', 'TRAIN_5218', 'TRAIN_5219', 'TRAIN_5220', 'TRAIN_5221', 'TRAIN_5222', 'TRAIN_5223', 'TRAIN_5224', 'TRAIN_5225', 'TRAIN_5226', 'TRAIN_5227', 'TRAIN_5228', 'TRAIN_5229', 'TRAIN_5230', 'TRAIN_5231', 'TRAIN_5232', 'TRAIN_5233', 'TRAIN_5234', 'TRAIN_5235', 'TRAIN_5236', 'TRAIN_5237', 'TRAIN_5238', 'TRAIN_5239', 'TRAIN_5240', 'TRAIN_5241', 'TRAIN_5242', 'TRAIN_5243', 'TRAIN_5244', 'TRAIN_5245', 'TRAIN_5246', 'TRAIN_5247', 'TRAIN_5248', 'TRAIN_5249', 'TRAIN_5250', 'TRAIN_5251', 'TRAIN_5252', 'TRAIN_5253', 'TRAIN_5254', 'TRAIN_5255', 'TRAIN_5256', 'TRAIN_5257', 'TRAIN_5258', 'TRAIN_5259', 'TRAIN_5260', 'TRAIN_5261', 'TRAIN_5262', 'TRAIN_5263', 'TRAIN_5264', 'TRAIN_5265', 'TRAIN_5266', 'TRAIN_5267', 'TRAIN_5268', 'TRAIN_5269', 'TRAIN_5270', 'TRAIN_5271', 'TRAIN_5272', 'TRAIN_5273', 'TRAIN_5274', 'TRAIN_5275', 'TRAIN_5276', 'TRAIN_5277', 'TRAIN_5278', 'TRAIN_5279', 'TRAIN_5280', 'TRAIN_5281', 'TRAIN_5282', 'TRAIN_5283', 'TRAIN_5284', 'TRAIN_5285', 'TRAIN_5286', 'TRAIN_5287', 'TRAIN_5288', 'TRAIN_5289', 'TRAIN_5290', 'TRAIN_5291', 'TRAIN_5292', 'TRAIN_5293', 'TRAIN_5294', 'TRAIN_5295', 'TRAIN_5296', 'TRAIN_5297', 'TRAIN_5298', 'TRAIN_5299', 'TRAIN_5300', 'TRAIN_5301', 'TRAIN_5302', 'TRAIN_5303', 'TRAIN_5304', 'TRAIN_5305', 'TRAIN_5306', 'TRAIN_5307', 'TRAIN_5308', 'TRAIN_5309', 'TRAIN_5310', 'TRAIN_5311', 'TRAIN_5312', 'TRAIN_5313', 'TRAIN_5314', 'TRAIN_5315', 'TRAIN_5316', 'TRAIN_5317', 'TRAIN_5318', 'TRAIN_5319', 'TRAIN_5320', 'TRAIN_5321', 'TRAIN_5322', 'TRAIN_5323', 'TRAIN_5324', 'TRAIN_5325', 'TRAIN_5326', 'TRAIN_5327', 'TRAIN_5328', 'TRAIN_5329', 'TRAIN_5330', 'TRAIN_5331', 'TRAIN_5332', 'TRAIN_5333', 'TRAIN_5334', 'TRAIN_5335', 'TRAIN_5336', 'TRAIN_5337', 'TRAIN_5338', 'TRAIN_5339', 'TRAIN_5340', 'TRAIN_5341', 'TRAIN_5342', 'TRAIN_5343', 'TRAIN_5344', 'TRAIN_5345', 'TRAIN_5346', 'TRAIN_5347', 'TRAIN_5348', 'TRAIN_5349', 'TRAIN_5350', 'TRAIN_5351', 'TRAIN_5352', 'TRAIN_5353', 'TRAIN_5354', 'TRAIN_5355', 'TRAIN_5356', 'TRAIN_5357', 'TRAIN_5358', 'TRAIN_5359', 'TRAIN_5360', 'TRAIN_5361', 'TRAIN_5362', 'TRAIN_5363', 'TRAIN_5364', 'TRAIN_5365', 'TRAIN_5366', 'TRAIN_5367', 'TRAIN_5368', 'TRAIN_5369', 'TRAIN_5370', 'TRAIN_5371', 'TRAIN_5372', 'TRAIN_5373', 'TRAIN_5374', 'TRAIN_5375', 'TRAIN_5376', 'TRAIN_5377', 'TRAIN_5378', 'TRAIN_5379', 'TRAIN_5380', 'TRAIN_5381', 'TRAIN_5382', 'TRAIN_5383', 'TRAIN_5384', 'TRAIN_5385', 'TRAIN_5386', 'TRAIN_5387', 'TRAIN_5388', 'TRAIN_5389', 'TRAIN_5390', 'TRAIN_5391', 'TRAIN_5392', 'TRAIN_5393', 'TRAIN_5394', 'TRAIN_5395', 'TRAIN_5396', 'TRAIN_5397', 'TRAIN_5398', 'TRAIN_5399', 'TRAIN_5400', 'TRAIN_5401', 'TRAIN_5402', 'TRAIN_5403', 'TRAIN_5404', 'TRAIN_5405', 'TRAIN_5406', 'TRAIN_5407', 'TRAIN_5408', 'TRAIN_5409', 'TRAIN_5410', 'TRAIN_5411', 'TRAIN_5412', 'TRAIN_5413', 'TRAIN_5414', 'TRAIN_5415', 'TRAIN_5416', 'TRAIN_5417', 'TRAIN_5418', 'TRAIN_5419', 'TRAIN_5420', 'TRAIN_5421', 'TRAIN_5422', 'TRAIN_5423', 'TRAIN_5424', 'TRAIN_5425', 'TRAIN_5426', 'TRAIN_5427', 'TRAIN_5428', 'TRAIN_5429', 'TRAIN_5430', 'TRAIN_5431', 'TRAIN_5432', 'TRAIN_5433', 'TRAIN_5434', 'TRAIN_5435', 'TRAIN_5436', 'TRAIN_5437', 'TRAIN_5438', 'TRAIN_5439', 'TRAIN_5440', 'TRAIN_5441', 'TRAIN_5442', 'TRAIN_5443', 'TRAIN_5444', 'TRAIN_5445', 'TRAIN_5446', 'TRAIN_5447', 'TRAIN_5448', 'TRAIN_5449', 'TRAIN_5450', 'TRAIN_5451', 'TRAIN_5452', 'TRAIN_5453', 'TRAIN_5454', 'TRAIN_5455', 'TRAIN_5456', 'TRAIN_5457', 'TRAIN_5458', 'TRAIN_5459', 'TRAIN_5460', 'TRAIN_5461', 'TRAIN_5462', 'TRAIN_5463', 'TRAIN_5464', 'TRAIN_5465', 'TRAIN_5466', 'TRAIN_5467', 'TRAIN_5468', 'TRAIN_5469', 'TRAIN_5470', 'TRAIN_5471', 'TRAIN_5472', 'TRAIN_5473', 'TRAIN_5474', 'TRAIN_5475', 'TRAIN_5476', 'TRAIN_5477', 'TRAIN_5478', 'TRAIN_5479', 'TRAIN_5480', 'TRAIN_5481', 'TRAIN_5482', 'TRAIN_5483', 'TRAIN_5484', 'TRAIN_5485', 'TRAIN_5486', 'TRAIN_5487', 'TRAIN_5488', 'TRAIN_5489', 'TRAIN_5490', 'TRAIN_5491', 'TRAIN_5492', 'TRAIN_5493', 'TRAIN_5494', 'TRAIN_5495', 'TRAIN_5496', 'TRAIN_5497', 'TRAIN_5498', 'TRAIN_5499', 'TRAIN_5500', 'TRAIN_5501', 'TRAIN_5502', 'TRAIN_5503', 'TRAIN_5504', 'TRAIN_5505', 'TRAIN_5506', 'TRAIN_5507', 'TRAIN_5508', 'TRAIN_5509', 'TRAIN_5510', 'TRAIN_5511', 'TRAIN_5512', 'TRAIN_5513', 'TRAIN_5514', 'TRAIN_5515', 'TRAIN_5516', 'TRAIN_5517', 'TRAIN_5518', 'TRAIN_5519', 'TRAIN_5520', 'TRAIN_5521', 'TRAIN_5522', 'TRAIN_5523', 'TRAIN_5524', 'TRAIN_5525', 'TRAIN_5526', 'TRAIN_5527', 'TRAIN_5528', 'TRAIN_5529', 'TRAIN_5530', 'TRAIN_5531', 'TRAIN_5532', 'TRAIN_5533', 'TRAIN_5534', 'TRAIN_5535', 'TRAIN_5536', 'TRAIN_5537', 'TRAIN_5538', 'TRAIN_5539', 'TRAIN_5540', 'TRAIN_5541', 'TRAIN_5542', 'TRAIN_5543', 'TRAIN_5544', 'TRAIN_5545', 'TRAIN_5546', 'TRAIN_5547', 'TRAIN_5548', 'TRAIN_5549', 'TRAIN_5550', 'TRAIN_5551', 'TRAIN_5552', 'TRAIN_5553', 'TRAIN_5554', 'TRAIN_5555', 'TRAIN_5556', 'TRAIN_5557', 'TRAIN_5558', 'TRAIN_5559', 'TRAIN_5560', 'TRAIN_5561', 'TRAIN_5562', 'TRAIN_5563', 'TRAIN_5564', 'TRAIN_5565', 'TRAIN_5566', 'TRAIN_5567', 'TRAIN_5568', 'TRAIN_5569', 'TRAIN_5570', 'TRAIN_5571', 'TRAIN_5572', 'TRAIN_5573', 'TRAIN_5574', 'TRAIN_5575', 'TRAIN_5576', 'TRAIN_5577', 'TRAIN_5578', 'TRAIN_5579', 'TRAIN_5580', 'TRAIN_5581', 'TRAIN_5582', 'TRAIN_5583', 'TRAIN_5584', 'TRAIN_5585', 'TRAIN_5586', 'TRAIN_5587', 'TRAIN_5588', 'TRAIN_5589', 'TRAIN_5590', 'TRAIN_5591', 'TRAIN_5592', 'TRAIN_5593', 'TRAIN_5594', 'TRAIN_5595', 'TRAIN_5596', 'TRAIN_5597', 'TRAIN_5598', 'TRAIN_5599', 'TRAIN_5600', 'TRAIN_5601', 'TRAIN_5602', 'TRAIN_5603', 'TRAIN_5604', 'TRAIN_5605', 'TRAIN_5606', 'TRAIN_5607', 'TRAIN_5608', 'TRAIN_5609', 'TRAIN_5610', 'TRAIN_5611', 'TRAIN_5612', 'TRAIN_5613', 'TRAIN_5614', 'TRAIN_5615', 'TRAIN_5616', 'TRAIN_5617', 'TRAIN_5618', 'TRAIN_5619', 'TRAIN_5620', 'TRAIN_5621', 'TRAIN_5622', 'TRAIN_5623', 'TRAIN_5624', 'TRAIN_5625', 'TRAIN_5626', 'TRAIN_5627', 'TRAIN_5628', 'TRAIN_5629', 'TRAIN_5630', 'TRAIN_5631', 'TRAIN_5632', 'TRAIN_5633', 'TRAIN_5634', 'TRAIN_5635', 'TRAIN_5636', 'TRAIN_5637', 'TRAIN_5638', 'TRAIN_5639', 'TRAIN_5640', 'TRAIN_5641', 'TRAIN_5642', 'TRAIN_5643', 'TRAIN_5644', 'TRAIN_5645', 'TRAIN_5646', 'TRAIN_5647', 'TRAIN_5648', 'TRAIN_5649', 'TRAIN_5650', 'TRAIN_5651', 'TRAIN_5652', 'TRAIN_5653', 'TRAIN_5654', 'TRAIN_5655', 'TRAIN_5656', 'TRAIN_5657', 'TRAIN_5658', 'TRAIN_5659', 'TRAIN_5660', 'TRAIN_5661', 'TRAIN_5662', 'TRAIN_5663', 'TRAIN_5664', 'TRAIN_5665', 'TRAIN_5666', 'TRAIN_5667', 'TRAIN_5668', 'TRAIN_5669', 'TRAIN_5670', 'TRAIN_5671', 'TRAIN_5672', 'TRAIN_5673', 'TRAIN_5674', 'TRAIN_5675', 'TRAIN_5676', 'TRAIN_5677', 'TRAIN_5678', 'TRAIN_5679', 'TRAIN_5680', 'TRAIN_5681', 'TRAIN_5682', 'TRAIN_5683', 'TRAIN_5684', 'TRAIN_5685', 'TRAIN_5686', 'TRAIN_5687', 'TRAIN_5688', 'TRAIN_5689', 'TRAIN_5690', 'TRAIN_5691', 'TRAIN_5692', 'TRAIN_5693', 'TRAIN_5694', 'TRAIN_5695', 'TRAIN_5696', 'TRAIN_5697', 'TRAIN_5698', 'TRAIN_5699', 'TRAIN_5700', 'TRAIN_5701', 'TRAIN_5702', 'TRAIN_5703', 'TRAIN_5704', 'TRAIN_5705', 'TRAIN_5706', 'TRAIN_5707', 'TRAIN_5708', 'TRAIN_5709', 'TRAIN_5710', 'TRAIN_5711', 'TRAIN_5712', 'TRAIN_5713', 'TRAIN_5714', 'TRAIN_5715', 'TRAIN_5716', 'TRAIN_5717', 'TRAIN_5718', 'TRAIN_5719', 'TRAIN_5720', 'TRAIN_5721', 'TRAIN_5722', 'TRAIN_5723', 'TRAIN_5724', 'TRAIN_5725', 'TRAIN_5726', 'TRAIN_5727', 'TRAIN_5728', 'TRAIN_5729', 'TRAIN_5730', 'TRAIN_5731', 'TRAIN_5732', 'TRAIN_5733', 'TRAIN_5734', 'TRAIN_5735', 'TRAIN_5736', 'TRAIN_5737', 'TRAIN_5738', 'TRAIN_5739', 'TRAIN_5740', 'TRAIN_5741', 'TRAIN_5742', 'TRAIN_5743', 'TRAIN_5744', 'TRAIN_5745', 'TRAIN_5746', 'TRAIN_5747', 'TRAIN_5748', 'TRAIN_5749', 'TRAIN_5750', 'TRAIN_5751', 'TRAIN_5752', 'TRAIN_5753', 'TRAIN_5754', 'TRAIN_5755', 'TRAIN_5756', 'TRAIN_5757', 'TRAIN_5758', 'TRAIN_5759', 'TRAIN_5760', 'TRAIN_5761', 'TRAIN_5762', 'TRAIN_5763', 'TRAIN_5764', 'TRAIN_5765', 'TRAIN_5766', 'TRAIN_5767', 'TRAIN_5768', 'TRAIN_5769', 'TRAIN_5770', 'TRAIN_5771', 'TRAIN_5772', 'TRAIN_5773', 'TRAIN_5774', 'TRAIN_5775', 'TRAIN_5776', 'TRAIN_5777', 'TRAIN_5778', 'TRAIN_5779', 'TRAIN_5780', 'TRAIN_5781', 'TRAIN_5782', 'TRAIN_5783', 'TRAIN_5784', 'TRAIN_5785', 'TRAIN_5786', 'TRAIN_5787', 'TRAIN_5788', 'TRAIN_5789', 'TRAIN_5790', 'TRAIN_5791', 'TRAIN_5792', 'TRAIN_5793', 'TRAIN_5794', 'TRAIN_5795', 'TRAIN_5796', 'TRAIN_5797', 'TRAIN_5798', 'TRAIN_5799', 'TRAIN_5800', 'TRAIN_5801', 'TRAIN_5802', 'TRAIN_5803', 'TRAIN_5804', 'TRAIN_5805', 'TRAIN_5806', 'TRAIN_5807', 'TRAIN_5808', 'TRAIN_5809', 'TRAIN_5810', 'TRAIN_5811', 'TRAIN_5812', 'TRAIN_5813', 'TRAIN_5814', 'TRAIN_5815', 'TRAIN_5816', 'TRAIN_5817', 'TRAIN_5818', 'TRAIN_5819', 'TRAIN_5820', 'TRAIN_5821', 'TRAIN_5822', 'TRAIN_5823', 'TRAIN_5824', 'TRAIN_5825', 'TRAIN_5826', 'TRAIN_5827', 'TRAIN_5828', 'TRAIN_5829', 'TRAIN_5830', 'TRAIN_5831', 'TRAIN_5832', 'TRAIN_5833', 'TRAIN_5834', 'TRAIN_5835', 'TRAIN_5836', 'TRAIN_5837', 'TRAIN_5838', 'TRAIN_5839', 'TRAIN_5840', 'TRAIN_5841', 'TRAIN_5842', 'TRAIN_5843', 'TRAIN_5844', 'TRAIN_5845', 'TRAIN_5846', 'TRAIN_5847', 'TRAIN_5848', 'TRAIN_5849', 'TRAIN_5850', 'TRAIN_5851', 'TRAIN_5852', 'TRAIN_5853', 'TRAIN_5854', 'TRAIN_5855', 'TRAIN_5856', 'TRAIN_5857', 'TRAIN_5858', 'TRAIN_5859', 'TRAIN_5860', 'TRAIN_5861', 'TRAIN_5862', 'TRAIN_5863', 'TRAIN_5864', 'TRAIN_5865', 'TRAIN_5866', 'TRAIN_5867', 'TRAIN_5868', 'TRAIN_5869', 'TRAIN_5870', 'TRAIN_5871', 'TRAIN_5872', 'TRAIN_5873', 'TRAIN_5874', 'TRAIN_5875', 'TRAIN_5876', 'TRAIN_5877', 'TRAIN_5878', 'TRAIN_5879', 'TRAIN_5880', 'TRAIN_5881', 'TRAIN_5882', 'TRAIN_5883', 'TRAIN_5884', 'TRAIN_5885', 'TRAIN_5886', 'TRAIN_5887', 'TRAIN_5888', 'TRAIN_5889', 'TRAIN_5890', 'TRAIN_5891', 'TRAIN_5892', 'TRAIN_5893', 'TRAIN_5894', 'TRAIN_5895', 'TRAIN_5896', 'TRAIN_5897', 'TRAIN_5898', 'TRAIN_5899', 'TRAIN_5900', 'TRAIN_5901', 'TRAIN_5902', 'TRAIN_5903', 'TRAIN_5904', 'TRAIN_5905', 'TRAIN_5906', 'TRAIN_5907', 'TRAIN_5908', 'TRAIN_5909', 'TRAIN_5910', 'TRAIN_5911', 'TRAIN_5912', 'TRAIN_5913', 'TRAIN_5914', 'TRAIN_5915', 'TRAIN_5916', 'TRAIN_5917', 'TRAIN_5918', 'TRAIN_5919', 'TRAIN_5920', 'TRAIN_5921', 'TRAIN_5922', 'TRAIN_5923', 'TRAIN_5924', 'TRAIN_5925', 'TRAIN_5926', 'TRAIN_5927', 'TRAIN_5928', 'TRAIN_5929', 'TRAIN_5930', 'TRAIN_5931', 'TRAIN_5932', 'TRAIN_5933', 'TRAIN_5934', 'TRAIN_5935', 'TRAIN_5936', 'TRAIN_5937', 'TRAIN_5938', 'TRAIN_5939', 'TRAIN_5940', 'TRAIN_5941', 'TRAIN_5942', 'TRAIN_5943', 'TRAIN_5944', 'TRAIN_5945', 'TRAIN_5946', 'TRAIN_5947', 'TRAIN_5948', 'TRAIN_5949', 'TRAIN_5950', 'TRAIN_5951', 'TRAIN_5952', 'TRAIN_5953', 'TRAIN_5954', 'TRAIN_5955', 'TRAIN_5956', 'TRAIN_5957', 'TRAIN_5958', 'TRAIN_5959', 'TRAIN_5960', 'TRAIN_5961', 'TRAIN_5962', 'TRAIN_5963', 'TRAIN_5964', 'TRAIN_5965', 'TRAIN_5966', 'TRAIN_5967', 'TRAIN_5968', 'TRAIN_5969', 'TRAIN_5970', 'TRAIN_5971', 'TRAIN_5972', 'TRAIN_5973', 'TRAIN_5974', 'TRAIN_5975', 'TRAIN_5976', 'TRAIN_5977', 'TRAIN_5978', 'TRAIN_5979', 'TRAIN_5980', 'TRAIN_5981', 'TRAIN_5982', 'TRAIN_5983', 'TRAIN_5984', 'TRAIN_5985', 'TRAIN_5986', 'TRAIN_5987', 'TRAIN_5988', 'TRAIN_5989', 'TRAIN_5990', 'TRAIN_5991', 'TRAIN_5992', 'TRAIN_5993', 'TRAIN_5994', 'TRAIN_5995', 'TRAIN_5996', 'TRAIN_5997', 'TRAIN_5998', 'TRAIN_5999', 'TRAIN_6000', 'TRAIN_6001', 'TRAIN_6002', 'TRAIN_6003', 'TRAIN_6004', 'TRAIN_6005', 'TRAIN_6006', 'TRAIN_6007', 'TRAIN_6008', 'TRAIN_6009', 'TRAIN_6010', 'TRAIN_6011', 'TRAIN_6012', 'TRAIN_6013', 'TRAIN_6014', 'TRAIN_6015', 'TRAIN_6016', 'TRAIN_6017', 'TRAIN_6018', 'TRAIN_6019', 'TRAIN_6020', 'TRAIN_6021', 'TRAIN_6022', 'TRAIN_6023', 'TRAIN_6024', 'TRAIN_6025', 'TRAIN_6026', 'TRAIN_6027', 'TRAIN_6028', 'TRAIN_6029', 'TRAIN_6030', 'TRAIN_6031', 'TRAIN_6032', 'TRAIN_6033', 'TRAIN_6034', 'TRAIN_6035', 'TRAIN_6036', 'TRAIN_6037', 'TRAIN_6038', 'TRAIN_6039', 'TRAIN_6040', 'TRAIN_6041', 'TRAIN_6042', 'TRAIN_6043', 'TRAIN_6044', 'TRAIN_6045', 'TRAIN_6046', 'TRAIN_6047', 'TRAIN_6048', 'TRAIN_6049', 'TRAIN_6050', 'TRAIN_6051', 'TRAIN_6052', 'TRAIN_6053', 'TRAIN_6054', 'TRAIN_6055', 'TRAIN_6056', 'TRAIN_6057', 'TRAIN_6058', 'TRAIN_6059', 'TRAIN_6060', 'TRAIN_6061', 'TRAIN_6062', 'TRAIN_6063', 'TRAIN_6064', 'TRAIN_6065', 'TRAIN_6066', 'TRAIN_6067', 'TRAIN_6068', 'TRAIN_6069', 'TRAIN_6070', 'TRAIN_6071', 'TRAIN_6072', 'TRAIN_6073', 'TRAIN_6074', 'TRAIN_6075', 'TRAIN_6076', 'TRAIN_6077', 'TRAIN_6078', 'TRAIN_6079', 'TRAIN_6080', 'TRAIN_6081', 'TRAIN_6082', 'TRAIN_6083', 'TRAIN_6084', 'TRAIN_6085', 'TRAIN_6086', 'TRAIN_6087', 'TRAIN_6088', 'TRAIN_6089', 'TRAIN_6090', 'TRAIN_6091', 'TRAIN_6092', 'TRAIN_6093', 'TRAIN_6094', 'TRAIN_6095', 'TRAIN_6096', 'TRAIN_6097', 'TRAIN_6098', 'TRAIN_6099', 'TRAIN_6100', 'TRAIN_6101', 'TRAIN_6102', 'TRAIN_6103', 'TRAIN_6104', 'TRAIN_6105', 'TRAIN_6106', 'TRAIN_6107', 'TRAIN_6108', 'TRAIN_6109', 'TRAIN_6110', 'TRAIN_6111', 'TRAIN_6112', 'TRAIN_6113', 'TRAIN_6114', 'TRAIN_6115', 'TRAIN_6116', 'TRAIN_6117', 'TRAIN_6118', 'TRAIN_6119', 'TRAIN_6120', 'TRAIN_6121', 'TRAIN_6122', 'TRAIN_6123', 'TRAIN_6124', 'TRAIN_6125', 'TRAIN_6126', 'TRAIN_6127', 'TRAIN_6128', 'TRAIN_6129', 'TRAIN_6130', 'TRAIN_6131', 'TRAIN_6132', 'TRAIN_6133', 'TRAIN_6134', 'TRAIN_6135', 'TRAIN_6136', 'TRAIN_6137', 'TRAIN_6138', 'TRAIN_6139', 'TRAIN_6140', 'TRAIN_6141', 'TRAIN_6142', 'TRAIN_6143', 'TRAIN_6144', 'TRAIN_6145', 'TRAIN_6146', 'TRAIN_6147', 'TRAIN_6148', 'TRAIN_6149', 'TRAIN_6150', 'TRAIN_6151', 'TRAIN_6152', 'TRAIN_6153', 'TRAIN_6154', 'TRAIN_6155', 'TRAIN_6156', 'TRAIN_6157', 'TRAIN_6158', 'TRAIN_6159', 'TRAIN_6160', 'TRAIN_6161', 'TRAIN_6162', 'TRAIN_6163', 'TRAIN_6164', 'TRAIN_6165', 'TRAIN_6166', 'TRAIN_6167', 'TRAIN_6168', 'TRAIN_6169', 'TRAIN_6170', 'TRAIN_6171', 'TRAIN_6172', 'TRAIN_6173', 'TRAIN_6174', 'TRAIN_6175', 'TRAIN_6176', 'TRAIN_6177', 'TRAIN_6178', 'TRAIN_6179', 'TRAIN_6180', 'TRAIN_6181', 'TRAIN_6182', 'TRAIN_6183', 'TRAIN_6184', 'TRAIN_6185', 'TRAIN_6186', 'TRAIN_6187', 'TRAIN_6188', 'TRAIN_6189', 'TRAIN_6190', 'TRAIN_6191', 'TRAIN_6192', 'TRAIN_6193', 'TRAIN_6194', 'TRAIN_6195', 'TRAIN_6196', 'TRAIN_6197', 'TRAIN_6198', 'TRAIN_6199', 'TRAIN_6200'], 'dev': [0, 1, 2, 4, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37, 38, 39, 41, 43, 44, 46, 47, 48, 49, 50, 51, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 67, 68, 69, 70, 72, 73, 74, 75, 76, 77, 78, 80, 81, 82, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 99, 100, 103, 104, 106, 107, 108, 109, 110, 111, 112, 115, 116, 117, 118, 120, 123, 124, 125, 126, 127, 128, 130, 131, 132, 134, 135, 136, 137, 138, 140, 141, 142, 143, 145, 146, 147, 148, 149, 150, 153, 154, 155, 157, 158, 159, 161, 162, 163, 164, 166, 167, 168, 169, 170, 172, 173, 174, 175, 178, 179, 180, 184, 185, 186, 188, 189, 191, 193, 194, 196, 197, 198, 199, 201, 202, 203, 204, 205, 206, 207, 208, 209, 211, 213, 214, 215, 216, 217, 218, 219, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 245, 247, 248, 250, 251, 253, 254, 255, 257, 258, 259, 260, 261, 262, 264, 265, 266, 267, 268, 270, 271, 272, 273, 274, 276, 277, 279, 280, 281, 282, 284, 285, 286, 287, 289, 290, 291, 292, 293, 294, 296, 297, 298, 299, 300, 301, 302, 304, 305, 307, 308, 309, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 348, 349, 350, 352, 355, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 369, 370, 371, 372, 374, 376, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 393, 394, 395, 396, 397, 399, 400, 401, 403, 404, 405, 406, 408, 409, 410, 411, 414, 415, 416, 417, 418, 419, 420, 422, 423, 424, 425, 426, 427, 428, 429, 430, 433, 434, 435, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 450, 452, 453, 454, 455, 456, 457, 458, 460, 464, 465, 467, 468, 469, 470, 472, 473, 474, 476, 477, 478, 479, 480, 481, 482, 483, 485, 486, 488, 490, 491, 492, 494, 495, 498, 500, 501, 502, 504, 506, 507, 509, 511, 513, 514, 515, 516, 517, 518, 519, 520, 521, 523, 524, 525, 527, 528, 529, 530, 533, 534, 535, 536, 537, 538, 539, 540, 541, 543, 544, 545, 546, 547, 549, 550, 551, 552, 553, 555, 556, 557, 558, 560, 561, 562, 563, 565, 567, 568, 569, 570, 571, 572, 573, 574, 575, 577, 578, 579, 580, 581, 583, 584, 587, 588, 589, 590, 591, 593, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 606, 607, 609, 611, 612, 613, 614, 615, 616, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 637, 638, 639, 640, 641, 642, 643, 644, 647, 648, 649, 650, 652, 653, 654, 655, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 672, 673, 674, 675, 676, 677, 678, 679, 680, 682, 684, 686, 687, 689, 690, 691, 692, 694, 695, 696, 698, 699, 700, 702, 703, 704, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 728, 730, 731, 732, 734, 735, 736, 737, 739, 742, 743, 744, 745, 747, 750, 751, 752, 753, 756, 757, 758, 759, 760, 761, 763, 765, 766, 767, 768, 769, 771, 774, 775, 776, 777, 778, 780, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792, 794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 806, 807, 808, 809, 810, 811, 812, 814, 815, 816, 817, 818, 819, 820, 821, 822, 823, 824, 826, 827, 828, 829, 830, 831, 833, 834, 835, 836, 837, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 849, 850, 851, 852, 853, 854, 855, 856, 857, 858, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 871, 872, 873, 874, 876, 877, 878, 879, 880, 881, 883, 884, 885, 888, 889, 890, 891, 892, 893, 895, 896, 899, 900, 901, 902, 905, 906, 907, 908, 909, 910, 911, 913, 914, 915, 916, 917, 918, 919, 920, 921, 922, 923, 924, 925, 926, 927, 928, 929, 931, 932, 934, 935, 936, 937, 938, 939, 940, 942, 943, 944, 945, 946, 947, 948, 949, 950, 951, 953, 954, 955, 956, 957, 958, 959, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 979, 980, 981, 983, 984, 985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 995, 996, 997, 998, 1000, 1001, 1002, 1003, 1004, 1006, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1023, 1024, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1048, 1049, 1050, 1051, 1052, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1070, 1071, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1082, 1083, 1084, 1086, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1096, 1097, 1098, 1099, 1100, 1101, 1102, 1103, 1107, 1109, 1110, 1111, 1113, 1115, 1116, 1117, 1118, 1119, 1120, 1122, 1123, 1125, 1126, 1127, 1128, 1129, 1130, 1131, 1132, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1144, 1145, 1146, 1148, 1149, 1150, 1151, 1152, 1154, 1155, 1156, 1157, 1158, 1159, 1161, 1162, 1163, 1164, 1165, 1166, 1167, 1168, 1169, 1170, 1172, 1174, 1176, 1177, 1178, 1179, 1181, 1182, 1183, 1184, 1185, 1187, 1188, 1189, 1190, 1191, 1192, 1193, 1194, 1195, 1196, 1197, 1199, 1201, 1202, 1203, 1205, 1206, 1207, 1208, 1209, 1210, 1211, 1213, 1215, 1216, 1217, 1218, 1219, 1220, 1221, 1222, 1225, 1226, 1227, 1228, 1230, 1231, 1232, 1233, 1234, 1236, 1237, 1238, 1240, 1242, 1243, 1244, 1245, 1246, 1247, 1248, 1249, 1251, 1253, 1256, 1257, 1258, 1259, 1260, 1261, 1262, 1263, 1264, 1265, 1266, 1268, 1269, 1270, 1273, 1274, 1275, 1276, 1277, 1278, 1279, 1280, 1282, 1283, 1284, 1285, 1286, 1287, 1288, 1290, 1291, 1292, 1293, 1294, 1295, 1296, 1297, 1298, 1299, 1300, 1301, 1302, 1303, 1304, 1305, 1306, 1307, 1309, 1310, 1311, 1313, 1314, 1315, 1316, 1317, 1318, 1320, 1322, 1323, 1324, 1325, 1326, 1327, 1328, 1329, 1330, 1331, 1332, 1333, 1334, 1335, 1336, 1337, 1338, 1339, 1340, 1341, 1342, 1343, 1344, 1345, 1346, 1347, 1349, 1350, 1351, 1352, 1353, 1354, 1355, 1356, 1357, 1358, 1360, 1361, 1363, 1364, 1365, 1366, 1367, 1368, 1369, 1370, 1372, 1374, 1376, 1379, 1380, 1381, 1383, 1385, 1387, 1388, 1389, 1390, 1392, 1394, 1395, 1397, 1398, 1399, 1401, 1402, 1403, 1404, 1405, 1407, 1410, 1412, 1413, 1414, 1415, 1416, 1417, 1418, 1419, 1420, 1421, 1422, 1423, 1424, 1426, 1427, 1428, 1429, 1430, 1431, 1432, 1433, 1434, 1435, 1437, 1438, 1439, 1441, 1442, 1443, 1444, 1445, 1446, 1448, 1449, 1450, 1451, 1453, 1455, 1457, 1458, 1459, 1461, 1462, 1464, 1465, 1466, 1467, 1468, 1469, 1470, 1471, 1472, 1473, 1474, 1475, 1476, 1478, 1479, 1481, 1483, 1485, 1486, 1488, 1489, 1490, 1492, 1496, 1498, 1499, 1500, 1501, 1502, 1503, 1504, 1505, 1506, 1507, 1508, 1510, 1513, 1516, 1518, 1519, 1520, 1521, 1522, 1523, 1524, 1528, 1529, 1530, 1531, 1532, 1533, 1535, 1536, 1537, 1538, 1540, 1541, 1542, 1543, 1544, 1545, 1546, 1547, 1548, 1549, 1550, 1551, 1552, 1553, 1554, 1556, 1557, 1558, 1559, 1560, 1562, 1563, 1564, 1565, 1566, 1568, 1571, 1572, 1573, 1574, 1575, 1576, 1577, 1578, 1580, 1581, 1582, 1583, 1584, 1586, 1588, 1589, 1590, 1591, 1592, 1593, 1595, 1596, 1597, 1599, 1600, 1601, 1603, 1604, 1605, 1606, 1607, 1608, 1610, 1611, 1612, 1613, 1614, 1615, 1616, 1617, 1618, 1619, 1620, 1621, 1625, 1626, 1627, 1629, 1630, 1631, 1632, 1634, 1635, 1636, 1637, 1638, 1639, 1640, 1641, 1642, 1643, 1644, 1645, 1646, 1647, 1648, 1650, 1651, 1652, 1653, 1654, 1656, 1657, 1658, 1659, 1661, 1662, 1663, 1664, 1667, 1669, 1670, 1671, 1672, 1673, 1674, 1675, 1677, 1678, 1679, 1680, 1681, 1682, 1684, 1685, 1686, 1687, 1689, 1690, 1691, 1692, 1693, 1694, 1695, 1696, 1697, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 1705, 1706, 1709, 1710, 1711, 1712, 1715, 1716, 1718, 1719, 1720, 1721, 1722, 1723, 1725, 1726, 1727, 1730, 1731, 1732, 1733, 1734, 1735, 1736, 1738, 1740, 1741, 1742, 1743, 1744, 1745, 1746, 1747, 1750, 1751, 1752, 1754, 1756, 1757, 1758, 1761, 1762, 1763, 1765, 1766, 1768, 1769, 1770, 1771, 1772, 1774, 1775, 1776, 1777, 1778, 1779, 1780, 1781, 1783, 1784, 1785, 1787, 1788, 1789, 1790, 1792, 1793, 1796, 1797, 1800, 1801, 1804, 1805, 1806, 1807, 1809, 1811, 1812, 1813, 1814, 1815, 1816, 1817, 1818, 1819, 1821, 1822, 1824, 1825, 1826, 1827, 1828, 1829, 1830, 1831, 1833, 1834, 1835, 1836, 1838, 1839, 1840, 1841, 1843, 1844, 1845, 1846, 1847, 1848, 1849, 1850, 1851, 1852, 1853, 1854, 1856, 1857, 1858, 1859, 1862, 1863, 1864, 1866, 1868, 1869, 1870, 1871, 1872, 1873, 1874, 1875, 1876, 1877, 1878, 1882, 1883, 1884, 1885, 1886, 1887, 1888, 1889, 1890, 1892, 1893, 1894, 1895, 1896, 1898, 1899, 1900, 1901, 1902, 1903, 1904, 1905, 1906, 1907, 1908, 1910, 1911, 1912, 1913, 1914, 1915, 1916, 1917, 1918, 1919, 1920, 1921, 1922, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1944, 1945, 1946, 1948, 1949, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1961, 1962, 1963, 1966, 1967, 1969, 1970, 1971, 1973, 1975, 1976, 1979, 1981, 1982, 1983, 1984, 1985, 1987, 1989, 1990, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2010, 2011, 2014, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2036, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046, 2047, 2049, 2051, 2052, 2053, 2054, 2055, 2056, 2057, 2058, 2059, 2060, 2061, 2062, 2063, 2064, 2065, 2066, 2067, 2068, 2069, 2070, 2072, 2073, 2074, 2075, 2077, 2078, 2080, 2081, 2084, 2087, 2088, 2089, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2102, 2104, 2105, 2106, 2109, 2110, 2111, 2112, 2113, 2114, 2116, 2117, 2118, 2119, 2121, 2123, 2124, 2125, 2126, 2127, 2128, 2129, 2130, 2131, 2132, 2133, 2134, 2135, 2137, 2138, 2139, 2140, 2141, 2142, 2143, 2144, 2145, 2146, 2147, 2148, 2150, 2151, 2152, 2153, 2154, 2155, 2156, 2157, 2159, 2160, 2162, 2164, 2165, 2166, 2167, 2168, 2169, 2170, 2171, 2172, 2173, 2174, 2175, 2176, 2179, 2181, 2183, 2184, 2185, 2186, 2187, 2188, 2190, 2191, 2192, 2195, 2196, 2197, 2198, 2199, 2200, 2202, 2204, 2205, 2206, 2209, 2210, 2213, 2214, 2216, 2217, 2218, 2219, 2220, 2221, 2223, 2224, 2225, 2226, 2227, 2228, 2229, 2230, 2231, 2232, 2233, 2235, 2236, 2238, 2239, 2240, 2243, 2244, 2245, 2246, 2247, 2248, 2249, 2250, 2251, 2252, 2253, 2254, 2255, 2256, 2257, 2259, 2260, 2261, 2262, 2263, 2264, 2265, 2266, 2267, 2268, 2269, 2270, 2271, 2272, 2273, 2275, 2276, 2278, 2279, 2280, 2282, 2283, 2285, 2286, 2287, 2288, 2289, 2290, 2292, 2293, 2294, 2296, 2298, 2299, 2300, 2301, 2303, 2304, 2306, 2307, 2308, 2309, 2310, 2312, 2313, 2316, 2318, 2319, 2320, 2322, 2325, 2326, 2327, 2329, 2330, 2331, 2332, 2333, 2334, 2335, 2336, 2337, 2338, 2339, 2340, 2341, 2342, 2343, 2345, 2346, 2347, 2350, 2351, 2352, 2353, 2356, 2358, 2359, 2360, 2362, 2363, 2364, 2366, 2367, 2369, 2370, 2371, 2373, 2374, 2375, 2376, 2377, 2379, 2380, 2381, 2382, 2383, 2384, 2385, 2386, 2387, 2388, 2389, 2390, 2393, 2394, 2395, 2397, 2398, 2399, 2400, 2401, 2402, 2403, 2404, 2405, 2406, 2407, 2409, 2410, 2411, 2412, 2413, 2414, 2415, 2416, 2417, 2418, 2419, 2420, 2421, 2422, 2423, 2424, 2425, 2427, 2429, 2430, 2431, 2433, 2434, 2435, 2436, 2437, 2438, 2439, 2441, 2442, 2443, 2445, 2446, 2447, 2448, 2449, 2451, 2452, 2453, 2454, 2455, 2456, 2457, 2458, 2459, 2460, 2462, 2463, 2464, 2465, 2466, 2468, 2469, 2470, 2471, 2472, 2473, 2474, 2475, 2476, 2478, 2479, 2480, 2481, 2482, 2483, 2484, 2485, 2486, 2487, 2488, 2489, 2490, 2491, 2494, 2495, 2496, 2497, 2498, 2499, 2500, 2502, 2503, 2504, 2505, 2506, 2507, 2508, 2509, 2510, 2511, 2512, 2513, 2516, 2517, 2520, 2521, 2522, 2523, 2524, 2525, 2526, 2527, 2528, 2529, 2530, 2532, 2533, 2534, 2535, 2536, 2537, 2538, 2539, 2540, 2541, 2542, 2543, 2544, 2546, 2548, 2549, 2551, 2552, 2554, 2555, 2556, 2558, 2559, 2560, 2563, 2564, 2565, 2567, 2568, 2569, 2571, 2572, 2573, 2574, 2575, 2576, 2577, 2578, 2580, 2581, 2582, 2583, 2585, 2586, 2587, 2588, 2589, 2590, 2592, 2593, 2594, 2595, 2596, 2597, 2598, 2600, 2601, 2602, 2603, 2604, 2605, 2606, 2607, 2608, 2609, 2610, 2611, 2612, 2614, 2617, 2618, 2619, 2620, 2621, 2622, 2623, 2624, 2625, 2626, 2629, 2631, 2632, 2634, 2636, 2637, 2640, 2641, 2642, 2643, 2648, 2649, 2650, 2651, 2652, 2653, 2654, 2655, 2657, 2658, 2659, 2660, 2661, 2662, 2663, 2664, 2665, 2666, 2667, 2669, 2670, 2671, 2674, 2676, 2677, 2678, 2679, 2681, 2682, 2684, 2685, 2686, 2687, 2688, 2689, 2690, 2691, 2692, 2693, 2694, 2695, 2696, 2697, 2698, 2699, 2700, 2701, 2702, 2703, 2704, 2705, 2707, 2708, 2710, 2712, 2713, 2715, 2716, 2717, 2718, 2719, 2720, 2721, 2722, 2723, 2724, 2726, 2727, 2728, 2729, 2730, 2731, 2733, 2734, 2735, 2736, 2737, 2738, 2740, 2741, 2742, 2743, 2744, 2746, 2748, 2749, 2750, 2751, 2752, 2753, 2754, 2755, 2756, 2757, 2758, 2759, 2760, 2762, 2763, 2764, 2766, 2768, 2769, 2770, 2773, 2774, 2775, 2777, 2779, 2780, 2783, 2785, 2786, 2787, 2789, 2790, 2791, 2792, 2793, 2795, 2798, 2799, 2800, 2802, 2803, 2806, 2808, 2810, 2811, 2812, 2813, 2814, 2816, 2817, 2818, 2819, 2820, 2822, 2824, 2825, 2826, 2827, 2828, 2829, 2832, 2833, 2835, 2836, 2838, 2839, 2840, 2841, 2842, 2843, 2845, 2846, 2847, 2848, 2850, 2851, 2852, 2853, 2854, 2855, 2856, 2858, 2859, 2860, 2861, 2864, 2865, 2866, 2867, 2868, 2869, 2870, 2875, 2876, 2879, 2880, 2881, 2882, 2884, 2885, 2886, 2887, 2889, 2890, 2891, 2893, 2894, 2895, 2896, 2897, 2898, 2899, 2900, 2902, 2904, 2906, 2907, 2908, 2909, 2910, 2911, 2912, 2914, 2916, 2917, 2918, 2919, 2920, 2921, 2922, 2923, 2924, 2925, 2926, 2928, 2929, 2930, 2931, 2932, 2933, 2934, 2935, 2936, 2937, 2938, 2939, 2940, 2941, 2942, 2943, 2944, 2945, 2947, 2948, 2949, 2950, 2951, 2952, 2953, 2954, 2955, 2956, 2957, 2958, 2959, 2960, 2962, 2963, 2964, 2965, 2966, 2967, 2968, 2969, 2970, 2972, 2973, 2974, 2975, 2976, 2977, 2979, 2980, 2981, 2983, 2984, 2985, 2987, 2988, 2989, 2990, 2991, 2992, 2994, 2995, 2996, 2998, 2999, 3000, 3001, 3002, 3004, 3005, 3006, 3007, 3009, 3010, 3011, 3013, 3015, 3017, 3018, 3021, 3022, 3023, 3024, 3025, 3026, 3027, 3028, 3029, 3030, 3031, 3032, 3033, 3034, 3035, 3036, 3037, 3038, 3039, 3040, 3041, 3045, 3046, 3048, 3050, 3051, 3052, 3053, 3054, 3056, 3057, 3058, 3060, 3062, 3063, 3065, 3066, 3067, 3068, 3069, 3070, 3071, 3072, 3073, 3075, 3076, 3077, 3079, 3080, 3081, 3082, 3083, 3084, 3085, 3086, 3087, 3088, 3090, 3091, 3092, 3094, 3095, 3096, 3097, 3098, 3099, 3100, 3101, 3102, 3103, 3104, 3105, 3106, 3107, 3109, 3110, 3111, 3113, 3114, 3115, 3116, 3117, 3118, 3119, 3120, 3121, 3122, 3124, 3125, 3126, 3127, 3128, 3130, 3131, 3132, 3133, 3134, 3135, 3137, 3138, 3140, 3141, 3142, 3143, 3146, 3147, 3153, 3154, 3155, 3156, 3157, 3159, 3161, 3162, 3164, 3165, 3166, 3167, 3168, 3169, 3172, 3173, 3174, 3175, 3176, 3177, 3178, 3179, 3181, 3182, 3183, 3184, 3185, 3188, 3189, 3190, 3191, 3192, 3193, 3194, 3195, 3196, 3197, 3199, 3201, 3202, 3203, 3205, 3206, 3208, 3209, 3211, 3212, 3214, 3215, 3216, 3217, 3218, 3219, 3221, 3222, 3223, 3224, 3225, 3227, 3228, 3229, 3230, 3231, 3232, 3233, 3235, 3237, 3238, 3240, 3241, 3242, 3243, 3245, 3246, 3247, 3248, 3249, 3250, 3251, 3252, 3253, 3254, 3255, 3256, 3257, 3259, 3260, 3261, 3262, 3263, 3265, 3267, 3268, 3269, 3270, 3272, 3273, 3274, 3275, 3276, 3277, 3278, 3279, 3280, 3281, 3282, 3283, 3285, 3286, 3287, 3288, 3289, 3290, 3291, 3292, 3293, 3295, 3297, 3298, 3300, 3301, 3302, 3303, 3304, 3305, 3306, 3308, 3309, 3310, 3311, 3312, 3313, 3314, 3315, 3316, 3317, 3319, 3320, 3321, 3322, 3325, 3328, 3329, 3330, 3332, 3333, 3334, 3335, 3336, 3337, 3338, 3339, 3340, 3341, 3343, 3344, 3345, 3346, 3347, 3348, 3349, 3350, 3351, 3352, 3353, 3354, 3355, 3356, 3357, 3358, 3359, 3360, 3361, 3362, 3364, 3366, 3367, 3368, 3369, 3371, 3372, 3373, 3375, 3378, 3379, 3380, 3381, 3382, 3383, 3384, 3385, 3386, 3388, 3389, 3390, 3391, 3393, 3394, 3395, 3396, 3397, 3398, 3399, 3400, 3401, 3402, 3404, 3405, 3406, 3407, 3409, 3411, 3412, 3413, 3414, 3415, 3416, 3417, 3418, 3419, 3420, 3421, 3422, 3423, 3425, 3426, 3427, 3430, 3433, 3434, 3435, 3436, 3437, 3438, 3440, 3441, 3442, 3443, 3444, 3445, 3448, 3449, 3450, 3453, 3454, 3455, 3456, 3458, 3459, 3460, 3461, 3462, 3463, 3464, 3465, 3466, 3470, 3471, 3474, 3475, 3477, 3478, 3479, 3480, 3481, 3482, 3484, 3485, 3486, 3487, 3488, 3489, 3490, 3491, 3492, 3493, 3494, 3495, 3497, 3498, 3499, 3500, 3501, 3502, 3505, 3506, 3507, 3509, 3510, 3511, 3512, 3514, 3516, 3518, 3519, 3520, 3521, 3522, 3523, 3524, 3525, 3526, 3529, 3530, 3531, 3532, 3534, 3535, 3536, 3537, 3538, 3539, 3540, 3541, 3542, 3543, 3544, 3545, 3547, 3548, 3549, 3550, 3551, 3552, 3553, 3554, 3555, 3557, 3558, 3561, 3562, 3563, 3564, 3565, 3566, 3567, 3568, 3570, 3573, 3574, 3575, 3576, 3577, 3578, 3579, 3580, 3581, 3582, 3584, 3587, 3588, 3589, 3590, 3591, 3592, 3593, 3595, 3596, 3599, 3600, 3601, 3602, 3603, 3604, 3605, 3606, 3609, 3610, 3611, 3612, 3613, 3614, 3615, 3617, 3619, 3620, 3622, 3623, 3624, 3627, 3628, 3629, 3630, 3631, 3633, 3634, 3635, 3636, 3637, 3641, 3642, 3644, 3645, 3647, 3648, 3649, 3650, 3651, 3652, 3653, 3654, 3655, 3656, 3657, 3658, 3659, 3661, 3664, 3665, 3666, 3667, 3669, 3670, 3671, 3672, 3673, 3674, 3675, 3676, 3678, 3679, 3680, 3681, 3682, 3683, 3684, 3685, 3686, 3687, 3688, 3689, 3691, 3692, 3694, 3695, 3696, 3697, 3698, 3699, 3701, 3702, 3703, 3705, 3706, 3707, 3708, 3709, 3710, 3711, 3712, 3713, 3714, 3715, 3716, 3717, 3719, 3720, 3722, 3726, 3729, 3730, 3731, 3732, 3733, 3734, 3737, 3738, 3739, 3740, 3741, 3742, 3743, 3745, 3746, 3747, 3749, 3750, 3751, 3752, 3753, 3754, 3755, 3757, 3758, 3759, 3760, 3761, 3763, 3764, 3765, 3767, 3768, 3769, 3770, 3772, 3773, 3775, 3776, 3778, 3779, 3780, 3781, 3782, 3783, 3784, 3785, 3786, 3787, 3788, 3791, 3793, 3794, 3795, 3796, 3797, 3798, 3799, 3800, 3802, 3804, 3805, 3806, 3807, 3808, 3810, 3811, 3812, 3813, 3814, 3815, 3816, 3817, 3819, 3820, 3822, 3823, 3824, 3825, 3826, 3828, 3829, 3830, 3831, 3832, 3833, 3835, 3837, 3838, 3839, 3841, 3842, 3845, 3846, 3847, 3848, 3849, 3850, 3852, 3853, 3854, 3858, 3859, 3860, 3862, 3863, 3864, 3865, 3867, 3868, 3869, 3870, 3872, 3874, 3875, 3877, 3879, 3880, 3881, 3883, 3884, 3885, 3886, 3889, 3890, 3891, 3892, 3893, 3895, 3896, 3897, 3898, 3899, 3900, 3901, 3902, 3903, 3905, 3907, 3909, 3910, 3912, 3913, 3914, 3915, 3916, 3918, 3919, 3920, 3921, 3922, 3923, 3925, 3926, 3927, 3928, 3929, 3931, 3932, 3933, 3935, 3936, 3937, 3938, 3941, 3942, 3943, 3945, 3946, 3947, 3948, 3950, 3951, 3952, 3953, 3954, 3956, 3957, 3958, 3959, 3961, 3962, 3963, 3964, 3965, 3967, 3968, 3969, 3970, 3971, 3972, 3973, 3974, 3975, 3976, 3977, 3979, 3980, 3981, 3982, 3983, 3984, 3985, 3986, 3987, 3989, 3991, 3993, 3995, 3996, 3997, 3998, 3999, 4000, 4001, 4003, 4005, 4006, 4007, 4008, 4009, 4010, 4011, 4012, 4013, 4014, 4015, 4016, 4017, 4018, 4019, 4020, 4022, 4024, 4025, 4026, 4028, 4030, 4033, 4035, 4036, 4038, 4039, 4040, 4041, 4042, 4045, 4046, 4048, 4049, 4050, 4051, 4052, 4053, 4054, 4057, 4058, 4059, 4060, 4061, 4062, 4063, 4064, 4066, 4067, 4069, 4071, 4072, 4073, 4074, 4075, 4078, 4079, 4081, 4083, 4085, 4086, 4087, 4088, 4089, 4090, 4091, 4092, 4093, 4094, 4095, 4096, 4097, 4098, 4102, 4103, 4104, 4105, 4108, 4109, 4112, 4113, 4116, 4117, 4118, 4119, 4120, 4121, 4123, 4124, 4126, 4128, 4129, 4130, 4131, 4132, 4133, 4134, 4135, 4136, 4138, 4139, 4142, 4143, 4144, 4145, 4146, 4147, 4148, 4149, 4150, 4151, 4152, 4153, 4154, 4155, 4156, 4157, 4159, 4162, 4163, 4164, 4165, 4166, 4167, 4168, 4170, 4172, 4173, 4175, 4176, 4177, 4178, 4179, 4180, 4183, 4184, 4185, 4187, 4189, 4190, 4192, 4194, 4195, 4197, 4198, 4199, 4200, 4201, 4203, 4204, 4205, 4206, 4208, 4209, 4210, 4211, 4212, 4215, 4216, 4217, 4218, 4219, 4220, 4222, 4223, 4224, 4225, 4226, 4227, 4229, 4231, 4232, 4234, 4237, 4238, 4239, 4240, 4241, 4242, 4245, 4246, 4247, 4248, 4249, 4250, 4251, 4252, 4253, 4254, 4255, 4256, 4257, 4258, 4259, 4260, 4261, 4262, 4263, 4264, 4265, 4266, 4267, 4270, 4271, 4274, 4275, 4276, 4277, 4278, 4279, 4280, 4281, 4282, 4283, 4284, 4285, 4286, 4287, 4288, 4289, 4290, 4292, 4293, 4296, 4298, 4299, 4300, 4301, 4302, 4303, 4304, 4305, 4306, 4307, 4308, 4309, 4311, 4312, 4313, 4314, 4316, 4318, 4319, 4320, 4322, 4323, 4324, 4325, 4326, 4328, 4329, 4330, 4331, 4332, 4334, 4335, 4336, 4338, 4339, 4340, 4341, 4342, 4343, 4344, 4345, 4346, 4347, 4348, 4349, 4350, 4351, 4352, 4353, 4354, 4355, 4356, 4359, 4360, 4361, 4362, 4363, 4364, 4366, 4367, 4368, 4369, 4370, 4371, 4372, 4373, 4374, 4376, 4377, 4379, 4380, 4381, 4382, 4383, 4384, 4385, 4386, 4388, 4390, 4391, 4392, 4394, 4395, 4396, 4397, 4398, 4399, 4400, 4401, 4402, 4403, 4404, 4405, 4406, 4408, 4409, 4410, 4411, 4412, 4414, 4415, 4417, 4418, 4419, 4420, 4422, 4423, 4424, 4425, 4426, 4427, 4429, 4430, 4431, 4432, 4433, 4434, 4435, 4436, 4437, 4438, 4440, 4441, 4442, 4443, 4444, 4445, 4447, 4448, 4452, 4453, 4454, 4455, 4456, 4457, 4458, 4460, 4461, 4463, 4464, 4465, 4468, 4469, 4470, 4471, 4472, 4473, 4475, 4477, 4478, 4480, 4481, 4482, 4484, 4485, 4486, 4488, 4489, 4490, 4493, 4494, 4496, 4497, 4498, 4499, 4500, 4501, 4502, 4503, 4504, 4506, 4507, 4508, 4510, 4511, 4512, 4513, 4514, 4516, 4517, 4518, 4519, 4520, 4521, 4522, 4523, 4526, 4527, 4528, 4531, 4532, 4533, 4535, 4537, 4538, 4539, 4540, 4541, 4542, 4543, 4544, 4545, 4547, 4549, 4550, 4551, 4552, 4553, 4555, 4556, 4557, 4558, 4560, 4562, 4563, 4564, 4565, 4566, 4567, 4569, 4570, 4571, 4574, 4576, 4579, 4581, 4582, 4583, 4584, 4586, 4587, 4588, 4589, 4590, 4591, 4592, 4595, 4596, 4597, 4598, 4599, 4601, 4602, 4603, 4605, 4606, 4608, 4609, 4610, 4612, 4613, 4614, 4615, 4616, 4617, 4618, 4620, 4621, 4622, 4623, 4627, 4628, 4629, 4631, 4633, 4634, 4635, 4636, 4637, 4638, 4639, 4640, 4641, 4643, 4644, 4645, 4646, 4647, 4650, 4651, 4653, 4654, 4656, 4657, 4658, 4659, 4660, 4661, 4662, 4663, 4664, 4665, 4666, 4667, 4669, 4672, 4673, 4675, 4676, 4677, 4678, 4679, 4680, 4681, 4684, 4685, 4687, 4688, 4689, 4690, 4691, 4692, 4693, 4694, 4695, 4696, 4697, 4698, 4699, 4700, 4701, 4703, 4704, 4705, 4706, 4707, 4708, 4709, 4710, 4712, 4714, 4715, 4716, 4717, 4718, 4719, 4720, 4721, 4722, 4723, 4724, 4725, 4727, 4728, 4729, 4731, 4732, 4733, 4737, 4738, 4739, 4740, 4741, 4742, 4743, 4744, 4745, 4746, 4747, 4748, 4749, 4750, 4751, 4752, 4753, 4754, 4755, 4756, 4758, 4759, 4761, 4762, 4763, 4764, 4765, 4767, 4769, 4770, 4771, 4773, 4775, 4776, 4777, 4778, 4779, 4780, 4781, 4782, 4783, 4784, 4785, 4786, 4790, 4791, 4792, 4793, 4794, 4797, 4798, 4800, 4801, 4803, 4804, 4805, 4806, 4807, 4808, 4809, 4810, 4811, 4813, 4814, 4815, 4816, 4817, 4818, 4820, 4821, 4822, 4824, 4826, 4828, 4829, 4830, 4831, 4832, 4833, 4834, 4835, 4836, 4838, 4839, 4842, 4843, 4844, 4847, 4848, 4849, 4850, 4851, 4852, 4853, 4854, 4855, 4856, 4857, 4858, 4859, 4860, 4861, 4862, 4863, 4865, 4866, 4867, 4868, 4869, 4871, 4872, 4873, 4874, 4875, 4876, 4877, 4880, 4881, 4884, 4887, 4889, 4890, 4891, 4892, 4894, 4895, 4896, 4897, 4899, 4900, 4901, 4905, 4906, 4907, 4908, 4909, 4910, 4911, 4913, 4914, 4915, 4916, 4917, 4918, 4920, 4922, 4923, 4924, 4926, 4927, 4928, 4931, 4932, 4933, 4934, 4936, 4939, 4940, 4942, 4945, 4946, 4947, 4948, 4949, 4950, 4951, 4953, 4955, 4957, 4958, 4959, 4961, 4963, 4964, 4965, 4967, 4968, 4969, 4970, 4971, 4973, 4974, 4975, 4977, 4979, 4980, 4981, 4982, 4983, 4984, 4985, 4987, 4988, 4989, 4990, 4991, 4992, 4993, 4994, 4995, 4996, 4997, 4998, 5000, 5004, 5006, 5008, 5009, 5010, 5012, 5013, 5014, 5015, 5016, 5017, 5018, 5019, 5020, 5021, 5022, 5023, 5024, 5025, 5026, 5028, 5030, 5031, 5032, 5033, 5034, 5036, 5037, 5038, 5039, 5040, 5044, 5045, 5046, 5047, 5048, 5049, 5050, 5052, 5054, 5055, 5056, 5057, 5058, 5060, 5061, 5062, 5063, 5064, 5065, 5066, 5067, 5069, 5070, 5071, 5073, 5075, 5078, 5079, 5080, 5081, 5082, 5083, 5085, 5086, 5087, 5089, 5090, 5091, 5092, 5093, 5094, 5095, 5096, 5098, 5099, 5100, 5102, 5105, 5106, 5107, 5108, 5109, 5111, 5112, 5113, 5115, 5116, 5117, 5119, 5121, 5122, 5123, 5124, 5125, 5126, 5128, 5129, 5130, 5131, 5132, 5135, 5136, 5137, 5139, 5140, 5141, 5143, 5145, 5146, 5147, 5148, 5149, 5150, 5151, 5152, 5153, 5154, 5155, 5158, 5159, 5160, 5161, 5162, 5163, 5165, 5166, 5167, 5168, 5169, 5170, 5171, 5172, 5173, 5174, 5175, 5176, 5178, 5179, 5181, 5182, 5183, 5184, 5185, 5186, 5188, 5189, 5190, 5191, 5194, 5195, 5196, 5197, 5199, 5200, 5201, 5202, 5203, 5204, 5205, 5206, 5208, 5209, 5211, 5212, 5214, 5215, 5216, 5217, 5218, 5219, 5220, 5221, 5222, 5223, 5225, 5226, 5227, 5228, 5229, 5230, 5232, 5233, 5234, 5235, 5236, 5237, 5238, 5239, 5240, 5242, 5243, 5244, 5245, 5248, 5249, 5251, 5252, 5253, 5254, 5255, 5257, 5258, 5259, 5261, 5262, 5263, 5264, 5265, 5266, 5268, 5269, 5270, 5271, 5272, 5274, 5275, 5276, 5277, 5278, 5279, 5281, 5282, 5283, 5284, 5285, 5286, 5287, 5288, 5289, 5290, 5292, 5294, 5295, 5296, 5298, 5299, 5300, 5302, 5303, 5304, 5305, 5306, 5307, 5308, 5309, 5310, 5311, 5312, 5314, 5315, 5316, 5319, 5320, 5321, 5323, 5324, 5325, 5328, 5330, 5331, 5332, 5333, 5334, 5335, 5336, 5339, 5340, 5341, 5342, 5343, 5344, 5346, 5347, 5348, 5350, 5352, 5353, 5355, 5356, 5357, 5361, 5363, 5364, 5365, 5366, 5367, 5368, 5369, 5371, 5374, 5375, 5376, 5378, 5379, 5380, 5381, 5382, 5384, 5385, 5386, 5387, 5388, 5389, 5390, 5392, 5393, 5394, 5395, 5396, 5398, 5400, 5401, 5402, 5404, 5405, 5406, 5407, 5409, 5410, 5411, 5412, 5413, 5414, 5415, 5416, 5417, 5418, 5419, 5420, 5421, 5422, 5423, 5425, 5426, 5427, 5428, 5429, 5430, 5431, 5432, 5433, 5435, 5436, 5438, 5439, 5441, 5442, 5443, 5445, 5446, 5447, 5448, 5449, 5450, 5451, 5452, 5453, 5454, 5455, 5456, 5457, 5458, 5459, 5460, 5462, 5464, 5465, 5466, 5467, 5468, 5469, 5470, 5471, 5472, 5473, 5474, 5475, 5476, 5477, 5478, 5479, 5481, 5482, 5485, 5486, 5487, 5488, 5490, 5491, 5494, 5495, 5496, 5497, 5498, 5499, 5500, 5501, 5502, 5503, 5504, 5505, 5506, 5508, 5510, 5511, 5512, 5513, 5514, 5515, 5516, 5517, 5518, 5519, 5520, 5521, 5522, 5523, 5524, 5525, 5530, 5532, 5533, 5535, 5536, 5537, 5538, 5539, 5540, 5541, 5542, 5544, 5545, 5546, 5547, 5548, 5549, 5550, 5551, 5552, 5553, 5554, 5555, 5556, 5558, 5560, 5562, 5563, 5564, 5565, 5566, 5567, 5568, 5569, 5570, 5572, 5574, 5575, 5577, 5578, 5580, 5581, 5582, 5583, 5584, 5585, 5586, 5587, 5588, 5589, 5590, 5591, 5592, 5593, 5594, 5595, 5597, 5598, 5599, 5602, 5604, 5605, 5606, 5607, 5609, 5610, 5611, 5612, 5613, 5614, 5615, 5616, 5617, 5618, 5620, 5621, 5622, 5624, 5626, 5627, 5628, 5629, 5630, 5632, 5633, 5634, 5635, 5636, 5639, 5642, 5643, 5644, 5646, 5647, 5649, 5650, 5651, 5653, 5654, 5657, 5658, 5659, 5660, 5661, 5662, 5663, 5664, 5666, 5667, 5669, 5670, 5671, 5672, 5673, 5675, 5676, 5677, 5678, 5679, 5682, 5683, 5684, 5685, 5686, 5688, 5689, 5691, 5692, 5693, 5695, 5696, 5697, 5698, 5699, 5700, 5701, 5703, 5704, 5706, 5709, 5711, 5712, 5713, 5714, 5715, 5716, 5717, 5718, 5719, 5720, 5721, 5722, 5723, 5724, 5725, 5726, 5727, 5728, 5730, 5731, 5732, 5733, 5734, 5735, 5736, 5737, 5739, 5740, 5741, 5742, 5743, 5744, 5745, 5746, 5747, 5748, 5750, 5751, 5752, 5754, 5755, 5756, 5757, 5758, 5759, 5760, 5761, 5762, 5763, 5764, 5765, 5766, 5767, 5769, 5770, 5772, 5773, 5774, 5777, 5779, 5780, 5782, 5783, 5784, 5785, 5786, 5787, 5789, 5790, 5791, 5792, 5793, 5794, 5795, 5797, 5799, 5800, 5801, 5803, 5805, 5806, 5807, 5808, 5809, 5810, 5811, 5812, 5813, 5814, 5815, 5817, 5819, 5821, 5823, 5824, 5825, 5826, 5827, 5828, 5829, 5831, 5832, 5833, 5834, 5835, 5836, 5837, 5839, 5840, 5841, 5842, 5844, 5845, 5846, 5847, 5848, 5849, 5851, 5852, 5853, 5854, 5855, 5856, 5857, 5859, 5860, 5861, 5862, 5863, 5865, 5866, 5867, 5869, 5870, 5871, 5872, 5873, 5874, 5876, 5877, 5878, 5879, 5880, 5881, 5882, 5883, 5884, 5885, 5886, 5887, 5889, 5890, 5891, 5893, 5894, 5895, 5896, 5897, 5898, 5899, 5900, 5901, 5902, 5903, 5904, 5907, 5908, 5909, 5910, 5911, 5912, 5914, 5916, 5917, 5921, 5922, 5923, 5924, 5926, 5927, 5928, 5929, 5930, 5931, 5932, 5934, 5935, 5937, 5939, 5940, 5942, 5943, 5944, 5945, 5946, 5947, 5948, 5950, 5951, 5952, 5953, 5954, 5955, 5956, 5960, 5961, 5963, 5964, 5965, 5966, 5967, 5968, 5969, 5970, 5971, 5974, 5976, 5977, 5978, 5980, 5981, 5982, 5983, 5984, 5986, 5987, 5989, 5990, 5991, 5992, 5993, 5994, 5995, 5996, 5997, 5998, 5999, 6000, 6001, 6002, 6004, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6012, 6013, 6014, 6015, 6016, 6017, 6018, 6020, 6021, 6022, 6023, 6024, 6025, 6026, 6027, 6029, 6030, 6031, 6032, 6033, 6035, 6036, 6037, 6038, 6039, 6040, 6041, 6042, 6043, 6044, 6045, 6046, 6047, 6048, 6049, 6050, 6051, 6052, 6054, 6055, 6056, 6059, 6060, 6061, 6063, 6064, 6065, 6067, 6069, 6070, 6071, 6072, 6074, 6076, 6077, 6078, 6080, 6081, 6083, 6084, 6085, 6088, 6089, 6091, 6092, 6093, 6094, 6095, 6096, 6098, 6099, 6100, 6101, 6102, 6103, 6104, 6105, 6106, 6107, 6108, 6109, 6110, 6111, 6112, 6113, 6114, 6115, 6118, 6119, 6121, 6122, 6123, 6124, 6125, 6126, 6128, 6129, 6130, 6131, 6132, 6133, 6135, 6136, 6139, 6140, 6141, 6142, 6143, 6144, 6145, 6146, 6148, 6149, 6150, 6152, 6153, 6154, 6155, 6156, 6157, 6158, 6159, 6160, 6164, 6165, 6166, 6167, 6168, 6169, 6170, 6171, 6172, 6174, 6175, 6176, 6177, 6178, 6179, 6182, 6183, 6184, 6185, 6186, 6187, 6188, 6190, 6191, 6192, 6193, 6194, 6195, 6196, 6197, 6198, 6200], 'groups': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '66', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '66', '123', '124', '125', '126', '127', '128', '66', '130', '131', '132', '66', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199', '200', '201', '202', '203', '204', '205', '206', '207', '208', '209', '210', '211', '212', '213', '214', '215', '216', '217', '218', '219', '220', '221', '222', '223', '33', '225', '226', '227', '228', '229', '230', '231', '232', '233', '234', '235', '236', '237', '238', '239', '240', '241', '242', '243', '244', '245', '246', '247', '248', '249', '250', '251', '252', '253', '254', '112', '66', '257', '258', '259', '260', '261', '262', '263', '264', '265', '266', '267', '268', '269', '270', '271', '272', '273', '274', '275', '276', '277', '278', '279', '280', '281', '282', '283', '284', '285', '286', '287', '288', '289', '290', '291', '292', '293', '294', '295', '296', '297', '298', '299', '300', '301', '302', '303', '304', '305', '306', '307', '308', '309', '310', '311', '312', '313', '314', '315', '316', '317', '318', '319', '320', '321', '322', '323', '324', '325', '326', '327', '328', '329', '330', '331', '332', '333', '334', '335', '336', '337', '338', '339', '340', '341', '342', '343', '344', '345', '346', '347', '348', '349', '350', '351', '352', '353', '354', '355', '356', '357', '358', '359', '360', '361', '362', '363', '364', '365', '366', '367', '368', '369', '370', '371', '372', '373', '374', '375', '376', '377', '378', '379', '380', '381', '382', '383', '384', '385', '386', '387', '388', '389', '390', '391', '392', '393', '394', '395', '396', '397', '398', '399', '400', '401', '402', '403', '404', '405', '406', '407', '408', '409', '410', '411', '66', '66', '414', '415', '416', '417', '418', '419', '420', '421', '422', '423', '424', '425', '426', '427', '428', '429', '430', '431', '432', '433', '434', '435', '436', '437', '438', '439', '440', '441', '442', '443', '444', '445', '446', '447', '448', '449', '450', '451', '452', '453', '454', '455', '456', '457', '458', '459', '460', '461', '462', '463', '464', '465', '466', '467', '468', '469', '470', '471', '472', '473', '474', '475', '476', '477', '478', '479', '480', '481', '482', '483', '484', '485', '486', '487', '488', '489', '490', '491', '492', '493', '494', '495', '496', '497', '498', '499', '500', '501', '502', '503', '504', '505', '506', '507', '508', '509', '510', '511', '512', '513', '514', '515', '516', '517', '518', '519', '520', '521', '522', '523', '524', '525', '526', '527', '528', '529', '530', '531', '532', '533', '534', '535', '536', '537', '538', '539', '540', '541', '542', '543', '544', '545', '546', '547', '548', '549', '550', '551', '552', '553', '554', '555', '556', '557', '558', '559', '560', '561', '562', '563', '564', '565', '566', '567', '568', '569', '570', '571', '572', '573', '574', '575', '66', '577', '578', '579', '580', '581', '582', '583', '584', '585', '586', '587', '588', '589', '590', '591', '592', '593', '594', '595', '596', '597', '598', '599', '600', '601', '602', '603', '604', '605', '606', '607', '608', '609', '610', '611', '612', '613', '614', '615', '616', '617', '618', '619', '620', '621', '622', '623', '624', '625', '626', '627', '628', '629', '630', '631', '632', '633', '634', '635', '66', '637', '638', '639', '640', '641', '642', '643', '644', '645', '646', '647', '648', '649', '650', '651', '652', '653', '654', '655', '656', '657', '658', '659', '660', '661', '662', '663', '664', '665', '666', '667', '668', '669', '670', '66', '672', '673', '674', '675', '676', '677', '678', '679', '680', '681', '682', '683', '684', '685', '686', '687', '688', '689', '690', '691', '692', '693', '694', '695', '696', '697', '698', '699', '700', '701', '702', '703', '704', '705', '706', '707', '708', '709', '710', '711', '712', '713', '714', '715', '716', '717', '718', '719', '720', '721', '722', '723', '724', '725', '726', '727', '728', '729', '730', '731', '732', '733', '734', '735', '736', '737', '738', '739', '740', '741', '742', '743', '744', '745', '746', '747', '748', '749', '750', '751', '752', '753', '754', '755', '756', '757', '758', '759', '760', '761', '762', '476', '764', '765', '766', '767', '768', '769', '770', '771', '772', '773', '774', '775', '776', '777', '778', '779', '780', '66', '782', '783', '784', '785', '786', '787', '788', '789', '790', '791', '792', '793', '794', '795', '796', '797', '798', '799', '800', '801', '802', '803', '804', '805', '806', '807', '808', '809', '810', '811', '812', '66', '814', '815', '816', '817', '818', '819', '820', '821', '822', '823', '824', '66', '826', '827', '828', '829', '830', '831', '832', '833', '834', '835', '836', '837', '838', '839', '840', '841', '842', '843', '844', '845', '846', '847', '848', '849', '850', '851', '852', '853', '854', '855', '422', '857', '858', '859', '860', '861', '862', '863', '864', '865', '866', '867', '868', '869', '59', '871', '872', '873', '874', '875', '876', '877', '878', '879', '880', '881', '882', '883', '884', '885', '886', '887', '888', '889', '890', '891', '892', '893', '66', '895', '896', '897', '898', '899', '900', '901', '902', '903', '904', '905', '906', '907', '908', '909', '910', '911', '912', '913', '914', '915', '916', '917', '918', '919', '920', '921', '922', '923', '924', '925', '926', '927', '928', '929', '930', '931', '932', '933', '934', '935', '936', '937', '938', '939', '940', '941', '942', '943', '944', '945', '946', '947', '948', '949', '950', '951', '952', '953', '954', '955', '956', '957', '958', '959', '960', '961', '962', '963', '964', '965', '966', '967', '968', '969', '970', '971', '972', '973', '974', '975', '976', '977', '978', '979', '980', '981', '982', '983', '984', '985', '986', '987', '988', '989', '990', '991', '992', '993', '994', '995', '996', '997', '998', '999', '1000', '1001', '1002', '1003', '1004', '1005', '1006', '66', '1008', '1009', '1010', '1011', '1012', '1013', '1014', '1015', '1016', '806', '1018', '1019', '1020', '1021', '1022', '1023', '1024', '1025', '1026', '1027', '1028', '1029', '1030', '1031', '1032', '1033', '1034', '1035', '1036', '1037', '1038', '1039', '1040', '1041', '1042', '1043', '1044', '1045', '1046', '1047', '1048', '1049', '1050', '1051', '1052', '1053', '1054', '1055', '1056', '1057', '1058', '1059', '1060', '1061', '1062', '1063', '1064', '1065', '1066', '1067', '1068', '1069', '1070', '1071', '1072', '1073', '1074', '1075', '1076', '1077', '1078', '1079', '1080', '1081', '1082', '1083', '1084', '1085', '1086', '1087', '1088', '1089', '1090', '1091', '1092', '1093', '1094', '1095', '1096', '1097', '1098', '1099', '1100', '1101', '1102', '1103', '1104', '1105', '1106', '1107', '1108', '1109', '1110', '1111', '1112', '1113', '1114', '1115', '1116', '1117', '1118', '1119', '1120', '66', '1122', '1123', '1124', '1125', '1126', '1127', '1128', '1129', '1130', '1131', '1132', '1133', '1134', '1135', '1136', '1137', '1138', '1139', '1140', '1141', '1142', '1143', '1144', '1145', '1146', '1147', '1148', '1149', '1150', '1151', '1152', '1153', '1154', '1155', '1156', '1157', '1158', '1159', '1160', '1161', '1162', '1163', '1164', '1165', '1166', '1167', '1168', '1169', '1170', '1171', '1172', '1173', '1174', '1175', '1176', '1177', '1178', '1179', '1180', '1181', '1182', '1183', '1184', '1185', '1186', '1187', '1188', '1189', '1190', '1191', '1192', '1193', '1194', '1195', '1196', '1197', '1143', '1199', '1200', '1201', '1202', '1203', '1204', '1205', '1206', '1207', '1208', '1209', '1210', '1211', '1212', '1213', '1214', '1215', '1216', '1217', '1218', '1219', '1220', '1221', '1222', '1223', '1224', '1225', '1226', '1227', '1228', '1229', '1230', '1231', '1232', '1233', '1234', '1235', '1236', '1237', '1238', '1239', '1240', '1241', '1242', '1243', '1244', '1245', '1246', '1247', '1248', '1249', '1250', '1251', '42', '1253', '1254', '1255', '1256', '1257', '1258', '1259', '1260', '1261', '1262', '1263', '1264', '1265', '1266', '1267', '1268', '1269', '1270', '1271', '1272', '1273', '1274', '1275', '1276', '1277', '1278', '1279', '1280', '346', '1282', '1283', '1284', '1285', '1286', '1287', '1288', '1289', '1290', '1291', '1292', '1293', '1294', '1295', '1296', '1297', '1298', '1299', '1300', '1301', '1302', '1303', '1304', '1305', '1306', '1307', '1308', '1309', '1310', '326', '1312', '1313', '1314', '1315', '1316', '1317', '1318', '1319', '1320', '1321', '1322', '1323', '1324', '1325', '1326', '1327', '1328', '1329', '1330', '1331', '1332', '1333', '1334', '1335', '1336', '1337', '1338', '1339', '1340', '1341', '1342', '1343', '1344', '1345', '1346', '1347', '1348', '1349', '1350', '1351', '1352', '1353', '1354', '1355', '1356', '1357', '1358', '1359', '1360', '1361', '1362', '1363', '1364', '1365', '1366', '1129', '1368', '1369', '1370', '1371', '1372', '1373', '1374', '1375', '728', '1377', '1378', '1379', '1380', '1381', '1382', '1383', '1384', '1385', '66', '1387', '1388', '1389', '1390', '1391', '1392', '1393', '1394', '1395', '1396', '1397', '1398', '1399', '1400', '1401', '1402', '1403', '1404', '1405', '1406', '267', '1408', '1409', '1410', '1411', '1412', '1413', '1414', '1415', '1416', '1417', '1418', '1419', '1420', '1421', '1422', '1423', '1424', '66', '1426', '1427', '1428', '1429', '1430', '1431', '1432', '1433', '1434', '1435', '1436', '1437', '1438', '1337', '1440', '1441', '1442', '1443', '1444', '1445', '1446', '1447', '1448', '1449', '1450', '1451', '1452', '1453', '1454', '1455', '1456', '1457', '1458', '1459', '66', '1461', '1462', '1463', '1464', '1465', '1466', '1467', '1468', '1469', '1470', '1471', '1472', '1473', '1474', '1475', '1476', '1477', '1478', '1479', '66', '1481', '1482', '1483', '1484', '1485', '1486', '1487', '1065', '1489', '1490', '1491', '1492', '1493', '1494', '1495', '1496', '1497', '450', '1499', '1500', '1501', '1502', '1503', '1504', '1505', '1506', '1507', '1508', '1509', '1510', '1511', '1512', '1513', '1514', '1515', '1516', '1517', '1518', '1519', '1520', '1521', '1522', '1523', '1524', '1525', '1526', '1527', '1528', '1529', '1530', '1531', '1532', '1533', '1534', '1535', '1536', '1537', '1538', '1539', '1540', '1541', '1542', '1543', '1544', '1545', '1546', '1547', '1548', '1549', '1550', '1551', '1552', '1553', '1554', '66', '1556', '1557', '1558', '1559', '1560', '1561', '1562', '1563', '1564', '1565', '1566', '1567', '1568', '1569', '1570', '1571', '1572', '1573', '300', '1575', '1576', '1577', '811', '1579', '1580', '1581', '1582', '1583', '1584', '1585', '1586', '66', '1588', '1589', '1590', '1591', '1592', '867', '151', '1595', '1596', '1597', '1598', '1599', '1600', '1601', '1602', '1603', '1604', '1605', '1606', '1576', '1608', '1609', '1610', '1611', '1612', '1613', '1614', '1615', '1616', '1617', '1618', '1619', '1620', '1621', '1622', '1623', '1624', '1625', '1626', '1627', '1628', '1629', '1630', '1631', '1632', '1633', '1634', '1635', '1636', '1637', '1638', '1639', '1640', '1641', '1642', '1643', '1644', '1645', '1646', '1647', '1648', '1649', '1650', '1651', '1652', '1653', '1654', '1655', '1656', '1657', '1658', '1659', '1660', '1661', '1662', '1663', '1664', '1665', '1666', '1667', '1668', '1669', '1670', '1671', '1672', '1341', '1674', '1675', '1676', '1677', '1678', '1679', '1680', '1681', '1682', '1683', '1684', '1685', '1686', '1687', '1688', '1689', '1690', '1691', '1692', '1351', '1694', '1695', '1696', '1697', '1698', '1699', '1700', '1701', '1702', '1703', '1704', '1705', '1706', '1707', '1272', '1709', '1710', '1711', '1712', '1713', '1714', '1715', '1716', '1717', '1718', '1719', '1720', '1721', '1722', '1723', '1724', '1725', '1726', '1727', '1728', '1729', '1730', '1731', '1732', '1733', '1734', '1735', '1736', '1737', '1738', '1739', '1740', '1741', '1742', '1743', '1744', '1745', '1746', '1747', '1748', '1749', '1750', '1751', '1752', '66', '1754', '1755', '1756', '1757', '1758', '66', '1760', '1761', '1762', '1763', '1764', '1765', '1766', '1767', '1768', '1769', '1770', '1771', '977', '1773', '1774', '1775', '1776', '1777', '1778', '1056', '1780', '1781', '1782', '1783', '1784', '1785', '66', '1787', '1788', '1789', '1790', '1791', '1792', '1793', '1794', '1795', '1796', '1797', '1798', '1799', '1800', '1801', '1802', '1803', '1804', '1805', '1806', '1807', '66', '1809', '1810', '1811', '1812', '1813', '1814', '1815', '1816', '1817', '1818', '1819', '1820', '1821', '1822', '1823', '1824', '1825', '1826', '1827', '1828', '1829', '1830', '1831', '1832', '1833', '1834', '1835', '1836', '1837', '1838', '1839', '1840', '1841', '1842', '1843', '1844', '1845', '1846', '1847', '1848', '1849', '1850', '1851', '1852', '1853', '1313', '1855', '1856', '1857', '1858', '1859', '1860', '1861', '1862', '1863', '1864', '1865', '1866', '1867', '1868', '1869', '1870', '1871', '1872', '1873', '1874', '1875', '1876', '1877', '1878', '1879', '1880', '1881', '1152', '1883', '1884', '1885', '1886', '1887', '1888', '1889', '1890', '1891', '1892', '1893', '1894', '1895', '1896', '1897', '1898', '1899', '790', '1901', '1902', '1903', '1904', '1905', '1906', '722', '1908', '1909', '1910', '1911', '1912', '34', '1914', '1915', '1916', '1917', '1918', '1919', '1920', '1921', '1922', '1923', '1924', '1925', '1926', '1927', '1928', '1929', '1790', '1931', '1723', '1933', '1934', '1935', '1936', '1937', '1938', '1939', '1940', '1941', '1942', '1943', '1944', '1945', '1946', '1947', '1948', '1949', '1950', '1951', '1952', '1953', '1954', '1955', '1956', '1957', '1958', '1959', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '1999', '2022', '2023', '2024', '2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '787', '2035', '2036', '2037', '2038', '2039', '2040', '2041', '2042', '811', '2044', '2045', '2046', '2047', '2048', '2049', '2050', '2051', '2052', '2053', '2054', '2055', '2056', '2057', '2058', '2059', '2060', '2061', '2062', '2063', '2064', '2065', '2066', '2067', '2068', '2069', '2070', '2071', '2072', '2073', '2074', '2075', '2076', '2077', '2078', '2079', '2080', '2081', '66', '2083', '2084', '2085', '2086', '2087', '61', '2089', '2090', '2091', '2092', '2093', '2094', '2095', '2096', '2097', '225', '2099', '2100', '2101', '2102', '2103', '2104', '2105', '2106', '2107', '2108', '2109', '2110', '2111', '2112', '2113', '2114', '2115', '2116', '2117', '2118', '2119', '2120', '2121', '278', '2123', '2124', '2125', '2126', '2127', '2128', '2129', '2130', '2131', '2132', '2133', '2134', '2135', '2136', '2137', '2138', '2139', '2140', '1888', '2142', '2143', '2144', '2145', '2146', '2147', '2148', '2149', '1829', '2151', '2152', '2153', '2154', '2155', '2156', '2157', '2158', '2159', '2160', '2161', '2162', '2163', '2164', '2165', '2166', '2167', '2168', '2169', '2170', '2171', '2172', '2173', '2174', '2175', '833', '2177', '2178', '2179', '2180', '2181', '2182', '2183', '2184', '2185', '2186', '2187', '2188', '2189', '2190', '2191', '2192', '2193', '2194', '2195', '2196', '2197', '2198', '2199', '2200', '1482', '2202', '2203', '2204', '2205', '2206', '2207', '2208', '2209', '2210', '2211', '2212', '2213', '2214', '2215', '2216', '2217', '2218', '2219', '2220', '2221', '2222', '69', '2224', '2225', '2226', '811', '2228', '2229', '2230', '2231', '2232', '2233', '2234', '2235', '1062', '2237', '2238', '2239', '2240', '2241', '2242', '2243', '2244', '2245', '2246', '2247', '2248', '2249', '2250', '2251', '2252', '2253', '1671', '2255', '2256', '2257', '2258', '2259', '2260', '2261', '2262', '1910', '2264', '2265', '2266', '2267', '2268', '2269', '2270', '2271', '2272', '2273', '2274', '2275', '2276', '2277', '1535', '2279', '2280', '2281', '2282', '2283', '66', '2285', '2286', '2287', '2288', '2289', '2256', '2291', '2292', '1711', '2294', '66', '2296', '2297', '2298', '2299', '2300', '2301', '2302', '2303', '2304', '2305', '2306', '2307', '2308', '2309', '2310', '2311', '2312', '2313', '2314', '2315', '2316', '66', '2318', '2319', '2320', '2321', '2322', '2323', '2324', '2325', '2326', '2327', '2328', '2329', '2330', '2331', '2332', '2333', '2334', '2335', '2336', '2337', '2338', '2339', '2340', '2341', '2342', '2343', '2344', '2345', '2346', '2347', '2348', '2349', '2350', '796', '2352', '1216', '2354', '2355', '811', '2357', '2358', '905', '218', '2361', '2362', '2363', '2364', '2365', '2366', '2367', '2368', '2369', '2370', '986', '2372', '2373', '2374', '2375', '2376', '680', '2378', '2379', '2380', '2381', '2382', '2383', '2384', '2385', '2386', '2387', '2388', '2389', '2390', '2391', '2392', '2393', '841', '2395', '2396', '2397', '2398', '2399', '2400', '2401', '2402', '2403', '2404', '2405', '2406', '2407', '2408', '2409', '2410', '2411', '2412', '2413', '2414', '2415', '2416', '2417', '2418', '2419', '2420', '2421', '2422', '2423', '2424', '2425', '66', '2427', '2428', '2429', '2430', '2431', '2432', '2433', '2434', '2435', '2436', '2437', '2438', '2439', '2440', '2441', '2442', '2443', '2444', '2445', '2446', '2447', '2448', '2449', '2450', '2451', '2452', '2453', '2454', '2455', '2456', '2457', '2458', '2459', '2460', '2461', '2462', '2463', '2464', '2465', '2466', '2467', '2468', '2469', '2470', '1506', '2472', '2473', '2474', '2475', '2476', '2477', '2478', '2479', '2480', '2481', '2482', '2483', '2484', '2485', '2486', '2487', '2488', '2489', '2490', '2491', '2492', '2493', '1937', '2495', '2496', '2497', '206', '2499', '2500', '2501', '2502', '2503', '2504', '2505', '2506', '2507', '2508', '2509', '2510', '2511', '2512', '2513', '2514', '66', '2516', '2517', '2518', '2519', '2520', '2521', '2522', '2523', '2524', '2525', '2526', '2527', '2528', '2529', '2530', '66', '2532', '2533', '2534', '2535', '2536', '2537', '2538', '2539', '2540', '2541', '2542', '2543', '2544', '66', '2546', '2547', '2548', '2549', '2550', '2551', '2552', '2553', '2554', '2555', '2556', '2557', '2558', '2559', '2560', '2561', '2562', '2563', '2564', '2565', '2566', '2567', '2568', '2569', '2570', '2571', '2572', '2573', '2574', '2575', '2576', '2577', '2578', '2579', '2580', '2581', '2582', '2583', '2584', '2585', '2586', '2587', '2588', '2589', '2590', '2591', '2592', '2593', '2594', '2595', '2596', '2597', '2598', '2599', '2600', '2601', '2602', '2603', '2604', '2605', '2606', '2607', '2608', '2609', '1389', '2611', '2612', '66', '2614', '2615', '2616', '1669', '2594', '2619', '2620', '2621', '2622', '2623', '2624', '2625', '2626', '2627', '66', '2629', '66', '2631', '2632', '2633', '2634', '2635', '2636', '2637', '2638', '66', '2640', '2641', '2642', '2643', '2644', '2645', '66', '2647', '2648', '2649', '2650', '2651', '2652', '2653', '2654', '2655', '2656', '2657', '2658', '2659', '2660', '2661', '2662', '2663', '2664', '2665', '2666', '2667', '2668', '2669', '2670', '2671', '2672', '1728', '2674', '2675', '2676', '2677', '2678', '2679', '2680', '2681', '2682', '2683', '2684', '2685', '2686', '2687', '2688', '2689', '2690', '2691', '2692', '2693', '2569', '2695', '2696', '2697', '2698', '2699', '2700', '2701', '2702', '2703', '2704', '2705', '2706', '2707', '2708', '2709', '2710', '2711', '2712', '2713', '2714', '2715', '2716', '2717', '2718', '290', '2720', '2721', '2722', '2723', '2724', '2725', '2726', '2727', '2728', '2729', '2730', '2731', '2732', '2716', '2734', '2735', '2736', '2737', '2738', '2739', '2740', '2741', '2742', '2743', '2744', '2745', '2746', '2747', '2748', '2548', '2750', '2751', '2752', '811', '2754', '2755', '2756', '2757', '2758', '2759', '2760', '2761', '2762', '2763', '2764', '2765', '2766', '2767', '2768', '2769', '2770', '2771', '2772', '2773', '2774', '2775', '66', '2777', '66', '2779', '2780', '2781', '2782', '2783', '2784', '2785', '2786', '2787', '2788', '1031', '2790', '2791', '2792', '2793', '66', '2795', '66', '2797', '2798', '2142', '2800', '2801', '2802', '2803', '2804', '2805', '2806', '2807', '2808', '373', '2810', '2811', '2812', '2813', '2814', '2815', '2816', '2817', '2818', '2819', '2820', '2821', '2822', '66', '2824', '2825', '2826', '2827', '507', '2829', '2830', '2831', '2832', '2833', '2834', '2835', '2836', '2837', '2838', '2839', '2840', '2841', '2842', '2843', '2844', '2845', '2846', '2847', '2848', '2849', '535', '2851', '2852', '453', '2854', '2855', '2856', '2857', '2858', '2859', '2860', '2861', '2862', '2863', '2864', '2865', '1300', '2867', '631', '2869', '2870', '2871', '2872', '2873', '121', '2875', '2876', '2877', '2878', '2879', '2880', '2881', '2882', '2883', '2884', '2885', '2886', '2887', '2888', '2889', '2890', '43', '2892', '835', '2894', '2895', '2192', '2897', '811', '2899', '2900', '2901', '2902', '2903', '2904', '2905', '2906', '2907', '2908', '2909', '2910', '2911', '2912', '2913', '2914', '2915', '381', '2917', '2918', '2919', '2920', '2921', '2922', '2923', '1611', '228', '2730', '2927', '1094', '2929', '1949', '2931', '2932', '2933', '2934', '2935', '2936', '2937', '2938', '2939', '2940', '2941', '2942', '2943', '2944', '2945', '2946', '2947', '2948', '2949', '2950', '2951', '2952', '2953', '2954', '2955', '1669', '2957', '2958', '2959', '2960', '2961', '2962', '2963', '2964', '2965', '2966', '2967', '2968', '2969', '2970', '1799', '2972', '2973', '2974', '2975', '2976', '2977', '2767', '2979', '2980', '2981', '2982', '2983', '2984', '2985', '2986', '2987', '2988', '2989', '2990', '2991', '2992', '2993', '2994', '2995', '2996', '2997', '2998', '2999', '3000', '3001', '3002', '3003', '3004', '3005', '3006', '3007', '3008', '3009', '3010', '3011', '3012', '3013', '2821', '3015', '66', '3017', '3018', '3019', '3020', '3021', '3022', '3023', '1731', '3025', '3026', '3027', '3028', '3029', '3004', '3031', '3032', '3033', '3034', '3035', '3036', '2793', '3038', '3039', '157', '3041', '3042', '3043', '3044', '2565', '3046', '3047', '3048', '3049', '3050', '3051', '3052', '3053', '3054', '66', '3056', '3057', '3058', '3059', '3060', '3061', '3062', '3063', '1371', '3065', '3066', '3067', '638', '3069', '3070', '3071', '3072', '3073', '3074', '3075', '3076', '3077', '3078', '3079', '1669', '3081', '3082', '3083', '3084', '3085', '3086', '3087', '3088', '3089', '3090', '3091', '3092', '66', '3094', '3095', '3096', '3097', '2214', '3099', '3100', '3101', '3102', '3103', '3104', '3105', '3106', '3107', '66', '3109', '3110', '3111', '3112', '3113', '3114', '3115', '3116', '3117', '3118', '3119', '3120', '3121', '3122', '3123', '3124', '3125', '3126', '3127', '3128', '3129', '3130', '3131', '3132', '3133', '3134', '3135', '3136', '3137', '3138', '3139', '3140', '3141', '3142', '3143', '3144', '3145', '3146', '3147', '3148', '2408', '3150', '3151', '3152', '3153', '3154', '3155', '3156', '3157', '3158', '3159', '3160', '3161', '3162', '3163', '3164', '3165', '3166', '3167', '3168', '3169', '3170', '3171', '3172', '3173', '3174', '3175', '3176', '3177', '3076', '3179', '3180', '3181', '3182', '3183', '3184', '1684', '2901', '3187', '3188', '3189', '3190', '3191', '3192', '3193', '3194', '3195', '3196', '3197', '3198', '3199', '3200', '3201', '3202', '3203', '3204', '3205', '3206', '3207', '3208', '3209', '3210', '3211', '3212', '3213', '3214', '3215', '3216', '3217', '3218', '3219', '3220', '3221', '1320', '3223', '3224', '3066', '3226', '3227', '3228', '3229', '3230', '3231', '3232', '3233', '3234', '3235', '2321', '3237', '3238', '3239', '3240', '3241', '3242', '3243', '3244', '3245', '3246', '2074', '3248', '3249', '3250', '3251', '3252', '3253', '3254', '3255', '811', '3257', '66', '3259', '3260', '3261', '3262', '3263', '3264', '2168', '3266', '3267', '3268', '3269', '3270', '3271', '1360', '2512', '3274', '3275', '3276', '3277', '3278', '3279', '3280', '3281', '2634', '3283', '3284', '3285', '3286', '3287', '3288', '3289', '3290', '3291', '3292', '3293', '3294', '3295', '3296', '3297', '3298', '3299', '3300', '3301', '3302', '3303', '3304', '3305', '330', '3307', '3308', '3309', '3310', '3311', '3312', '1851', '3314', '3315', '3316', '3317', '3318', '3319', '3320', '454', '3322', '3323', '66', '3325', '3326', '3327', '3328', '3329', '3330', '3331', '3332', '3333', '3334', '3335', '3336', '3337', '3338', '1441', '3340', '3341', '3342', '2620', '3344', '3345', '3346', '3347', '3348', '3349', '3350', '3351', '3352', '3353', '3354', '3355', '3356', '3357', '3358', '3359', '3360', '3361', '3362', '3363', '3364', '3365', '3366', '3367', '3368', '3369', '3370', '3371', '3372', '3373', '3374', '3375', '3376', '3377', '2451', '3379', '3380', '3381', '3382', '3383', '3384', '3385', '3386', '3387', '3388', '2204', '3390', '3391', '3392', '3393', '202', '1619', '3396', '3397', '3398', '3399', '3400', '3401', '3402', '3403', '3404', '3405', '3406', '3407', '3408', '3409', '3410', '3411', '2555', '3413', '3414', '3415', '3416', '3417', '3418', '3419', '3420', '3421', '3422', '3423', '3424', '3425', '3426', '3427', '3428', '3429', '3430', '3431', '3432', '3433', '3434', '3435', '3436', '3437', '3438', '3439', '3440', '3441', '3442', '3443', '3444', '3445', '3446', '3447', '3448', '3449', '3450', '3451', '3452', '3453', '1357', '3455', '3456', '3457', '3458', '3459', '3460', '3461', '3462', '3463', '3464', '3465', '3466', '3467', '3468', '66', '3470', '3471', '3472', '3473', '3474', '3475', '66', '3477', '3478', '3479', '3480', '3481', '3482', '3483', '3484', '3485', '3486', '3487', '3488', '3489', '3490', '3491', '3492', '3493', '2660', '3495', '3496', '3497', '1639', '3499', '3500', '3501', '3502', '3503', '3504', '3505', '3506', '3507', '3266', '3509', '3510', '3511', '3512', '3513', '3514', '3515', '3516', '3517', '3518', '3519', '3520', '3521', '2951', '3523', '3524', '3525', '3526', '3527', '3528', '3529', '3530', '3531', '3532', '3533', '3534', '3535', '3536', '3537', '1118', '3539', '3540', '3541', '3542', '3543', '3544', '3545', '3546', '3547', '3548', '3549', '2269', '3551', '3552', '3553', '3554', '3555', '3556', '3557', '3558', '3559', '3560', '3561', '3562', '3563', '3564', '3565', '3566', '3567', '3568', '3569', '3094', '3571', '3572', '3573', '3574', '3575', '3576', '3577', '1873', '3579', '3580', '3581', '3582', '66', '3584', '3585', '3586', '3587', '3588', '3589', '3590', '3591', '3592', '3593', '2615', '3595', '2506', '66', '3598', '3599', '3600', '3601', '3602', '3603', '3604', '3605', '3606', '3607', '3608', '3609', '3610', '3611', '3612', '3613', '709', '3615', '3616', '3617', '3618', '3619', '3620', '3621', '3622', '3623', '3624', '3625', '3626', '3627', '3628', '3629', '3630', '3631', '3632', '3633', '3634', '3635', '3636', '893', '3638', '3639', '3640', '3641', '3642', '3643', '3644', '3645', '3646', '3647', '3648', '3649', '3650', '3651', '3652', '3653', '3654', '3655', '3656', '3657', '3658', '3659', '3660', '3661', '3662', '66', '3664', '3665', '3666', '1691', '66', '3669', '3670', '3671', '3672', '3673', '3674', '3675', '3676', '3677', '3678', '3679', '3680', '3681', '3682', '3683', '3684', '3685', '3686', '3687', '3688', '3689', '3690', '3691', '3692', '3693', '3694', '3695', '3696', '3697', '3698', '3699', '3700', '3701', '3702', '3703', '3704', '811', '3706', '3707', '3708', '3709', '3710', '3711', '3712', '3713', '3714', '3715', '3716', '3717', '3718', '3719', '3720', '3721', '3722', '3365', '3724', '3725', '3726', '3727', '3728', '3729', '3730', '3731', '3732', '1722', '3734', '3735', '3736', '2078', '2960', '3739', '3740', '3741', '3742', '3743', '3043', '3745', '3746', '3747', '3748', '3749', '3750', '2476', '3752', '3753', '3754', '3530', '3756', '3757', '3758', '3759', '3760', '3761', '3762', '3763', '3764', '3765', '3766', '3767', '3768', '3769', '724', '3771', '3772', '3773', '3774', '3775', '492', '3777', '3778', '3779', '3780', '3781', '2736', '1344', '3784', '3785', '3786', '3787', '3788', '66', '3790', '3791', '1943', '3793', '3794', '3795', '3796', '3797', '3798', '3799', '3800', '3801', '1332', '3803', '3804', '2004', '3806', '3807', '3808', '805', '3810', '3811', '3812', '3813', '3814', '3815', '3816', '3817', '3818', '3819', '3820', '3821', '3822', '3823', '3824', '3825', '3826', '3827', '3828', '3829', '3830', '3831', '3832', '3833', '3834', '3835', '3836', '3837', '3838', '3839', '3840', '3841', '3842', '1897', '3844', '3845', '1432', '3847', '3848', '3849', '3850', '3851', '3852', '3853', '3854', '2461', '3856', '3857', '3858', '3859', '3860', '3861', '3862', '3863', '3864', '3865', '3866', '3867', '3868', '3869', '3870', '3871', '3872', '3873', '3874', '3875', '3876', '3877', '3878', '3879', '3880', '3881', '3882', '3883', '3884', '1256', '712', '3887', '3888', '3889', '3890', '3891', '3892', '3893', '3894', '3895', '3896', '3897', '3898', '3899', '3900', '3901', '3902', '3903', '66', '3905', '904', '3907', '3908', '3909', '3910', '3911', '3912', '3913', '3914', '3915', '1374', '3917', '3918', '3919', '3920', '3465', '3922', '3923', '3924', '3925', '3926', '3927', '3928', '3929', '3930', '3931', '3345', '3933', '3934', '3935', '3936', '811', '3938', '3939', '3940', '580', '3942', '3943', '3944', '3945', '1608', '3947', '3948', '3949', '3950', '3951', '3952', '3953', '662', '3955', '3956', '593', '500', '3959', '66', '3961', '3962', '2535', '3964', '3965', '3966', '3967', '3968', '811', '3970', '3971', '3972', '3973', '3974', '3975', '3976', '3977', '3978', '3979', '3980', '3981', '546', '3983', '3984', '3985', '3986', '3987', '3988', '3989', '3990', '169', '3992', '3993', '3994', '3995', '3996', '3997', '3998', '3999', '4000', '4001', '4002', '4003', '66', '4005', '4006', '4007', '568', '4009', '4010', '850', '4012', '4013', '4014', '4015', '4016', '4017', '4018', '4019', '4020', '4021', '4022', '4023', '4024', '1356', '4026', '4027', '4028', '4029', '2437', '4031', '4032', '3274', '313', '4035', '1356', '4037', '4038', '4039', '4040', '799', '4042', '4043', '66', '4045', '4046', '4047', '4048', '4049', '3056', '4051', '4052', '4053', '4054', '4055', '2241', '4057', '1868', '1775', '4060', '4061', '4062', '4063', '3356', '4065', '4066', '4067', '471', '4069', '4070', '4071', '4072', '1197', '4074', '4075', '1933', '4077', '3438', '2981', '4080', '4081', '4082', '4083', '1628', '4085', '4086', '4087', '4088', '4089', '4090', '4091', '4092', '4093', '4094', '4095', '4096', '4097', '4098', '4099', '4100', '4101', '4102', '4103', '423', '4105', '4106', '4107', '4108', '4109', '2815', '4111', '4112', '640', '4114', '4115', '2341', '4117', '4118', '4119', '4120', '4121', '66', '2852', '4124', '4125', '4126', '4127', '4128', '4129', '4130', '4131', '4132', '4133', '4134', '4135', '4136', '4137', '4138', '4139', '2878', '2566', '4142', '4143', '4144', '4145', '4146', '581', '802', '3520', '4150', '4151', '4152', '4153', '4154', '4155', '4156', '4157', '4158', '4159', '4160', '4161', '4162', '4163', '4164', '4165', '4166', '4167', '4168', '4169', '4170', '4171', '4172', '419', '4174', '4175', '4176', '4177', '4178', '4179', '4180', '4181', '4182', '4183', '4184', '4185', '4186', '4187', '4188', '4189', '4190', '4191', '4192', '4193', '4194', '4195', '4196', '4197', '4198', '2607', '4200', '4201', '3693', '4203', '4204', '4205', '4206', '52', '4208', '4209', '4210', '4211', '4212', '4213', '4214', '4215', '4216', '4217', '4218', '4219', '4220', '4221', '4222', '4223', '4224', '4225', '4226', '4227', '4228', '4229', '4230', '892', '4232', '4233', '149', '4235', '4236', '4237', '4238', '4239', '4240', '4241', '4242', '4243', '4244', '811', '4246', '4247', '4248', '4249', '4250', '4251', '4252', '4253', '4254', '4255', '2160', '4257', '1433', '4259', '4260', '4261', '4262', '4263', '4264', '4265', '4266', '3947', '4268', '4269', '4270', '4271', '2120', '4273', '4274', '4275', '4276', '4277', '4278', '2195', '4280', '4281', '4282', '4283', '4284', '4285', '4286', '2418', '4288', '4289', '4290', '4291', '4292', '4293', '4294', '4295', '2101', '4297', '4298', '4299', '4300', '4301', '4302', '4303', '4304', '4305', '4306', '4307', '4308', '4309', '66', '4311', '4312', '4313', '4314', '4315', '4316', '3468', '4318', '4319', '4320', '4321', '4322', '3974', '4324', '4325', '4326', '4327', '4328', '2331', '4330', '4331', '4332', '4333', '4334', '4335', '4336', '4337', '4338', '4339', '4340', '4341', '4342', '4343', '4344', '4345', '4346', '4347', '4348', '4349', '4350', '4351', '4352', '4353', '4354', '4355', '3901', '4357', '4358', '4359', '4360', '4361', '4362', '4363', '4364', '4365', '2431', '4367', '4368', '4369', '4370', '4371', '4372', '2143', '4374', '347', '3759', '4377', '4378', '4379', '4380', '4381', '4382', '4383', '4384', '4385', '4386', '4387', '4388', '66', '3980', '4391', '4392', '4393', '4394', '4395', '4396', '2720', '4398', '4399', '4400', '4401', '4402', '4403', '2580', '4405', '4406', '4407', '4408', '4409', '3609', '4411', '4412', '4413', '4414', '4415', '4416', '4417', '4418', '4419', '4420', '66', '4422', '4423', '4424', '4425', '4426', '4427', '4428', '987', '4430', '4431', '1243', '4433', '4434', '4435', '4436', '4437', '4438', '4439', '4440', '1156', '4442', '4443', '4444', '4445', '4446', '4447', '4448', '4449', '4450', '3392', '3360', '4453', '4454', '4455', '4456', '4457', '4458', '4459', '4460', '4461', '4462', '4463', '4464', '4465', '4466', '66', '4468', '4469', '833', '4471', '4472', '4473', '4474', '4475', '4476', '4477', '4478', '4479', '4480', '4481', '4482', '1105', '4484', '4485', '4486', '4487', '4488', '4489', '4490', '4491', '4492', '4493', '4494', '1069', '4496', '4497', '4498', '4499', '4500', '4501', '4502', '4503', '4504', '4505', '4506', '3536', '4508', '4509', '4510', '2235', '4512', '4513', '4514', '4515', '4516', '376', '4135', '4519', '4520', '4521', '4522', '4523', '740', '4525', '4526', '4527', '3754', '4529', '4530', '4531', '4532', '4533', '4534', '4535', '4536', '4537', '2605', '4539', '4540', '4541', '4542', '4543', '4544', '4545', '4546', '2336', '4548', '4549', '574', '4551', '1895', '1437', '4554', '4555', '2755', '4557', '4558', '3370', '4560', '4561', '1571', '3881', '4564', '4565', '4566', '4567', '4568', '2590', '4570', '1975', '66', '683', '1953', '4575', '4576', '4577', '4578', '4579', '4580', '4581', '4582', '4583', '852', '4585', '4586', '4587', '4588', '4589', '4590', '4591', '4592', '4593', '4594', '4595', '4596', '4597', '1599', '4599', '4600', '4601', '4602', '4052', '4604', '2423', '4606', '4607', '4608', '4609', '3300', '4611', '4612', '31', '4614', '4615', '4616', '4617', '4618', '4619', '4620', '4621', '4622', '4623', '4624', '4625', '4626', '4627', '4628', '4629', '4630', '4631', '1396', '4633', '4634', '2952', '4636', '4637', '4638', '4639', '4640', '4641', '1660', '3602', '776', '1066', '4646', '4647', '4648', '4649', '4650', '4651', '1791', '4653', '4654', '66', '4656', '4657', '4658', '4659', '1249', '4661', '4662', '4663', '4664', '4665', '4666', '4667', '4668', '4669', '4670', '1991', '4672', '4673', '4674', '4675', '4676', '811', '630', '1427', '4680', '4681', '4682', '4683', '4684', '4685', '66', '357', '4688', '4689', '4690', '4691', '4692', '4693', '4694', '1061', '4696', '4697', '4698', '4699', '4700', '4701', '4702', '4703', '4704', '4705', '4706', '4707', '4708', '4709', '4710', '1527', '4712', '779', '4714', '4715', '4716', '2280', '4718', '4719', '4720', '4721', '4722', '4723', '4724', '4725', '4530', '4727', '4728', '4729', '4730', '4731', '4732', '4733', '4734', '4735', '4736', '2909', '2895', '4739', '4740', '854', '4742', '4743', '4744', '4745', '4746', '4747', '4748', '4749', '4750', '1824', '4752', '4753', '4754', '4755', '4756', '4757', '4758', '4759', '4760', '1280', '4762', '4763', '4764', '4765', '4766', '4767', '4768', '4769', '4770', '4771', '4772', '4773', '4774', '4775', '1052', '4028', '4778', '4779', '4780', '4781', '4782', '3535', '4784', '4785', '4786', '4787', '66', '2857', '4790', '4791', '4792', '537', '4794', '4795', '4796', '4797', '4798', '4799', '4800', '4801', '4802', '3285', '4804', '4805', '4806', '4807', '4808', '4809', '811', '4811', '4812', '4813', '4814', '4815', '4816', '4817', '4818', '4819', '4820', '4821', '4822', '66', '4824', '4825', '4826', '4827', '4828', '4829', '4830', '4831', '4832', '4833', '4834', '4835', '4836', '4648', '4838', '4839', '4840', '4841', '4842', '4843', '966', '4845', '4846', '673', '4848', '4849', '4850', '4851', '4852', '4853', '4854', '2798', '4856', '4857', '4858', '4859', '4860', '4861', '4862', '4863', '4864', '4865', '4866', '4867', '4868', '4869', '4870', '4871', '4872', '4873', '4874', '4875', '4876', '4877', '4878', '66', '4880', '929', '4882', '4883', '2056', '4885', '4886', '3576', '4888', '4889', '4890', '3062', '4892', '4893', '4894', '4895', '4896', '1657', '2745', '4899', '4900', '4901', '4902', '4903', '66', '4905', '4906', '4907', '4908', '4909', '4910', '4911', '4912', '4913', '4914', '4915', '1919', '4917', '4918', '4919', '4920', '4921', '4922', '4923', '4924', '2444', '4926', '4927', '4928', '4929', '4930', '1537', '4932', '4933', '4934', '4935', '4936', '4937', '4938', '4939', '4940', '66', '4942', '4943', '693', '4945', '4946', '4947', '4948', '4949', '2097', '1730', '4952', '4953', '4954', '4955', '4956', '4957', '4589', '2419', '4960', '4961', '4962', '4963', '4964', '4965', '4966', '4967', '4968', '4969', '2472', '4971', '4972', '4973', '4974', '4975', '4976', '4977', '4978', '4979', '4980', '4981', '4982', '4983', '4984', '4985', '4986', '4987', '4988', '4989', '4990', '4991', '1864', '4993', '4994', '4995', '4996', '4997', '4998', '3638', '5000', '1598', '5002', '5003', '5004', '5005', '5006', '5007', '5008', '5009', '5010', '5011', '5012', '271', '5014', '5015', '5016', '5017', '5018', '5019', '5020', '5021', '5022', '5023', '5024', '3587', '5026', '5027', '5028', '5029', '5030', '5031', '5032', '5033', '5034', '5035', '5036', '2327', '5038', '5039', '5040', '5041', '5042', '5043', '5044', '5045', '5046', '5047', '5048', '5049', '5050', '5051', '1242', '5053', '5054', '5055', '5056', '5057', '5058', '5059', '5060', '5061', '587', '5063', '5064', '5065', '5066', '5067', '5068', '3669', '5070', '5071', '5072', '5073', '5074', '5075', '5076', '5077', '1366', '5079', '5080', '5081', '5082', '5083', '66', '5085', '5086', '5087', '5088', '5089', '5090', '4519', '5092', '5093', '5094', '5095', '5096', '5097', '5098', '5099', '5100', '5101', '5102', '3861', '5104', '5105', '5106', '5107', '5108', '5109', '5110', '5111', '5112', '5113', '66', '5115', '5116', '5117', '5118', '3903', '5120', '5121', '5122', '5123', '5124', '5125', '97', '5127', '5128', '5129', '5130', '5131', '5132', '5133', '5134', '3391', '5136', '5137', '66', '5139', '2810', '5141', '5142', '5143', '5144', '5145', '5146', '5147', '5148', '5149', '2954', '5151', '5152', '5153', '5154', '5155', '5156', '5157', '5158', '5159', '5160', '5161', '5162', '5163', '5164', '5165', '5166', '5167', '5168', '5169', '5170', '1819', '5172', '5173', '5174', '1700', '5176', '5177', '5178', '5179', '5180', '5181', '5182', '5183', '1310', '628', '5186', '5187', '5188', '5189', '5190', '5191', '5192', '66', '5194', '5195', '4533', '5197', '3376', '5199', '5200', '5201', '5202', '5203', '5204', '5205', '1648', '66', '5004', '5209', '5210', '5163', '5212', '5213', '5214', '811', '5216', '5217', '5218', '5219', '5220', '5221', '2684', '5223', '5224', '5225', '343', '5227', '5228', '4988', '5230', '5231', '5232', '5233', '5234', '5235', '5236', '5237', '1027', '5239', '5240', '5241', '5242', '5243', '5244', '5245', '5246', '5247', '5248', '5249', '5250', '5251', '5252', '5253', '5254', '4092', '5256', '5257', '5258', '5259', '5260', '5261', '5262', '4386', '5264', '5265', '5266', '5267', '5268', '1236', '5270', '5271', '1032', '5273', '5274', '5275', '390', '5277', '5278', '5279', '5280', '5281', '5282', '5283', '5284', '5285', '5286', '5287', '5288', '5289', '3034', '66', '5292', '5293', '5294', '5295', '5296', '66', '811', '5299', '5300', '886', '5302', '5303', '5304', '5305', '5306', '5307', '5308', '5309', '3291', '5311', '5312', '5313', '5314', '5315', '5316', '5317', '5318', '5319', '5320', '5321', '5322', '1380', '5324', '5325', '1393', '5327', '5328', '5329', '5330', '5331', '5332', '5333', '5334', '5335', '5336', '3888', '5338', '5339', '5340', '5341', '5342', '5343', '5344', '66', '5346', '5347', '5348', '5349', '5350', '5351', '5352', '5353', '66', '5355', '5356', '5357', '5358', '5359', '5360', '5361', '5362', '5363', '5364', '5365', '5366', '5367', '5368', '2665', '5370', '5371', '5372', '5373', '5374', '5375', '5376', '5377', '5378', '5379', '5380', '5381', '5382', '5383', '5384', '5385', '5386', '5387', '5388', '5389', '2536', '5391', '5392', '5393', '5394', '5395', '829', '5397', '5398', '5399', '5400', '5401', '5402', '5403', '36', '5405', '5406', '5407', '5408', '1981', '1126', '5411', '3421', '5413', '5414', '5415', '5416', '5417', '5418', '5419', '5420', '5421', '2742', '5423', '5424', '5425', '5426', '5427', '5428', '5429', '5430', '5431', '5432', '2816', '5434', '4394', '5436', '5437', '5438', '5439', '5440', '2865', '5442', '5443', '5444', '5306', '5446', '5447', '5448', '5449', '5450', '5451', '5452', '5453', '5454', '5455', '5456', '5457', '5458', '4794', '5460', '5461', '5462', '5463', '5464', '5465', '5466', '5467', '4348', '5469', '5470', '5471', '5472', '5473', '5474', '5350', '5476', '5477', '5478', '5479', '5480', '5050', '2320', '5483', '4378', '5485', '5486', '5487', '5488', '5489', '5490', '5491', '4029', '5493', '5494', '5495', '5496', '5497', '5498', '5499', '5500', '5501', '5502', '5503', '4144', '5505', '5506', '5507', '811', '5509', '5510', '5511', '5512', '5513', '5514', '5515', '5516', '5517', '5518', '5519', '5520', '5521', '5522', '5523', '5524', '5525', '5526', '5527', '5528', '66', '5530', '5531', '5532', '5533', '5534', '1395', '5536', '2130', '5538', '5539', '5540', '5541', '5542', '5543', '5544', '785', '5546', '5547', '5548', '5549', '5550', '5551', '4042', '5553', '5554', '5555', '4828', '252', '4981', '5559', '5560', '5561', '5562', '5563', '5564', '5565', '5566', '5567', '3892', '5569', '5570', '5571', '5572', '5573', '5574', '5575', '4505', '5577', '956', '5579', '5580', '5581', '5582', '5583', '5584', '5585', '5586', '5587', '5588', '5589', '5590', '5591', '5592', '5593', '5594', '5595', '5596', '5597', '5131', '5599', '5600', '5601', '5602', '5603', '5604', '5605', '5606', '4987', '5608', '5609', '5610', '5611', '5612', '5613', '5614', '4109', '138', '5617', '5618', '5619', '5620', '5621', '5622', '4321', '5624', '66', '641', '5627', '5628', '5629', '5630', '5631', '5632', '5633', '5634', '5635', '5524', '5637', '5638', '811', '5640', '5641', '5642', '5643', '5644', '3', '5646', '5647', '66', '5649', '5650', '5651', '5652', '5653', '5654', '5655', '2761', '5657', '692', '5659', '5660', '5661', '5662', '5663', '5664', '5665', '5666', '5667', '1022', '5669', '5670', '5671', '5672', '5673', '5674', '5675', '5676', '945', '5678', '5679', '5680', '5681', '5682', '5683', '5684', '5685', '5686', '3639', '5688', '5689', '5690', '5691', '1825', '5693', '5694', '5695', '5696', '5697', '5698', '5699', '5700', '5701', '5702', '120', '5704', '5705', '5706', '5707', '66', '5709', '5710', '5711', '4369', '5713', '5714', '5715', '5716', '5717', '5718', '5719', '5720', '5721', '5722', '5723', '5724', '5725', '5726', '5727', '2144', '5729', '5730', '5731', '5732', '5733', '5734', '5735', '5736', '5737', '5738', '5739', '5740', '5741', '5742', '5743', '5744', '5745', '5647', '5747', '5748', '66', '5750', '607', '823', '5267', '5754', '5676', '5756', '5757', '5758', '5759', '5760', '5232', '5762', '5763', '5764', '5765', '5766', '17', '5768', '5769', '2095', '5771', '5772', '4848', '5774', '5775', '5776', '5777', '5778', '5763', '5780', '5781', '5782', '5783', '5784', '5785', '5786', '5787', '5788', '5789', '5790', '5791', '5792', '5793', '5794', '5795', '5035', '5797', '66', '5799', '5800', '5801', '5802', '5803', '66', '5805', '5806', '5807', '5808', '5809', '148', '5811', '5812', '5813', '5814', '5815', '5816', '1093', '5818', '5819', '5820', '5821', '5822', '5823', '5824', '5825', '5826', '5827', '5828', '5829', '5830', '5831', '5832', '5833', '5834', '5835', '5836', '5837', '5838', '5839', '5840', '2806', '5842', '5843', '5844', '5845', '5846', '5316', '5848', '5849', '5850', '5851', '5852', '5853', '5854', '5855', '5856', '2529', '5858', '5859', '5860', '5861', '5862', '5863', '5864', '5865', '5866', '5867', '5868', '5869', '3027', '1073', '5872', '5115', '5874', '5875', '5876', '1830', '5878', '5879', '5880', '5881', '3951', '5883', '1003', '5587', '5886', '5887', '160', '5889', '5890', '5891', '5892', '4473', '5894', '5895', '5896', '5897', '5898', '5899', '5900', '5901', '5902', '5903', '4187', '5905', '5906', '5907', '5908', '5909', '5910', '5911', '5912', '5913', '5914', '5915', '5916', '5917', '5631', '5919', '5920', '5921', '5922', '5923', '5924', '5925', '5926', '5927', '3023', '5929', '5930', '1990', '5932', '5933', '5934', '5935', '5391', '5937', '5938', '5939', '5940', '1972', '5942', '5943', '5944', '5945', '5946', '2188', '5948', '5949', '5950', '5951', '5952', '5953', '5954', '5955', '5956', '5957', '754', '5959', '1293', '5961', '5962', '5963', '5964', '5965', '5966', '5967', '5968', '5969', '5970', '5971', '66', '5973', '4592', '5975', '5976', '5977', '5978', '3020', '5980', '5981', '5982', '5983', '5984', '5985', '5986', '5987', '5988', '5989', '5990', '5991', '5992', '5993', '5994', '5995', '727', '5997', '5998', '5999', '6000', '6001', '6002', '6003', '6004', '6005', '6006', '6007', '6008', '4685', '2040', '6011', '6012', '6013', '6014', '6015', '6016', '6017', '6018', '6019', '6020', '4849', '6022', '6023', '6024', '6025', '6026', '513', '6028', '6029', '6030', '2271', '6032', '6033', '6034', '6035', '3540', '6037', '6038', '6039', '6040', '5249', '6042', '6043', '6044', '6045', '6046', '6047', '6048', '6049', '6050', '6051', '6052', '1271', '5190', '6055', '6056', '4670', '6058', '6059', '6060', '6061', '6062', '3419', '4490', '6065', '6066', '6067', '6068', '6069', '6070', '6071', '6072', '6073', '6074', '1567', '6076', '6077', '6078', '6079', '6080', '6081', '6082', '6083', '6084', '6085', '6086', '6087', '6088', '6089', '6090', '833', '6092', '511', '6094', '6095', '4691', '6097', '6098', '6099', '6100', '6101', '6102', '6103', '6104', '6105', '6106', '4867', '6108', '2496', '6110', '1532', '6112', '6113', '6114', '6115', '66', '6117', '6118', '4054', '3318', '6121', '6122', '6123', '6124', '2963', '6126', '6127', '6128', '6129', '668', '6131', '6132', '6133', '66', '6135', '6136', '6137', '6138', '6139', '6140', '6141', '2476', '6143', '6144', '6145', '6146', '6147', '6148', '6149', '6150', '6151', '6152', '6153', '6154', '6155', '6156', '6157', '6158', '6159', '5836', '6161', '6162', '192', '6164', '6165', '6166', '5852', '6168', '6169', '6170', '5172', '6172', '6173', '6174', '6175', '6176', '1901', '6178', '6179', '6180', '6181', '6182', '6183', '6184', '6185', '1805', '6187', '6188', '6189', '6190', '6191', '6192', '6193', '6194', '6195', '6196', '179', '6198', '6199', '6200']}
import os,sys,re,json,time,hashlib,warnings,platform,gc
from pathlib import Path
from collections import Counter
import numpy as np,pandas as pd
from scipy import sparse
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import itertools
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
import cloudpickle
threadpool_limits(2)
CONFIGS30=[{'name':'lr_strong','algo':'lr','view':'count','C':.1}, {'name':'lr_light','algo':'lr','view':'count','C':1.}, {'name':'lgb15','algo':'lgb','view':'count','leaves':15,'objective':'multiclass'}, {'name':'lgb31_ova','algo':'lgb','view':'count','leaves':31,'objective':'multiclassova','augment':True}, {'name':'xgb4','algo':'xgb','view':'count'}, {'name':'mlp128','algo':'mlp','view':'count','augment':True}]
def combinations30():
 return [tuple(c['name'] for c in combo) for k in range(1,5) for combo in itertools.combinations(CONFIGS30,k) if len(set(c['algo'] for c in combo))==k]
COMBINATIONS30=combinations30()
PROTOCOL30={'version':30,'purpose':'from-scratch compact ensemble of1..4 classifiers; no inherited fitted classifiers','candidates':CONFIGS30,'max_final_classifiers':4,'five_requires_user_permission':True,'combination_count':len(COMBINATIONS30),'revision':'initial single-only interrupted on user clarification; later string-group-ID pilot exposed split mismatch and weak partial outer outcomes; final design corrected uint64 groups, shared AA/bin features and train-only augmentation after those observations; logs retained','outer_folds':5,'outer_seed':29032,'inner_folds':2,'inner_seed':30030,'features':'gene binary/count/type; shared gene-AA and position bins64/256; recurrent exact event; non-normalized burden summaries','fit_scope':'vocabulary, DF, chi2, MaxAbs, IDF only training fold','selection':'fixed equal weights; all35 family-distinct subsets of1..4; choose fewest models within0.002 of best mean inner F1, tie highest mean then fixed order; no outer-based selection','final_selection':'same two-fold inner procedure on development only, then selected configuration freshly fit all6201','exact_min_df':3,'exact_max_features':2048,'general_min_df':2,'general_max_features':6144,'legacy_X':'only full single-AA substitution A123X maps A123*; historical-stop assumption, not all X; retain raw files','augmentation':{'members':['lgb31_ova','mlp128'],'gene_mask_probability':.1,'rare_nuisance_poisson':10,'pool_df_max':2,'types':['missense','synonymous'],'weight':.5,'vocabulary_fit':'original training rows only','seed':30030},'test_for_research':False,'old_holdout_scored':False,'external_pretrained':False,'twin_rule':False,'historical_bias':'same repeatedly observed development patients; nested CV does not remove prior research decisions','reference':'v29 genuine out-of-fold cache only, exact training/validation and train hash checked; cache optional'}
def locate30():
 root=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/train.csv').exists()),None)
 if root is None:raise FileNotFoundError('data/train.csv required')
 work=root/'models/혜진' if (root/'models/혜진').is_dir() else root
 return root,work,work/'artifacts/v30'
def canonical_token30(token):
 token=token.strip()
 if re.fullmatch(r'[ACDEFGHIKLMNPQRSTVWY]\d+X',token):token=token[:-1]+'*'
 return token
SUB30=re.compile(r'^([ACDEFGHIKLMNPQRSTVWY*])(\d+)([ACDEFGHIKLMNPQRSTVWY*])$')
def kind30(token):
 if 'fs' in token:return 'frameshift'
 if 'delins' in token:return 'delins'
 if 'del' in token:return 'deletion'
 if 'ins' in token:return 'insertion'
 m=SUB30.fullmatch(token)
 if m:
  a,_,b=m.groups()
  if a==b:return 'stop_retained' if a=='*' else 'synonymous'
  if b=='*':return 'stop_gained'
  if a=='*':return 'stop_lost'
  return 'missense'
 return 'unparsed'
def row_events30(events,missing=()):
 d={k:1. for k in missing};events=sorted(set(events));pergene=Counter();types=Counter();unparsed=0
 for gene,token in events:
  kind=kind30(token);pergene[gene]+=1;types[kind]+=1;unparsed+=kind=='unparsed';d['gene|'+gene]=1.;d['type|'+gene+'|'+kind]=1.;d['event|'+gene+'|'+token]=1.
  m=SUB30.fullmatch(token)
  if m:
   aa,pos,bb=m.groups();d['meta|'+gene+'|aa|'+aa+'>'+bb]=1.;d['meta|'+gene+'|bin64|'+str((int(pos)-1)//64)]=1.;d['meta|'+gene+'|bin256|'+str((int(pos)-1)//256)]=1.
 for gene,n in pergene.items():d['count|'+gene]=float(np.log1p(n))
 d['summary|log_mutated_genes']=float(np.log1p(len(pergene)));d['summary|log_unique_events']=float(np.log1p(len(events)));d['summary|log_missing']=float(np.log1p(len(missing)));d['summary|log_unparsed']=float(np.log1p(unparsed))
 for kind,n in types.items():d['summary|log_type_'+kind]=float(np.log1p(n))
 return d
def parse30(frame,genes):
 if set(genes)-set(frame.columns):raise ValueError('Missing gene columns')
 rows=[]
 for vals in frame.loc[:,genes].itertuples(index=False,name=None):
  events=[];missing=[]
  for gene,cell in zip(genes,vals):
   if pd.isna(cell) or str(cell).strip()=='':missing.append('missing|'+gene);continue
   if str(cell).strip()=='WT':continue
   events.extend((gene,canonical_token30(t)) for t in re.split(r'[\s;,]+',str(cell).strip()) if t)
  rows.append(row_events30(events,missing))
 return rows
def augment30(rows):
 rng=np.random.default_rng(30030);df=Counter(k for r in rows for k in r if k.startswith('event|'));pool=[tuple(k.split('|',2)[1:]) for k,n in df.items() if n<=2 and kind30(k.split('|',2)[2]) in ['missense','synonymous']];aug=[]
 for r in rows:
  events=[tuple(k.split('|',2)[1:]) for k in r if k.startswith('event|')];masked={g for g in {g for g,t in events} if rng.random()<.1};events=[e for e in events if e[0] not in masked]
  if pool:events.extend(pool[i] for i in rng.integers(0,len(pool),size=rng.poisson(10)))
  aug.append(row_events30(events,[k for k in r if k.startswith('missing|')]))
 return aug
class Features30:
 def fit(self,rows,y):
  self.vectorizer=DictVectorizer(dtype=np.float32,sort=True);m=self.vectorizer.fit_transform(rows).tocsr();names=np.asarray(self.vectorizer.get_feature_names_out());df=np.asarray((m>0).sum(0)).ravel();score=chi2(m,y)[0];score=np.nan_to_num(score,nan=-np.inf)
  selected=[]
  for prefix,min_df,budget in [('event|',3,2048),('meta|',3,2048),('general',2,6144)]:
   mask=np.char.startswith(names.astype(str),prefix) if prefix!='general' else ~np.char.startswith(names.astype(str),'event|') & ~np.char.startswith(names.astype(str),'meta|') & ~np.char.startswith(names.astype(str),'summary|')
   ix=np.flatnonzero(mask & (df>=min_df));ix=ix[np.argsort(-score[ix],kind='stable')[:budget]];selected.extend(ix.tolist())
  selected.extend(np.flatnonzero(np.char.startswith(names.astype(str),'summary|')).tolist());self.selected_=np.sort(np.array(selected,dtype=int));self.names_=names[self.selected_];self.df_=df[self.selected_];self.scores_=score[self.selected_];return self
 def transform(self,rows):
  m=self.vectorizer.transform(rows).tocsr()[:,self.selected_].astype(np.float32);m.eliminate_zeros();return m
class Single30:
 def __init__(self,config,genes,labels):self.config=dict(config);self.genes_=list(genes);self.classes_=np.asarray(labels)
 def fit(self,rows,y):
  self.pre=Features30().fit(rows,y);m=self.pre.transform(rows);self.transformer=TfidfTransformer(norm=None,sublinear_tf=True) if self.config['view']=='tfidf' else MaxAbsScaler();m=self.transformer.fit_transform(m)
  c=self.config;weights=compute_sample_weight('balanced',y);target=np.asarray(y)
  if c.get('augment',False):
   aug=self.transformer.transform(self.pre.transform(augment30(rows)));m=sparse.vstack([m,aug],format='csr');target=np.concatenate([target,target]);weights=np.concatenate([weights,.5*weights])
  if c['algo']=='lr':self.estimator=LogisticRegression(C=c['C'],solver='lbfgs',max_iter=700,class_weight='balanced',random_state=30030)
  elif c['algo']=='cnb':self.estimator=ComplementNB(alpha=c['alpha'])
  elif c['algo']=='xgb':self.estimator=XGBClassifier(n_estimators=300,max_depth=4,learning_rate=.05,min_child_weight=3,subsample=.85,colsample_bytree=.7,reg_lambda=5.,objective='multi:softprob',num_class=len(self.classes_),tree_method='hist',n_jobs=2,random_state=30030)
  elif c['algo']=='mlp':self.estimator=MLPClassifier(hidden_layer_sizes=(128,),alpha=1.,batch_size=128,learning_rate_init=.001,max_iter=150,early_stopping=False,random_state=30030)
  else:self.estimator=LGBMClassifier(objective=c['objective'],num_class=len(self.classes_),n_estimators=400 if c.get('augment') else 260,learning_rate=.05,num_leaves=c['leaves'],max_depth=-1,min_child_samples=25,colsample_bytree=.7,subsample=.85,subsample_freq=1,reg_alpha=.1,reg_lambda=3.,class_weight=None if c.get('augment') else 'balanced',n_jobs=2,random_state=30030,verbosity=-1,force_col_wise=True,zero_as_missing=False)
  # Numeric feature names avoid LightGBM JSON-character restrictions on mutation tokens.
  if c['algo'] in ['xgb','mlp'] or c.get('augment',False):self.estimator.fit(m,target,sample_weight=weights)
  else:self.estimator.fit(m,target)
  self.feature_count_=m.shape[1];return self
 def predict_rows(self,rows):
  p=self.estimator.predict_proba(self.transformer.transform(self.pre.transform(rows)));p=np.maximum(np.asarray(p,dtype=float),1e-15);p/=p.sum(1,keepdims=True);assert p.shape==(len(rows),len(self.classes_)) and np.isfinite(p).all();return p
 def predict_frame(self,frame):return self.predict_rows(parse30(frame,self.genes_))
def score30(y,p):return float(f1_score(y,p.argmax(1),labels=np.arange(26),average='macro',zero_division=0))
class Compact30:
 def __init__(self,selected,genes,labels):self.selected_=list(selected);self.genes_=list(genes);self.classes_=np.asarray(labels)
 def fit(self,rows,y):
  self.members_={};self.fit_seconds_={}
  for name in self.selected_:
   cfg=next(c for c in CONFIGS30 if c['name']==name);a=time.perf_counter();self.members_[name]=Single30(cfg,self.genes_,self.classes_).fit(rows,y);self.fit_seconds_[name]=time.perf_counter()-a
  assert 1<=len(self.members_)<=4;self.feature_count_=sum(m.feature_count_ for m in self.members_.values());return self
 def predict_rows(self,rows):return np.mean([m.predict_rows(rows) for m in self.members_.values()],axis=0)
 def predict_frame(self,frame):return self.predict_rows(parse30(frame,self.genes_))
def choose30(rows,y,groups,genes,labels,seed,log=None):
 metrics=[];combometrics=[]
 for fold,(a,b) in enumerate(StratifiedGroupKFold(2,shuffle=True,random_state=seed).split(np.zeros(len(y)),y,groups)):
  assert not set(groups[a])&set(groups[b]);ra=[rows[i] for i in a];rb=[rows[i] for i in b];probs={}
  for cfg in CONFIGS30:
   start=time.perf_counter();model=Single30(cfg,genes,labels).fit(ra,y[a]);p=model.predict_rows(rb);probs[cfg['name']]=p;record={'inner_fold':fold,'candidate':cfg['name'],'f1':score30(y[b],p),'accuracy':float(accuracy_score(y[b],p.argmax(1))),'seconds':time.perf_counter()-start,'features':model.feature_count_};metrics.append(record);print('INNER',record,flush=True)
   if log:pd.DataFrame(metrics).to_csv(log,index=False)
   del model;gc.collect()
  for names in COMBINATIONS30:
   p=np.mean([probs[k] for k in names],axis=0);combometrics.append({'inner_fold':fold,'members':'|'.join(names),'count':len(names),'f1':score30(y[b],p)})
  if log:pd.DataFrame(combometrics).to_csv(Path(log).with_name(Path(log).stem+'_combinations.csv'),index=False)
 mean={names:float(np.mean([r['f1'] for r in combometrics if r['members']=='|'.join(names)])) for names in COMBINATIONS30};best=max(mean.values());eligible=[n for n in COMBINATIONS30 if mean[n]>=best-.002];selected=min(eligible,key=lambda n:(len(n),-mean[n],COMBINATIONS30.index(n)));print('INNER SELECTION',selected,'mean',mean[selected],'best',best,flush=True);return {'name':'|'.join(selected),'members':list(selected),'count':len(selected),'inner_f1':mean[selected],'inner_best_f1':best},metrics


In [2]:
import subprocess
from IPython.display import display
ROOT30,WORK30,OUT30=locate30();OUT30.mkdir(parents=True,exist_ok=True)
def guard30(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('v30 research forbids test/sample')
sys.addaudithook(guard30)
(OUT30/'protocol_v30.json').write_text(json.dumps(PROTOCOL30,ensure_ascii=False,indent=2));(OUT30/'core30.py').write_text(CORE30);(OUT30/'worker30.py').write_text(WORKER30)
train30=pd.read_csv(ROOT30/'data/train.csv');sha30=hashlib.sha256((ROOT30/'data/train.csv').read_bytes()).hexdigest();assert sha30=='92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab';assert train30.ID.astype(str).tolist()==COHORT30['ids'];genes30=[c for c in train30 if c not in ['ID','SUBCLASS']];labels30=np.sort(train30.SUBCLASS.unique());y30=pd.Categorical(train30.SUBCLASS,categories=labels30).codes.astype(int);rows30=parse30(train30,genes30)
# Union earlier conservative groups and newly canonicalized complete profiles, without labels.
parent30=list(range(len(train30)))
def find30(i):
 while parent30[i]!=i:parent30[i]=parent30[parent30[i]];i=parent30[i]
 return i
def union30(a,b):
 a,b=find30(a),find30(b)
 if a!=b:parent30[max(a,b)]=min(a,b)
for keys30 in [COHORT30['groups'],[hashlib.sha256(json.dumps(r,sort_keys=True).encode()).hexdigest() for r in rows30]]:
 seen30={}
 for i,key in enumerate(keys30):
  if key in seen30:union30(i,seen30[key])
  else:seen30[key]=i
groups30=np.array([str(find30(i)) for i in range(len(train30))]);dev30=np.array(COHORT30['dev']);hold30=np.setdiff1d(np.arange(len(train30)),dev30);holdgroups30=set(groups30[hold30]);excluded30=[int(i) for i in dev30 if groups30[i] in holdgroups30];dev30=dev30[~np.isin(dev30,excluded30)];assert not set(groups30[dev30])&holdgroups30
# Preserve old numeric grouping for exact paired reference if union did not merge any old groups.
old30=np.asarray(COHORT30['groups'],dtype=np.uint64);merged30=any(len(set(old30[groups30==g]))>1 for g in np.unique(groups30));groups30=groups30 if merged30 else old30
assert canonical_token30('R1450X')==canonical_token30('R1450*');assert canonical_token30('X12R')=='X12R';assert kind30('*123Q')=='stop_lost'
probe30=pd.DataFrame('WT',index=[0],columns=genes30);probe30.loc[0,genes30[0]]='R12X R12*';assert parse30(probe30,genes30)[0]==parse30(probe30.assign(**{genes30[0]:'R12*'}),genes30)[0]
audit30={'train_sha256':sha30,'rows':len(train30),'genes':len(genes30),'classes':len(labels30),'development':len(dev30),'old_holdout_not_scored':len(hold30),'additional_dev_exclusions':excluded30,'new_groups_merged':bool(merged30),'test_opened_for_research':False}
(OUT30/'data_audit.json').write_text(json.dumps(audit30,indent=2));print('DATA AUDIT',audit30,flush=True)
with (OUT30/'parsed_train.pkl').open('wb') as f:cloudpickle.dump({'rows':rows30,'y':y30,'groups':groups30,'dev':dev30,'genes':genes30,'labels':labels30,'sha':sha30,'ids':train30.ID.tolist()},f)
# EDA is development-only, never hidden test or old holdout labels.
burden30=pd.DataFrame([{'ID':train30.ID.iloc[i],'SUBCLASS':train30.SUBCLASS.iloc[i],'mutated_genes':sum(k.startswith('gene|') for k in rows30[i]),'unique_events':sum(k.startswith('event|') for k in rows30[i]),'unparsed_events':sum(k.startswith('event|') and kind30(k.split('|',2)[2])=='unparsed' for k in rows30[i])} for i in dev30]);burden30.to_csv(OUT30/'eda_development.csv',index=False);display(burden30.groupby('SUBCLASS')[['mutated_genes','unique_events']].agg(['count','median','mean']))
for fold,(a,b) in enumerate(StratifiedGroupKFold(5,shuffle=True,random_state=29032).split(np.zeros(len(dev30)),y30[dev30],groups30[dev30])):
 tr,va=dev30[a],dev30[b];assert not set(groups30[tr])&set(groups30[va]);np.savez_compressed(OUT30/f'split{fold}.npz',tr=tr,va=va)
 reference_split=WORK30/f'artifacts/v29/followup_split{fold}.npz'
 if reference_split.exists():
  with np.load(reference_split) as ref:assert np.array_equal(tr,ref['tr']) and np.array_equal(va,ref['va']), 'paired split mismatch'
active30=[];pending30=list(range(5))
while pending30 or active30:
 while pending30 and len(active30)<2:
  fold=pending30.pop(0);handle=(OUT30/f'worker{fold}.log').open('w');env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2');p=subprocess.Popen([sys.executable,str(OUT30/'worker30.py'),str(OUT30),str(fold)],stdout=handle,stderr=subprocess.STDOUT,env=env);active30.append((p,handle,fold));print('START OUTER',fold,flush=True)
 done=[]
 for item in active30:
  p,h,f=item
  if p.poll() is not None:
   h.close()
   if p.returncode:raise RuntimeError('v30 fold failed: '+str(OUT30/f'worker{f}.log'))
   done.append(item);print('DONE OUTER',f,flush=True)
 active30=[i for i in active30 if i not in done]
 if active30:time.sleep(2)
records30=[];oof30=np.zeros((len(dev30),26));position30={v:k for k,v in enumerate(dev30)};ref30=np.zeros_like(oof30);reference_ok30=True
for fold in range(5):
 info=json.loads((OUT30/f'fold{fold}.json').read_text());records30.append(info)
 with np.load(OUT30/f'fold{fold}.npz') as q:
  va=q['va'];tr=q['tr'];positions=np.array([position30[i] for i in va]);oof30[positions]=q['probabilities']
  refpath=WORK30/f'artifacts/v29/followup_fold{fold}.npz';meta=WORK30/'artifacts/v29/verification_v29.json'
  if refpath.exists() and meta.exists() and json.loads(meta.read_text())['train_sha256']==sha30:
   with np.load(refpath) as old:
    if np.array_equal(tr,old['tr']) and np.array_equal(va,old['va']) and np.array_equal(labels30.astype(str),old['classes'].astype(str)):
     ref30[positions]=old['candidate'];info['v29_f1']=score30(y30[va],old['candidate']);info['delta_vs_v29']=info['f1']-info['v29_f1']
    else:reference_ok30=False
  else:reference_ok30=False
pd.DataFrame(records30).to_csv(OUT30/'nested_outer_results.csv',index=False);np.savez_compressed(OUT30/'nested_oof.npz',dev=dev30,probabilities=oof30,classes=labels30,reference=ref30 if reference_ok30 else np.array([]))
pd.DataFrame(classification_report(y30[dev30],oof30.argmax(1),labels=np.arange(26),target_names=labels30,output_dict=True,zero_division=0)).T.to_csv(OUT30/'classification_report.csv');pd.DataFrame(confusion_matrix(y30[dev30],oof30.argmax(1),labels=np.arange(26)),index=labels30,columns=labels30).to_csv(OUT30/'confusion_matrix.csv')
summary30={'version':30,'max_classifiers':4,'nested_mean_f1':float(np.mean([r['f1'] for r in records30])),'nested_mean_accuracy':float(np.mean([r['accuracy'] for r in records30])),'nested_pooled_f1':score30(y30[dev30],oof30),'reference_paired':reference_ok30,'public_score':None,'model_saved':False,'test_used_for_research':False,'train_sha256':sha30,'fold_selected_candidates':[r['selected'] for r in records30]}
if reference_ok30:
 summary30.update(v29_mean_f1=float(np.mean([r['v29_f1'] for r in records30])),mean_delta_vs_v29=float(np.mean([r['delta_vs_v29'] for r in records30])),all_fold_deltas=[r['delta_vs_v29'] for r in records30]);summary30['improvement_gate_passed']=bool(summary30['mean_delta_vs_v29']>=.003 and min(summary30['all_fold_deltas'])>=-.003)
else:summary30['improvement_gate_passed']=False
(OUT30/'verification_v30.json').write_text(json.dumps(summary30,indent=2));print('NESTED RESULT',json.dumps(summary30,indent=2),flush=True)
# Final configuration is chosen by development inner CV only, never by outer validation results.
finalcfg30,finalinner30=choose30([rows30[i] for i in dev30],y30[dev30],groups30[dev30],genes30,labels30,30030,OUT30/'final_inner_results.csv');(OUT30/'final_selection_frozen.json').write_text(json.dumps(finalcfg30,indent=2));print('FINAL CONFIG FROZEN',finalcfg30,flush=True)
a=time.perf_counter();final30=Compact30(finalcfg30['members'],genes30,labels30).fit(rows30,y30);fitseconds30=time.perf_counter()-a;final30.training_data_sha256_=sha30;final30.training_protocol_=PROTOCOL30
modelpath30=OUT30/'cancer_model_v30_compact_experimental.pkl'
with modelpath30.open('wb') as f:cloudpickle.dump(final30,f)
with modelpath30.open('rb') as f:loaded30=cloudpickle.load(f)
probe30=train30.loc[:15,genes30].copy();assert np.allclose(final30.predict_frame(probe30),loaded30.predict_frame(probe30),atol=1e-8);assert np.allclose(loaded30.predict_frame(probe30)[:1],loaded30.predict_frame(probe30.iloc[:1]),atol=1e-7)
star30=probe30.copy();x30=probe30.copy();star30.loc[0,genes30[0]]='R1450*';x30.loc[0,genes30[0]]='R1450X';assert np.allclose(loaded30.predict_frame(star30),loaded30.predict_frame(x30),atol=1e-8)
pd.concat([pd.DataFrame({'member':k,'feature':m.pre.names_,'train_df':m.pre.df_,'chi2_training':m.pre.scores_}) for k,m in final30.members_.items()],ignore_index=True).to_csv(OUT30/'selected_variables.csv',index=False)
summary30.update(final_config=finalcfg30,classifiers=len(final30.members_),fit_breakdown=final30.fit_seconds_,model_saved=True,model_bytes=modelpath30.stat().st_size,model_sha256=hashlib.sha256(modelpath30.read_bytes()).hexdigest(),final_features=final30.feature_count_,full_train_fit_seconds=fitseconds30,reload_notation_batch_checks=True,python=platform.python_version());(OUT30/'verification_v30.json').write_text(json.dumps(summary30,indent=2));print('V30 COMPLETE',json.dumps(summary30,indent=2),flush=True)


DATA AUDIT {'train_sha256': '92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab', 'rows': 6201, 'genes': 4384, 'classes': 26, 'development': 4960, 'old_holdout_not_scored': 1241, 'additional_dev_exclusions': [], 'new_groups_merged': False, 'test_opened_for_research': False}
START OUTER 0
START OUTER 1
DONE OUTER 1
START OUTER 2
DONE OUTER 0
START OUTER 3
DONE OUTER 2
START OUTER 4
DONE OUTER 3
DONE OUTER 4
NESTED RESULT {
  "version": 30,
  "max_classifiers": 4,
  "nested_mean_f1": 0.4330610502318445,
  "nested_mean_accuracy": 0.4372983870967742,
  "nested_pooled_f1": 0.43584059003238956,
  "reference_paired": true,
  "public_score": null,
  "model_saved": false,
  "test_used_for_research": false,
  "train_sha256": "92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab",
  "fold_selected_candidates": [
    "lr_light|lgb31_ova|xgb4",
    "lr_light|xgb4",
    "lgb31_ova|xgb4|mlp128",
    "lr_strong|lgb31_ova|xgb4|mlp128",
    "lgb31_ova|xgb4|mlp128"
  ],
  "v29_m

mutated_genes                    unique_events                   
                 count median        mean         count median        mean
SUBCLASS                                                                  
ACC                 58   24.0   36.172414            58   26.0   39.155172
BLCA                84   57.5   75.011905            84   60.0   81.000000
BRCA               629    9.0   15.763116           629   10.0   16.483307
CESC               124   18.5   44.258065           124   19.0   48.185484
COAD               179   24.0   81.357542           179   24.0   97.324022
DLBC                31   30.0   49.354839            31   32.0   54.677419
GBMLGG             369   11.0   12.658537           369   12.0   13.000000
HNSC               179   32.0   40.949721           179   33.0   43.016760
KIPAN              412   15.0   16.371359           412   15.0   16.844660
KIRC               267   13.0   13.258427           267   13.0   13.565543
LAML               127    3.0    3.330709           127    3.0    3.385827
LGG                183    8.0    9.043716           183    8.0    9.382514
LIHC               127   24.0   31.393701           127   25.0   32.267717
LUAD               148   41.5   58.128378           148   42.0   61.283784
LUSC               143   59.0   71.083916           143   60.0   75.251748
OV                 203   12.0   14.330049           203   12.0   14.443350
PAAD                96   14.5   41.562500            96   14.5   66.500000
PCPG               118    4.0    4.500000           118    4.0    4.618644
PRAD               213    6.0    7.906103           213    6.0    8.122066
SARC               159    9.0   13.440252           159    9.0   14.572327
SKCM               221   83.0  133.850679           221   87.0  164.180995
STES               303   41.0   81.438944           303   42.0   89.036304
TGCT               100   16.0   17.240000           100   16.0   17.460000
THCA               260    4.0    4.457692           260    4.0    4.469231
THYM                68    3.0    6.308824            68    3.0    6.558824
UCEC               159   18.0  121.226415           159   19.0  160.069182

## B: 희귀 변이 공유 표현을 보강한 고정3개 후속

앞선 A의 일부 약한 결과와 기존 문헌/실패 기록을 본 뒤 설계한 적응적 후속입니다. A 결과는 primary_a에 보존하며, 새 고정3개 조합은 가중치·구성을 B outer 점수로 조정하지 않습니다. 같은 과거 환자의 평가로 외부 독립 확인이 아닙니다.

In [3]:
CORE30B="# Adaptive follow-up after primary partial outer results; fixed3 model recipe before any B scores.\nPROTOCOL30B={'version':30,'phase':'B fixed3 rare-aware followup','adaptation':'motivated by weak primary partial outer scores and existing literature/V23 lessons; same repeatedly observed patients, not independent confirmation','members':['rare_lr','rare_ova','rare_mlp'],'weights':[1/3,1/3,1/3],'folds':5,'seed':29032,'features':'current pointwise parser plus train-only rare/supported gene counts, LOO rarity summaries, global-AA counts and type counts/fractions; global chi2 up to8192, minDF3, MaxAbs then exact residual df/(df+5)','augmentation':'OVA/MLP train-only gene mask0.1 plus Poisson20 rare nuisance; original train vocabulary only; original1,aug0.5','fit':'only3 classifiers fresh; same representation and parser shared','validation':'fixed3, no weight tuning or extra pair experts; no selection on B folds','max_classifiers':3,'five_requires_user_permission':True,'test_for_research':False,'external_patient_or_pretrained_data':False,'twin_rule':False,'gate':'mean_delta_vs_v29>=0.003; each fold_delta>=-0.003; exploratory due adaptive same-patient design'}\ndef augment30b(rows):\n rng=np.random.default_rng(30031);df=Counter(k for r in rows for k in r if k.startswith('event|'));pool=[tuple(k.split('|',2)[1:]) for k,n in df.items() if n<=2 and kind30(k.split('|',2)[2]) in ['missense','synonymous']];aug=[]\n for r in rows:\n  events=[tuple(k.split('|',2)[1:]) for k in r if k.startswith('event|')];genes={g for g,t in events};masked={g for g in genes if rng.random()<.1}\n  if genes and genes<=masked:masked.remove(sorted(genes)[int(rng.integers(len(genes)))])\n  events=[e for e in events if e[0] not in masked];missing=[k for k in r if k.startswith('missing|')];missing_genes={k.split('|',1)[1] for k in missing}\n  if pool:events.extend(pool[i] for i in rng.integers(0,len(pool),size=rng.poisson(20)) if pool[i][0] not in missing_genes)\n  aug.append(row_events30(events,missing))\n return aug\nclass RareFeatures30:\n def records(self,rows,training=False):\n  result=[]\n  for raw in rows:\n   d=dict(raw);events=[k for k in raw if k.startswith('event|')];rare=0;unseen=0;types=Counter();local=Counter();global_aa=Counter()\n   for key in events:\n    _,g,t=key.split('|',2);support=max(0,self.event_df_.get(key,0)-int(training));rare+=support<3;unseen+=support==0;bucket='rare' if support<3 else 'supported';d[bucket+'|'+g]=d.get(bucket+'|'+g,0)+1.;kind=kind30(t);types[kind]+=1;local[(g,kind)]+=1;m=SUB30.fullmatch(t)\n    if m:global_aa[m.group(1)+'>'+m.group(3)]+=1\n    if self.event_df_.get(key,0)<3:d.pop(key,None)\n   for (g,kind),n in local.items():d['effectcount|'+g+'|'+kind]=float(np.log1p(n))\n   for aa,n in global_aa.items():d['aa_global|'+aa]=float(n)\n   for kind,n in types.items():d['summary|fraction_'+kind]=n/max(len(events),1)\n   d['summary|log_rare']=float(np.log1p(rare));d['summary|rare_fraction']=rare/max(len(events),1);d['summary|log_unseen']=float(np.log1p(unseen));result.append(d)\n  return result\n def fit(self,rows,y):\n  self.event_df_=Counter(k for r in rows for k in r if k.startswith('event|'));self.vectorizer=DictVectorizer(dtype=np.float32,sort=True);a=self.vectorizer.fit_transform(self.records(rows,training=True)).tocsr();a.eliminate_zeros();names=np.asarray(self.vectorizer.get_feature_names_out());df=np.asarray((a>0).sum(0)).ravel();valid=(df>=3)|np.char.startswith(names.astype(str),'summary|');ix=np.flatnonzero(valid);scores=np.nan_to_num(chi2(a[:,ix],y)[0],nan=0.,posinf=0.,neginf=0.);chosen=np.argsort(-scores,kind='stable')[:8192];self.selected_=np.sort(ix[chosen]);self.names_=names[self.selected_];self.df_=df[self.selected_];allscore=np.zeros(a.shape[1]);allscore[ix]=scores;self.scores_=allscore[self.selected_];self.scale=MaxAbsScaler().fit(a[:,self.selected_]);self.residual_weights_=np.array([self.event_df_[n]/(self.event_df_[n]+5.) if n.startswith('event|') else 1. for n in self.names_],dtype=np.float32);return self\n def transform(self,rows,training=False):\n  a=self.vectorizer.transform(self.records(rows,training=training)).tocsr()[:,self.selected_];a.eliminate_zeros();return sparse.csr_matrix(self.scale.transform(a).multiply(self.residual_weights_),dtype=np.float32)\nclass RareCompact30:\n def __init__(self,genes,labels):self.genes_=list(genes);self.classes_=np.asarray(labels);self.selected_=['rare_lr','rare_ova','rare_mlp'];self.members_={}\n def fit(self,rows,y):\n  self.pre=RareFeatures30().fit(rows,y);a=self.pre.transform(rows,training=True);aug=self.pre.transform(augment30b(rows));at=sparse.vstack([a,aug],format='csr');yt=np.r_[y,y];self.fit_seconds_={}\n  models={'rare_lr':LogisticRegression(C=1.,class_weight='balanced',max_iter=1600), 'rare_ova':LGBMClassifier(objective='multiclassova',is_unbalance=True,num_class=len(self.classes_),n_estimators=500,learning_rate=.04,num_leaves=15,max_depth=5,min_child_samples=20,reg_lambda=10.,colsample_bytree=.85,n_jobs=2,verbosity=-1,force_col_wise=True,zero_as_missing=False,random_state=30031), 'rare_mlp':MLPClassifier(hidden_layer_sizes=(128,),alpha=1.,batch_size=128,learning_rate_init=.001,max_iter=180,early_stopping=False,random_state=30031)}\n  for name,est in models.items():\n   start=time.perf_counter()\n   if name=='rare_lr':est.fit(a,y)\n   elif name=='rare_ova':est.fit(at,yt,sample_weight=np.r_[np.ones(len(y)),.5*np.ones(len(y))]/1.5)\n   else:\n    w=compute_sample_weight('balanced',y);est.fit(at,yt,sample_weight=np.r_[w,.5*w]/1.5)\n   self.members_[name]=est;self.fit_seconds_[name]=time.perf_counter()-start\n  self.feature_count_=len(self.pre.names_);return self\n def predict_rows(self,rows):\n  a=self.pre.transform(rows);ps=[]\n  for m in self.members_.values():\n   p=np.maximum(m.predict_proba(a),1e-15);p/=p.sum(1,keepdims=True);ps.append(p)\n  return np.mean(ps,axis=0)\n def predict_frame(self,frame):return self.predict_rows(parse30(frame,self.genes_))\n"
WORKER30B="import sys,json,time\nfrom pathlib import Path\nout=Path(sys.argv[1]);fold=int(sys.argv[2]);exec((out/'core30.py').read_text());exec((out/'core30b.py').read_text())\ndef guard_b(event,args):\n if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('Research forbids test/sample')\nsys.addaudithook(guard_b)\nwith (out/'parsed_train.pkl').open('rb') as f:data=cloudpickle.load(f)\nwith np.load(out/f'split{fold}.npz') as q:tr=q['tr'];va=q['va']\nassert not set(data['groups'][tr])&set(data['groups'][va]);start=time.perf_counter();model=RareCompact30(data['genes'],data['labels']).fit([data['rows'][i] for i in tr],data['y'][tr]);fitseconds=time.perf_counter()-start;p=model.predict_rows([data['rows'][i] for i in va]);f1=score30(data['y'][va],p)\nnp.savez_compressed(out/f'b_fold{fold}.npz',tr=tr,va=va,probabilities=p,classes=data['labels']);result={'train_sha256':data['sha'],'protocol_sha256':hashlib.sha256(json.dumps(PROTOCOL30B,sort_keys=True).encode()).hexdigest(),'fold':fold,'selected':'rare_lr|rare_ova|rare_mlp','f1':f1,'accuracy':float(accuracy_score(data['y'][va],p.argmax(1))),'features':model.feature_count_,'fit_seconds':fitseconds};(out/f'b_fold{fold}.json').write_text(json.dumps(result,indent=2));print('B OUTER',result,flush=True)\n"
# Adaptive follow-up after primary partial outer results; fixed3 model recipe before any B scores.
PROTOCOL30B={'version':30,'phase':'B fixed3 rare-aware followup','adaptation':'motivated by weak primary partial outer scores and existing literature/V23 lessons; same repeatedly observed patients, not independent confirmation','members':['rare_lr','rare_ova','rare_mlp'],'weights':[1/3,1/3,1/3],'folds':5,'seed':29032,'features':'current pointwise parser plus train-only rare/supported gene counts, LOO rarity summaries, global-AA counts and type counts/fractions; global chi2 up to8192, minDF3, MaxAbs then exact residual df/(df+5)','augmentation':'OVA/MLP train-only gene mask0.1 plus Poisson20 rare nuisance; original train vocabulary only; original1,aug0.5','fit':'only3 classifiers fresh; same representation and parser shared','validation':'fixed3, no weight tuning or extra pair experts; no selection on B folds','max_classifiers':3,'five_requires_user_permission':True,'test_for_research':False,'external_patient_or_pretrained_data':False,'twin_rule':False,'gate':'mean_delta_vs_v29>=0.003; each fold_delta>=-0.003; exploratory due adaptive same-patient design'}
def augment30b(rows):
 rng=np.random.default_rng(30031);df=Counter(k for r in rows for k in r if k.startswith('event|'));pool=[tuple(k.split('|',2)[1:]) for k,n in df.items() if n<=2 and kind30(k.split('|',2)[2]) in ['missense','synonymous']];aug=[]
 for r in rows:
  events=[tuple(k.split('|',2)[1:]) for k in r if k.startswith('event|')];genes={g for g,t in events};masked={g for g in genes if rng.random()<.1}
  if genes and genes<=masked:masked.remove(sorted(genes)[int(rng.integers(len(genes)))])
  events=[e for e in events if e[0] not in masked];missing=[k for k in r if k.startswith('missing|')];missing_genes={k.split('|',1)[1] for k in missing}
  if pool:events.extend(pool[i] for i in rng.integers(0,len(pool),size=rng.poisson(20)) if pool[i][0] not in missing_genes)
  aug.append(row_events30(events,missing))
 return aug
class RareFeatures30:
 def records(self,rows,training=False):
  result=[]
  for raw in rows:
   d=dict(raw);events=[k for k in raw if k.startswith('event|')];rare=0;unseen=0;types=Counter();local=Counter();global_aa=Counter()
   for key in events:
    _,g,t=key.split('|',2);support=max(0,self.event_df_.get(key,0)-int(training));rare+=support<3;unseen+=support==0;bucket='rare' if support<3 else 'supported';d[bucket+'|'+g]=d.get(bucket+'|'+g,0)+1.;kind=kind30(t);types[kind]+=1;local[(g,kind)]+=1;m=SUB30.fullmatch(t)
    if m:global_aa[m.group(1)+'>'+m.group(3)]+=1
    if self.event_df_.get(key,0)<3:d.pop(key,None)
   for (g,kind),n in local.items():d['effectcount|'+g+'|'+kind]=float(np.log1p(n))
   for aa,n in global_aa.items():d['aa_global|'+aa]=float(n)
   for kind,n in types.items():d['summary|fraction_'+kind]=n/max(len(events),1)
   d['summary|log_rare']=float(np.log1p(rare));d['summary|rare_fraction']=rare/max(len(events),1);d['summary|log_unseen']=float(np.log1p(unseen));result.append(d)
  return result
 def fit(self,rows,y):
  self.event_df_=Counter(k for r in rows for k in r if k.startswith('event|'));self.vectorizer=DictVectorizer(dtype=np.float32,sort=True);a=self.vectorizer.fit_transform(self.records(rows,training=True)).tocsr();a.eliminate_zeros();names=np.asarray(self.vectorizer.get_feature_names_out());df=np.asarray((a>0).sum(0)).ravel();valid=(df>=3)|np.char.startswith(names.astype(str),'summary|');ix=np.flatnonzero(valid);scores=np.nan_to_num(chi2(a[:,ix],y)[0],nan=0.,posinf=0.,neginf=0.);chosen=np.argsort(-scores,kind='stable')[:8192];self.selected_=np.sort(ix[chosen]);self.names_=names[self.selected_];self.df_=df[self.selected_];allscore=np.zeros(a.shape[1]);allscore[ix]=scores;self.scores_=allscore[self.selected_];self.scale=MaxAbsScaler().fit(a[:,self.selected_]);self.residual_weights_=np.array([self.event_df_[n]/(self.event_df_[n]+5.) if n.startswith('event|') else 1. for n in self.names_],dtype=np.float32);return self
 def transform(self,rows,training=False):
  a=self.vectorizer.transform(self.records(rows,training=training)).tocsr()[:,self.selected_];a.eliminate_zeros();return sparse.csr_matrix(self.scale.transform(a).multiply(self.residual_weights_),dtype=np.float32)
class RareCompact30:
 def __init__(self,genes,labels):self.genes_=list(genes);self.classes_=np.asarray(labels);self.selected_=['rare_lr','rare_ova','rare_mlp'];self.members_={}
 def fit(self,rows,y):
  self.pre=RareFeatures30().fit(rows,y);a=self.pre.transform(rows,training=True);aug=self.pre.transform(augment30b(rows));at=sparse.vstack([a,aug],format='csr');yt=np.r_[y,y];self.fit_seconds_={}
  models={'rare_lr':LogisticRegression(C=1.,class_weight='balanced',max_iter=1600), 'rare_ova':LGBMClassifier(objective='multiclassova',is_unbalance=True,num_class=len(self.classes_),n_estimators=500,learning_rate=.04,num_leaves=15,max_depth=5,min_child_samples=20,reg_lambda=10.,colsample_bytree=.85,n_jobs=2,verbosity=-1,force_col_wise=True,zero_as_missing=False,random_state=30031), 'rare_mlp':MLPClassifier(hidden_layer_sizes=(128,),alpha=1.,batch_size=128,learning_rate_init=.001,max_iter=180,early_stopping=False,random_state=30031)}
  for name,est in models.items():
   start=time.perf_counter()
   if name=='rare_lr':est.fit(a,y)
   elif name=='rare_ova':est.fit(at,yt,sample_weight=np.r_[np.ones(len(y)),.5*np.ones(len(y))]/1.5)
   else:
    w=compute_sample_weight('balanced',y);est.fit(at,yt,sample_weight=np.r_[w,.5*w]/1.5)
   self.members_[name]=est;self.fit_seconds_[name]=time.perf_counter()-start
  self.feature_count_=len(self.pre.names_);return self
 def predict_rows(self,rows):
  a=self.pre.transform(rows);ps=[]
  for m in self.members_.values():
   p=np.maximum(m.predict_proba(a),1e-15);p/=p.sum(1,keepdims=True);ps.append(p)
  return np.mean(ps,axis=0)
 def predict_frame(self,frame):return self.predict_rows(parse30(frame,self.genes_))

def guard30b_parent(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('B research forbids test/sample')
sys.addaudithook(guard30b_parent)
# Keep the primary result and model; fixed3 B is a disclosed adaptive follow-up, not a relabelled A success.
primarydir30=OUT30/'primary_a';primarydir30.mkdir(exist_ok=True)
for name in ['verification_v30.json','nested_outer_results.csv','nested_oof.npz','classification_report.csv','confusion_matrix.csv','selected_variables.csv','final_selection_frozen.json','cancer_model_v30_compact_experimental.pkl']:
 src=OUT30/name
 if src.exists():shutil.copy2(src,primarydir30/name)
primary30=json.loads((primarydir30/'verification_v30.json').read_text());(OUT30/'protocol_v30b.json').write_text(json.dumps(PROTOCOL30B,indent=2));(OUT30/'core30b.py').write_text(CORE30B);(OUT30/'worker30b.py').write_text(WORKER30B)
active30=[];pending30=[]
for fold in range(5):
 meta=OUT30/f'b_fold{fold}.json';pack=OUT30/f'b_fold{fold}.npz'
 if meta.exists() and pack.exists():
  info=json.loads(meta.read_text());assert info['train_sha256']==primary30['train_sha256'] and info['protocol_sha256']==hashlib.sha256(json.dumps(PROTOCOL30B,sort_keys=True).encode()).hexdigest()
  with np.load(pack) as q,np.load(OUT30/f'split{fold}.npz') as sp:assert np.array_equal(q['tr'],sp['tr']) and np.array_equal(q['va'],sp['va'])
  print('REUSE completed exact B fold',fold,flush=True)
 else:pending30.append(fold)
while pending30 or active30:
 while pending30 and len(active30)<2:
  fold=pending30.pop(0);handle=(OUT30/f'b_worker{fold}.log').open('w');env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2');p=subprocess.Popen([sys.executable,str(OUT30/'worker30b.py'),str(OUT30),str(fold)],stdout=handle,stderr=subprocess.STDOUT,env=env);active30.append((p,handle,fold));print('START FIXED3 B',fold,flush=True)
 done=[]
 for item in active30:
  p,h,f=item
  if p.poll() is not None:
   h.close()
   if p.returncode:raise RuntimeError('v30 B failed: '+str(OUT30/f'b_worker{f}.log'))
   done.append(item);print('DONE B',f,flush=True)
 active30=[i for i in active30 if i not in done]
 if active30:time.sleep(2)
with (OUT30/'parsed_train.pkl').open('rb') as f:d30=cloudpickle.load(f)
dev30=d30['dev'];y30=d30['y'];positions30={i:j for j,i in enumerate(dev30)};oof30=np.zeros((len(dev30),26));reference30=np.zeros_like(oof30);records30=[];paired30=True
for fold in range(5):
 info=json.loads((OUT30/f'b_fold{fold}.json').read_text());records30.append(info)
 with np.load(OUT30/f'b_fold{fold}.npz') as q:
  va=q['va'];tr=q['tr'];ix=np.array([positions30[i] for i in va]);oof30[ix]=q['probabilities'];refpath=WORK30/f'artifacts/v29/followup_fold{fold}.npz'
  if refpath.exists():
   with np.load(refpath) as old:
    assert np.array_equal(old['tr'],tr) and np.array_equal(old['va'],va) and np.array_equal(old['classes'].astype(str),d30['labels'].astype(str));reference30[ix]=old['candidate'];info['v29_f1']=score30(y30[va],old['candidate']);info['delta_vs_v29']=info['f1']-info['v29_f1']
  else:paired30=False
pd.DataFrame(records30).to_csv(OUT30/'b_outer_results.csv',index=False);np.savez_compressed(OUT30/'b_oof.npz',dev=dev30,probabilities=oof30,reference=reference30,classes=d30['labels']);pd.DataFrame(classification_report(y30[dev30],oof30.argmax(1),labels=np.arange(26),target_names=d30['labels'],output_dict=True,zero_division=0)).T.to_csv(OUT30/'b_classification_report.csv');pd.DataFrame(confusion_matrix(y30[dev30],oof30.argmax(1),labels=np.arange(26)),index=d30['labels'],columns=d30['labels']).to_csv(OUT30/'b_confusion_matrix.csv')
summary30={'version':30,'phase':'B fixed3 rare-aware adaptive follow-up','classifiers':3,'max_classifiers':4,'final_config':{'name':'rare_lr|rare_ova|rare_mlp','members':['rare_lr','rare_ova','rare_mlp'],'count':3,'weights':[1/3]*3,'selection':'fixed B before B scores; not selected with B outer outcomes'},'mean_f1':float(np.mean([r['f1'] for r in records30])),'mean_accuracy':float(np.mean([r['accuracy'] for r in records30])),'pooled_f1':score30(y30[dev30],oof30),'reference_paired':paired30,'primary_a_result':primary30,'test_used_for_research':False,'public_score':None,'train_sha256':d30['sha'],'model_saved':False,'limitations':'adaptive followup after weak primary partial scores; same repeatedly observed patients; no independent external confirmation'}
if paired30:
 summary30.update(v29_mean_f1=float(np.mean([r['v29_f1'] for r in records30])),mean_delta_vs_v29=float(np.mean([r['delta_vs_v29'] for r in records30])),all_fold_deltas=[r['delta_vs_v29'] for r in records30]);summary30['improvement_gate_passed']=bool(summary30['mean_delta_vs_v29']>=.003 and min(summary30['all_fold_deltas'])>=-.003)
else:summary30['improvement_gate_passed']=False
(OUT30/'verification_v30b.json').write_text(json.dumps(summary30,indent=2));print('FIXED3 B RESULT',json.dumps(summary30,indent=2),flush=True)
# Final deployment fit only the three fixed members, with one shared preprocessor.
start=time.perf_counter();final30=RareCompact30(d30['genes'],d30['labels']).fit(d30['rows'],y30);fitseconds30=time.perf_counter()-start;final30.training_protocol_=PROTOCOL30B;final30.training_data_sha256_=d30['sha'];modelpath30=OUT30/'cancer_model_v30_compact_experimental.pkl'
with modelpath30.open('wb') as f:cloudpickle.dump(final30,f)
with modelpath30.open('rb') as f:loaded30=cloudpickle.load(f)
assert np.allclose(final30.predict_rows(d30['rows'][:8]),loaded30.predict_rows(d30['rows'][:8]),atol=1e-8)
probe30=pd.DataFrame('WT',index=[0,1],columns=d30['genes']);probe30.loc[0,d30['genes'][0]]='R1450X';probe30.loc[1,d30['genes'][0]]='R1450*';probe_p30=loaded30.predict_frame(probe30);assert np.allclose(probe_p30[0],probe_p30[1],atol=1e-8);assert np.allclose(probe_p30[:1],loaded30.predict_frame(probe30.iloc[:1]),atol=1e-8)
pd.DataFrame({'feature':final30.pre.names_,'train_df':final30.pre.df_,'chi2_training':final30.pre.scores_,'exact_residual_weight':final30.pre.residual_weights_}).to_csv(OUT30/'b_selected_variables.csv',index=False)
summary30.update(model_saved=True,model_bytes=modelpath30.stat().st_size,model_sha256=hashlib.sha256(modelpath30.read_bytes()).hexdigest(),final_features=final30.feature_count_,full_train_fit_seconds=fitseconds30,fit_breakdown=final30.fit_seconds_,reload_notation_batch_checks=True,python=platform.python_version());(OUT30/'verification_v30b.json').write_text(json.dumps(summary30,indent=2));(OUT30/'verification_v30.json').write_text(json.dumps(summary30,indent=2));print('B MODEL SAVED',json.dumps(summary30,indent=2),flush=True)


REUSE completed exact B fold 0
REUSE completed exact B fold 1
REUSE completed exact B fold 2
REUSE completed exact B fold 3
START FIXED3 B 4
DONE B 4
FIXED3 B RESULT {
  "version": 30,
  "phase": "B fixed3 rare-aware adaptive follow-up",
  "classifiers": 3,
  "max_classifiers": 4,
  "final_config": {
    "name": "rare_lr|rare_ova|rare_mlp",
    "members": [
      "rare_lr",
      "rare_ova",
      "rare_mlp"
    ],
    "count": 3,
    "weights": [
      0.3333333333333333,
      0.3333333333333333,
      0.3333333333333333
    ],
    "selection": "fixed B before B scores; not selected with B outer outcomes"
  },
  "mean_f1": 0.4405478661587071,
  "mean_accuracy": 0.4504032258064516,
  "pooled_f1": 0.4440895058942286,
  "reference_paired": true,
  "primary_a_result": {
    "version": 30,
    "max_classifiers": 4,
    "nested_mean_f1": 0.4330610502318445,
    "nested_mean_accuracy": 0.4372983870967742,
    "nested_pooled_f1": 0.43584059003238956,
    "reference_paired": true,
    "public

## C: 선택적 네 번째 분류기 검증

과거 v19의 희귀 특징 CNB 다양성을 참고한 고정비중(.15) 실험입니다. 최고 비중 탐색은 하지 않습니다. B 대비 평균+0.003/최악-0.003을 만족할 때만 네 번째를 추가합니다. 같은 개발 결과로 선택하므로 최종 선택 점수는 탐색적이며 독립 평가가 아닙니다. 최대4개,5개는 미사용입니다.

In [4]:
# Optional fourth classifier, fixed low weight before C scores. No fifth member.
PROTOCOL30C={'version':30,'phase':'C optional fourth ComplementNB','members':'fixed B3 plus ComplementNB(alpha1,normFalse,temperature2)','weights':'0.85*fixedB +0.15*CNB, fixed before C scores; no grid','selection':'add fourth only if mean delta C-B>=0.003 and worst fold delta>=-0.003; otherwise keep B3; this adaptive development selection is exploratory, not unbiased final model estimate','max_final_classifiers':4,'test_for_research':False,'rationale':'historical v19 rare-feature CNB diversity; unlike aborted raw TFIDF CNB this uses rare-aware shared representation'}
def cnb_probability30(est,a):
 from scipy.special import softmax
 return softmax(est.predict_log_proba(a)/2.,axis=1)
class OptionalFourth30:
 def __init__(self,base,cnb):
  self.pre=base.pre;self.members_=dict(base.members_);self.members_['rare_cnb']=cnb;self.genes_=base.genes_;self.classes_=base.classes_;self.weights_={'rare_lr':.85/3,'rare_ova':.85/3,'rare_mlp':.85/3,'rare_cnb':.15};self.feature_count_=base.feature_count_;self.training_protocol_={'base':PROTOCOL30B,'optional_fourth':PROTOCOL30C};self.training_data_sha256_=base.training_data_sha256_
 def predict_rows(self,rows):
  a=self.pre.transform(rows);out=np.zeros((len(rows),len(self.classes_)))
  for name,m in self.members_.items():
   if name=='rare_cnb':p=cnb_probability30(m,a)
   else:p=m.predict_proba(a);p/=p.sum(1,keepdims=True)
   out+=self.weights_[name]*p
  return out
 def predict_frame(self,frame):return self.predict_rows(parse30(frame,self.genes_))

# Existing research guards remain active in a full notebook run.
def guard30c(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('C research forbids test/sample')
sys.addaudithook(guard30c)
(OUT30/'protocol_v30c.json').write_text(json.dumps(PROTOCOL30C,indent=2));bproof30=json.loads((OUT30/'verification_v30b.json').read_text())
with (OUT30/'parsed_train.pkl').open('rb') as f:data30=cloudpickle.load(f)
records_c30=[];dev_c30=data30['dev'];position_c30={v:j for j,v in enumerate(dev_c30)};oof_c30=np.zeros((len(dev_c30),26))
for fold in range(5):
 with np.load(OUT30/f'b_fold{fold}.npz') as q:tr=q['tr'];va=q['va'];bp=q['probabilities']
 rows=[data30['rows'][i] for i in tr];pre=RareFeatures30().fit(rows,data30['y'][tr]);est=ComplementNB(alpha=1.,norm=False).fit(pre.transform(rows,training=True),data30['y'][tr]);cp=.85*bp+.15*cnb_probability30(est,pre.transform([data30['rows'][i] for i in va]));ix=[position_c30[i] for i in va];oof_c30[ix]=cp
 record={'fold':fold,'b_f1':score30(data30['y'][va],bp),'c_f1':score30(data30['y'][va],cp),'c_accuracy':float(accuracy_score(data30['y'][va],cp.argmax(1)))};record['delta_c_minus_b']=record['c_f1']-record['b_f1'];records_c30.append(record);np.savez_compressed(OUT30/f'c_fold{fold}.npz',tr=tr,va=va,probabilities=cp);print('OPTIONAL FOURTH',record,flush=True)
table_c30=pd.DataFrame(records_c30);table_c30.to_csv(OUT30/'c_outer_results.csv',index=False);use_c30=bool(table_c30.delta_c_minus_b.mean()>=.003 and table_c30.delta_c_minus_b.min()>=-.003);cproof30={'mean_f1':float(table_c30.c_f1.mean()),'mean_accuracy':float(table_c30.c_accuracy.mean()),'mean_delta_c_minus_b':float(table_c30.delta_c_minus_b.mean()),'worst_delta_c_minus_b':float(table_c30.delta_c_minus_b.min()),'add_fourth':use_c30,'protocol':PROTOCOL30C};(OUT30/'verification_v30c.json').write_text(json.dumps(cproof30,indent=2));print('FOURTH DECISION',cproof30,flush=True)
finalproof30=dict(bproof30);finalproof30['optional_fourth_result']=cproof30;finalproof30['selected_after_optional_fourth']='C4' if use_c30 else 'B3'
if use_c30:
 bdir=OUT30/'fixed3_b';bdir.mkdir(exist_ok=True);shutil.copy2(OUT30/'cancer_model_v30_compact_experimental.pkl',bdir/'cancer_model_v30_fixed3.pkl')
 with (OUT30/'cancer_model_v30_compact_experimental.pkl').open('rb') as f:base30=cloudpickle.load(f)
 a=time.perf_counter();cnb30=ComplementNB(alpha=1.,norm=False).fit(base30.pre.transform(data30['rows'],training=True),data30['y']);full30=OptionalFourth30(base30,cnb30);seconds30=time.perf_counter()-a
 path=OUT30/'cancer_model_v30_compact_experimental.pkl'
 with path.open('wb') as f:cloudpickle.dump(full30,f)
 with path.open('rb') as f:reloaded_c30=cloudpickle.load(f)
 assert np.allclose(full30.predict_rows(data30['rows'][:8]),reloaded_c30.predict_rows(data30['rows'][:8]),atol=1e-8)
 probe_c30=pd.DataFrame('WT',index=[0,1],columns=data30['genes']);probe_c30.loc[0,data30['genes'][0]]='R1450X';probe_c30.loc[1,data30['genes'][0]]='R1450*';p_c30=reloaded_c30.predict_frame(probe_c30);assert np.allclose(p_c30[:1],p_c30[1:],atol=1e-8);assert np.allclose(p_c30[:1],reloaded_c30.predict_frame(probe_c30.iloc[:1]),atol=1e-8)
 finalproof30.update(phase='C optional-fourth adaptive selection',classifiers=4,mean_f1=cproof30['mean_f1'],mean_accuracy=cproof30['mean_accuracy'],pooled_f1=score30(data30['y'][dev_c30],oof_c30),final_config={'name':'rare_lr|rare_ova|rare_mlp|rare_cnb','members':list(full30.members_),'count':4,'weights':full30.weights_},model_bytes=path.stat().st_size,model_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),full_train_fit_seconds=bproof30['full_train_fit_seconds']+seconds30)
 if bproof30['reference_paired']:
  finalproof30['mean_delta_vs_v29']=finalproof30['mean_f1']-bproof30['v29_mean_f1'];reference_rows=pd.read_csv(OUT30/'b_outer_results.csv');finalproof30['all_fold_deltas']=(table_c30.c_f1-reference_rows.v29_f1).tolist();finalproof30['improvement_gate_passed']=bool(finalproof30['mean_delta_vs_v29']>=.003 and min(finalproof30['all_fold_deltas'])>=-.003)
 else:finalproof30.update(v29_mean_f1=float('nan'),mean_delta_vs_v29=float('nan'),all_fold_deltas=[],improvement_gate_passed=False)
 finalproof30['fit_breakdown']={**bproof30['fit_breakdown'],'rare_cnb':seconds30}
 pd.DataFrame(classification_report(data30['y'][dev_c30],oof_c30.argmax(1),labels=np.arange(26),target_names=data30['labels'],output_dict=True,zero_division=0)).T.to_csv(OUT30/'c_classification_report.csv');pd.DataFrame(confusion_matrix(data30['y'][dev_c30],oof_c30.argmax(1),labels=np.arange(26)),index=data30['labels'],columns=data30['labels']).to_csv(OUT30/'c_confusion_matrix.csv')
(OUT30/'verification_v30.json').write_text(json.dumps(finalproof30,indent=2));print('FINAL V30',json.dumps(finalproof30,indent=2),flush=True)


OPTIONAL FOURTH {'fold': 0, 'b_f1': 0.4626296782034291, 'c_f1': 0.46341098053571145, 'c_accuracy': 0.46875, 'delta_c_minus_b': 0.000781302332282352}
OPTIONAL FOURTH {'fold': 1, 'b_f1': 0.41962004395815505, 'c_f1': 0.44174230559617644, 'c_accuracy': 0.44556451612903225, 'delta_c_minus_b': 0.02212226163802139}
OPTIONAL FOURTH {'fold': 2, 'b_f1': 0.41660531392184585, 'c_f1': 0.42237343897664503, 'c_accuracy': 0.4344758064516129, 'delta_c_minus_b': 0.005768125054799178}
OPTIONAL FOURTH {'fold': 3, 'b_f1': 0.4494346892040945, 'c_f1': 0.44580108766707954, 'c_accuracy': 0.4647177419354839, 'delta_c_minus_b': -0.0036336015370149743}
OPTIONAL FOURTH {'fold': 4, 'b_f1': 0.4544496055060107, 'c_f1': 0.4594356471383904, 'c_accuracy': 0.4536290322580645, 'delta_c_minus_b': 0.004986041632379701}
FOURTH DECISION {'mean_f1': 0.44655269198280056, 'mean_accuracy': 0.4534274193548387, 'mean_delta_c_minus_b': 0.00600482582409353, 'worst_delta_c_minus_b': -0.0036336015370149743, 'add_fourth': False, 'protoc

## D: 좋았던 문헌 전처리를 차용한 공유4개

A/B의 약한 결과 뒤 고정한 적응적 후속입니다. v16의 reference-aware가 아닌 보수적 parser와 rare_backoff 전처리 원리를 차용하고4개 분류기는 새로 fit합니다. 이전 학습 피클/14개 모델을 불러오지 않습니다. 전처리는1개 공유하며 기존 v25의4개 leaf 비중을 고정 정규화합니다. 이4개 recipe는 과거 v29에서도 연구한 이력이 있으므로 새롭고 독립적인 발견이라고 주장하지 않습니다. D가 현재 소형 후보보다 평균0.003 이상 높을 때 실험용 활성 후보로 선택하며, 개발 결과로 선택한 편향을 기록합니다.

In [5]:
CORE30D="PANELS16={}\n# P43: HGVS 2016 principles; this is a conservative dataset parser, not a full HGVS validator.\nAA16=set('ACDEFGHIKLMNPQRSTVWY');THREE16=dict(zip(['Ala','Arg','Asn','Asp','Cys','Gln','Glu','Gly','His','Ile','Leu','Lys','Met','Phe','Pro','Ser','Thr','Trp','Tyr','Val','Ter','Xaa'],list('ARNDCQEGHILKMFPSTWYV')+['*','X']))\n# Each event: key, kind, position, components(position,ref,alt,kind), flags, fs_stop_distance.\ndef parse_token16(raw):\n s=str(raw).strip();core=s[2:] if s.startswith('p.') else s;uncertain=core.startswith('(') and core.endswith(')');core=core[1:-1] if uncertain else core\n if re.match('^(?:'+'|'.join(THREE16)+')[0-9]+',core):core=re.sub('|'.join(THREE16),lambda m:THREE16[m[0]],core)\n key=('uncertain|' if uncertain else '')+core;flags=['uncertain'] if uncertain else [];pos=None;parts=();dist=None;kind='unparsed'\n def effect(a,b):\n  if a=='*' and b=='*':return 'stop_retained'\n  if a=='*':return 'stop_lost' if b in AA16 else 'ambiguous_residue'\n  if b=='*':return 'stop_gained' if a in AA16 else 'ambiguous_residue'\n  if a not in AA16 or b not in AA16:return 'ambiguous_residue'\n  return 'synonymous' if a==b else 'missense'\n m=re.fullmatch(r'([A-Z*])([1-9][0-9]*)([A-Z*])',core)\n if m:\n  a,p,b=m.groups();pos=int(p);kind=effect(a,b);parts=((pos,a,b,kind),)\n  if 'X' in [a,b]:flags.append('ambiguous_X')\n else:\n  m=re.fullmatch(r'([1-9][0-9]*)_([1-9][0-9]*)([A-Z*]+)>([A-Z*]+)',core)\n  if m:\n   start,end,ref,alt=m.groups();start=int(start);end=int(end);pos=start;span=end-start+1\n   if span>0 and span==len(ref)==len(alt) and all(a in AA16 or a=='*' for a in ref+alt):\n    parts=tuple((start+j,a,b,effect(a,b)) for j,(a,b) in enumerate(zip(ref,alt)));kind='synonymous_block' if all(a==b for a,b in zip(ref,alt)) else 'block_change';flags.append('resolved_equal_length_block')\n   else:kind='complex_range';flags.append('unresolved_range')\n  elif 'fs' in core:\n   kind='frameshift';m=re.match(r'([A-Z*]+)([1-9][0-9]*)',core);pos=int(m[2]) if m else None;m=re.search(r'fs(?:\\*|Ter)([1-9][0-9]*)$',core);dist=int(m[1]) if m else None\n  elif 'delins' in core:kind='delins'\n  elif 'del' in core:kind='deletion'\n  elif 'ins' in core:kind='insertion'\n  elif 'dup' in core:kind='duplication'\n  elif 'ext' in core:kind='extension'\n  elif core=='=':kind='no_change_unspecified'\n  elif re.fullmatch(r'[A-Z*]+[1-9][0-9]*',core):kind='incomplete';flags.append('missing_alternate')\n  if pos is None:\n   # Protein coordinate only: do not mistake fs termination distance for the affected residue.\n   m=re.match(r'(?:[A-Z*]+)?([1-9][0-9]*)',core);pos=int(m[1]) if m else None\n return (key,kind,pos,parts,tuple(flags),dist)\n\ndef parse_rows16(frame):\n cols=list(frame.columns);cache={};rows=[]\n for row in frame.itertuples(index=False,name=None):\n  mutations=[];missing=[];raw_n=0;duplicates=0\n  for gene,value in zip(cols,row):\n   if value is None or pd.isna(value) or (isinstance(value,str) and not value.strip()):missing.append(gene);continue\n   if str(value).strip()=='WT':continue\n   tokens=str(value).split();raw_n+=len(tokens);unique=list(dict.fromkeys(tokens));duplicates+=len(tokens)-len(unique);events=[]\n   for token in unique:\n    if token not in cache:cache[token]=parse_token16(token)\n    events.append(cache[token])\n   mutations.append((gene,tuple(events)))\n  rows.append({'mutations':tuple(mutations),'missing':tuple(missing),'raw_tokens':raw_n,'duplicate_tokens':duplicates})\n return rows\n\n\nfrom sklearn.feature_extraction import DictVectorizer\nfrom sklearn.feature_selection import chi2\nfrom sklearn.preprocessing import MaxAbsScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom scipy import sparse\nfrom collections import Counter\nimport copy\nVIEWS16=['legacy_raw','hierarchical','rare_backoff','rare_backoff_panels']\n# Literature-derived panel snapshots are explicit priors, not learned class labels.\nclass Preprocess16:\n def __init__(self,view,min_df=3,k=4096):self.view=view;self.min_df=min_df;self.k=k\n def fit(self,rows,target):\n  self.df=Counter((g,e[0]) for row in rows for g,evs in row['mutations'] for e in {e[0]:e for e in evs}.values())\n  records=self.records(rows,training=True);self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(records);names=self.vec.get_feature_names_out();support=np.asarray((a!=0).sum(0)).ravel();keep=(support>=self.min_df)|np.array([s.startswith(('num:','module:','numeric:','panel:')) for s in names]);self.keep=np.flatnonzero(keep);a=a[:,self.keep]\n  with warnings.catch_warnings():\n   warnings.simplefilter('ignore');scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)\n  self.cols=np.sort(np.argsort(-scores,kind='stable')[:min(self.k,a.shape[1])]);self.names=names[self.keep[self.cols]];selected=a[:,self.cols];self.scale=MaxAbsScaler().fit(selected);self.residual_weights=np.ones(len(self.names),np.float32)\n  if self.view.startswith('rare_backoff'):\n   for j,name in enumerate(self.names):\n    if name.startswith('variant:'):\n     _,gene,key=name.split(':',2);n=self.df.get((gene,key),0);self.residual_weights[j]=n/(n+5.)\n  return self\n def records(self,rows,training=False):\n  # legacy records were computed pointwise using the frozen v6 raw encoder.\n  if self.view=='legacy_raw':return [r['legacy'] for r in rows]\n  result=[]\n  for row in rows:\n   d={};types=Counter();gene_effects={};n=0;rare=0;new=0;uncertain=0;incomplete=0;components=0;ambiguous_x=0\n   for g,evs in row['mutations']:\n    evs=list({e[0]:e for e in evs}.values());d['gene:'+g]=1.;d['gene_event_count:'+g]=np.log1p(len(evs));local=Counter();bin64=set();bin256=set()\n    for key,kind,pos,parts,flags,fsdist in evs:\n     n+=1;support=max(0,self.df.get((g,key),0)-(int(training)));israre=support<3;rare+=israre;new+=support==0;uncertain+='uncertain' in flags;incomplete+=kind in ['incomplete','unparsed','complex_range'];ambiguous_x+='ambiguous_X' in flags\n     effects=[p[3] for p in parts] or [kind];types.update(effects);local.update(effects);d['format:'+g+':'+kind]=1.;components+=len(parts)\n     if self.df.get((g,key),0)>=3:\n      d['variant:'+g+':'+key]=1.\n     if self.view.startswith('rare_backoff'):\n      bucket='rare' if israre else 'supported';d[bucket+'_gene:'+g]=d.get(bucket+'_gene:'+g,0)+1\n     for p,a,b,effect in parts:\n      d['aa_meta:'+g+':'+a+'>'+b]=d.get('aa_meta:'+g+':'+a+'>'+b,0)+1;d['aa_global:'+a+'>'+b]=d.get('aa_global:'+a+'>'+b,0)+1\n     if pos is not None:bin64.add((pos-1)//64);bin256.add((pos-1)//256)\n     # Absolute protein bins are an engineering heuristic, not genomic bins or biological domains.\n     if fsdist is not None:d['fs_stop_distance:'+g]=np.log1p(fsdist)\n    for kind,count in local.items():d['effect:'+g+':'+kind]=np.log1p(count)\n    for b in bin64:d[f'protein_bin64:{g}:{b}']=1.\n    for b in bin256:d[f'protein_bin256:{g}:{b}']=1.\n    gene_effects[g]=local\n   for name,count in [('observed_genes',len(row['mutations'])),('unique_events',n),('raw_tokens',row['raw_tokens']),('duplicate_tokens',row['duplicate_tokens']),('missing_genes',len(row['missing'])),('uncertain_events',uncertain),('incomplete_events',incomplete),('ambiguous_X',ambiguous_x),('resolved_components',components)]:d['num:log_'+name]=np.log1p(count)\n   for kind in ['synonymous','missense','stop_gained','stop_lost','stop_retained','frameshift','deletion','insertion','delins','complex_range','incomplete','unparsed','ambiguous_residue']:\n    d['num:log_'+kind]=np.log1p(types[kind]);d['num:fraction_'+kind]=types[kind]/max(sum(types.values()),1)\n   if self.view.startswith('rare_backoff'):d['num:log_rare_events']=np.log1p(rare);d['num:rare_fraction']=rare/max(n,1);d['num:log_unseen_events']=np.log1p(new)\n   if self.view.endswith('panels'):\n    observed=set(gene_effects)\n    for panel,gs in PANELS16.items():\n     count=len(observed.intersection(gs));d['panel:'+panel+':mutated_genes']=np.log1p(count)\n     flags=sum(bool(set(gene_effects.get(g,{})).intersection({'stop_gained','frameshift','deletion','delins'})) for g in gs);d['panel:'+panel+':truncation_notation_genes']=np.log1p(flags)\n   result.append(d)\n  return result\n def transform(self,rows,training=False):\n  a=self.vec.transform(self.records(rows,training=training));a=a[:,self.keep][:,self.cols];return sparse.csr_matrix(self.scale.transform(a).multiply(self.residual_weights),dtype=np.float32)\n\n\ndef corrupt23(rows):\n # Support and nuisance pool come from the current original training fold only, without labels.\n df=Counter((g,e[0]) for row in rows for g,ev in row['mutations'] for e in {e[0]:e for e in ev}.values());pool={(g,e[0]):(g,e) for row in rows for g,ev in row['mutations'] for e in ev if df[(g,e[0])]<=2 and e[1] in ['missense','synonymous','synonymous_block']};pool=list(pool.values());assert pool;rng=np.random.default_rng(23023);aug=[]\n for row in rows:\n  mutations=row['mutations'];keep=rng.random(len(mutations))>=.1\n  if len(keep) and not keep.any():keep[rng.integers(len(keep))]=True\n  retained={g:{e[0]:e for e in ev} for (g,ev),flag in zip(mutations,keep) if flag}\n  for _ in range(int(rng.poisson(20))):\n   g,e=pool[rng.integers(len(pool))]\n   if g in row['missing']:continue\n   retained.setdefault(g,{})[e[0]]=e\n  evs=tuple((g,tuple(events23.values())) for g,events23 in retained.items());aug.append({'mutations':evs,'missing':row['missing'],'raw_tokens':sum(len(ev) for g,ev in evs),'duplicate_tokens':0})\n return aug\n\nPROTOCOL30D={'version':30,'phase':'D shared literature-preprocessor fixed4','motivation':'after weak new-parser A/B outcomes, reuse proven v16 parser and rare-backoff feature principles; all4 estimators freshly trained, no legacy model pickle','members':['rich_lr','rich_lgb','rare_cnb','aug_ova'],'weights':[.108/.476,.108/.476,.06/.476,.2/.476],'weights_origin':'fixed renormalized v25 leaf weights, not new search','features':'one shared Preprocess16 rare_backoff minDF3 k8192; no panel inputs; exact legacy parsing/canonicalization','lr':{'C':1,'max_iter':1600,'balanced':True},'lgb':{'trees':300,'leaves':15,'depth':5,'lambda':10,'seed':17017},'ova':{'trees':500,'leaves':15,'depth':5,'lambda':10,'is_unbalance':True,'seed':23023},'cnb':{'alpha':1,'norm':False,'temperature':2},'augmentation':'OVA only exact train-only corrupt23 recipe; same shared preprocessor fit originals only','cv_folds':5,'cv_seed':29032,'max_final_classifiers':4,'test_for_research':False,'external_patient_pretrained_or_panels':False,'twin_rule':False,'limitations':'adaptive same-patient comparison after A/B; D historical four-member recipe previously investigated, not independent novel confirmation'}\ndef canonical_frame30d(frame):\n def one(v):\n  if not isinstance(v,str) or 'X' not in v:return v\n  return ' '.join(re.sub(r'^(p\\.)?([ACDEFGHIKLMNPQRSTVWY])([1-9][0-9]*)X$',lambda m:(m[1] or '')+m[2]+m[3]+'*',t) for t in v.split())\n return frame.map(one)\nclass SharedFour30:\n def __init__(self,genes,labels):self.genes_=list(genes);self.classes_=np.asarray(labels);self.weights_={'rich_lr':.108/.476,'rich_lgb':.108/.476,'rare_cnb':.06/.476,'aug_ova':.2/.476};self.members_={}\n def fit(self,rows,y):\n  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,y);a=self.pre.transform(rows,training=True);aug=self.pre.transform(corrupt23(rows),training=False);at=sparse.vstack([a,aug],format='csr');yt=np.r_[y,y];self.fit_seconds_={}\n  models={'rich_lr':LogisticRegression(C=1.,class_weight='balanced',max_iter=1600),'rich_lgb':LGBMClassifier(n_estimators=300,num_leaves=15,max_depth=5,learning_rate=.04,reg_lambda=10.,min_child_samples=20,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=17017),'rare_cnb':ComplementNB(alpha=1.,norm=False),'aug_ova':LGBMClassifier(objective='multiclassova',is_unbalance=True,n_estimators=500,learning_rate=.04,num_leaves=15,max_depth=5,min_child_samples=20,reg_lambda=10.,colsample_bytree=.85,verbosity=-1,n_jobs=2,random_state=23023)}\n  for name,m in models.items():\n   start=time.perf_counter()\n   if name=='aug_ova':m.fit(at,yt,sample_weight=np.r_[np.ones(len(y)),.5*np.ones(len(y))]/1.5)\n   else:m.fit(a,y)\n   self.members_[name]=m;self.fit_seconds_[name]=time.perf_counter()-start\n  self.feature_count_=len(self.pre.names);return self\n def predict_rows(self,rows):\n  from scipy.special import softmax\n  a=self.pre.transform(rows);p=np.zeros((len(rows),len(self.classes_)))\n  for name,m in self.members_.items():\n   q=softmax(m.predict_log_proba(a)/2.,axis=1) if name=='rare_cnb' else m.predict_proba(a);q/=q.sum(1,keepdims=True);p+=self.weights_[name]*q\n  return p\n def predict_frame(self,frame):return self.predict_rows(parse_rows16(canonical_frame30d(frame.loc[:,self.genes_])))\n"
WORKER30D="import sys,json,time\nfrom pathlib import Path\nout=Path(sys.argv[1]);fold=int(sys.argv[2]);exec((out/'core30.py').read_text());exec((out/'core30d.py').read_text())\ndef guard_d(event,args):\n if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('D research forbids test/sample')\nsys.addaudithook(guard_d)\nwith (out/'parsed_train.pkl').open('rb') as f:d=cloudpickle.load(f)\nwith (out/'parsed_train_d.pkl').open('rb') as f:rows=cloudpickle.load(f)\nwith np.load(out/f'split{fold}.npz') as q:tr=q['tr'];va=q['va']\nassert not set(d['groups'][tr])&set(d['groups'][va]);start=time.perf_counter();model=SharedFour30(d['genes'],d['labels']).fit([rows[i] for i in tr],d['y'][tr]);p=model.predict_rows([rows[i] for i in va]);np.savez_compressed(out/f'd_fold{fold}.npz',tr=tr,va=va,probabilities=p);record={'fold':fold,'selected':'rich_lr|rich_lgb|rare_cnb|aug_ova','f1':score30(d['y'][va],p),'accuracy':float(accuracy_score(d['y'][va],p.argmax(1))),'features':model.feature_count_,'fit_predict_seconds':time.perf_counter()-start};(out/f'd_fold{fold}.json').write_text(json.dumps(record,indent=2));print('SHARED4 D',record,flush=True)\n"
PANELS16={}
# P43: HGVS 2016 principles; this is a conservative dataset parser, not a full HGVS validator.
AA16=set('ACDEFGHIKLMNPQRSTVWY');THREE16=dict(zip(['Ala','Arg','Asn','Asp','Cys','Gln','Glu','Gly','His','Ile','Leu','Lys','Met','Phe','Pro','Ser','Thr','Trp','Tyr','Val','Ter','Xaa'],list('ARNDCQEGHILKMFPSTWYV')+['*','X']))
# Each event: key, kind, position, components(position,ref,alt,kind), flags, fs_stop_distance.
def parse_token16(raw):
 s=str(raw).strip();core=s[2:] if s.startswith('p.') else s;uncertain=core.startswith('(') and core.endswith(')');core=core[1:-1] if uncertain else core
 if re.match('^(?:'+'|'.join(THREE16)+')[0-9]+',core):core=re.sub('|'.join(THREE16),lambda m:THREE16[m[0]],core)
 key=('uncertain|' if uncertain else '')+core;flags=['uncertain'] if uncertain else [];pos=None;parts=();dist=None;kind='unparsed'
 def effect(a,b):
  if a=='*' and b=='*':return 'stop_retained'
  if a=='*':return 'stop_lost' if b in AA16 else 'ambiguous_residue'
  if b=='*':return 'stop_gained' if a in AA16 else 'ambiguous_residue'
  if a not in AA16 or b not in AA16:return 'ambiguous_residue'
  return 'synonymous' if a==b else 'missense'
 m=re.fullmatch(r'([A-Z*])([1-9][0-9]*)([A-Z*])',core)
 if m:
  a,p,b=m.groups();pos=int(p);kind=effect(a,b);parts=((pos,a,b,kind),)
  if 'X' in [a,b]:flags.append('ambiguous_X')
 else:
  m=re.fullmatch(r'([1-9][0-9]*)_([1-9][0-9]*)([A-Z*]+)>([A-Z*]+)',core)
  if m:
   start,end,ref,alt=m.groups();start=int(start);end=int(end);pos=start;span=end-start+1
   if span>0 and span==len(ref)==len(alt) and all(a in AA16 or a=='*' for a in ref+alt):
    parts=tuple((start+j,a,b,effect(a,b)) for j,(a,b) in enumerate(zip(ref,alt)));kind='synonymous_block' if all(a==b for a,b in zip(ref,alt)) else 'block_change';flags.append('resolved_equal_length_block')
   else:kind='complex_range';flags.append('unresolved_range')
  elif 'fs' in core:
   kind='frameshift';m=re.match(r'([A-Z*]+)([1-9][0-9]*)',core);pos=int(m[2]) if m else None;m=re.search(r'fs(?:\*|Ter)([1-9][0-9]*)$',core);dist=int(m[1]) if m else None
  elif 'delins' in core:kind='delins'
  elif 'del' in core:kind='deletion'
  elif 'ins' in core:kind='insertion'
  elif 'dup' in core:kind='duplication'
  elif 'ext' in core:kind='extension'
  elif core=='=':kind='no_change_unspecified'
  elif re.fullmatch(r'[A-Z*]+[1-9][0-9]*',core):kind='incomplete';flags.append('missing_alternate')
  if pos is None:
   # Protein coordinate only: do not mistake fs termination distance for the affected residue.
   m=re.match(r'(?:[A-Z*]+)?([1-9][0-9]*)',core);pos=int(m[1]) if m else None
 return (key,kind,pos,parts,tuple(flags),dist)

def parse_rows16(frame):
 cols=list(frame.columns);cache={};rows=[]
 for row in frame.itertuples(index=False,name=None):
  mutations=[];missing=[];raw_n=0;duplicates=0
  for gene,value in zip(cols,row):
   if value is None or pd.isna(value) or (isinstance(value,str) and not value.strip()):missing.append(gene);continue
   if str(value).strip()=='WT':continue
   tokens=str(value).split();raw_n+=len(tokens);unique=list(dict.fromkeys(tokens));duplicates+=len(tokens)-len(unique);events=[]
   for token in unique:
    if token not in cache:cache[token]=parse_token16(token)
    events.append(cache[token])
   mutations.append((gene,tuple(events)))
  rows.append({'mutations':tuple(mutations),'missing':tuple(missing),'raw_tokens':raw_n,'duplicate_tokens':duplicates})
 return rows


from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from scipy import sparse
from collections import Counter
import copy
VIEWS16=['legacy_raw','hierarchical','rare_backoff','rare_backoff_panels']
# Literature-derived panel snapshots are explicit priors, not learned class labels.
class Preprocess16:
 def __init__(self,view,min_df=3,k=4096):self.view=view;self.min_df=min_df;self.k=k
 def fit(self,rows,target):
  self.df=Counter((g,e[0]) for row in rows for g,evs in row['mutations'] for e in {e[0]:e for e in evs}.values())
  records=self.records(rows,training=True);self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(records);names=self.vec.get_feature_names_out();support=np.asarray((a!=0).sum(0)).ravel();keep=(support>=self.min_df)|np.array([s.startswith(('num:','module:','numeric:','panel:')) for s in names]);self.keep=np.flatnonzero(keep);a=a[:,self.keep]
  with warnings.catch_warnings():
   warnings.simplefilter('ignore');scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)
  self.cols=np.sort(np.argsort(-scores,kind='stable')[:min(self.k,a.shape[1])]);self.names=names[self.keep[self.cols]];selected=a[:,self.cols];self.scale=MaxAbsScaler().fit(selected);self.residual_weights=np.ones(len(self.names),np.float32)
  if self.view.startswith('rare_backoff'):
   for j,name in enumerate(self.names):
    if name.startswith('variant:'):
     _,gene,key=name.split(':',2);n=self.df.get((gene,key),0);self.residual_weights[j]=n/(n+5.)
  return self
 def records(self,rows,training=False):
  # legacy records were computed pointwise using the frozen v6 raw encoder.
  if self.view=='legacy_raw':return [r['legacy'] for r in rows]
  result=[]
  for row in rows:
   d={};types=Counter();gene_effects={};n=0;rare=0;new=0;uncertain=0;incomplete=0;components=0;ambiguous_x=0
   for g,evs in row['mutations']:
    evs=list({e[0]:e for e in evs}.values());d['gene:'+g]=1.;d['gene_event_count:'+g]=np.log1p(len(evs));local=Counter();bin64=set();bin256=set()
    for key,kind,pos,parts,flags,fsdist in evs:
     n+=1;support=max(0,self.df.get((g,key),0)-(int(training)));israre=support<3;rare+=israre;new+=support==0;uncertain+='uncertain' in flags;incomplete+=kind in ['incomplete','unparsed','complex_range'];ambiguous_x+='ambiguous_X' in flags
     effects=[p[3] for p in parts] or [kind];types.update(effects);local.update(effects);d['format:'+g+':'+kind]=1.;components+=len(parts)
     if self.df.get((g,key),0)>=3:
      d['variant:'+g+':'+key]=1.
     if self.view.startswith('rare_backoff'):
      bucket='rare' if israre else 'supported';d[bucket+'_gene:'+g]=d.get(bucket+'_gene:'+g,0)+1
     for p,a,b,effect in parts:
      d['aa_meta:'+g+':'+a+'>'+b]=d.get('aa_meta:'+g+':'+a+'>'+b,0)+1;d['aa_global:'+a+'>'+b]=d.get('aa_global:'+a+'>'+b,0)+1
     if pos is not None:bin64.add((pos-1)//64);bin256.add((pos-1)//256)
     # Absolute protein bins are an engineering heuristic, not genomic bins or biological domains.
     if fsdist is not None:d['fs_stop_distance:'+g]=np.log1p(fsdist)
    for kind,count in local.items():d['effect:'+g+':'+kind]=np.log1p(count)
    for b in bin64:d[f'protein_bin64:{g}:{b}']=1.
    for b in bin256:d[f'protein_bin256:{g}:{b}']=1.
    gene_effects[g]=local
   for name,count in [('observed_genes',len(row['mutations'])),('unique_events',n),('raw_tokens',row['raw_tokens']),('duplicate_tokens',row['duplicate_tokens']),('missing_genes',len(row['missing'])),('uncertain_events',uncertain),('incomplete_events',incomplete),('ambiguous_X',ambiguous_x),('resolved_components',components)]:d['num:log_'+name]=np.log1p(count)
   for kind in ['synonymous','missense','stop_gained','stop_lost','stop_retained','frameshift','deletion','insertion','delins','complex_range','incomplete','unparsed','ambiguous_residue']:
    d['num:log_'+kind]=np.log1p(types[kind]);d['num:fraction_'+kind]=types[kind]/max(sum(types.values()),1)
   if self.view.startswith('rare_backoff'):d['num:log_rare_events']=np.log1p(rare);d['num:rare_fraction']=rare/max(n,1);d['num:log_unseen_events']=np.log1p(new)
   if self.view.endswith('panels'):
    observed=set(gene_effects)
    for panel,gs in PANELS16.items():
     count=len(observed.intersection(gs));d['panel:'+panel+':mutated_genes']=np.log1p(count)
     flags=sum(bool(set(gene_effects.get(g,{})).intersection({'stop_gained','frameshift','deletion','delins'})) for g in gs);d['panel:'+panel+':truncation_notation_genes']=np.log1p(flags)
   result.append(d)
  return result
 def transform(self,rows,training=False):
  a=self.vec.transform(self.records(rows,training=training));a=a[:,self.keep][:,self.cols];return sparse.csr_matrix(self.scale.transform(a).multiply(self.residual_weights),dtype=np.float32)


def corrupt23(rows):
 # Support and nuisance pool come from the current original training fold only, without labels.
 df=Counter((g,e[0]) for row in rows for g,ev in row['mutations'] for e in {e[0]:e for e in ev}.values());pool={(g,e[0]):(g,e) for row in rows for g,ev in row['mutations'] for e in ev if df[(g,e[0])]<=2 and e[1] in ['missense','synonymous','synonymous_block']};pool=list(pool.values());assert pool;rng=np.random.default_rng(23023);aug=[]
 for row in rows:
  mutations=row['mutations'];keep=rng.random(len(mutations))>=.1
  if len(keep) and not keep.any():keep[rng.integers(len(keep))]=True
  retained={g:{e[0]:e for e in ev} for (g,ev),flag in zip(mutations,keep) if flag}
  for _ in range(int(rng.poisson(20))):
   g,e=pool[rng.integers(len(pool))]
   if g in row['missing']:continue
   retained.setdefault(g,{})[e[0]]=e
  evs=tuple((g,tuple(events23.values())) for g,events23 in retained.items());aug.append({'mutations':evs,'missing':row['missing'],'raw_tokens':sum(len(ev) for g,ev in evs),'duplicate_tokens':0})
 return aug

PROTOCOL30D={'version':30,'phase':'D shared literature-preprocessor fixed4','motivation':'after weak new-parser A/B outcomes, reuse proven v16 parser and rare-backoff feature principles; all4 estimators freshly trained, no legacy model pickle','members':['rich_lr','rich_lgb','rare_cnb','aug_ova'],'weights':[.108/.476,.108/.476,.06/.476,.2/.476],'weights_origin':'fixed renormalized v25 leaf weights, not new search','features':'one shared Preprocess16 rare_backoff minDF3 k8192; no panel inputs; exact legacy parsing/canonicalization','lr':{'C':1,'max_iter':1600,'balanced':True},'lgb':{'trees':300,'leaves':15,'depth':5,'lambda':10,'seed':17017},'ova':{'trees':500,'leaves':15,'depth':5,'lambda':10,'is_unbalance':True,'seed':23023},'cnb':{'alpha':1,'norm':False,'temperature':2},'augmentation':'OVA only exact train-only corrupt23 recipe; same shared preprocessor fit originals only','cv_folds':5,'cv_seed':29032,'max_final_classifiers':4,'test_for_research':False,'external_patient_pretrained_or_panels':False,'twin_rule':False,'limitations':'adaptive same-patient comparison after A/B; D historical four-member recipe previously investigated, not independent novel confirmation'}
def canonical_frame30d(frame):
 def one(v):
  if not isinstance(v,str) or 'X' not in v:return v
  return ' '.join(re.sub(r'^(p\.)?([ACDEFGHIKLMNPQRSTVWY])([1-9][0-9]*)X$',lambda m:(m[1] or '')+m[2]+m[3]+'*',t) for t in v.split())
 return frame.map(one)
class SharedFour30:
 def __init__(self,genes,labels):self.genes_=list(genes);self.classes_=np.asarray(labels);self.weights_={'rich_lr':.108/.476,'rich_lgb':.108/.476,'rare_cnb':.06/.476,'aug_ova':.2/.476};self.members_={}
 def fit(self,rows,y):
  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,y);a=self.pre.transform(rows,training=True);aug=self.pre.transform(corrupt23(rows),training=False);at=sparse.vstack([a,aug],format='csr');yt=np.r_[y,y];self.fit_seconds_={}
  models={'rich_lr':LogisticRegression(C=1.,class_weight='balanced',max_iter=1600),'rich_lgb':LGBMClassifier(n_estimators=300,num_leaves=15,max_depth=5,learning_rate=.04,reg_lambda=10.,min_child_samples=20,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=17017),'rare_cnb':ComplementNB(alpha=1.,norm=False),'aug_ova':LGBMClassifier(objective='multiclassova',is_unbalance=True,n_estimators=500,learning_rate=.04,num_leaves=15,max_depth=5,min_child_samples=20,reg_lambda=10.,colsample_bytree=.85,verbosity=-1,n_jobs=2,random_state=23023)}
  for name,m in models.items():
   start=time.perf_counter()
   if name=='aug_ova':m.fit(at,yt,sample_weight=np.r_[np.ones(len(y)),.5*np.ones(len(y))]/1.5)
   else:m.fit(a,y)
   self.members_[name]=m;self.fit_seconds_[name]=time.perf_counter()-start
  self.feature_count_=len(self.pre.names);return self
 def predict_rows(self,rows):
  from scipy.special import softmax
  a=self.pre.transform(rows);p=np.zeros((len(rows),len(self.classes_)))
  for name,m in self.members_.items():
   q=softmax(m.predict_log_proba(a)/2.,axis=1) if name=='rare_cnb' else m.predict_proba(a);q/=q.sum(1,keepdims=True);p+=self.weights_[name]*q
  return p
 def predict_frame(self,frame):return self.predict_rows(parse_rows16(canonical_frame30d(frame.loc[:,self.genes_])))

def guard30d(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('D research forbids test/sample')
sys.addaudithook(guard30d)
(OUT30/'core30d.py').write_text(CORE30D);(OUT30/'worker30d.py').write_text(WORKER30D)
with (OUT30/'parsed_train.pkl').open('rb') as f:data30=cloudpickle.load(f)
sha_d30=hashlib.sha256((ROOT30/'data/train.csv').read_bytes()).hexdigest();assert sha_d30==data30['sha'];source_sha_d30=hashlib.sha256(CORE30D.encode()).hexdigest();meta_path_d30=OUT30/'d_cache_identity.json';valid_cache_d30=False
if meta_path_d30.exists() and (OUT30/'parsed_train_d.pkl').exists():
 oldmeta=json.loads(meta_path_d30.read_text());valid_cache_d30=bool(oldmeta['train_sha256']==sha_d30 and oldmeta['source_sha256']==source_sha_d30 and oldmeta['protocol']==PROTOCOL30D and oldmeta['parsed_sha256']==hashlib.sha256((OUT30/'parsed_train_d.pkl').read_bytes()).hexdigest())
if not valid_cache_d30:
 train_d30=pd.read_csv(ROOT30/'data/train.csv');assert train_d30.ID.tolist()==data30['ids'];rows_d30=parse_rows16(canonical_frame30d(train_d30.loc[:,data30['genes']]))
 with (OUT30/'parsed_train_d.pkl').open('wb') as f:cloudpickle.dump(rows_d30,f)
 meta={'train_sha256':sha_d30,'source_sha256':source_sha_d30,'protocol':PROTOCOL30D,'parsed_sha256':hashlib.sha256((OUT30/'parsed_train_d.pkl').read_bytes()).hexdigest()};meta_path_d30.write_text(json.dumps(meta,indent=2));del train_d30,rows_d30
(OUT30/'protocol_v30d.json').write_text(json.dumps(PROTOCOL30D,indent=2));pending_d30=[];active_d30=[]
for fold in range(5):
 if valid_cache_d30 and (OUT30/f'd_fold{fold}.json').exists() and (OUT30/f'd_fold{fold}.npz').exists():
  with np.load(OUT30/f'd_fold{fold}.npz') as q,np.load(OUT30/f'split{fold}.npz') as sp:assert np.array_equal(q['tr'],sp['tr']) and np.array_equal(q['va'],sp['va'])
  print('VALIDATED COMPLETED D FOLD',fold,flush=True)
 else:pending_d30.append(fold)
while pending_d30 or active_d30:
 while pending_d30 and len(active_d30)<2:
  fold=pending_d30.pop(0);h=(OUT30/f'd_worker{fold}.log').open('w');env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2');p=subprocess.Popen([sys.executable,str(OUT30/'worker30d.py'),str(OUT30),str(fold)],stdout=h,stderr=subprocess.STDOUT,env=env);active_d30.append((p,h,fold));print('START D',fold,flush=True)
 done=[]
 for item in active_d30:
  p,h,fold=item
  if p.poll() is not None:
   h.close()
   if p.returncode:raise RuntimeError('D failed '+str(fold))
   done.append(item)
 active_d30=[x for x in active_d30 if x not in done]
 if active_d30:time.sleep(2)
records_d30=[];dev_d30=data30['dev'];pos_d30={i:j for j,i in enumerate(dev_d30)};oof_d30=np.zeros((len(dev_d30),26));paired_d30=True
for fold in range(5):
 record=json.loads((OUT30/f'd_fold{fold}.json').read_text());records_d30.append(record)
 with np.load(OUT30/f'd_fold{fold}.npz') as q:
  tr=q['tr'];va=q['va'];oof_d30[[pos_d30[i] for i in va]]=q['probabilities'];refpath=WORK30/f'artifacts/v29/followup_fold{fold}.npz'
  if refpath.exists():
   with np.load(refpath) as ref:assert np.array_equal(tr,ref['tr']) and np.array_equal(va,ref['va']) and np.array_equal(ref['classes'].astype(str),data30['labels'].astype(str));record['v29_f1']=score30(data30['y'][va],ref['candidate']);record['delta_vs_v29']=record['f1']-record['v29_f1']
  else:paired_d30=False
pd.DataFrame(records_d30).to_csv(OUT30/'d_outer_results.csv',index=False);np.savez_compressed(OUT30/'d_oof.npz',dev=dev_d30,probabilities=oof_d30,classes=data30['labels']);before_d30=json.loads((OUT30/'verification_v30.json').read_text());dproof30={'mean_f1':float(np.mean([r['f1'] for r in records_d30])),'mean_accuracy':float(np.mean([r['accuracy'] for r in records_d30])),'pooled_f1':score30(data30['y'][dev_d30],oof_d30),'reference_paired':paired_d30,'protocol':PROTOCOL30D};dproof30['select_d']=bool(dproof30['mean_f1']>=before_d30['mean_f1']+.003)
if paired_d30:
 dproof30.update(v29_mean_f1=float(np.mean([r['v29_f1'] for r in records_d30])),mean_delta_vs_v29=float(np.mean([r['delta_vs_v29'] for r in records_d30])),all_fold_deltas=[r['delta_vs_v29'] for r in records_d30]);dproof30['improvement_gate_passed']=bool(dproof30['mean_delta_vs_v29']>=.003 and min(dproof30['all_fold_deltas'])>=-.003)
else:dproof30['improvement_gate_passed']=False
(OUT30/'verification_v30d.json').write_text(json.dumps(dproof30,indent=2));print('SHARED4 D RESULT',json.dumps(dproof30,indent=2),flush=True)
finalproof30=dict(before_d30);finalproof30['shared_four_result']=dproof30
if dproof30['select_d']:
 preserve=OUT30/'before_d';preserve.mkdir(exist_ok=True);shutil.copy2(OUT30/'cancer_model_v30_compact_experimental.pkl',preserve/'cancer_model_v30_before_d.pkl');(preserve/'verification.json').write_text(json.dumps(before_d30,indent=2))
 with (OUT30/'parsed_train_d.pkl').open('rb') as f:rows_d30=cloudpickle.load(f)
 start=time.perf_counter();final_d30=SharedFour30(data30['genes'],data30['labels']).fit(rows_d30,data30['y']);fit_d30=time.perf_counter()-start;final_d30.training_data_sha256_=data30['sha'];final_d30.training_protocol_=PROTOCOL30D;path=OUT30/'cancer_model_v30_compact_experimental.pkl'
 with path.open('wb') as f:cloudpickle.dump(final_d30,f)
 with path.open('rb') as f:reload_d30=cloudpickle.load(f)
 assert np.allclose(final_d30.predict_rows(rows_d30[:8]),reload_d30.predict_rows(rows_d30[:8]),atol=1e-8)
 probe=pd.DataFrame('WT',index=[0,1],columns=data30['genes']);probe.loc[0,data30['genes'][0]]='R1450X';probe.loc[1,data30['genes'][0]]='R1450*';p=reload_d30.predict_frame(probe);assert np.allclose(p[:1],p[1:],atol=1e-8) and np.allclose(p[:1],reload_d30.predict_frame(probe.iloc[:1]),atol=1e-8)
 names=final_d30.pre.names;weights=final_d30.pre.residual_weights;df=np.array([final_d30.pre.df.get(tuple(n.split(':',2)[1:]),np.nan) if n.startswith('variant:') else np.nan for n in names]);unscaled_d30=final_d30.pre.vec.transform(final_d30.pre.records(rows_d30,training=True))[:,final_d30.pre.keep][:,final_d30.pre.cols];support_d30=np.asarray((unscaled_d30>0).sum(0)).ravel();chi_d30=np.nan_to_num(chi2(unscaled_d30,data30['y'])[0],nan=0.,posinf=0.,neginf=0.);pd.DataFrame({'feature':names,'train_df':support_d30,'chi2_training':chi_d30,'exact_event_train_df':df,'exact_residual_weight':weights}).to_csv(OUT30/'d_selected_variables.csv',index=False)
 pd.DataFrame(classification_report(data30['y'][dev_d30],oof_d30.argmax(1),labels=np.arange(26),target_names=data30['labels'],output_dict=True,zero_division=0)).T.to_csv(OUT30/'d_classification_report.csv');pd.DataFrame(confusion_matrix(data30['y'][dev_d30],oof_d30.argmax(1),labels=np.arange(26)),index=data30['labels'],columns=data30['labels']).to_csv(OUT30/'d_confusion_matrix.csv')
 finalproof30.update(phase='D shared literature-preprocessor fixed4 adaptive followup',classifiers=4,final_config={'name':'rich_lr|rich_lgb|rare_cnb|aug_ova','members':list(final_d30.members_),'count':4,'weights':final_d30.weights_},mean_f1=dproof30['mean_f1'],mean_accuracy=dproof30['mean_accuracy'],pooled_f1=dproof30['pooled_f1'],reference_paired=paired_d30,v29_mean_f1=dproof30.get('v29_mean_f1',float('nan')),mean_delta_vs_v29=dproof30.get('mean_delta_vs_v29',float('nan')),all_fold_deltas=dproof30.get('all_fold_deltas',[]),improvement_gate_passed=dproof30['improvement_gate_passed'],model_bytes=path.stat().st_size,model_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),final_features=final_d30.feature_count_,full_train_fit_seconds=fit_d30,fit_breakdown=final_d30.fit_seconds_,reload_notation_batch_checks=True,model_saved=True,selected_after_shared_four='D4')
 finalproof30['limitations']='adaptive comparison after A/B/C and historical4 recipe investigations; same development patients; selected with development comparisons, not unbiased final or external score'
(OUT30/'verification_v30.json').write_text(json.dumps(finalproof30,indent=2));print('FINAL ACTIVE V30',json.dumps(finalproof30,indent=2),flush=True)


VALIDATED COMPLETED D FOLD 0
VALIDATED COMPLETED D FOLD 1
VALIDATED COMPLETED D FOLD 2
VALIDATED COMPLETED D FOLD 3
VALIDATED COMPLETED D FOLD 4
SHARED4 D RESULT {
  "mean_f1": 0.4892161920883584,
  "mean_accuracy": 0.4903225806451613,
  "pooled_f1": 0.49265246573881966,
  "reference_paired": true,
  "protocol": {
    "version": 30,
    "phase": "D shared literature-preprocessor fixed4",
    "motivation": "after weak new-parser A/B outcomes, reuse proven v16 parser and rare-backoff feature principles; all4 estimators freshly trained, no legacy model pickle",
    "members": [
      "rich_lr",
      "rich_lgb",
      "rare_cnb",
      "aug_ova"
    ],
    "weights": [
      0.226890756302521,
      0.226890756302521,
      0.12605042016806722,
      0.4201680672268908
    ],
    "weights_origin": "fixed renormalized v25 leaf weights, not new search",
    "features": "one shared Preprocess16 rare_backoff minDF3 k8192; no panel inputs; exact legacy parsing/canonicalization",
    "lr": {
  

## 최종 고정 모델 추론과 보고서

연구 중 test/sample 접근 차단을 유지하며, 별도 추론 프로세스만 test/sample을 읽습니다. 추론 프로세스는 train을 읽거나 fit하지 않습니다. 실패한 비교도 기존 버전과 별도로 보존합니다.

In [6]:
import subprocess,tempfile
ROOT30,WORK30,OUT30=locate30()
EXPORT_SOURCE30="import os,sys,json,hashlib,time\nfrom pathlib import Path\nimport numpy as np,pandas as pd,cloudpickle\nroot=Path(sys.argv[1]);work=Path(sys.argv[2]);out=work/'artifacts/v30'\ndef guard_export30(event,args):\n if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name=='train.csv':raise RuntimeError('Final inference must not load training data')\nsys.addaudithook(guard_export30)\nproof=json.loads((out/'verification_v30.json').read_text());path=out/'cancer_model_v30_compact_experimental.pkl';assert hashlib.sha256(path.read_bytes()).hexdigest()==proof['model_sha256']\nwith path.open('rb') as f:model=cloudpickle.load(f)\nassert len(model.members_)==proof['classifiers']<=4\nx=pd.read_csv(root/'data/test.csv');template=pd.read_csv(root/'data/sample_submission.csv');assert x.ID.is_unique and template.ID.is_unique;assert x.ID.astype(str).tolist()==template.ID.astype(str).tolist();assert len(x)==2546\n# No fitting, thresholds, cohort matching or test-based distribution correction here.\na=time.perf_counter();p=model.predict_frame(x);seconds=time.perf_counter()-a;assert np.allclose(p.sum(1),1) and np.isfinite(p).all();assert np.allclose(p[:7],model.predict_frame(x.iloc[:7]),atol=1e-7)\nsubmission=template.copy();submission['SUBCLASS']=model.classes_[p.argmax(1)];assert set(submission.SUBCLASS)<=set(model.classes_);target=root/'submission/혜진_submission_v30_compact_experimental.csv';target.parent.mkdir(exist_ok=True);submission.to_csv(target,index=False)\nrecord={'version':30,'rows':len(submission),'classifiers':len(model.members_),'members':list(model.members_),'phase':'final inference only','test_for_training_or_validation':False,'id_order_checked':True,'batch_invariance':True,'test_prediction_seconds':seconds,'model_sha256':proof['model_sha256'],'csv_sha256':hashlib.sha256(target.read_bytes()).hexdigest(),'public_score':None};(out/'submission_verification.json').write_text(json.dumps(record,ensure_ascii=False,indent=2));print(json.dumps(record,ensure_ascii=False,indent=2))\n"
(OUT30/'export30.py').write_text(EXPORT_SOURCE30)
env30=os.environ.copy();env30.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2')
with tempfile.TemporaryDirectory(prefix='v30-infer-') as empty_cwd30:
 r30=subprocess.run([sys.executable,str(OUT30/'export30.py'),str(ROOT30),str(WORK30)],check=True,capture_output=True,text=True,env=env30,cwd=empty_cwd30)
print(r30.stdout)
def markdown30(frame,index=False):
 safe=frame.copy()
 for column in safe.columns:
  if safe[column].dtype==object or str(safe[column].dtype).startswith('str'):safe[column]=safe[column].map(lambda v:v.replace('|',r'\|') if isinstance(v,str) else v)
 return safe.to_markdown(index=index)
active30=json.loads((OUT30/'verification_v30.json').read_text());v30=json.loads((OUT30/'verification_v30b.json').read_text());a30=v30['primary_a_result'];outer30=pd.read_csv(OUT30/'b_outer_results.csv').drop(columns=['train_sha256','protocol_sha256'],errors='ignore');aouter30=pd.read_csv(OUT30/'primary_a/nested_outer_results.csv');features30=pd.read_csv(OUT30/'b_selected_variables.csv');eda30=pd.read_csv(OUT30/'eda_development.csv');classes_report30=pd.read_csv(OUT30/'b_classification_report.csv',index_col=0);families30=features30.assign(family=features30.feature.str.split('|').str[0]).groupby('family').size().rename('features').reset_index();export30=json.loads((OUT30/'submission_verification.json').read_text());cm30=pd.read_csv(OUT30/'b_confusion_matrix.csv',index_col=0);confusions30=[]
for true30 in cm30.index:
 for pred30 in cm30.columns:
  if true30!=pred30 and cm30.loc[true30,pred30]>0:confusions30.append({'true':true30,'predicted':pred30,'patients':int(cm30.loc[true30,pred30]),'fraction_within_true_class':float(cm30.loc[true30,pred30]/cm30.loc[true30].sum())})
confusions30=pd.DataFrame(confusions30).sort_values('patients',ascending=False).head(15)
refs30='''| 문헌 검토 ID/출처 | 참고한 내용 | 실제 적용·한계 |
|---|---|---|
| [P03](https://doi.org/10.3389/fgene.2020.00674) | mutation 선별·단순 LR 기준선 | LR 비교/고정3개 구성. 저자의 gene 목록/성능 차용 아님 |
| [P05](https://doi.org/10.1111/biom.13367) | 미관측 희귀 변이를 공유 meta-feature에 연결 | gene/type/AA/bin + exact residual + train DF 희귀 back-off. Bayesian/group-lasso 재현 아님 |
| [P06](https://doi.org/10.1038/s41598-019-53034-3), [P10](https://doi.org/10.1016/j.jbi.2019.103247) | 관측 event count·mutation 기반 분류 | gene binary/log unique count와 burden 보존. CNV·NMF·기능 점수 미사용 |
| [P17](https://doi.org/10.1038/nature12213), [P19](https://doi.org/10.1038/nature12634), [P20](https://doi.org/10.1016/j.cell.2017.09.042) | 변이량 이질성·유형·동의 변이 구분 | 부담량/유형 수·분율 별도 유지. synonymous 미삭제. driver·병원성 판정 아님 |
| [P41](https://doi.org/10.1093/nar/gkq603), [P42](https://doi.org/10.1186/s13059-016-0974-4), [P43](https://doi.org/10.1002/humu.22981) | 참조정보·표기와 불확실성 구별 | stop gained/lost/retained, unparsed 보존. 정식 HGVS validator가 아니며 위치 이동 금지 |
| [P47](https://doi.org/10.1073/pnas.102102699), [P48](https://doi.org/10.1186/1471-2105-7-91), [P49](https://www.jmlr.org/papers/v11/cawley10a.html), [P50](https://doi.org/10.1016/j.patter.2023.100804) | 선별 누수·선택 편향·재현성 | 어휘/DF/chi2/스케일을 학습부에서만 fit. A nested 선택 평가와 B 고정 조합 평가를 구분. 적응적 후속 편향은 남음 |
'''
report30=f'''# v30 — 처음부터 다시 구성한 소형 앙상블과 실패/후속 기록

## 최종 결과와 채택 판단

최종 B 모델은 **3개**: `rare_lr + rare_ova + rare_mlp`, **1/3씩 균등 확률 평균**입니다. 하나의 parser·희귀 변이 전처리·선택 변수 행렬을 공유합니다. 기존 v25/v29 학습 분류기나 피클을 가져와 감싼 모델이 아닙니다.

| 비교 | 평균 Macro F1 | 평균 accuracy | 평가 의미 |
|---|---:|---:|---|
| v29 9개 | {v30.get('v29_mean_f1',float('nan')):.6f} | 기존 v29 보고서 참조 | 동일 train/validation의 실제 OOF |
| v30 A: 35개 소형 조합 선택 과정 | {a30['nested_mean_f1']:.6f} | {a30['nested_mean_accuracy']:.4%} | inner2/outer5 nested, 조합 선택 과정의 성능 |
| v30 B: 고정3개 희귀 변이 후속 | **{v30['mean_f1']:.6f}** | **{v30['mean_accuracy']:.4%}** | A 일부 결과를 보고 설계한 적응적 후속의 그룹5fold |

- B−v29 평균 **{v30.get('mean_delta_vs_v29',float('nan')):+.6f}**, fold 차이 `{v30.get('all_fold_deltas')}`, 짝지은 참조 확인 **{v30['reference_paired']}**.
- 공학적 개선 기준: 평균 +0.003 이상·모든 fold -0.003 이상. 통과 **{v30['improvement_gate_passed']}**. 실패하면 기존 성능 모델의 교체/개선을 주장하지 않습니다.
- B pooled OOF Macro F1 **{v30['pooled_f1']:.6f}**. B는 고정3개 구성이지만 adaptive history가 있어 외부 독립 확인이나 선택 편향 없는 최고점으로 표현하지 않습니다.
- Public 미제출/미확인. v25 실제 **0.390417094**, v19 실제 **0.3664759128**과 로컬 점수를 구분합니다.
- 모델 {v30['model_bytes']:,} bytes, 공유 선택 변수 {v30['final_features']}개. 최종 train6201명 학습·피클 저장·재로딩·제출 CSV까지 완료했습니다.

## 요구사항과 범위 변경을 숨기지 않기

사용자는 '처음부터 단순하게, 단일 모델일 필요는 없고 최대4개까지, 5개는 먼저 문의'라고 정했습니다. 최종은3개이며5개를 추가하지 않았습니다. 기존 모델은 보존합니다.

초기에는 단일 모델 제한으로 시작했다가 사용자 설명에 따라 outer 점수 전 중단했습니다. 일부 inner 결과는 확인했고 로그를 `aborted_single_only/`에 보관했습니다. 그 다음 문자열 group-ID pilot은 SGKF 정렬이 달라져 기존 v29와 분할이 일치하지 않았습니다. 일부 pilot outer 점수를 확인한 뒤 중단하고, group ID의 원래 숫자 정렬을 복구했습니다. 해당 partial 결과는 `pilot_string_group_ids/`에 남기며 최종 대조에 섞지 않습니다.

수정한 A는 공유 AA/bin과 일부 train-only 증강을 추가한 후 nested5fold로 평가했습니다. B는 A 일부 outer 결과가 약한 것을 본 뒤, 과거 v16 문헌 설계와 v23~25의 희귀 변이 처리에서 배운 점을 새 코드로 구현한 후속입니다. B의3개 구성·파라미터·동일 가중치는 **B 점수를 보기 전에 고정**했습니다. 같은 과거 환자의 반복 평가와 적응적 설계 이력을 감추지 않습니다. 새 seed만으로 연구자 편향을 없앨 수 없습니다.

## 실패 기록에서 무엇을 배웠나

| 과거/이번 관찰 | 반영 |
|---|---|
| v19 로컬 개선이 Public에서 그대로 유지되지 않음 | 실제 Public와 로컬 점수 분리, 리더보드 향상 보장 안 함 |
| v25 표기 정합성 | 단일 치환 A123X만 A123*로 통일하는 기존 legacy 가정 유지. 모든 X를 stop으로 변환하지 않음 |
| v29 4개 pruning 후보가 추가 검증에서 탈락 | 소수 모델이라는 이유만으로 성능 유지 선언 안 함. 새 표현과 모델을 다시 학습 |
| v30 A의 단순 특징만으로 기존 앙상블을 넘지 못할 수 있음 | B에 학습 DF 기반 희귀/지원 표현·미관측 요약 보강. 기존 좋은 전처리 관점은 살리되 학습 모델은 재사용 안 함 |
| 무조건 행 정규화는 부담량 신호를 없앨 수 있음 | L1/L2 행 정규화 없이 burden와 유형 분율 모두 유지 |
| rare exact는 미래 환자에서 안 나올 수 있음 | gene/type/AA/bin back-off + residual attenuation. df/(df+5)는 자체 휴리스틱이지 병원성 점수 아님 |
| 많은 모델/비중 최적화는 운영 부담·선택 편향 | B3개 고정·동일 비중·shared preprocessor·메타 모델 없음 |
| 형제 라벨/쌍둥이 규칙의 일반화 위험 | 그룹 분리만 사용, 라벨 반전·정답 복사·강제 암종 비율 보정 미사용 |

## 논문 50편 기록을 다시 참고한 범위

`LITERATURE_REVIEW_AND_PREPROCESSING_V16.md`의50편별 요약·검토 범위·구현/미구현 경계를 재검토했습니다. 이번에50편 원문을 전부 새로 정독한 것은 아닙니다. 실제 v30 구현 연결은 아래와 같습니다.

{refs30}

외부 환자·정답·pretrained weights·PPI·CGC 테이블을 새로 추가하지 않았습니다. Geneformer를 사용하지 않습니다. 논문 방법을 참고하는 것과 외부 데이터 사용 규정은 구분합니다. DNA ref/alt·assembly·transcript·coverage가 없으므로 SBS96·CNV·MSI·TMB/Mb·SIFT/PolyPhen/CADD·단백질 위치 보정을 만들지 않습니다. 문헌상의 다른 cohort/accuracy를 본26종 MacroF1 기대치로 쓰지 않습니다.

## 입력·EDA·중복 그룹

원본 train6201행,4384개 gene,26개 SUBCLASS입니다. ID는 입력에서 제외하며 SUBCLASS는 정답으로만 사용합니다. 개발4960명에서 EDA/CV, 기존 보류1241명은 평가하지 않습니다. 최종 배포 학습에만 전체6201명을 사용합니다. 원본 CSV 수정 없음.

기존 보수적 그룹과 새 canonical profile을 union해 같은 프로필의 fold 간 중복을 방지했습니다. 추가 합쳐진 그룹/개발 제외는 없었고 v29의5개 train/validation 인덱스와 전부 일치하는 것을 확인했습니다. train checksum `{v30['train_sha256']}`. 그룹은 정답/test 없이 원본 프로필에서만 구성했습니다. 프로필 일치가 같은 환자임을 증명하는 것은 아닙니다.

{eda30.groupby('SUBCLASS')[['mutated_genes','unique_events']].agg(['count','median','mean']).to_markdown()}

## 파서와 변수군의 의미

공백/세미콜론/쉼표로 cell 토큰을 분리하고 pointwise canonicalization 후 환자·gene 내 같은 토큰을1개로 셉니다. 단일 AA substitution·stop gained/lost/retained·fs/delins/del/ins를 구분합니다. 복합/불완전 표기는 unparsed/exact로 보존하며 임의 위치 이동·서열 추정·분해를 하지 않습니다. 이 파서는 정식 HGVS validator가 아니고 새로운 표기에서는 미해석률을 확인해야 합니다.

| 특징군 | 정의/의미 |
|---|---|
| gene/count | gene별 변이 있음 binary·서로 다른 canonical 토큰 수 log1p |
| type/effectcount | gene별 표기유형 binary 및 유형별 event 수 log1p |
| event | gene×canonical exact 표기. 학습 DF3 이상만 어휘 후보 |
| meta | gene×AA 치환, 단백질 표기 위치 bin64/256. 실제 domain·genomic bin 아님 |
| aa_global | gene을 공유한 AA 치환 수. DNA mutation signature 아님 |
| rare/supported | 해당 gene의 사건 중 학습 참조 DF<3/≥3 개수. 병적/양성 분류 아님 |
| summary | gene/event/미해석/결측/유형별 log 수·유형 분율·rare/unseen 요약 |
| missing | 결측 gene binary. WT와 구별 |

rare/unseen training 특징은 각 event DF에서 자기 자신1을 빼서 계산하고, validation/inference는 학습 DF만 사용합니다. labels로 rare pool/DF를 정하지 않습니다. 원본 학습부에서 DictVectorizer→DF≥3(summary 예외)→global chi2 상위 최대8192→MaxAbs를 fit합니다. exact event 열은 **MaxAbs 후** df/(df+5)를 곱합니다. 이는 임의 조합별 분류기를 더 넣는 것이 아니라 하나의 공유 전처리입니다.

{markdown30(families30)}

최종 전체 변수·원래 학습 DF·chi2·residual weight는 `b_selected_variables.csv`와 `selected_variables_v30.md`에 기록했습니다. chi2는 표본 내 라벨 연관성이지 병원성·인과성이나 임상 진단 근거가 아닙니다.

## 학습부 내부 증강

OVA/MLP만 원본 행 + 증강 행으로 fit합니다. 원본 학습 fold에서 DF2 이하 missense/synonymous event를 nuisance pool로 정하고, gene mask0.1 및 Poisson20 nuisance를 사용합니다. 원래 gene이 있으면 최소1개를 남기고, missing gene에는 nuisance를 넣지 않습니다. 변형된 event에서 gene/type/AA/bin/count/summary를 다시 계산해 채널이 서로 모순되지 않게 합니다.

어휘/DF/chi2/MaxAbs는 원래 학습 행에서만 fit하고, 증강에는 transform만 수행합니다. 원본1/증강0.5 가중치이며 MLP는 원본 train class-balanced weight를 적용합니다. OVA는 is_unbalance를 사용합니다. LR는 증강 없이 원본 class-balanced fit입니다. 증강이 실제 질병 생물학을 재현하거나 nuisance가 germline임을 뜻하지 않으며, label-preserving augmentation은 모델링 가정입니다. test/validation 원본은 증강하지 않습니다.

## 최종 B 모델 설정과 구조

1. LR: C1, balanced, max_iter1600.
2. LightGBM: multiclassova/is_unbalance,500trees,leaves15,depth5,min_child20,lambda10,learning_rate.04,colsample.85.
3. MLP: hidden128,alpha1,batch128,learning_rate.001,max_iter180,early_stoppingFalse,balanced sample weight.

세 확률을 각각 행합1로 맞춘 뒤 동일 비중으로 평균하고 argmax합니다. OVA는 API 객체1개 안에서26개 one-vs-all 문제를 학습하며 트리·파라미터 자체가3개뿐이라는 뜻은 아닙니다. MLP180epoch 상한은 수렴 보증이 아니며, 예측 확률은 임상적 확률로 calibration한 결과가 아닙니다. 모든 model seed는30031이며 MaxAbs/DF/어휘/선택 변수는 공유합니다.

v25/v29의 classifier나 pretrained 모델을 불러와 학습/추론하는 구조가 아닙니다. 일반 알고리즘과 과거 전처리의 좋은 원리를 차용해 새 코드로 전체 학습했습니다. 유지보수 대상은 parser1개·전처리1개·학습기3종·균등 평균이며 이전9개보다 개념 수를 줄였습니다. 라이브러리/표기 스키마 관리는 여전히 필요합니다.

## A의 nested 결과와 B 결과를 섞지 않기

A는 기본 설정6개와 family-distinct1~4개 조합35개를 균등 평균으로 비교했습니다. 매 outer 학습부 inner2-fold에서 최고점0.002 이내의 최소 모델 수를 선택했습니다. 아래 A 점수는 조합 선택 과정의 성능이며 마지막 배포 조합의 독립 점수가 아닙니다.

{markdown30(aouter30)}

A 마지막 구성 `{a30['final_config']['name']}`, 모델 수 {a30['classifiers']}. 피클과 모든 결과는 `artifacts/v30/primary_a/`에 보존했습니다. B 결과로 A를 성공으로 고치지 않습니다.

B는 고정3개를 동일한 그룹5fold에서 새로 fit했습니다. v29 참조는 전체 train 모델 예측이 아니라 당시의 실제 OOF만 읽고 인덱스·class 순서·train SHA를 확인합니다. 참조 캐시가 없어도 B 학습은 가능하며 대조만 생략합니다.

{markdown30(outer30)}

## 오류 진단

{markdown30(classes_report30,index=True)}

큰 혼동 흐름:

{markdown30(confusions30)}

같은 입력의 다른 라벨·암종 간 공유 변이·관측 정보 부족이 가능한 한계지만, 이 표만으로 원인을 확정하거나 진짜 오진이라고 부르지 않습니다. 결과를 보고 라벨 반전/쌍별 추가 전문가를 붙이지 않았습니다. adaptive 같은 환자 반복 평가라 외부 코호트 일반화·임상적 안전성·리더보드 향상을 보증하지 않습니다.

## 실행·저장·추론

`cencer_v30.ipynb`에 A와 B의 source·EDA·선별·검증·학습·모델 저장·최종 추론·문서 생성이 모두 있습니다. 외부 프로젝트 py를 요구하지 않으며 worker는 artifacts 아래 자동 생성합니다. 원본 data와 requirements 환경이 필요합니다. 참조 v29 캐시는 대조용 선택 사항입니다.

- Python {v30['python']}, requirements-v30.txt.
- 피클 `artifacts/v30/cancer_model_v30_compact_experimental.pkl`, {v30['model_bytes']:,} bytes, checksum `{v30['model_sha256']}`.
- 전체 train6201명에서 고정3개만 새로 fit. fit시간 {v30['full_train_fit_seconds']:.2f}초(파싱·nested 탐색 제외), 개별 `{v30['fit_breakdown']}`.
- 독립 추론 프로세스에서 모델만 로드해 test2546명을 예측. 해당 프로세스는 train open을 차단하고 fit하지 않음. 연구 프로세스는 test/sample open을 차단.
- 최종 `submission/혜진_submission_v30_compact_experimental.csv`, {export30['rows']}행. ID·순서·클래스·확률합·배치 불변성·X/* 예측 일치·재로딩·모델/CSV hash 검사.
- test 빈도/분포·정답·형제 label 규칙 미사용. raw CSV와 이전 모델/제출 파일 보존. Public 미확인으로 실험용 파일을 기존 최고 Public 모델로 덮어쓰지 않음.
- 공유 표기 parser와 train DF back-off 외에 Geneformer/외부 환자/정답 사전/COAD prior를 넣지 않음. 기존 가정의 규정 적합성과 임상 안전성은 별도 확인 대상.
'''
c30=json.loads((OUT30/'verification_v30c.json').read_text());ctable30=pd.read_csv(OUT30/'c_outer_results.csv')
if c30['add_fourth']:
 report30=report30.replace('최종 B 모델은','B 후보 모델은').replace('## 최종 결과와 채택 판단','## B 고정3개 후보 결과와 판단').replace('artifacts/v30/cancer_model_v30_compact_experimental.pkl','artifacts/v30/fixed3_b/cancer_model_v30_fixed3.pkl')
activeheader30=f'''# 최종 활성 v30: {active30['classifiers']}개 분류기

구성 `{active30['final_config']['name']}`. 동일 개발 그룹5-fold 관측 평균 MacroF1 **{active30['mean_f1']:.6f}**, accuracy **{active30['mean_accuracy']:.4%}**, v29 대비 **{active30.get('mean_delta_vs_v29',float('nan')):+.6f}**. 개선 기준 통과 **{active30['improvement_gate_passed']}**. Public 미확인. 아래 B와 선택적 C를 비교해 선택한 개발 결과이며 독립 최종 성능 추정치가 아닙니다. 미통과면 기존 모델의 성능 개선/교체를 주장하지 않습니다.

활성 모델 `artifacts/v30/cancer_model_v30_compact_experimental.pkl`, **{active30['model_bytes']:,} bytes**, checksum `{active30['model_sha256']}`. 실제 최종 CSV는 이 활성 모델에서 생성했습니다.

## C: 선택적 네 번째 CNB를 넣을 가치가 있는가

B3의 확률에0.85, RareFeatures 기반 ComplementNB(alpha1,normFalse,temperature2)에0.15를 고정했습니다. 이 네 번째 후보는 raw-TFIDF CNB와 다른 공유 희귀 표현입니다. 비중 grid 탐색은 하지 않았습니다. C의 CNB 전처리는 각 fold 학습 행에서 다시 fit하고, 전체train B 전처리를 CV에 사용하지 않습니다. B 대비 평균+0.003 이상/최악fold-0.003 이상일 때만 추가하는 개발 판단이며, 같은 결과로 선택한 편향이 남습니다. protocol은 C 점수 전에 저장했고 A/B의 일부 약한 결과를 본 뒤 설계한 적응적 후속입니다.

C 평균 F1 **{c30['mean_f1']:.6f}**, B 대비 **{c30['mean_delta_c_minus_b']:+.6f}**, 최악 차이 **{c30['worst_delta_c_minus_b']:+.6f}**. 네 번째 추가 **{c30['add_fourth']}**. 5개는 미사용입니다.

{markdown30(ctable30)}

---

'''
report30=activeheader30+report30
(WORK30/'analysis_report_v30.md').write_text(report30)
variables30=f'''# v30 B 최종 공유 변수

활성 {active30['classifiers']}개 `{active30['final_config']['name']}`가 아래 동일 전처리/변수 {v30['final_features']}개를 공유합니다. ID는 입력에서 제외하고 SUBCLASS는 train 정답으로만 사용합니다. 학습 DF·chi2·MaxAbs는 fold 학습부에서만 fit합니다. rare/unseen train 통계는 자기 자신 제외, 추론에는 학습 DF만 씁니다. exact weight는 MaxAbs 후 df/(df+5)이며 다른 열은1입니다. 병원성/인과성 값이 아닙니다.

{markdown30(families30)}

{markdown30(features30)}
'''
(WORK30/'selected_variables_v30.md').write_text(variables30);print('v30 A/B full report, selected features, model and submission saved.');display(outer30)

d30=json.loads((OUT30/'verification_v30d.json').read_text());active30=json.loads((OUT30/'verification_v30.json').read_text())
if d30['select_d']:
 dtable30=pd.read_csv(OUT30/'d_outer_results.csv');dfeatures30=pd.read_csv(OUT30/'d_selected_variables.csv');dclasses30=pd.read_csv(OUT30/'d_classification_report.csv',index_col=0);dfamilies30=dfeatures30.assign(family=dfeatures30.feature.str.split(':').str[0]).groupby('family').size().rename('features').reset_index()
 ci_boot30=[]
 if active30['reference_paired'] and all((WORK30/f'artifacts/v29/followup_fold{i}.npz').exists() for i in range(5)):
  with (OUT30/'parsed_train.pkl').open('rb') as f:bootstrap_data30=cloudpickle.load(f)
  with np.load(OUT30/'d_oof.npz') as q:dev_boot30=q['dev'];dp_boot30=q['probabilities']
  rp_boot30=np.zeros_like(dp_boot30);pos_boot30={i:j for j,i in enumerate(dev_boot30)}
  for fold in range(5):
   with np.load(WORK30/f'artifacts/v29/followup_fold{fold}.npz') as q:rp_boot30[[pos_boot30[i] for i in q['va']]]=q['candidate']
  group_boot30=bootstrap_data30['groups'][dev_boot30];ug_boot30=np.unique(group_boot30);gpositions30={g:np.flatnonzero(group_boot30==g) for g in ug_boot30};rng_boot30=np.random.default_rng(30034);deltas_boot30=[]
  for _ in range(500):
   ix=np.concatenate([gpositions30[g] for g in rng_boot30.choice(ug_boot30,len(ug_boot30),replace=True)]);target=bootstrap_data30['y'][dev_boot30[ix]];deltas_boot30.append(score30(target,dp_boot30[ix])-score30(target,rp_boot30[ix]))
  ci_boot30=np.quantile(deltas_boot30,[.025,.975]).tolist();active30['conditional_pooled_delta_ci95']=ci_boot30;active30['bootstrap_note']='conditional fixed OOF, grouped resampling500 seed30034; excludes fitting/selection/adaptive-design uncertainty';(OUT30/'verification_v30.json').write_text(json.dumps(active30,indent=2))
 dsection30=f'''# v30 최종 활성 후보 — 공유 전처리1개 + 분류기4개

## 결과와 한계

| 이번 동일 그룹5-fold 비교 | 분류기 수 | 평균 Macro F1 | 평균 accuracy |
|---|---:|---:|---:|
| v29 기존 구성 | 9 | {active30['v29_mean_f1']:.6f} | 48.6694% |
| A: 새 단순 표현·nested 소형 조합 선택 | 2~4 | {a30['nested_mean_f1']:.6f} | {a30['nested_mean_accuracy']:.4%} |
| B: 새 희귀 표현·고정3개 | 3 | {v30['mean_f1']:.6f} | {v30['mean_accuracy']:.4%} |
| C: B + 보조 CNB 고정15% | 4 | {c30['mean_f1']:.6f} | {c30['mean_accuracy']:.4%} |
| **D: 검증된 문헌 전처리 차용·공유4개** | **4** | **{active30['mean_f1']:.6f}** | **{active30['mean_accuracy']:.4%}** |

D−v29 평균 차이는 **{active30['mean_delta_vs_v29']:+.6f}**, pooled F1 **{active30['pooled_f1']:.6f}**입니다. 평균+0.003/최악fold-0.003의 기존 공학적 개선 기준 통과 **{active30['improvement_gate_passed']}**. 작은 평균 상승을 확실한 성능 개선이라고 주장하지 않습니다. 고정 OOF의 조건부 그룹 bootstrap95% pooled 차이 구간은 `{ci_boot30}`입니다. 이는 학습·선택·적응적 설계 불확실성을 포함하지 않고 fold 평균 차이의 CI도 아닙니다.

A/B/C가 기대보다 낮아 기존 전처리의 좋은 부분까지 버리지 않고 차용했습니다. D의4개 recipe/기존 비중은 과거 v29의4개 후보에서도 연구했던 이력이 있습니다. 그 과거 후보는 다른 split에서14개 baseline 대비 기준을 통과하지 못했습니다. 이번 결과가 과거 실패를 취소하거나 독립 재발견을 뜻하지 않습니다. 새로운 클래스에서 하나의 전처리를 공유하고4개를 **새로 fit**했으며, v25/v29 학습 피클이나14개 전체 모델을 감싸지 않습니다. D와 앞선 후보를 같은 개발 결과로 비교해 선택했으므로 최고 관측값에는 선택 편향이 남습니다.

{markdown30(dtable30)}

Public는 미확인입니다. v25 실제 **0.390417094**를 넘었다거나 리더보드0.50을 달성했다고 말하지 않습니다. 실험용 후보로 저장하며 기존 파일/모델은 보존합니다. 같은 과거 개발4960명 재평가이고 보류1241명은 평가하지 않았으며, 최종 배포 fit에서만 전체6201명을 합칩니다.

## 최종 구조

`원본 gene 문자열 → 고정 X/* canonicalization → 문헌 기반 보수적 파서 → 공유 rare_backoff 전처리/8192개 선택 변수 → LR·LightGBM(multiclass)·ComplementNB·LightGBM(OVA) → 고정 가중 확률 평균 → SUBCLASS`

| 분류기 | 설정 | 비중 |
|---|---|---:|
| rich_lr | LR C1/balanced/max_iter1600 | {.108/.476:.6%} |
| rich_lgb | LightGBM300trees/leaves15/depth5/lr.04/lambda10/min_child20/colsample.85/balanced/seed17017 | {.108/.476:.6%} |
| rare_cnb | ComplementNB alpha1/normFalse/temperature2 | {.06/.476:.6%} |
| aug_ova | LightGBM OVA500trees/leaves15/depth5/lr.04/lambda10/min_child20/colsample.85/is_unbalance/seed23023 | {.2/.476:.6%} |

비중은 v25의 해당4개 leaf 비중 .108/.108/.06/.2를 합 .476으로 나눈 고정값입니다. 이번 결과로 비중을 탐색하지 않았습니다. 최종4개 객체지만 OVA는 내부적으로26개 이진 목표, LightGBM에는 많은 트리가 존재합니다. 학습기 종류는3종(LR/LGB/CNB), 전처리는1개이며 NN/Geneformer/XGBoost는 최종 D 추론에 포함하지 않습니다. 전체 notebook의 A/B 연구 재현에는 관련 패키지가 필요합니다.

이 구성은 기존 분류기의 일부를 불러와 추론 때만 숨긴 축소가 아닙니다. 공유 표현을 원래 train에서 새로 적합하고4개 estimator만 train6201명에서 새로 학습했습니다. OVA를 제외한3개는 원래 행만, OVA는 아래 train-only 증강도 사용합니다. 모델 수4개 상한을 지켰으며5개는 추가하지 않았습니다.

## 차용한 전처리와 새 표현의 차이

v16의 `parse_token16/parse_rows16`, `Preprocess16('rare_backoff',min_df=3,k=8192)` 정의를 notebook에 직접 넣었습니다. 외부 프로젝트 py를 import하지 않습니다. 과거의 **unfitted 정의/원리** 차용이며 기존 fitted state 차용은 아닙니다. 문헌 gene panel branch는 사용하지 않고 빈 panel dict로 둡니다. 외부 환자·사전학습·외부 gene 목록을 새로 불러오지 않습니다.

- p./3-letter AA/불확실성 괄호, 동일 길이 block, stop gained/lost/retained, fs 종료거리, del/ins/delins, 불완전/미해석 표기를 구분합니다. 같은 길이 block을 성분 속성으로 풀어도 원래 event는1개입니다. 복합표기를 무조건 지우거나 DNA 사건 수로 재해석하지 않습니다.
- `R1450X/p.R1450X`형 단일-AA 치환은 원래 v25의 dataset legacy-stop 가정을 유지해 *로 통일합니다. 모든 X의 보편적 HGVS 의미가 stop이라는 뜻은 아닙니다.
- gene 유무/log event count, effect별 log count, 원래 format, AA gene/global count, protein bin64/256, fs stop distance, rare/supported gene count, 관측/원문/중복/불확실/미해석/유형 분율/rare/unseen 요약을 공유합니다.
- 학습 DF는 gene×canonical event로 계산합니다. training rarity/unseen은 자기 자신1을 제외, validation/inference는 학습 DF만 사용합니다.
- 각 학습 fold의 원래 행에서 어휘·DF≥3(일부 numeric 예외)·chi2상위8192·MaxAbs를 fit하고 exact residual만 MaxAbs 후 df/(df+5)로 완화합니다. 이 값은 병원성/driver/기능 점수가 아닙니다.
- OVA에만 기존 corrupt23를 차용: 해당 train 내 DF2 이하 missense/synonymous/synonymous_block pool, gene mask0.1, Poisson20 nuisance. 원본1/증강0.5, 전처리 fit은 원래 train만, 증강에는 transform만 수행. 이 증강은 label-preserving 모델링 가정이지 실제 정상/암 변이 구분이 아닙니다.
- raw_tokens/duplicate_tokens를 별도 기록하므로 중복 토큰 수까지 무조건 불변이라고 주장하지 않습니다. X/* canonicalization과 배치 독립성은 저장 모델에서 실제 검사했습니다.

공유 변수군:

{markdown30(dfamilies30)}

정확한 변수8192개·학습지원도·원래 unscaled chi2·exact DF·감쇠비중은 `d_selected_variables.csv`와 `selected_variables_v30.md`에 기록했습니다. exact DF 칸의 비-event NaN은 해당 없음이며 환자 데이터의 결측 처리와 다릅니다.

## D의 오류 진단

{markdown30(dclasses30,index=True)}

동일 profile의 라벨 충돌·공유 분자 특징·미관측 임상 정보는 가능한 한계지만, 이 표로 정답 오류/진짜 환자 중복을 단정하거나 형제 라벨을 뒤집지 않습니다. 임상적 안전성/외부 코호트 일반화/개별 환자 진단 성능을 입증한 모델이 아닙니다.

## 재현·저장·최종 추론

- 활성 피클 `artifacts/v30/cancer_model_v30_compact_experimental.pkl`, **{active30['model_bytes']:,} bytes**, model SHA `{active30['model_sha256']}`.
- full train 선택4개 fit {active30['full_train_fit_seconds']:.2f}초, 분류기별 `{active30['fit_breakdown']}`. 탐색/로딩/파싱 제외이며 다른 CPU 부하의 과거 시간과 직접 배수 비교 안 함.
- A/B/C/D source·선별·학습·진짜 OOF·실패 및 cache 출처 모두 notebook/artifacts에 보존. D 완료 CV cache는 train/source/protocol/parsed-cache SHA 및 정확한 fold 인덱스를 확인해 재사용하며, 전체 train 모델로 CV하지 않습니다. cache가 없으면 notebook에서 새로 학습합니다.
- 최종 CSV `submission/혜진_submission_v30_compact_experimental.csv`,2546행. 선택 고정 뒤 **test에 추론만** 적용했습니다. test/sample은 연구 open 차단, 추론 child는 train open 차단. fit/선별/비중/암종 비율 정렬에 test 미사용.
- 독립 child 재로딩, ID/순서/클래스/확률합, batch invariance, X/* 동등 예측, model/CSV hash 확인. 기존 원본·모델·제출 보존, commit/push하지 않음.
- 최종 후보는 적은 모델로 대등한 평균 성능을 보인 실험용이며, 사전 개선 기준 미통과라면 기존 모델 교체나 확실한 성능 향상을 주장하지 않습니다.

---

## A/B/C 상세 실패·문헌 연결 기록

'''
 body30=report30[report30.index('## C:'):];body30=body30.replace('최종 B 모델은','B 후보 모델은').replace('## 최종 결과와 채택 판단','## B 후보 결과와 판단').replace('artifacts/v30/cancer_model_v30_compact_experimental.pkl','artifacts/v30/before_d/cancer_model_v30_before_d.pkl');report30=dsection30+body30;(WORK30/'analysis_report_v30.md').write_text(report30)
 (WORK30/'selected_variables_v30.md').write_text(f'''# v30 D 활성 모델의 공유 변수

최종4개 rich_lr/rich_lgb/rare_cnb/aug_ova가 동일 전처리/변수 {active30['final_features']}개를 공유합니다. ID는 입력에서 제외, SUBCLASS는 정답으로만 사용합니다. 학습 지원도와 chi2는 원래 train의 unscaled 선택 행렬에서 다시 계산했습니다. exact_event_train_df는 exact 열만 해당하며 나머지 NaN은 해당 없음입니다. exact_residual_weight는 MaxAbs 후 적용하는 df/(df+5), 비exact는1입니다. 임상 병원성/인과성 값이 아닙니다.

{markdown30(dfamilies30)}

{markdown30(dfeatures30)}
''')
 print('Final D4 report and selected8192 features saved; former A/B/C results retained.');display(dtable30)


{
  "version": 30,
  "rows": 2546,
  "classifiers": 4,
  "members": [
    "rich_lr",
    "rich_lgb",
    "rare_cnb",
    "aug_ova"
  ],
  "phase": "final inference only",
  "test_for_training_or_validation": false,
  "id_order_checked": true,
  "batch_invariance": true,
  "test_prediction_seconds": 11.200826083979337,
  "model_sha256": "176769082696bd4c95b1f3645c15bffb30dfc01842cd57ab84f7383defd8770d",
  "csv_sha256": "cfbd7073a9bbe0c46e9c531c879382ee020b10546e22180c3f46b0179e4757af",
  "public_score": null
}

v30 A/B full report, selected features, model and submission saved.
Final D4 report and selected8192 features saved; former A/B/C results retained.


,fold,selected,f1,accuracy,features,fit_seconds,v29_f1,delta_vs_v29
0,0,rare_lr|rare_ova|rare_mlp,0.462630,0.467742,8192,81.848638,0.522736,-0.060106
1,1,rare_lr|rare_ova|rare_mlp,0.419620,0.437500,8192,72.664188,0.464869,-0.045249
2,2,rare_lr|rare_ova|rare_mlp,0.416605,0.431452,8192,84.407104,0.469139,-0.052534
3,3,rare_lr|rare_ova|rare_mlp,0.449435,0.465726,8192,79.780629,0.496357,-0.046923
4,4,rare_lr|rare_ova|rare_mlp,0.454450,0.449597,8192,70.893038,0.485161,-0.030711


,fold,selected,f1,accuracy,features,fit_predict_seconds,v29_f1,delta_vs_v29
0,0,rich_lr|rich_lgb|rare_cnb|aug_ova,0.527756,0.512097,8192,57.359796,0.522736,0.005020
1,1,rich_lr|rich_lgb|rare_cnb|aug_ova,0.461104,0.477823,8192,58.427773,0.464869,-0.003765
2,2,rich_lr|rich_lgb|rare_cnb|aug_ova,0.468022,0.468750,8192,59.874857,0.469139,-0.001117
3,3,rich_lr|rich_lgb|rare_cnb|aug_ova,0.507956,0.514113,8192,58.732306,0.496357,0.011599
4,4,rich_lr|rich_lgb|rare_cnb|aug_ova,0.481243,0.478831,8192,54.973144,0.485161,-0.003918
